# Bouton-Specific Diversity of Glutamate Release from Single Axons


This notebook assembles the analyses and figure panels supporting the manuscript on bouton-specific diversity of glutamate release along cerebellar parallel fibers. WT boutons define the reference description of release strength and short-term plasticity, and perturbation datasets are interpreted relative to that common framework.

The notebook is organized to follow the logic of the Results section: establish the WT reference space, define WT bouton classes, interpret those classes in mechanistic terms, then examine fiber organization, target identity, calcium, stimulation frequency, Synapsin II loss, and recording stability.


## 1. Data Foundations and Shared Resources

The opening section loads the processed traces, bouton-level metrics, target annotations, and shared plotting utilities used throughout the notebook. These cells establish the common data structures that support the WT reference analyses and all later perturbation comparisons.

Methodological note: the analyses combine bouton-level summary metrics extracted from linescan recordings with normalized average traces and trial-level event tables. This makes it possible to move from descriptive fluorescence features to mechanistic interpretations of release probability, apparent release-site occupancy, and short-term plasticity.


### 1.1 General Imports

The import cell collects the numerical, statistical, geometric, and plotting tools needed for the full notebook. Optional dependencies are handled explicitly so that the main manuscript analyses remain executable even when secondary plotting packages are unavailable.


In [ ]:
## Standard library imports. PCA, Clustering, Ellipse/boundary, Alpha-shape, k-NN boundary, Stability/plasticity

# Standard library imports
import json
import os, re
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

# Scientific computing and data analysis
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage, set_link_color_palette
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Data visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.colors import ListedColormap, to_hex
from matplotlib.cm import Set1
from matplotlib.patches import Ellipse
from matplotlib.widgets import Button

# Geometric and statistical analysis tools
try:
    import alphashape
except ModuleNotFoundError:
    alphashape = None

from shapely.geometry import MultiPolygon, Point, Polygon as ShapelyPolygon
from shapely.affinity import scale as shp_scale
from shapely.prepared import prep

try:
    from statannotations.Annotator import Annotator
except ModuleNotFoundError:
    Annotator = None

### 1.2 Data Sources and Paths

The file paths connect the notebook to the processed feature tables, trace summaries, and target-annotation tables generated upstream. Keeping the paths explicit makes it clear which tables define the WT reference dataset and which tables are only used for perturbation overlays.


In [ ]:
## File paths and directory structure. Failures/reliability, PCA, Clustering, Extracellular Ca²+, Stability/plasticity, Temporal traces

# File paths and directory structure
#BASE_DIR            = Path(r"C:\Users\Anthime.PERROT\PPR_DATA_AND_CODE\Stability_After_temp_t_delete_later")
BASE_DIR            = Path(r"C:\Users\Antoine.Valera\Desktop\PPR_DATA_FINAL")  # Base directory for data and code

PPR_FILENAME        = 'summary.xlsx'           # PCA features file
PPR_TRIALS_FILENAME = 'summary_trials.xlsx'  # Per-trial failure data
PPR_TRIALS_NNLS_NULL_FILENAME = 'summary_trials_nnls_null.xlsx'  # Packed per-trial sliding-NNLS baseline amplitudes
TRACES_FILENAME     = 'summary_traces.xlsx'  # Preprocessed average traces (generated by extract_metrics)
TIMES_FILENAME      = 'summary_times.xlsx'   # Time vectors for traces (generated by extract_metrics)
TARGET_MAP_FILENAME = 'Target_WT_pooled.xlsx'  # Bouton target identity mapping
OUTPUT_DIR          = BASE_DIR / 'output'

# ------------------------------------------------------------------
# Central condition registry (single source of truth)
# ------------------------------------------------------------------
STABILITY_BEFORE_CONDITIONS = ['Stability_Before', 'Stability_Before_05']
STABILITY_AFTER_CONDITIONS = ['Stability_After', 'Stability_After_05']
WT_2_5_20HZ_CONDITIONS = ['WT_Theo', 'WT_Theo_1scd', 'WT_Anthime'] + STABILITY_BEFORE_CONDITIONS
STABILITY_2_5_20HZ_CONDITIONS = STABILITY_BEFORE_CONDITIONS + STABILITY_AFTER_CONDITIONS
STABILITY_OUTLIER_IDS = ['241212_Fibre1_PortionB_bouton9', '241212_Fibre1_PortionB_Bouton_9_bis']
CALCIUM_2_5_20HZ_ALL_CONDITIONS = WT_2_5_20HZ_CONDITIONS + STABILITY_AFTER_CONDITIONS
CALCIUM_LEVEL_ALIASES = {
    '1.5': '1.5mM',
    '2.5': '2.5mM',
    '4': '4mM',
    '4.0': '4mM',
}


def _ordered_unique(seq):
    return list(dict.fromkeys(seq))


def _duplicates(seq):
    seen = set()
    dup = []
    for item in seq:
        if item in seen and item not in dup:
            dup.append(item)
        seen.add(item)
    return dup


def _assert_no_duplicates(group_name, seq):
    dup = _duplicates(seq)
    if dup:
        raise ValueError(f"Duplicate conditions in '{group_name}': {dup}")


def combine_condition_groups(*groups):
    merged = []
    for group in groups:
        merged.extend(list(group))
    _assert_no_duplicates('combined groups', merged)
    return merged


_assert_no_duplicates('WT_2_5_20HZ_CONDITIONS', WT_2_5_20HZ_CONDITIONS)
_assert_no_duplicates('STABILITY_BEFORE_CONDITIONS', STABILITY_BEFORE_CONDITIONS)
_assert_no_duplicates('STABILITY_AFTER_CONDITIONS', STABILITY_AFTER_CONDITIONS)
_assert_no_duplicates('CALCIUM_2_5_20HZ_ALL_CONDITIONS', CALCIUM_2_5_20HZ_ALL_CONDITIONS)

CALCIUM_CONDITION_GROUPS = {
    '1.5mM': {'20Hz': ['Theo_1_5Ca'], '50Hz': ['Theo_1_5_50Hz']},
    '2.5mM': {
        '20Hz': CALCIUM_2_5_20HZ_ALL_CONDITIONS,
        '50Hz': ['Theo_2_5_50Hz'],
    },
    '4mM': {'20Hz': ['Theo_4Ca'], '50Hz': ['Theo_4_50Hz']},
}
SYNAPSIN_CONDITIONS = ['SynII']

# ------------------------------------------------------------------
# Central display palette (single source of truth)
# ------------------------------------------------------------------
WT_CA_COLORS = {
    '1.5mM': "#59baff",
    '2.5mM': "#ffa759",
    '4mM': "#ff3030",
}
FREQ_50_COLORS = {
    '1.5mM': "#2048fa",
    '2.5mM': "#c97018",
    '4mM': "#c00000",
}
SYNII_COLOR = "#af37ff"
WT_THEO_COLOR = "#727272"
WT_ANTHIME_COLOR = "#b9b9b9"
WT_THEO_1SCD_COLOR = WT_CA_COLORS['2.5mM']
STABILITY_BEFORE_COLOR = WT_CA_COLORS['2.5mM']
STABILITY_AFTER_COLOR = "#ffd037"
PC_TARGET_COLOR = "#13ca02"
PC_TARGET_MEDIAN_COLOR = "#0fb400"
IN_TARGET_COLOR = "#ba67bd"
IN_TARGET_MEDIAN_COLOR = "#955597"

def _normalize_calcium_key(calcium_level):
    calcium_raw = str(calcium_level).strip().replace(' ', '')
    calcium_key = calcium_raw.lower()
    for suffix in ('mmca', 'mm'):
        if calcium_key.endswith(suffix):
            calcium_key = calcium_key[:-len(suffix)]
            break
    return CALCIUM_LEVEL_ALIASES.get(calcium_key, calcium_raw)

def get_wt_ca_color(calcium_level):
    return WT_CA_COLORS[_normalize_calcium_key(calcium_level)]

def get_50hz_ca_color(calcium_level):
    return FREQ_50_COLORS[_normalize_calcium_key(calcium_level)]

def get_source_condition_color(condition_name):
    source_colors = {
        'WT_Theo': WT_THEO_COLOR,
        'WT_Anthime': WT_ANTHIME_COLOR,
        'WT_Theo_1scd': WT_THEO_1SCD_COLOR,
        'SynII': SYNII_COLOR,
    }
    return source_colors[condition_name]

def get_stability_condition_color(condition_name):
    if 'Before' in str(condition_name):
        return STABILITY_BEFORE_COLOR
    if 'After' in str(condition_name):
        return STABILITY_AFTER_COLOR
    raise KeyError(condition_name)

def get_target_identity_color(target_name):
    target_key = str(target_name).strip().upper()
    if target_key == 'PC' or target_key.endswith('_PC') or target_key.startswith('PC_'):
        return PC_TARGET_COLOR
    if target_key == 'IN' or target_key.endswith('_IN') or target_key.startswith('IN_'):
        return IN_TARGET_COLOR
    raise KeyError(target_name)

def get_calcium_conditions(calcium_level, frequency='all'):
    calcium_raw = str(calcium_level).strip().replace(' ', '')
    calcium_key = calcium_raw.lower()
    for suffix in ('mmca', 'mm'):
        if calcium_key.endswith(suffix):
            calcium_key = calcium_key[:-len(suffix)]
            break

    calcium_norm = CALCIUM_LEVEL_ALIASES.get(calcium_key, calcium_raw)
    if calcium_norm not in CALCIUM_CONDITION_GROUPS:
        raise KeyError(f"Unknown calcium level: {calcium_level}")

    freq_key = str(frequency).strip().lower().replace(' ', '')
    if freq_key in ('all', '*', 'any'):
        return _ordered_unique(
            CALCIUM_CONDITION_GROUPS[calcium_norm]['20Hz']
            + CALCIUM_CONDITION_GROUPS[calcium_norm]['50Hz']
        )
    if freq_key in ('20hz', '20'):
        return list(CALCIUM_CONDITION_GROUPS[calcium_norm]['20Hz'])
    if freq_key in ('50hz', '50'):
        return list(CALCIUM_CONDITION_GROUPS[calcium_norm]['50Hz'])
    raise KeyError(f"Unknown frequency selector: {frequency}")


def get_synapsin_conditions():
    return list(SYNAPSIN_CONDITIONS)


# ------------------------------------------------------------------
# Central condition/group aliases
# ------------------------------------------------------------------
CONDITION_POOLS = {
    'WT_pooled': WT_2_5_20HZ_CONDITIONS,
    'stability_before': STABILITY_BEFORE_CONDITIONS,
    'stability_after': STABILITY_AFTER_CONDITIONS,
}

FRIENDLY_CONDITION_GROUPS = {
    '1.5mM_20Hz': get_calcium_conditions('1.5mM', '20Hz'),
    '1.5mM_50Hz': get_calcium_conditions('1.5mM', '50Hz'),
    '1.5mM_all': get_calcium_conditions('1.5mM', 'all'),
    '2.5mM_20Hz': get_calcium_conditions('2.5mM', '20Hz'),
    '2.5mM_20Hz_WT': WT_2_5_20HZ_CONDITIONS,
    '2.5mM_20Hz_Stability': STABILITY_2_5_20HZ_CONDITIONS,
    '2.5mM_50Hz': get_calcium_conditions('2.5mM', '50Hz'),
    '2.5mM_all': get_calcium_conditions('2.5mM', 'all'),
    '4mM_20Hz': get_calcium_conditions('4mM', '20Hz'),
    '4mM_50Hz': get_calcium_conditions('4mM', '50Hz'),
    '4mM_all': get_calcium_conditions('4mM', 'all'),
    'Synapsin': get_synapsin_conditions(),
}

def get_friendly_conditions(group_name):
    if group_name not in FRIENDLY_CONDITION_GROUPS:
        raise KeyError(f"Unknown group '{group_name}'")
    return list(FRIENDLY_CONDITION_GROUPS[group_name])

for _name, _group in FRIENDLY_CONDITION_GROUPS.items():
    _assert_no_duplicates(_name, _group)


def filter_df_by_conditions(dataframe, conditions, condition_col='Condition'):
    return dataframe[dataframe[condition_col].isin(list(conditions))].copy()


def filter_df_by_calcium(dataframe, calcium_level, frequency='all', include_synapsin=False, condition_col='Condition'):
    selected = get_calcium_conditions(calcium_level, frequency)
    if include_synapsin:
        selected = _ordered_unique(selected + get_synapsin_conditions())
    return filter_df_by_conditions(dataframe, selected, condition_col=condition_col)



# ------------------------------------------------------------------
# DataFrame enrichment and selection utilities
# ------------------------------------------------------------------

def _build_condition_metadata_map():
    metadata = {}

    for cond in get_calcium_conditions('1.5mM', '20Hz'):
        metadata[cond] = {'Ca_mM': 1.5, 'Freq_Hz': 20, 'ConditionFamily': 'Calcium'}
    for cond in get_calcium_conditions('1.5mM', '50Hz'):
        metadata[cond] = {'Ca_mM': 1.5, 'Freq_Hz': 50, 'ConditionFamily': 'Calcium'}

    for cond in WT_2_5_20HZ_CONDITIONS:
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'WT'}
    for cond in STABILITY_BEFORE_CONDITIONS:
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityBefore'}
    for cond in STABILITY_AFTER_CONDITIONS:
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityAfter'}
    for cond in get_calcium_conditions('2.5mM', '50Hz'):
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 50, 'ConditionFamily': 'Calcium'}

    for cond in get_calcium_conditions('4mM', '20Hz'):
        metadata[cond] = {'Ca_mM': 4.0, 'Freq_Hz': 20, 'ConditionFamily': 'Calcium'}
    for cond in get_calcium_conditions('4mM', '50Hz'):
        metadata[cond] = {'Ca_mM': 4.0, 'Freq_Hz': 50, 'ConditionFamily': 'Calcium'}

    for cond in get_synapsin_conditions():
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'Synapsin'}

    # Pooled labels
    metadata['WT_pooled'] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'WTPooled'}
    metadata['stability_before'] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityBeforePooled'}
    metadata['stability_after'] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityAfterPooled'}
    return metadata


def _normalize_bouton_id(value):
    if pd.isna(value):
        return value
    return str(value).strip().replace('_traces_converted', '')


def _extract_fiber_id(value, n_chars=22):
    if pd.isna(value):
        return value
    s = _normalize_bouton_id(value)
    m = re.match(r'^(.*?)(?:[_\s]*Bouton[_\s]*\d+.*|[_\s]*bouton\d+.*)$', s, flags=re.IGNORECASE)
    if m:
        return m.group(1).rstrip('_ ')
    return str(s)


def add_condition_metadata(dataframe, condition_col='Condition'):
    d = dataframe.copy()
    condition_meta = _build_condition_metadata_map()
    cond_series = d[condition_col].astype(str)
    d['Ca_mM'] = cond_series.map(lambda c: condition_meta.get(c, {}).get('Ca_mM', np.nan))
    d['Freq_Hz'] = cond_series.map(lambda c: condition_meta.get(c, {}).get('Freq_Hz', np.nan))
    d['ConditionFamily'] = cond_series.map(lambda c: condition_meta.get(c, {}).get('ConditionFamily', 'Unknown'))
    return d


def enrich_features_dataframe(dataframe):
    d = dataframe.copy()
    d['ID'] = d['ID'].astype(str).str.strip()
    d['BaseID'] = d['ID'].map(_normalize_bouton_id)
    d['FiberID'] = d['ID'].map(_extract_fiber_id)
    d = add_condition_metadata(d, condition_col='Condition')
    return d


def enrich_traces_dataframe(dataframe):
    d = dataframe.copy()
    d['ID'] = d['ID'].astype(str).str.strip()
    d['BaseID'] = d['ID'].map(_normalize_bouton_id)
    d['FiberID'] = d['ID'].map(_extract_fiber_id)
    d = add_condition_metadata(d, condition_col='Condition')
    return d


def build_condition_views(dataframe, condition_col='Condition'):
    return {
        cond: sub.copy()
        for cond, sub in dataframe.groupby(condition_col, sort=False)
    }


def make_pooled_features_dataframe(features_raw, pools):
    d = features_raw.copy()
    existing_conditions = set(d['Condition'].unique())
    for pool_name, source_conditions in pools.items():
        if pool_name in existing_conditions:
            continue
        available_sources = [c for c in source_conditions if c in existing_conditions]
        if not available_sources:
            continue
        pooled_parts = []
        seen_ids = set()
        for source_condition in available_sources:
            source_rows = d[d['Condition'] == source_condition].copy()
            source_norm_ids = source_rows['ID'].map(_normalize_bouton_id)
            keep_mask = ~source_norm_ids.isin(seen_ids)
            source_rows = source_rows[keep_mask].copy()
            seen_ids.update(source_rows['ID'].map(_normalize_bouton_id))
            pooled_parts.append(source_rows)
        if len(pooled_parts) == 0:
            continue
        pooled_data = pd.concat(pooled_parts, ignore_index=True)
        pooled_data['Condition'] = pool_name
        d = pd.concat([d, pooled_data], ignore_index=True)
        existing_conditions.add(pool_name)
    return d


def recompute_failure_summary_from_trials(trials_df, n_fail_cols=3, threshold_col='thr_shared'):
    """Temporary notebook-side recomputation of %Fail from summary_trials.xlsx."""
    d = trials_df.copy()
    if 'condition' not in d.columns or 'file' not in d.columns:
        raise KeyError("Trials dataframe must contain 'condition' and 'file' columns")

    d['Condition'] = d['condition'].astype(str).str.strip()
    d['BaseID'] = d['file'].map(_normalize_bouton_id)

    rows = []
    for (condition_name, base_id), grp in d.groupby(['Condition', 'BaseID'], sort=False):
        out = {'Condition': condition_name, 'BaseID': base_id}
        for k in range(1, n_fail_cols + 1):
            corr_col = f'AMP{k}_CORR'
            uncorr_col = f'AMP{k}_UNCORR'
            n_fail = 0
            n_valid = 0
            for _, row in grp.iterrows():
                thr = pd.to_numeric(pd.Series([row.get(threshold_col, np.nan)]), errors='coerce').iloc[0]
                if not np.isfinite(thr):
                    continue
                vals = []
                if corr_col in grp.columns:
                    v_corr = pd.to_numeric(pd.Series([row.get(corr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_corr):
                        vals.append(float(v_corr))
                if uncorr_col in grp.columns:
                    v_uncorr = pd.to_numeric(pd.Series([row.get(uncorr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_uncorr):
                        vals.append(float(v_uncorr))
                if len(vals) == 0:
                    continue
                n_valid += 1
                if np.min(vals) < float(thr):
                    n_fail += 1
            out[f'%Fail{k}'] = round((100.0 * n_fail / n_valid), 2) if n_valid else np.nan
        rows.append(out)
    return pd.DataFrame(rows)


def recompute_event_failure_rates_from_trials(trials_df, max_pulse_number=10, threshold_col='thr_shared'):
    """Compute canonical eventwise failure rates (0-1) from trials for pulses 1..max_pulse_number."""
    d = trials_df.copy()
    if 'condition' not in d.columns or 'file' not in d.columns:
        raise KeyError("Trials dataframe must contain 'condition' and 'file' columns")

    d['Condition'] = d['condition'].astype(str).str.strip()
    d['BaseID'] = d['file'].map(_normalize_bouton_id)

    rows = []
    for (condition_name, base_id), grp in d.groupby(['Condition', 'BaseID'], sort=False):
        out = {'Condition': condition_name, 'BaseID': base_id}
        for k in range(1, max_pulse_number + 1):
            corr_col = f'AMP{k}_CORR'
            uncorr_col = f'AMP{k}_UNCORR'
            fail_count = 0
            valid_count = 0
            for _, row in grp.iterrows():
                thr = pd.to_numeric(pd.Series([row.get(threshold_col, np.nan)]), errors='coerce').iloc[0]
                if not np.isfinite(thr):
                    continue
                vals = []
                if corr_col in grp.columns:
                    v_corr = pd.to_numeric(pd.Series([row.get(corr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_corr):
                        vals.append(float(v_corr))
                if uncorr_col in grp.columns:
                    v_uncorr = pd.to_numeric(pd.Series([row.get(uncorr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_uncorr):
                        vals.append(float(v_uncorr))
                if len(vals) == 0:
                    continue
                valid_count += 1
                if np.min(vals) < float(thr):
                    fail_count += 1
            out[f'FailRate{k}'] = (fail_count / valid_count) if valid_count else np.nan
        rows.append(out)
    return pd.DataFrame(rows)


def attach_event_failure_rates_to_features(features_df, trials_df=None, max_pulse_number=10, verbose=True):
    """Attach canonical FailRate1..N columns to the feature dataframe without touching %Fail columns."""
    d = features_df.copy()
    if 'Condition' not in d.columns:
        raise KeyError("Features dataframe must contain a 'Condition' column")
    if 'BaseID' not in d.columns:
        if 'ID' not in d.columns:
            raise KeyError("Features dataframe must contain either 'BaseID' or 'ID'")
        d['BaseID'] = d['ID'].map(_normalize_bouton_id)

    if trials_df is None:
        trials_df = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)

    fail_rates = recompute_event_failure_rates_from_trials(
        trials_df,
        max_pulse_number=max_pulse_number,
    )
    d = d.merge(fail_rates, on=['Condition', 'BaseID'], how='left')

    if verbose:
        rate_cols = [f'FailRate{k}' for k in range(1, max_pulse_number + 1)]
        matched = pd.to_numeric(d[rate_cols[0]], errors='coerce').notna().sum() if rate_cols[0] in d.columns else 0
        print('Canonical trial-derived failure rates attached to features:')
        print(f'  matched feature rows: {matched}/{len(d)}')
        print(f'  added columns: {rate_cols[0]} .. {rate_cols[-1]}')

    return d, fail_rates


def apply_recomputed_failures_to_features(features_df, trials_df=None, n_fail_cols=3, verbose=True):
    """Patch summary-derived feature rows with trial-recomputed %Fail columns."""
    d = features_df.copy()
    if 'Condition' not in d.columns:
        raise KeyError("Features dataframe must contain a 'Condition' column")
    if 'BaseID' not in d.columns:
        if 'ID' not in d.columns:
            raise KeyError("Features dataframe must contain either 'BaseID' or 'ID'")
        d['BaseID'] = d['ID'].map(_normalize_bouton_id)

    if trials_df is None:
        trials_df = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)

    fail_summary = recompute_failure_summary_from_trials(
        trials_df,
        n_fail_cols=n_fail_cols,
    )

    before_cols = {col: d[col].copy() for col in [f'%Fail{k}' for k in range(1, n_fail_cols + 1)] if col in d.columns}
    d = d.merge(fail_summary, on=['Condition', 'BaseID'], how='left', suffixes=('', '_recomputed'))

    updated_counts = {}
    for k in range(1, n_fail_cols + 1):
        col = f'%Fail{k}'
        new_col = f'{col}_recomputed'
        if new_col not in d.columns:
            continue
        old_vals = before_cols.get(col, pd.Series(np.nan, index=d.index))
        has_new = np.isfinite(pd.to_numeric(d[new_col], errors='coerce'))
        d[col] = d[new_col].where(has_new, d[col] if col in d.columns else np.nan)
        updated_counts[col] = int(np.sum(has_new & (~np.isclose(pd.to_numeric(old_vals, errors='coerce'), pd.to_numeric(d[col], errors='coerce'), equal_nan=True))))
        d = d.drop(columns=[new_col])

    if verbose:
        n_match = int(d[['Condition', 'BaseID']].merge(fail_summary[['Condition', 'BaseID']].drop_duplicates(), on=['Condition', 'BaseID'], how='inner').shape[0])
        print("Temporary failure correction from trials:")
        print(f"  matched feature rows: {n_match}/{len(d)}")
        for col in sorted(updated_counts):
            print(f"  {col}: updated {updated_counts[col]} rows")

    return d, fail_summary


def get_ppr_columns(dataframe, max_pulse=10):
    return [
        f'PPR{i}/1'
        for i in range(2, max_pulse + 1)
        if f'PPR{i}/1' in dataframe.columns
    ]


# Data filtering and analysis parameters
EXCEPTIONAL_CONDITIONS = combine_condition_groups(
    STABILITY_BEFORE_CONDITIONS[1:],
    STABILITY_AFTER_CONDITIONS[1:],
    get_calcium_conditions('1.5mM', 'all'),
    get_calcium_conditions('4mM', 'all'),
    get_calcium_conditions('2.5mM', '50Hz'),
    ['WT_Theo'],
)
PCA_DROP_COLS          = [f'AMP{i}' for i in range(3, 11)] + [f'FailRate{i}' for i in range(1, 11)] + ['measurement', 'Condition', 'condition', 'ID', 'Target', '%Fail3', 'Sexe', 'Ca_mM', 'Freq_Hz']
N_CLUSTERS             = 6                          # Number of clusters for analysis

# Trace processing parameters (used for visualization only - traces are pre-processed)
STIM_SHIFT  = 0.5                        # Stimulus time offset (seconds)
CROP_END    = 2.0                          # Trace duration to keep (seconds)
SAMPLE_RATE = 1000                      # Target sampling rate (Hz)
N_SAMPLES   = int(CROP_END * SAMPLE_RATE) + 1
COMMON_TIME = np.linspace(0, CROP_END, N_SAMPLES)  # Standardized time vector

# Shared trace-source switches
TRACE_MEAN_SOURCE = 'raw_nearest'   # 'normalized' or 'raw_nearest'
TRACE_SINGLE_SOURCE = 'raw'         # 'raw' or 'normalized'
TRACE_RAW_NEAREST_TOL_FACTOR = 0.51
TRACE_XLIM_20HZ = (0.8, 1.6)


### 1.3 Output Directory and Shared Helpers

Figure and table export helpers are defined once here so that all later panels are saved reproducibly. The output names are aligned to the notebook order to simplify manuscript assembly and figure curation.


In [ ]:
# Create output directory and define helper functions
OUTPUT_FALLBACK_DIR = Path.cwd() / '_exported_figures'


def _probe_writable_directory(directory):
    directory = Path(directory)
    try:
        directory.mkdir(parents=True, exist_ok=True)
        probe = directory / '.__write_probe__.tmp'
        probe.write_text('ok', encoding='utf-8')
        probe.unlink()
        return True
    except Exception:
        return False


def _timestamped_path(path_obj, tag='autosave'):
    path_obj = Path(path_obj)
    stem = path_obj.stem
    suffix = path_obj.suffix
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    candidate = path_obj.with_name(f'{stem}__{tag}_{timestamp}{suffix}')
    counter = 1
    while candidate.exists():
        candidate = path_obj.with_name(f'{stem}__{tag}_{timestamp}_{counter}{suffix}')
        counter += 1
    return candidate


def _as_pathlike_target(target):
    try:
        return Path(target)
    except Exception:
        return None


def _replace_savefig_target(args, kwargs, new_target):
    args = list(args)
    kwargs = dict(kwargs)
    if len(args) >= 1:
        args[0] = new_target
    else:
        kwargs['fname'] = new_target
    return tuple(args), kwargs


def _permission_retry_path(target, prefer_same_dir=True):
    path_obj = _as_pathlike_target(target)
    if path_obj is None:
        return None
    if prefer_same_dir:
        try:
            path_obj.parent.mkdir(parents=True, exist_ok=True)
            return _timestamped_path(path_obj, tag='retry')
        except Exception:
            pass
    fallback_dir = OUTPUT_FALLBACK_DIR
    fallback_dir.mkdir(parents=True, exist_ok=True)
    return _timestamped_path(fallback_dir / path_obj.name, tag='fallback')


if not _probe_writable_directory(OUTPUT_DIR):
    OUTPUT_FALLBACK_DIR.mkdir(parents=True, exist_ok=True)
    print(f'OUTPUT_DIR not writable, switching to fallback: {OUTPUT_FALLBACK_DIR}')
    OUTPUT_DIR = OUTPUT_FALLBACK_DIR
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def is_bouton_file(file_path: Path) -> bool:
    """Check if Excel file contains bouton trace data (excludes metadata files)"""
    metadata_files = {PPR_FILENAME.lower(), PPR_TRIALS_FILENAME.lower(), TARGET_MAP_FILENAME.lower()}
    return (file_path.suffix.lower() == '.xlsx' and 
            not file_path.name.startswith('~$') and 
            file_path.name.lower() not in metadata_files)

def clean_bouton_id(raw_bouton_id: str) -> str:
    """Remove file suffixes from bouton IDs for consistent matching"""
    return raw_bouton_id.replace('_traces_converted', '')

### 1.4 Load Preprocessed Traces

The trace-loading step reconstructs the average glutamate transients used for the trace panels throughout the notebook. These traces retain the timing structure of the stimulation trains and provide the most direct view of bouton-level glutamate release dynamics.


In [ ]:
## Load preprocessed average traces from extract_metrics output. Ellipse/boundary, Temporal traces
## Traces are loaded from TRACES_FILENAME and time vectors from TIMES_FILENAME
## (generated by export_folders_to_excel with save_traces=True)

traces_file = BASE_DIR / TRACES_FILENAME
times_file = BASE_DIR / TIMES_FILENAME

if not traces_file.exists():
    raise FileNotFoundError(
        f"Traces file not found: {traces_file}\n"
        "Run extract_metrics.export_folders_to_excel() with save_traces=True to generate it."
    )
if not times_file.exists():
    raise FileNotFoundError(
        f"Times file not found: {times_file}\n"
        "Run extract_metrics.export_folders_to_excel() with save_traces=True to generate it."
    )

# Load all sheets (one per condition) from both Excel files
excel_traces = pd.ExcelFile(traces_file)
excel_times = pd.ExcelFile(times_file)
experimental_conditions = excel_traces.sheet_names
raw_traces_data = []

for condition_name in experimental_conditions:
    trace_df = pd.read_excel(traces_file, sheet_name=condition_name)
    time_df = pd.read_excel(times_file, sheet_name=condition_name)
    
    # All columns are bouton IDs (no Time column in traces file)
    bouton_columns = list(trace_df.columns)
    
    print(f"Loading {condition_name}: {len(bouton_columns)} boutons")
    
    for bouton_id in bouton_columns:
        avg_trace = trace_df[bouton_id].to_numpy(float)
        # Get corresponding time vector from times file
        if bouton_id in time_df.columns:
            time_array = time_df[bouton_id].to_numpy(float)
        else:
            # Fallback: use index as time (should not happen with consistent files)
            print(f"Warning: {bouton_id} not found in times file, using index")
            time_array = np.arange(len(avg_trace)) / SAMPLE_RATE
        
        raw_traces_data.append({
            'ID': str(bouton_id),
            'Condition': condition_name,
            'Time': time_array.tolist(),
            'Avg': avg_trace.tolist(),
            'n_trials': 1,  # Already averaged
            'FilePath': str(traces_file),
        })

excel_traces.close()
excel_times.close()
RAW_TRACES_DF = pd.DataFrame(raw_traces_data)
CONDITIONS = experimental_conditions

print(f"\n=== Loaded {len(RAW_TRACES_DF)} preprocessed traces from {TRACES_FILENAME} and {TIMES_FILENAME} ===")

### 1.5 Plotting and Statistical Helpers

Shared formatting, PCA, and PPR helper functions are centralized here so that later figures are stylistically consistent and analytically comparable. This cell also contains the small utilities that make the same computations reusable across WT, calcium, frequency, SynII, and stability sections.


In [ ]:
# PNAS-aligned figure defaults and style clamps
PNAS_FIGURE_WIDTH_CM = {'single': 8.7, 'one_half': 11.4, 'double': 17.8}
PNAS_MAX_HEIGHT_CM = 22.5
PNAS_FIGURE_WIDTH_IN = {k: v / 2.54 for k, v in PNAS_FIGURE_WIDTH_CM.items()}
PNAS_MAX_HEIGHT_IN = PNAS_MAX_HEIGHT_CM / 2.54
PNAS_PANEL_ASPECT = {
    'simple': 0.78,
    'elongated': 0.40,
    'pca': 0.78,
    'ppr': 0.60,
    'trace': 0.42,
    'hist': 0.72,
}

# Canonical grid unit: one data panel should match the current visual size of
# the single-panel hierarchical-cluster PPR figure. Legends and inter-panel
# spacing are budgeted separately, never by shrinking the panel footprint.
GRID_PANEL_UNIT_W_IN = float(PNAS_FIGURE_WIDTH_IN['single'])
GRID_PANEL_UNIT_H_IN = GRID_PANEL_UNIT_W_IN * float(PNAS_PANEL_ASPECT['simple'])
GRID_WSPACE_IN = 0.32
GRID_HSPACE_IN = 0.26
GRID_MARGIN_LEFT_IN = 0.48
GRID_MARGIN_RIGHT_IN = 0.10
GRID_MARGIN_BOTTOM_IN = 0.42
GRID_MARGIN_TOP_IN = 0.18
GRID_LEGEND_STRIP_IN = 0.92
GRID_SUPTITLE_IN = 0.24
PNAS_FONT_MIN_PT = 6.0
PNAS_FONT_MAX_PT = 12.0
PNAS_LINEWIDTH_MIN_PT = 0.25
PNAS_LINEWIDTH_MAX_PT = 1.5
PNAS_MARKER_MIN_PT = 4.0
PNAS_MARKER_MAX_PT = 11.0
PNAS_SCATTER_AREA_MIN = PNAS_MARKER_MIN_PT ** 2
PNAS_SCATTER_AREA_MAX = 30.0
PNAS_SCATTER_AREA_DEFAULT = 30.0

def _clip_style_value(value, vmin, vmax):
    arr = np.asarray(value, dtype=float)
    clipped = np.clip(arr, vmin, vmax)
    if np.isscalar(value):
        return float(clipped)
    return clipped

def clamp_fontsize(value):
    return _clip_style_value(value, PNAS_FONT_MIN_PT, PNAS_FONT_MAX_PT)

def clamp_linewidth(value):
    return _clip_style_value(value, PNAS_LINEWIDTH_MIN_PT, PNAS_LINEWIDTH_MAX_PT)

def clamp_markersize(value):
    return _clip_style_value(value, PNAS_MARKER_MIN_PT, PNAS_MARKER_MAX_PT)

def clamp_scatter_area(value):
    """Pass-through by default: marker area remains user-controlled."""
    return value

def _guess_panel_kind(panel_kind=None, nrows=1, ncols=1, figsize=None):
    if panel_kind is not None:
        return panel_kind
    if figsize is not None:
        try:
            w, h = float(figsize[0]), float(figsize[1])
            if h > 0 and (w / h) >= 2.2:
                return 'elongated'
        except Exception:
            pass
    if int(ncols) >= 4:
        return 'elongated'
    return 'simple'


def grid_figsize(nrows=1, ncols=1, panel_kind='simple', *, has_legend=False, has_suptitle=False):
    """Return a figure size whose data-panel grid is built from one fixed panel unit."""
    nrows = max(1, int(nrows))
    ncols = max(1, int(ncols))

    total_w = (GRID_PANEL_UNIT_W_IN * ncols) + (GRID_WSPACE_IN * max(0, ncols - 1))
    total_w += GRID_MARGIN_LEFT_IN + GRID_MARGIN_RIGHT_IN
    if has_legend:
        total_w += GRID_LEGEND_STRIP_IN

    total_h = (GRID_PANEL_UNIT_H_IN * nrows) + (GRID_HSPACE_IN * max(0, nrows - 1))
    total_h += GRID_MARGIN_BOTTOM_IN + GRID_MARGIN_TOP_IN
    if has_suptitle:
        total_h += GRID_SUPTITLE_IN

    return total_w, total_h


# Backward-compatible names used later in notebook.
def pnas_figsize(width='single', aspect=0.70, nrows=1):
    if isinstance(width, (int, float)):
        w = float(width)
        h = w * float(aspect) * max(1, int(nrows))
        return w, h
    return grid_figsize(nrows=max(1, int(nrows)), ncols=1, panel_kind='simple')


def recommended_panel_figsize(nrows=1, ncols=1, panel_kind='simple'):
    return grid_figsize(nrows=nrows, ncols=ncols, panel_kind=panel_kind)


def clamp_figsize_to_pnas(figsize, *, nrows=1, ncols=1, panel_kind=None):
    # Intentionally ignore any ad-hoc requested figsize: only grid helper is authoritative.
    return grid_figsize(nrows=nrows, ncols=ncols, panel_kind=_guess_panel_kind(panel_kind, nrows=nrows, ncols=ncols, figsize=figsize))


def _parse_subplots_shape(args, kwargs):
    if len(args) >= 2:
        nrows, ncols = args[0], args[1]
    elif len(args) == 1:
        nrows, ncols = args[0], 1
    else:
        nrows, ncols = kwargs.get('nrows', 1), kwargs.get('ncols', 1)
    nrows = kwargs.get('nrows', nrows)
    ncols = kwargs.get('ncols', ncols)
    return int(nrows), int(ncols)


def _figure_grid_shape(fig):
    """Infer logical panel grid from non-colorbar axes."""
    row_starts = []
    col_starts = []

    for axis in getattr(fig, 'axes', []):
        # Ignore colorbar axes: they are layout helpers, not data panels.
        if str(getattr(axis, 'get_label', lambda: '')()) == '<colorbar>':
            continue
        try:
            ss = axis.get_subplotspec()
            row_starts.append(int(ss.rowspan.start))
            col_starts.append(int(ss.colspan.start))
        except Exception:
            pass

    if row_starts and col_starts:
        nrows = len(sorted(set(row_starts)))
        ncols = len(sorted(set(col_starts)))
        return max(1, int(nrows)), max(1, int(ncols))

    # Fallback
    return 1, 1


def _outer_axis_label_visibility(ax):
    """Return shared outer-label visibility for aligned subplot grids."""
    try:
        ss = ax.get_subplotspec()
        gs = ss.get_gridspec()
        row_mid = int((gs.nrows - 1) // 2)
        col_mid = int((gs.ncols - 1) // 2)
        show_xlabel = (int(ss.rowspan.stop) == int(gs.nrows)) and (int(ss.colspan.start) <= col_mid < int(ss.colspan.stop))
        show_ylabel = (int(ss.colspan.start) == 0) and (int(ss.rowspan.start) <= row_mid < int(ss.rowspan.stop))
        if int(gs.nrows) == 1:
            show_xlabel = True
        if int(gs.ncols) == 1:
            show_ylabel = True
        return show_xlabel, show_ylabel
    except Exception:
        return True, True

def _display_layout_rect(fig, *, has_external_legend=None, has_suptitle=None):
    """Return a tight-layout rect from fixed inch budgets instead of scaled fractions."""
    if fig is None:
        return [0.03, 0.03, 0.98, 0.99]
    if has_external_legend is None:
        has_external_legend = len(getattr(fig, 'legends', [])) > 0
    if has_suptitle is None:
        has_suptitle = getattr(fig, '_suptitle', None) is not None

    w, h = fig.get_size_inches()
    w = max(float(w), 1e-6)
    h = max(float(h), 1e-6)

    left = GRID_MARGIN_LEFT_IN / w
    bottom = GRID_MARGIN_BOTTOM_IN / h
    right = 1.0 - ((GRID_MARGIN_RIGHT_IN + (GRID_LEGEND_STRIP_IN if has_external_legend else 0.0)) / w)
    top = 1.0 - ((GRID_MARGIN_TOP_IN + (GRID_SUPTITLE_IN if has_suptitle else 0.0)) / h)

    left = min(max(left, 0.01), 0.45)
    bottom = min(max(bottom, 0.01), 0.30)
    right = min(max(right, left + 0.20), 0.99)
    top = min(max(top, bottom + 0.20), 0.99)
    return [left, bottom, right, top]

def prepare_figure_for_display(fig):
    """Apply one shared layout pass for both notebook display and export."""
    if fig is None:
        return None
    apply_grid_size(fig, panel_kind=getattr(fig, '_grid_panel_kind', 'simple'))
    rect = _display_layout_rect(fig)
    if hasattr(type(fig), '_grid_original_tight_layout'):
        try:
            type(fig)._grid_original_tight_layout(fig, rect=rect)
        except Exception:
            pass
    try:
        fig.canvas.draw()
    except Exception:
        pass
    return fig


def apply_grid_size(fig, panel_kind='simple'):
    if fig is None:
        return None
    nrows, ncols = _figure_grid_shape(fig)
    if panel_kind is None:
        panel_kind = getattr(fig, '_grid_panel_kind', 'simple')
    w, h = grid_figsize(
        nrows=nrows,
        ncols=ncols,
        panel_kind=panel_kind,
        has_legend=len(getattr(fig, 'legends', [])) > 0,
        has_suptitle=getattr(fig, '_suptitle', None) is not None,
    )
    fig.set_size_inches(w, h, forward=True)
    return fig.get_size_inches()


def shrink_figure_to_pnas(fig, panel_kind='simple'):
    # Kept for compatibility with existing code paths.
    return apply_grid_size(fig, panel_kind=panel_kind)


def _install_pnas_matplotlib_sizers():
    from matplotlib.axes import Axes as _MplAxes
    from matplotlib.figure import Figure as _MplFigure

    if not hasattr(plt, '_grid_original_subplots'):
        plt._grid_original_subplots = plt.subplots
        plt._grid_original_figure = plt.figure
        plt._grid_original_show = plt.show

        def _grid_subplots(*args, **kwargs):
            panel_kind = kwargs.pop('panel_kind', None)
            nrows, ncols = _parse_subplots_shape(args, kwargs)
            kind = _guess_panel_kind(panel_kind, nrows=nrows, ncols=ncols)

            # Enforce single helper for all figure sizes.
            kwargs.pop('figsize', None)
            kwargs['figsize'] = grid_figsize(nrows=nrows, ncols=ncols, panel_kind=kind)

            if (nrows * ncols) > 1:
                gkw = dict(kwargs.get('gridspec_kw', {}))
                gkw.setdefault('wspace', GRID_WSPACE_IN / max(GRID_PANEL_UNIT_W_IN, 1e-6))
                gkw.setdefault('hspace', GRID_HSPACE_IN / max(GRID_PANEL_UNIT_H_IN, 1e-6))
                kwargs['gridspec_kw'] = gkw

            prev = getattr(plt, '_grid_in_subplots', False)
            plt._grid_in_subplots = True
            try:
                fig, axes = plt._grid_original_subplots(*args, **kwargs)
            finally:
                plt._grid_in_subplots = prev

            fig._grid_helper_applied = True
            fig._grid_panel_kind = kind
            return fig, axes

        def _grid_figure(*args, **kwargs):
            # If called via subplots(), keep the subplots-computed figsize.
            called_from_subplots = getattr(plt, '_grid_in_subplots', False)
            panel_kind = kwargs.pop('panel_kind', None)
            if called_from_subplots:
                return plt._grid_original_figure(*args, **kwargs)

            # Standalone figure(): use standard single-panel base size.
            kwargs.pop('figsize', None)
            fig = plt._grid_original_figure(*args, **kwargs)
            kind = _guess_panel_kind(panel_kind, nrows=1, ncols=1)
            fig.set_size_inches(*grid_figsize(1, 1, panel_kind=kind), forward=True)
            fig._grid_panel_kind = kind
            return fig

        def _grid_show(*args, **kwargs):
            try:
                from matplotlib._pylab_helpers import Gcf
                for manager in Gcf.get_all_fig_managers():
                    fig = manager.canvas.figure
                    apply_grid_size(fig, panel_kind=getattr(fig, '_grid_panel_kind', 'simple'))
                    if '_style_boxplot_axis' in globals():
                        for ax in getattr(fig, 'axes', []):
                            _style_boxplot_axis(ax)
                    if 'apply_external_legend' in globals():
                        apply_external_legend(fig)
                    if 'sanitize_figure_text' in globals():
                        sanitize_figure_text(fig)
                    if 'prepare_figure_for_display' in globals():
                        prepare_figure_for_display(fig)
            except Exception:
                pass
            return plt._grid_original_show(*args, **kwargs)

        plt.subplots = _grid_subplots
        plt.figure = _grid_figure
        plt.show = _grid_show

    if not hasattr(_MplFigure, '_grid_original_add_subplot'):
        _MplFigure._grid_original_add_subplot = _MplFigure.add_subplot

        def _grid_add_subplot(self, *args, **kwargs):
            ax = _MplFigure._grid_original_add_subplot(self, *args, **kwargs)
            try:
                apply_grid_size(self, panel_kind=getattr(self, '_grid_panel_kind', 'simple'))
            except Exception:
                pass
            return ax

        _MplFigure.add_subplot = _grid_add_subplot

    if not hasattr(plt, '_pnas_original_scatter'):
        plt._pnas_original_scatter = plt.scatter
        plt._pnas_original_axes_scatter = _MplAxes.scatter

        def _pnas_plt_scatter(*args, **kwargs):
            kwargs = dict(kwargs)
            kwargs.setdefault('s', PNAS_SCATTER_AREA_DEFAULT)
            return plt._pnas_original_scatter(*args, **kwargs)

        def _pnas_axes_scatter(self, *args, **kwargs):
            kwargs = dict(kwargs)
            kwargs.setdefault('s', PNAS_SCATTER_AREA_DEFAULT)
            return plt._pnas_original_axes_scatter(self, *args, **kwargs)

        plt.scatter = _pnas_plt_scatter
        _MplAxes.scatter = _pnas_axes_scatter

    def _pnas_boxplot_defaults(kwargs=None):
        kw = dict(kwargs or {})
        kw.setdefault('patch_artist', True)
        kw.setdefault('showmeans', True)
        kw.setdefault('meanline', True)
        kw.setdefault('showcaps', False)

        boxprops = dict(kw.get('boxprops', {}))
        boxprops.setdefault('edgecolor', 'none')
        boxprops.setdefault('linewidth', 0.0)
        kw['boxprops'] = boxprops

        medianprops = dict(kw.get('medianprops', {}))
        medianprops.setdefault('color', 'none')
        medianprops.setdefault('linewidth', 0.0)
        kw['medianprops'] = medianprops

        meanprops = dict(kw.get('meanprops', {}))
        meanprops.setdefault('color', 'white')
        meanprops.setdefault('linewidth', clamp_linewidth(2.2) if 'clamp_linewidth' in globals() else 2.2)
        meanprops.setdefault('linestyle', '-')
        kw['meanprops'] = meanprops

        whiskerprops = dict(kw.get('whiskerprops', {}))
        whiskerprops.setdefault('linewidth', clamp_linewidth(1.8) if 'clamp_linewidth' in globals() else 1.8)
        kw['whiskerprops'] = whiskerprops

        capprops = dict(kw.get('capprops', {}))
        capprops.setdefault('linewidth', 0.0)
        capprops.setdefault('alpha', 0.0)
        kw['capprops'] = capprops
        return kw

    if not hasattr(plt, '_pnas_original_boxplot'):
        plt._pnas_original_boxplot = plt.boxplot
        plt._pnas_original_axes_boxplot = _MplAxes.boxplot
        plt._pnas_original_axes_bxp = _MplAxes.bxp

        def _pnas_plt_boxplot(*args, **kwargs):
            return plt._pnas_original_boxplot(*args, **_pnas_boxplot_defaults(kwargs))

        def _pnas_axes_boxplot(self, *args, **kwargs):
            return plt._pnas_original_axes_boxplot(self, *args, **_pnas_boxplot_defaults(kwargs))

        def _pnas_axes_bxp(self, *args, **kwargs):
            return plt._pnas_original_axes_bxp(self, *args, **_pnas_boxplot_defaults(kwargs))

        plt.boxplot = _pnas_plt_boxplot
        _MplAxes.boxplot = _pnas_axes_boxplot
        _MplAxes.bxp = _pnas_axes_bxp

    if not hasattr(_MplFigure, '_grid_original_tight_layout'):
        _MplFigure._grid_original_tight_layout = _MplFigure.tight_layout

        def _grid_tight_layout(self, *args, **kwargs):
            try:
                if '_style_boxplot_axis' in globals():
                    for ax in getattr(self, 'axes', []):
                        _style_boxplot_axis(ax)
                if 'apply_external_legend' in globals():
                    apply_external_legend(self)
                if 'sanitize_figure_text' in globals():
                    sanitize_figure_text(self)
                if 'prepare_figure_for_display' in globals():
                    prepare_figure_for_display(self)
                    return self
            except Exception:
                pass
            return _MplFigure._grid_original_tight_layout(self, *args, **kwargs)

        _MplFigure.tight_layout = _grid_tight_layout

    if not hasattr(_MplFigure, '_grid_original_savefig'):
        _MplFigure._grid_original_savefig = _MplFigure.savefig

        def _grid_savefig(self, *args, **kwargs):
            try:
                if '_style_boxplot_axis' in globals():
                    for ax in getattr(self, 'axes', []):
                        _style_boxplot_axis(ax)
                if 'apply_external_legend' in globals():
                    apply_external_legend(self)
                if 'sanitize_figure_text' in globals():
                    sanitize_figure_text(self)
            except Exception:
                pass

            kwargs = dict(kwargs)
            extra_artists = list(kwargs.get('bbox_extra_artists', []))
            extra_artists.extend([lg for lg in getattr(self, 'legends', []) if lg is not None])
            if extra_artists:
                kwargs['bbox_extra_artists'] = extra_artists

            try:
                return _MplFigure._grid_original_savefig(self, *args, **kwargs)
            except PermissionError as exc:
                target = args[0] if len(args) >= 1 else kwargs.get('fname', None)
                retry_path = _permission_retry_path(target, prefer_same_dir=True) if '_permission_retry_path' in globals() else None
                if retry_path is not None:
                    print(f'Permission denied saving to {target}; retrying at {retry_path}')
                    retry_args, retry_kwargs = _replace_savefig_target(args, kwargs, retry_path)
                    try:
                        return _MplFigure._grid_original_savefig(self, *retry_args, **retry_kwargs)
                    except PermissionError:
                        fallback_path = _permission_retry_path(target, prefer_same_dir=False) if '_permission_retry_path' in globals() else None
                        if fallback_path is not None:
                            print(f'Permission denied saving to {target}; using fallback export {fallback_path}')
                            fallback_args, fallback_kwargs = _replace_savefig_target(args, kwargs, fallback_path)
                            return _MplFigure._grid_original_savefig(self, *fallback_args, **fallback_kwargs)
                raise exc

        _MplFigure.savefig = _grid_savefig

    if not hasattr(_MplFigure, '_grid_original_show'):
        _MplFigure._grid_original_show = _MplFigure.show

        def _grid_figure_show(self, *args, **kwargs):
            try:
                apply_grid_size(self, panel_kind=getattr(self, '_grid_panel_kind', 'simple'))
                if '_style_boxplot_axis' in globals():
                    for ax in getattr(self, 'axes', []):
                        _style_boxplot_axis(ax)
                if 'apply_external_legend' in globals():
                    apply_external_legend(self)
                if 'sanitize_figure_text' in globals():
                    sanitize_figure_text(self)
                if 'prepare_figure_for_display' in globals():
                    prepare_figure_for_display(self)
            except Exception:
                pass
            return _MplFigure._grid_original_show(self, *args, **kwargs)

        _MplFigure.show = _grid_figure_show


def _boxplot_face_rgba(patch):
    if not hasattr(patch, 'get_facecolor'):
        return None
    fc = patch.get_facecolor()
    if fc is None:
        return None
    if hasattr(fc, '__len__') and len(np.shape(fc)) > 1:
        fc = fc[0]
    if len(fc) < 4:
        return None
    return tuple(fc)


def _boxplot_center_x(patch):
    try:
        verts = np.asarray(patch.get_path().vertices, dtype=float)
    except Exception:
        return np.nan
    if verts.ndim != 2 or verts.shape[1] < 2:
        return np.nan
    x = verts[:, 0]
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.nanmean(x))


def _style_boxplot_axis(ax):
    if ax is None:
        return
    patches = [p for p in getattr(ax, 'patches', []) if _boxplot_face_rgba(p) is not None]
    if not patches:
        return

    for patch in patches:
        patch.set_edgecolor('none')
        patch.set_linewidth(0.0)

    line_groups = {}
    for line in getattr(ax, 'lines', []):
        try:
            x = np.asarray(line.get_xdata(), dtype=float)
            y = np.asarray(line.get_ydata(), dtype=float)
        except Exception:
            continue
        if x.size == 0 or y.size == 0:
            continue
        if not np.isfinite(np.nanmean(x)):
            continue
        center = round(float(np.nanmean(x)), 3)
        line_groups.setdefault(center, []).append(line)

    centers = sorted(line_groups)
    for patch in patches:
        color = _boxplot_face_rgba(patch)
        if color is None or not centers:
            continue
        patch_center = _boxplot_center_x(patch)
        if not np.isfinite(patch_center):
            continue
        nearest_center = min(centers, key=lambda c: abs(float(c) - patch_center))
        group = line_groups.get(nearest_center, [])
        horiz = []
        vert = []
        for line in group:
            x = np.asarray(line.get_xdata(), dtype=float)
            y = np.asarray(line.get_ydata(), dtype=float)
            xr = float(np.nanmax(x) - np.nanmin(x)) if x.size else 0.0
            yr = float(np.nanmax(y) - np.nanmin(y)) if y.size else 0.0
            if xr >= yr:
                horiz.append((line, xr))
            else:
                vert.append(line)

        for line in vert:
            line.set_color(color)
            line.set_linewidth(clamp_linewidth(1.8))
            line.set_linestyle('-')

        if horiz:
            horiz = sorted(horiz, key=lambda item: float(item[1]), reverse=True)
            keep_width = horiz[0][1] if horiz else 0.0
            kept = False
            for line, xr in horiz:
                if keep_width > 0 and xr < (0.75 * keep_width):
                    line.set_visible(False)
                    continue
                if not kept:
                    line.set_color('white')
                    line.set_linewidth(clamp_linewidth(2.2))
                    line.set_linestyle('-')
                    kept = True
                else:
                    line.set_visible(False)


def _enforce_pnas_artist_style(ax):
    if '_style_boxplot_axis' in globals():
        _style_boxplot_axis(ax)
    for line in ax.lines:
        if hasattr(line, 'get_linewidth') and line.get_linewidth() is not None:
            line.set_linewidth(clamp_linewidth(line.get_linewidth()))
        if hasattr(line, 'get_markersize') and line.get_markersize() is not None:
            line.set_markersize(clamp_markersize(line.get_markersize()))

    for collection in ax.collections:
        if hasattr(collection, 'get_linewidths'):
            widths = collection.get_linewidths()
            if widths is not None and len(widths) > 0:
                collection.set_linewidths(clamp_linewidth(widths))
        if hasattr(collection, 'get_sizes'):
            sizes = collection.get_sizes()
            if sizes is not None and len(sizes) > 0:
                collection.set_sizes(clamp_scatter_area(sizes))

if 'plt' in globals():
    plt.rcParams['axes.grid'] = False
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.spines.right'] = False
    plt.rcParams['legend.frameon'] = False
    plt.rcParams['pdf.fonttype'] = 42
    plt.rcParams['ps.fonttype'] = 42
    plt.rcParams['svg.fonttype'] = 'none'
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = clamp_fontsize(8.0)
    plt.rcParams['axes.labelsize'] = clamp_fontsize(8.0)
    plt.rcParams['axes.titlesize'] = clamp_fontsize(9.0)
    plt.rcParams['xtick.labelsize'] = clamp_fontsize(7.0)
    plt.rcParams['ytick.labelsize'] = clamp_fontsize(7.0)
    plt.rcParams['legend.fontsize'] = clamp_fontsize(7.0)
    plt.rcParams['lines.linewidth'] = clamp_linewidth(0.9)
    plt.rcParams['lines.markersize'] = clamp_markersize(5.0)
    plt.rcParams['contour.linewidth'] = clamp_linewidth(0.8)
    _install_pnas_matplotlib_sizers()

# Helper function for cleaning bouton IDs
def clean_bouton_id(raw_bouton_id: str) -> str:
    """Remove file suffixes from bouton IDs for consistent matching"""
    return raw_bouton_id.replace('_traces_converted', '')

# Savitzky-Golay smoothing function (for additional analysis if needed)
def sg_smooth(y: np.ndarray, window_length: int = 9, polyorder: int = 2) -> np.ndarray:
    """Apply Savitzky-Golay filter, handling NaN gracefully."""
    from scipy.signal import savgol_filter
    if np.all(np.isnan(y)):
        return y.copy()
    valid = np.isfinite(y)
    if valid.sum() < max(window_length, polyorder + 2):
        return y.copy()
    result = y.copy()
    result[valid] = savgol_filter(y[valid], min(window_length, valid.sum() // 2 * 2 - 1), polyorder)
    return result

# Unified trace selection / stats / plotting API

def _normalize_trace_source(source=None):
    src = TRACE_MEAN_SOURCE if source is None else source
    src = str(src).strip().lower()
    if src not in {'normalized', 'raw', 'raw_nearest'}:
        raise ValueError(f'Unsupported trace source: {source}')
    return src


def _trace_source_dataframe(source=None):
    return NORM_TRACES_DATAFRAME if _normalize_trace_source(source) == 'normalized' else RAW_TRACES_DF


def _normalize_trace_alignment(alignment=None, source=None):
    src = _normalize_trace_source(source)
    align = 'auto' if alignment is None else str(alignment).strip().lower()
    if align == 'auto':
        return 'nearest' if src == 'raw_nearest' else 'grid'
    if align not in {'grid', 'nearest'}:
        raise ValueError(f'Unsupported trace alignment: {alignment}')
    return align


def select_traces(trace_ids=None, condition_names=None, rows=None, source=None, filters=None):
    selected = rows.copy() if rows is not None else _trace_source_dataframe(source).copy()
    if 'BaseID' not in selected.columns and 'ID' in selected.columns:
        selected['BaseID'] = selected['ID'].map(_normalize_bouton_id)
    if trace_ids is not None:
        ids = trace_ids if isinstance(trace_ids, (list, tuple, set, pd.Index, np.ndarray, pd.Series)) else [trace_ids]
        norm_ids = pd.Index([_normalize_bouton_id(v) for v in ids])
        selected = selected[selected['BaseID'].astype(str).isin(norm_ids.astype(str))].copy()
    if condition_names is not None:
        conds = [condition_names] if isinstance(condition_names, str) else list(condition_names)
        selected = selected[selected['Condition'].isin(conds)].copy()
        if trace_ids is not None and 'BaseID' in selected.columns:
            cond_priority = {cond: idx for idx, cond in enumerate(conds)}
            selected['_cond_priority'] = selected['Condition'].map(lambda c: cond_priority.get(c, len(cond_priority)))
            selected = selected.sort_values(['_cond_priority']).drop_duplicates(subset='BaseID', keep='first').drop(columns=['_cond_priority']).reset_index(drop=True)
    if filters is not None:
        if callable(filters):
            selected = selected[np.asarray(filters(selected), dtype=bool)].copy()
        elif isinstance(filters, dict):
            for col, expected in filters.items():
                if callable(expected):
                    selected = selected[np.asarray(expected(selected[col]), dtype=bool)].copy()
                elif isinstance(expected, (list, tuple, set, pd.Index, np.ndarray, pd.Series)):
                    selected = selected[selected[col].isin(list(expected))].copy()
                else:
                    selected = selected[selected[col] == expected].copy()
        else:
            raise ValueError('filters must be a callable or dict')
    return selected


def _trace_pairs(rows, source=None):
    src = _normalize_trace_source(source)
    pairs = []
    for _, row in rows.iterrows():
        t = np.asarray(row['Time'], float)
        y = np.asarray(row['Avg'], float)
        cond = row['Condition'] if 'Condition' in row.index else None
        if src != 'normalized' and cond in EXCEPTIONAL_CONDITIONS:
            t = t + EXCEPTIONAL_BASELINE_OFFSET
        pairs.append((t, y))
    return pairs


def _reference_time(pairs, resample=1.0):
    scale = 1.0 if resample is None else float(resample)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f'resample must be positive, got {resample!r}')
    for t_src, _ in pairs:
        t_src = np.asarray(t_src, float)
        t_valid = t_src[np.isfinite(t_src)]
        if t_valid.size < 2:
            continue
        diffs = np.diff(t_valid)
        diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
        if diffs.size:
            dt = float(np.nanmedian(diffs)) / scale
            return np.arange(t_valid[0], t_valid[-1] + dt / 2.0, dt, dtype=float)
    raise ValueError('No valid time vector available for trace resampling')


def _trace_matrix(rows, source=None, alignment='auto', resample=1.0, tol_factor=None):
    pairs = _trace_pairs(rows, source=source)
    if not pairs:
        return None, None
    t_ref = _reference_time(pairs, resample=resample)
    align = _normalize_trace_alignment(alignment, source=source)
    if align == 'grid':
        def _interp(t_src, y_src):
            finite = np.isfinite(t_src) & np.isfinite(y_src)
            if finite.sum() < 2:
                return np.full_like(t_ref, np.nan, dtype=float)
            t_valid = t_src[finite]
            y_valid = y_src[finite]
            out = np.interp(t_ref, t_valid, y_valid)
            out[(t_ref < t_valid[0]) | (t_ref > t_valid[-1])] = np.nan
            return out
        return t_ref, np.vstack([_interp(np.asarray(t, float), np.asarray(y, float)) for t, y in pairs])
    tol_factor = TRACE_RAW_NEAREST_TOL_FACTOR if tol_factor is None else float(tol_factor)
    diffs = [np.diff(np.asarray(t, float)) for t, _ in pairs if len(t) > 1]
    diffs = [d[np.isfinite(d) & (d > 0)] for d in diffs if np.any(np.isfinite(d))]
    tol = tol_factor * float(np.nanmedian(np.concatenate(diffs))) if diffs else np.inf
    mat = np.full((len(pairs), len(t_ref)), np.nan, dtype=float)
    for i, (t_src, y_src) in enumerate(pairs):
        t_src = np.asarray(t_src, float)
        y_src = np.asarray(y_src, float)
        finite = np.isfinite(t_src) & np.isfinite(y_src)
        if finite.sum() == 0:
            continue
        t_valid = t_src[finite]
        y_valid = y_src[finite]
        idx = np.searchsorted(t_valid, t_ref)
        idx0 = np.clip(idx - 1, 0, len(t_valid) - 1)
        idx1 = np.clip(idx, 0, len(t_valid) - 1)
        d0 = np.abs(t_valid[idx0] - t_ref)
        d1 = np.abs(t_valid[idx1] - t_ref)
        use1 = d1 < d0
        best = np.where(use1, idx1, idx0)
        dist = np.where(use1, d1, d0)
        valid = dist <= tol
        mat[i, valid] = y_valid[best[valid]]
    return t_ref, mat


def _aggregate_trace_matrix(matrix, aggregation='mean'):
    agg = 'mean' if aggregation is None else str(aggregation).strip().lower()
    if agg == 'median':
        return np.nanmedian(matrix, axis=0)
    if agg == 'max':
        return np.nanmax(matrix, axis=0)
    if agg == 'std':
        return np.nanstd(matrix, axis=0)
    return np.nanmean(matrix, axis=0)


def compute_trace_stats(trace_ids=None, condition_names=None, rows=None, source=None, *, filters=None, alignment='auto', resample=1, aggregation='mean', smooth=None, tol_factor=None, return_matrix=False):
    selected = select_traces(trace_ids=trace_ids, condition_names=condition_names, rows=rows, source=source, filters=filters)
    if selected is None or len(selected) == 0:
        raise ValueError('No traces matched the requested selection')
    t_ref, matrix = _trace_matrix(selected, source=source, alignment=alignment, resample=resample, tol_factor=tol_factor)
    average = _aggregate_trace_matrix(matrix, aggregation=aggregation)
    if smooth:
        cfg = smooth if isinstance(smooth, dict) else {}
        average = sg_smooth(average, window_length=int(cfg.get('window_length', cfg.get('window', 9))), polyorder=int(cfg.get('polyorder', 2)))
    n_eff = np.sum(np.isfinite(matrix), axis=0)
    sem = np.nanstd(matrix, axis=0, ddof=1) / np.sqrt(np.maximum(n_eff, 1))
    return {'time': t_ref, 'average': average, 'sem': sem, 'n': int(matrix.shape[0]), 'rows': selected, 'matrix': matrix if return_matrix else None}


def plot_traces(ax=None, trace_ids=None, condition_names=None, rows=None, source=None, *, filters=None, show_average=True, show_sem=True, show_individuals=False, alignment='auto', resample=1, aggregation='mean', smooth=None, tol_factor=None, color='k', individual_color=None, sem_color=None, label=None, linewidth=2.0, linestyle='-', sem_alpha=0.25, individual_alpha=0.12, individual_lw=0.6, stim_times=None, stim_kwargs=None, zero_line=False, zero_kwargs=None, hlines=None, hline_kwargs=None, event_time=None, event_kwargs=None, xlim=None, ylim=None, xlabel=None, ylabel=None, title=None, legend=False, legend_kwargs=None, style_axis=True, return_data=False):
    """Unified trace plotting helper.

    Data selection:
    - provide one of `rows`, `trace_ids`, or `condition_names`
    - `source`: None | 'normalized' | 'raw' | 'raw_nearest'
    - `filters`: optional extra row filter(s) passed to `select_traces`

    Trace computation:
    - `alignment`: 'auto' | 'grid' | 'nearest'
    - `resample`: grid resampling factor
    - `aggregation`: 'mean' | 'median' | 'robust_mean' | 'max' | 'std'
    - `smooth`: optional post-aggregation smoothing spec
    - `tol_factor`: nearest-sample tolerance multiplier

    What to draw:
    - `show_average`, `show_sem`, `show_individuals`
    - `color`, `individual_color`, `sem_color`, `label`
    - `linewidth`, `linestyle`, `sem_alpha`, `individual_alpha`, `individual_lw`

    Common trace decorators:
    - `stim_times` with `stim_kwargs`
    - `zero_line` with `zero_kwargs`
    - `hlines` with `hline_kwargs`
    - `event_time` with `event_kwargs`
    - axis labels / limits / title / legend through `xlim`, `ylim`, `xlabel`, `ylabel`, `title`, `legend`, `legend_kwargs`

    Return value:
    - returns `ax` by default
    - returns the computed stats dict when `return_data=True`
    """
    if ax is None:
        _, ax = make_figure_grid(panel_kind='trace')
    stats = compute_trace_stats(trace_ids=trace_ids, condition_names=condition_names, rows=rows, source=source, filters=filters, alignment=alignment, resample=resample, aggregation=aggregation, smooth=smooth, tol_factor=tol_factor, return_matrix=show_individuals)
    stats['ax'] = ax
    plot_lw = clamp_linewidth(linewidth)
    ind_lw = clamp_linewidth(individual_lw)
    ind_color = individual_color if individual_color is not None else color
    fill_color = sem_color if sem_color is not None else color
    if show_individuals and stats['matrix'] is not None:
        for trace in stats['matrix']:
            ax.plot(stats['time'], trace, color=ind_color, alpha=individual_alpha, linewidth=ind_lw)
    if show_sem and stats['sem'] is not None:
        ax.fill_between(stats['time'], stats['average'] - stats['sem'], stats['average'] + stats['sem'], color=fill_color, alpha=sem_alpha)
    if show_average:
        ax.plot(stats['time'], stats['average'], color=color, linewidth=plot_lw, linestyle=linestyle, label=label)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if stim_times is not None:
        add_stimulus_ticks(ax, stim_times, **(stim_kwargs or {}))
    if zero_line:
        zero_defaults = {'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1}
        zero_defaults.update(zero_kwargs or {})
        ax.axhline(0, **zero_defaults)
    if hlines is not None:
        ref_defaults = {'color': 'black', 'linestyle': ':', 'alpha': 0.5, 'linewidth': 1}
        ref_defaults.update(hline_kwargs or {})
        for y_value in hlines:
            ax.axhline(y_value, **ref_defaults)
    if event_time is not None:
        event_defaults = {'color': 'gray', 'linestyle': '--', 'alpha': 0.5, 'linewidth': 1}
        event_defaults.update(event_kwargs or {})
        ax.axvline(event_time, **event_defaults)
    if style_axis:
        style_trace_axis(ax)
    if legend:
        add_legend(ax, frameon=False, **(legend_kwargs or {}))
    return stats if return_data else ax


def build_trace_lookup(traces_dataframe, id_col='ID', time_col='Time', avg_col='Avg'):
    lookup = {}
    if traces_dataframe is None or len(traces_dataframe) == 0:
        return lookup
    for _, trace_row in traces_dataframe.iterrows():
        lookup[str(trace_row[id_col])] = {'Time': trace_row[time_col], 'Avg': trace_row[avg_col]}
    return lookup


def build_trace_lookup_from_source(source=None, condition_names=None):
    src = _normalize_trace_source(source)
    rows = select_traces(condition_names=condition_names, source=src)
    lookup = {}
    for _, trace_row in rows.iterrows():
        time_values = np.asarray(trace_row['Time'], dtype=float)
        if src != 'normalized' and trace_row['Condition'] in EXCEPTIONAL_CONDITIONS:
            time_values = time_values + EXCEPTIONAL_BASELINE_OFFSET
        lookup[str(trace_row['ID'])] = {'Time': time_values, 'Avg': trace_row['Avg'], 'Condition': trace_row.get('Condition', None)}
    return lookup

print("✓ Helper functions loaded")

# Shared trace helper utilities

def add_stimulus_ticks(
    ax,
    stim_times,
    *,
    mode='top',
    y_span=None,
    tick_ratio=0.03,
    color='black',
    linewidth=1.5,
):
    """Add stimulus ticks either at top of axes or at a fixed y-span."""
    if mode == 'fixed':
        if y_span is None:
            raise ValueError("y_span must be provided when mode='fixed'")
        y0, y1 = y_span
        clip_on = True
    else:
        y_min, y_max = ax.get_ylim()
        tick_height = (y_max - y_min) * tick_ratio
        y0, y1 = y_max - tick_height, y_max
        clip_on = False

    tick_lw = clamp_linewidth(linewidth) if 'clamp_linewidth' in globals() else linewidth
    for stim_time in stim_times:
        ax.plot([stim_time, stim_time], [y0, y1], color=color, linewidth=tick_lw, clip_on=clip_on)

def _apply_clean_axes_style(ax, panel_kind='simple'):
    """Apply no-grid and no top/right spines style."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    tick_w = clamp_linewidth(0.8) if 'clamp_linewidth' in globals() else 0.8
    ax.tick_params(direction='out', length=4, width=tick_w)
    ax.grid(False)
    if 'shrink_figure_to_pnas' in globals():
        shrink_figure_to_pnas(ax.figure, panel_kind=panel_kind)
    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)

def style_trace_axis(ax):
    """Apply consistent style for trace plots."""
    _apply_clean_axes_style(ax, panel_kind='trace')
    legend = ax.get_legend()
    if legend is not None:
        legend.set_frame_on(False)

def style_hist_axis(ax):
    """Apply consistent style for histogram plots."""
    _apply_clean_axes_style(ax, panel_kind='hist')
    legend = ax.get_legend()
    if legend is not None:
        legend.set_frame_on(False)



# ===== Shared PPR Helpers (moved from later cell) =====
def ppr_profile_stats(condition_dataframe, ppr_column_names=None, max_pulse_number=10):
    """Return pulse numbers, means, SEMs, and n for a condition dataframe."""
    if ppr_column_names is None:
        ppr_column_names = [
            f'PPR{pulse_num}/1'
            for pulse_num in range(2, max_pulse_number + 1)
            if f'PPR{pulse_num}/1' in condition_dataframe.columns
        ]

    # Keep only columns present in dataframe, then drop columns with no finite values.
    ppr_column_names = [col for col in ppr_column_names if col in condition_dataframe.columns]
    if len(ppr_column_names) == 0:
        return [1], [1.0], [0.0], len(condition_dataframe)

    ppr_data = condition_dataframe[ppr_column_names].apply(pd.to_numeric, errors='coerce')
    valid_cols = [col for col in ppr_column_names if ppr_data[col].notna().any()]
    if len(valid_cols) == 0:
        return [1], [1.0], [0.0], len(condition_dataframe)

    means = [1.0] + ppr_data[valid_cols].mean().tolist()
    sems = [0.0] + ppr_data[valid_cols].sem().fillna(0.0).tolist()
    pulse_numbers = list(range(1, len(valid_cols) + 2))
    return pulse_numbers, means, sems, len(condition_dataframe)


def extract_ppr_profile(condition_dataframe, max_pulse_number=10):
    """Backward-compatible wrapper used in older cells."""
    pulse_numbers, means, sems, _ = ppr_profile_stats(
        condition_dataframe,
        max_pulse_number=max_pulse_number,
    )
    return means, sems, len(pulse_numbers)


def ppr_profiles_matrix(condition_dataframe, ppr_column_names=None, max_pulse_number=10):
    """Return matrix with pulse 1 forced to 1.0, then valid PPR2..PPRn columns."""
    if ppr_column_names is None:
        ppr_column_names = [
            f'PPR{pulse_num}/1'
            for pulse_num in range(2, max_pulse_number + 1)
            if f'PPR{pulse_num}/1' in condition_dataframe.columns
        ]

    ppr_column_names = [col for col in ppr_column_names if col in condition_dataframe.columns]
    n_rows = len(condition_dataframe)
    if len(ppr_column_names) == 0:
        return np.ones((n_rows, 1), dtype=float) if n_rows > 0 else np.empty((0, 1), dtype=float)

    ppr_data = condition_dataframe[ppr_column_names].apply(pd.to_numeric, errors='coerce')
    valid_cols = [col for col in ppr_column_names if ppr_data[col].notna().any()]
    if len(valid_cols) == 0:
        return np.ones((n_rows, 1), dtype=float) if n_rows > 0 else np.empty((0, 1), dtype=float)

    arr = ppr_data[valid_cols].to_numpy(dtype=float)
    return np.hstack([np.ones((arr.shape[0], 1), dtype=float), arr])




def failure_profiles_matrix(
    condition_dataframe,
    trials_df=None,
    *,
    threshold_col='thr_shared',
    max_pulse_number=10,
    min_trials=1,
):
    """Return bouton × pulse failure-rate matrix using explicit per-event failures."""
    if condition_dataframe is None or len(condition_dataframe) == 0:
        return np.empty((0, int(max_pulse_number)), dtype=float)

    failrate_cols = [f'FailRate{k}' for k in range(1, int(max_pulse_number) + 1)]
    if all(col in condition_dataframe.columns for col in failrate_cols):
        return condition_dataframe[failrate_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)

    if trials_df is None:
        if 'trials_all' not in globals():
            raise RuntimeError('trials_all is required to build explicit failure-rate profiles.')
        trials_df = trials_all

    if 'ID' not in condition_dataframe.columns:
        raise ValueError("condition_dataframe must contain an 'ID' column")

    cond_col = 'Condition' if 'Condition' in condition_dataframe.columns else None
    trial_df = trials_df.copy()

    if cond_col is not None and 'condition' in trial_df.columns:
        wanted_conditions = (
            condition_dataframe[cond_col]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        trial_df = trial_df[trial_df['condition'].astype(str).str.strip().isin(wanted_conditions)]

    fail_lookup = defaultdict(list)
    for _, row in trial_df.iterrows():
        if 'file' not in row.index:
            continue
        trial_id = extract_base_name(str(row['file']).strip())
        trial_cond = str(row['condition']).strip() if 'condition' in row.index else ''
        fail_mask, _ = get_failure_mask_row(row, threshold_col=threshold_col)
        fail_mask = np.asarray(fail_mask, dtype=float)[:max_pulse_number]
        if not np.isfinite(fail_mask).any():
            continue
        key = (trial_cond, trial_id) if cond_col is not None else trial_id
        fail_lookup[key].append(fail_mask)

    profiles = []
    for _, row in condition_dataframe.iterrows():
        bouton_id = extract_base_name(str(row['ID']).strip())
        bouton_cond = str(row[cond_col]).strip() if cond_col is not None else ''
        key = (bouton_cond, bouton_id) if cond_col is not None else bouton_id
        trial_masks = fail_lookup.get(key, [])
        if len(trial_masks) < int(min_trials):
            profiles.append(np.full(int(max_pulse_number), np.nan, dtype=float))
            continue
        mat = np.asarray(trial_masks, dtype=float)
        profiles.append(np.nanmean(mat, axis=0))

    if len(profiles) == 0:
        return np.empty((0, int(max_pulse_number)), dtype=float)
    return np.vstack(profiles)

def plot_ppr_mean_sem(
    ax,
    pulse_numbers,
    means,
    sems,
    *,
    color,
    marker,
    label,
    linewidth=2.0,
    markersize=6,
    sem_alpha=0.2,
):
    """Plot one mean PPR trajectory with SEM shading (robust to NaN segments)."""
    x = np.asarray(pulse_numbers, dtype=float)
    means_arr = np.asarray(means, dtype=float)
    sems_arr = np.asarray(sems, dtype=float)

    if len(x) != len(means_arr):
        n = min(len(x), len(means_arr), len(sems_arr))
        x, means_arr, sems_arr = x[:n], means_arr[:n], sems_arr[:n]

    valid_line = np.isfinite(x) & np.isfinite(means_arr)
    if not np.any(valid_line):
        return

    linewidth = clamp_linewidth(linewidth) if 'clamp_linewidth' in globals() else linewidth
    markersize = clamp_markersize(markersize) if 'clamp_markersize' in globals() else markersize

    ax.plot(
        x[valid_line],
        means_arr[valid_line],
        marker=marker,
        color=color,
        linewidth=linewidth,
        markersize=markersize,
        label=label,
    )

    lower = means_arr - sems_arr
    upper = means_arr + sems_arr
    valid_fill = np.isfinite(x) & np.isfinite(lower) & np.isfinite(upper)
    if np.any(valid_fill):
        ax.fill_between(x, lower, upper, where=valid_fill, alpha=sem_alpha, color=color)


def plot_ppr_profiles_overlay(
    ax,
    profiles_matrix,
    *,
    color,
    label='Mean',
    individual_alpha=0.25,
    individual_lw=1.0,
    mean_lw=2.5,
    sem_alpha=0.18,
):
    """Plot individual PPR profiles + mean ± SEM."""
    if profiles_matrix is None or profiles_matrix.size == 0:
        return None

    pulse_numbers = list(range(1, profiles_matrix.shape[1] + 1))
    x = np.asarray(pulse_numbers, dtype=float)

    individual_lw = clamp_linewidth(individual_lw) if 'clamp_linewidth' in globals() else individual_lw
    mean_lw = clamp_linewidth(mean_lw) if 'clamp_linewidth' in globals() else mean_lw

    for i in range(profiles_matrix.shape[0]):
        ax.plot(pulse_numbers, profiles_matrix[i], color=color, alpha=individual_alpha, linewidth=individual_lw)

    mean_profile = np.nanmean(profiles_matrix, axis=0)
    n_eff = np.sum(np.isfinite(profiles_matrix), axis=0)
    if profiles_matrix.shape[0] > 1:
        sem_profile = np.nanstd(profiles_matrix, axis=0, ddof=1) / np.sqrt(np.maximum(n_eff, 1))
        sem_profile[n_eff < 2] = 0.0
    else:
        sem_profile = np.zeros_like(mean_profile)

    valid_line = np.isfinite(x) & np.isfinite(mean_profile)
    if np.any(valid_line):
        ax.plot(x[valid_line], mean_profile[valid_line], color=color, linewidth=mean_lw, label=label)

    lower = mean_profile - sem_profile
    upper = mean_profile + sem_profile
    valid_fill = np.isfinite(x) & np.isfinite(lower) & np.isfinite(upper)
    if np.any(valid_fill):
        ax.fill_between(x, lower, upper, where=valid_fill, color=color, alpha=sem_alpha)

    return pulse_numbers, mean_profile, sem_profile


def finalize_ppr_axis(
    ax,
    pulse_numbers,
    *,
    title=None,
    xlabel='Pulse Number',
    ylabel='PPR (A_n/A_1)',
    show_xlabel='auto',
    show_ylabel='auto',
    ylim=None,
    unity_line=True,
    unity_kwargs=None,
    legend=True,
    legend_loc='best',
    legend_fontsize=None,
    legend_frame=False,
    grid=True,
    grid_alpha=0.3,
    xlabel_fontsize=None,
    ylabel_fontsize=None,
    title_fontsize=None,
    title_fontweight=None,
):
    """Apply shared PPR axis formatting across notebook figures."""
    if unity_line:
        line_kwargs = dict(color='gray', linestyle='dotted', linewidth=2)
        if unity_kwargs:
            line_kwargs.update(unity_kwargs)
        if 'clamp_linewidth' in globals() and 'linewidth' in line_kwargs:
            line_kwargs['linewidth'] = clamp_linewidth(line_kwargs['linewidth'])
        ax.axhline(1.0, **line_kwargs)

    if show_xlabel == 'auto' or show_ylabel == 'auto':
        auto_x, auto_y = _outer_axis_label_visibility(ax) if '_outer_axis_label_visibility' in globals() else (True, True)
        if show_xlabel == 'auto':
            show_xlabel = auto_x
        if show_ylabel == 'auto':
            show_ylabel = auto_y

    if xlabel is not None:
        if show_xlabel:
            if xlabel_fontsize is None:
                ax.set_xlabel(xlabel)
            else:
                fs = clamp_fontsize(xlabel_fontsize) if 'clamp_fontsize' in globals() else xlabel_fontsize
                ax.set_xlabel(xlabel, fontsize=fs)
        else:
            ax.set_xlabel('')

    if ylabel is not None:
        if show_ylabel:
            if ylabel_fontsize is None:
                ax.set_ylabel(ylabel)
            else:
                fs = clamp_fontsize(ylabel_fontsize) if 'clamp_fontsize' in globals() else ylabel_fontsize
                ax.set_ylabel(ylabel, fontsize=fs)
        else:
            ax.set_ylabel('')

    if pulse_numbers is not None:
        ax.set_xticks(list(pulse_numbers))

    if ylim is not None:
        ax.set_ylim(ylim)

    if title is not None:
        title_kwargs = {}
        if title_fontsize is not None:
            title_kwargs['fontsize'] = clamp_fontsize(title_fontsize) if 'clamp_fontsize' in globals() else title_fontsize
        if title_fontweight is not None:
            title_kwargs['fontweight'] = title_fontweight
        ax.set_title(title, **title_kwargs)

    if legend:
        handles, _labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            legend_kwargs = dict(loc=legend_loc, frameon=legend_frame)
            if legend_fontsize is not None:
                legend_kwargs['fontsize'] = clamp_fontsize(legend_fontsize) if 'clamp_fontsize' in globals() else legend_fontsize
            lg = ax.legend(**legend_kwargs)
            if lg is not None:
                for txt in lg.get_texts():
                    txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    tick_w = clamp_linewidth(0.8) if 'clamp_linewidth' in globals() else 0.8
    ax.tick_params(direction='out', length=4, width=tick_w)
    ax.grid(False)
    if 'shrink_figure_to_pnas' in globals():
        shrink_figure_to_pnas(ax.figure, panel_kind='ppr')
    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)


# ===== Shared PCA Helpers (moved from later cell) =====
def get_cluster_color(cluster_id):
    cluster_id = int(cluster_id)
    if 'cluster_color_lookup' in globals() and cluster_id in cluster_color_lookup:
        return cluster_color_lookup[cluster_id]
    cmap = plt.get_cmap('tab10')
    return cmap((cluster_id - 1) % 10)

def get_cluster_colors(labels):
    return [get_cluster_color(cid) for cid in labels]

def get_cluster_hex_color(cluster_id):
    import matplotlib.colors as mcolors

    cluster_id = int(cluster_id)
    if 'cluster_hex_colors' in globals() and 1 <= cluster_id <= len(cluster_hex_colors):
        return cluster_hex_colors[cluster_id - 1]
    return mcolors.to_hex(get_cluster_color(cluster_id))

def plot_pca_background(
    ax,
    *,
    coords=None,
    labels=None,
    use_cluster_colors=True,
    color='gray',
    alpha=0.3,
    s=30,
    marker='o',
    edgecolors='none',
    linewidths=0.0,
    label='WT pooled',
    zorder=1,
):
    """Plot WT-like PCA background cloud with optional cluster colors."""
    if coords is None:
        coords = pca_coordinates
    if coords is None or len(coords) == 0:
        return None

    if use_cluster_colors:
        if labels is None:
            labels = cluster_assignments
        colors = get_cluster_colors(labels)
    else:
        colors = color

    s = clamp_scatter_area(s) if 'clamp_scatter_area' in globals() else s
    linewidths = clamp_linewidth(linewidths) if ('clamp_linewidth' in globals() and linewidths is not None) else linewidths
    kwargs = dict(c=colors, alpha=alpha, s=s, marker=marker, edgecolors=edgecolors, linewidths=linewidths, zorder=zorder)
    if label is not None:
        kwargs['label'] = label
    return ax.scatter(coords[:, 0], coords[:, 1], **kwargs)


def plot_pca_overlay_points(
    ax,
    coords,
    *,
    y=None,
    color='red',
    marker='o',
    s=30,
    alpha=0.8,
    edgecolors='black',
    linewidths=0.5,
    label=None,
    zorder=3,
    **kwargs,
):
    """Plot overlay points in PCA space from Nx2 coords or x/y vectors."""
    if coords is None:
        return None

    if y is not None:
        x_vals = np.asarray(coords)
        y_vals = np.asarray(y)
        if x_vals.size == 0 or y_vals.size == 0:
            return None
        coords = np.column_stack([x_vals.reshape(-1), y_vals.reshape(-1)])
    else:
        coords = np.asarray(coords)
        if coords.size == 0:
            return None
        if coords.ndim == 1 and coords.size == 2:
            coords = coords.reshape(1, 2)

    scatter_kwargs = dict(
        marker=marker,
        s=s,
        alpha=alpha,
        zorder=zorder,
    )
    if edgecolors is not None and 'edgecolors' not in kwargs and 'edgecolor' not in kwargs:
        scatter_kwargs['edgecolors'] = edgecolors
    if linewidths is not None and 'linewidths' not in kwargs and 'linewidth' not in kwargs and 'lw' not in kwargs:
        scatter_kwargs['linewidths'] = linewidths
    if color is not None and 'c' not in kwargs and 'color' not in kwargs:
        scatter_kwargs['c'] = color
    scatter_kwargs.update(kwargs)

    # Normalize matplotlib aliases to avoid conflicts (e.g., lw + linewidths)
    if 'lw' in scatter_kwargs:
        if 'linewidths' not in scatter_kwargs and 'linewidth' not in scatter_kwargs:
            scatter_kwargs['linewidths'] = scatter_kwargs['lw']
        scatter_kwargs.pop('lw', None)
    if label is not None:
        scatter_kwargs['label'] = label
    if 'clamp_scatter_area' in globals() and 's' in scatter_kwargs:
        scatter_kwargs['s'] = clamp_scatter_area(scatter_kwargs['s'])
    if 'clamp_linewidth' in globals():
        if 'linewidths' in scatter_kwargs and scatter_kwargs['linewidths'] is not None:
            scatter_kwargs['linewidths'] = clamp_linewidth(scatter_kwargs['linewidths'])
        if 'linewidth' in scatter_kwargs and scatter_kwargs['linewidth'] is not None:
            scatter_kwargs['linewidth'] = clamp_linewidth(scatter_kwargs['linewidth'])
    return ax.scatter(coords[:, 0], coords[:, 1], **scatter_kwargs)


def get_reference_pca_limits(pad=0.5, fallback_x=(-7.0, 10.0), fallback_y=(-6.0, 6.0), pad_frac=0.05):
    """Return the WT reference PCA window from the WT cloud itself, with a small margin."""
    coords = None
    if 'pca_data' in globals() and isinstance(globals().get('pca_data'), dict) and 'WT_pooled' in globals().get('pca_data', {}):
        arr = np.asarray(globals()['pca_data']['WT_pooled'], float)
        if arr.ndim == 2 and arr.shape[0] > 0 and arr.shape[1] >= 2:
            coords = arr[:, :2]
    if coords is None and 'pca_coordinates' in globals() and globals().get('pca_coordinates') is not None:
        arr = np.asarray(globals().get('pca_coordinates'), float)
        if arr.ndim == 2 and arr.shape[0] > 0 and arr.shape[1] >= 2:
            coords = arr[:, :2]
    if coords is None:
        return tuple(float(v) for v in fallback_x), tuple(float(v) for v in fallback_y)
    x = coords[:, 0][np.isfinite(coords[:, 0])]
    y = coords[:, 1][np.isfinite(coords[:, 1])]
    if len(x) == 0 or len(y) == 0:
        return tuple(float(v) for v in fallback_x), tuple(float(v) for v in fallback_y)
    xmn, xmx = float(np.min(x)), float(np.max(x))
    ymn, ymx = float(np.min(y)), float(np.max(y))
    xpad = max(float(pad), float((xmx - xmn) * pad_frac)) if np.isfinite(xmx - xmn) else float(pad)
    ypad = max(float(pad), float((ymx - ymn) * pad_frac)) if np.isfinite(ymx - ymn) else float(pad)
    return (xmn - xpad, xmx + xpad), (ymn - ypad, ymx + ypad)

def _draw_reference_pca_box(ax, ref_xlim, ref_ylim):
    """Show the WT pooled reference PCA footprint when current axes extend beyond it."""
    for ln in list(ax.lines):
        try:
            if ln.get_gid() == 'wt_ref_pca_box':
                ln.remove()
        except Exception:
            pass
    cur_xlim = ax.get_xlim()
    cur_ylim = ax.get_ylim()
    bigger = (
        (cur_xlim[0] < ref_xlim[0] - 1e-9) or (cur_xlim[1] > ref_xlim[1] + 1e-9)
        or (cur_ylim[0] < ref_ylim[0] - 1e-9) or (cur_ylim[1] > ref_ylim[1] + 1e-9)
    )
    if not bigger:
        return None
    keep_xlim = ax.get_xlim()
    keep_ylim = ax.get_ylim()
    xs = [ref_xlim[0], ref_xlim[1], ref_xlim[1], ref_xlim[0], ref_xlim[0]]
    ys = [ref_ylim[0], ref_ylim[0], ref_ylim[1], ref_ylim[1], ref_ylim[0]]
    line, = ax.plot(xs, ys, ls=':', lw=0.9, color='0.55', alpha=0.8, zorder=2)
    line.set_gid('wt_ref_pca_box')
    ax.set_xlim(keep_xlim)
    ax.set_ylim(keep_ylim)
    return line

def _pca_artist_data_limits(ax):
    """Return finite PCA point/curve bounds from plotted artists only."""
    xs = []
    ys = []
    for coll in getattr(ax, 'collections', []):
        try:
            offs = coll.get_offsets()
        except Exception:
            offs = None
        if offs is None:
            continue
        offs = np.asarray(offs, float)
        if offs.ndim != 2 or offs.shape[1] < 2 or offs.size == 0:
            continue
        ok = np.isfinite(offs[:, 0]) & np.isfinite(offs[:, 1])
        if np.any(ok):
            xs.append(offs[ok, 0])
            ys.append(offs[ok, 1])
    for ln in getattr(ax, 'lines', []):
        try:
            if ln.get_gid() == 'wt_ref_pca_box':
                continue
        except Exception:
            pass
        try:
            xdat = np.asarray(ln.get_xdata(orig=False), float)
            ydat = np.asarray(ln.get_ydata(orig=False), float)
        except Exception:
            continue
        ok = np.isfinite(xdat) & np.isfinite(ydat)
        if np.any(ok):
            xs.append(xdat[ok])
            ys.append(ydat[ok])
    if not xs or not ys:
        return None
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    if x.size == 0 or y.size == 0:
        return None
    return float(np.min(x)), float(np.max(x)), float(np.min(y)), float(np.max(y))

def style_pca_axes(
    ax,
    *,
    title=None,
    xlim=None,
    ylim=None,
    explained_variance=None,
    grid=False,
    legend=True,
    legend_frame=False,
    show_xlabel='auto',
    show_ylabel='auto',
    extend_limits=True,
    align_to_background=True,
):
    """Apply consistent PCA axis formatting with extend-only limits."""
    if explained_variance is None and 'PCA_RESULTS' in globals():
        explained_variance = PCA_RESULTS.get('explained_variance')

    image_extent = None
    if align_to_background and len(ax.images) > 0:
        ext = ax.images[0].get_extent()
        if ext is not None and len(ext) == 4:
            image_extent = (
                float(min(ext[0], ext[1])),
                float(max(ext[0], ext[1])),
                float(min(ext[2], ext[3])),
                float(max(ext[2], ext[3])),
            )

    ref_xlim, ref_ylim = get_reference_pca_limits(pad=0.5, fallback_x=(-7.0, 10.0), fallback_y=(-6.0, 6.0))
    if xlim is None:
        xlim = tuple(ref_xlim)
    if ylim is None:
        ylim = tuple(ref_ylim)
    if image_extent is not None:
        xlim = (float(image_extent[0]), float(image_extent[1]))
        ylim = (float(image_extent[2]), float(image_extent[3]))

    if show_xlabel == 'auto' or show_ylabel == 'auto':
        auto_x, auto_y = _outer_axis_label_visibility(ax) if '_outer_axis_label_visibility' in globals() else (True, True)
        if show_xlabel == 'auto':
            show_xlabel = auto_x
        if show_ylabel == 'auto':
            show_ylabel = auto_y

    if explained_variance is not None and len(explained_variance) >= 2:
        if show_xlabel:
            ax.set_xlabel(f"PC1 ({explained_variance[0]:.1%} variance)")
        else:
            ax.set_xlabel('')
        if show_ylabel:
            ax.set_ylabel(f"PC2 ({explained_variance[1]:.1%} variance)")
        else:
            ax.set_ylabel('')
    else:
        if not show_xlabel:
            ax.set_xlabel('')
        if not show_ylabel:
            ax.set_ylabel('')

    if xlim is not None:
        x0, x1 = float(xlim[0]), float(xlim[1])
        x0 = min(x0, float(ref_xlim[0]))
        x1 = max(x1, float(ref_xlim[1]))
        if extend_limits:
            data_limits = _pca_artist_data_limits(ax)
            if data_limits is not None:
                x0 = min(x0, float(data_limits[0]))
                x1 = max(x1, float(data_limits[1]))
        ax.set_xlim((x0, x1))

    if ylim is not None:
        y0, y1 = float(ylim[0]), float(ylim[1])
        y0 = min(y0, float(ref_ylim[0]))
        y1 = max(y1, float(ref_ylim[1]))
        if extend_limits:
            data_limits = _pca_artist_data_limits(ax)
            if data_limits is not None:
                y0 = min(y0, float(data_limits[2]))
                y1 = max(y1, float(data_limits[3]))
        ax.set_ylim((y0, y1))

    _draw_reference_pca_box(ax, ref_xlim, ref_ylim)

    if title is not None:
        title_fs = clamp_fontsize(9.0) if 'clamp_fontsize' in globals() else None
        if title_fs is None:
            ax.set_title(title)
        else:
            ax.set_title(title, fontsize=title_fs)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    tick_w = clamp_linewidth(0.8) if 'clamp_linewidth' in globals() else 0.8
    ax.tick_params(direction='out', length=4, width=tick_w)
    ax.grid(False)
    if 'shrink_figure_to_pnas' in globals():
        shrink_figure_to_pnas(ax.figure, panel_kind='pca')
    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)
    if legend:
        lg = ax.legend(frameon=legend_frame)
        if lg is not None:
            lg.set_frame_on(legend_frame)

def plot_pca_value_overlay(
    ax,
    coords,
    mask,
    values,
    *,
    cmap,
    vmin,
    vmax,
    s=30,
    edgecolors='none',
    zorder=3,
):
    """Overlay PCA points colored by scalar values on top of a map."""
    if coords is None or values is None:
        return None
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0 or np.sum(mask) == 0:
        return None
    point_size = clamp_scatter_area(s) if 'clamp_scatter_area' in globals() else s
    return ax.scatter(
        coords[mask, 0],
        coords[mask, 1],
        c=np.asarray(values)[mask],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=point_size,
        edgecolors=edgecolors,
        zorder=zorder,
    )

def make_figure(panel_kind='simple', **kwargs):
    """Unified single-figure factory using the notebook sizing rules."""
    kwargs = dict(kwargs)
    kwargs.pop('figsize', None)
    return plt.figure(panel_kind=panel_kind, **kwargs)

def make_figure_grid(nrows=1, ncols=1, panel_kind='simple', **kwargs):
    """Unified subplot factory using the notebook grid sizing rules."""
    kwargs = dict(kwargs)
    kwargs.pop('figsize', None)
    return plt.subplots(nrows, ncols, panel_kind=panel_kind, **kwargs)

def finalize_figure(fig, title=None, *, rect=None, save_path=None, dpi=300, show=True):
    """Apply common suptitle, layout, save, and display handling."""
    if title:
        fig.suptitle(title, fontsize=clamp_fontsize(12.0), fontweight='bold')
    if rect is None and title is not None:
        rect = [0, 0, 1, 0.97]
    if 'apply_grid_size' in globals():
        apply_grid_size(fig, panel_kind=getattr(fig, '_grid_panel_kind', 'simple'))
    if 'apply_external_legend' in globals():
        apply_external_legend(fig)
    if 'sanitize_figure_text' in globals():
        sanitize_figure_text(fig)
    if rect is None:
        fig.tight_layout()
    else:
        fig.tight_layout(rect=rect)
    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    if show:
        plt.show()
    return fig

def _display_cmap(cmap, bad_color='#e6e6e6'):
    """Return a colormap copy with unsupported / missing regions shown in light gray."""
    cm = plt.get_cmap(cmap).copy() if isinstance(cmap, str) else cmap.copy()
    cm.set_bad(bad_color)
    return cm

def add_scalar_colorbar(fig, axes, *, cmap, vmin, vmax, label, shrink=0.7):
    """Attach one shared scalar colorbar to one axis or a group of axes."""
    axes_arr = np.atleast_1d(np.asarray(axes, dtype=object)).ravel().tolist()
    sm = plt.cm.ScalarMappable(cmap=_display_cmap(cmap), norm=plt.Normalize(vmin, vmax))
    return fig.colorbar(sm, ax=axes_arr, shrink=shrink, label=label)

def render_pca_scalar_panel(
    ax,
    xy,
    arr,
    *,
    cmap,
    vmin,
    vmax,
    title,
    pad=0.5,
    point_size=14,
    min_points=3,
    empty_label='too few',
    add_colorbar=False,
    colorbar_label=None,
    colorbar_shrink=0.7,
):
    """Render one PCA scalar map panel with consistent smoothing and fallback text."""
    xy = np.asarray(xy) if xy is not None else np.empty((0, 2))
    arr = np.asarray(arr, dtype=float)
    ok = np.isfinite(arr)
    n_ok = int(np.sum(ok))

    if xy.size:
        _, _, ext = setup_pca_grid(xy, pad=pad)
    else:
        ext = (-7, 10, -6, 6)

    if n_ok < min_points:
        ax.text(0.5, 0.5, f'{title}\n{empty_label}', ha='center', va='center',
                transform=ax.transAxes, fontsize=clamp_fontsize(8.5))
        ax.set_xlim(ext[0], ext[1])
        ax.set_ylim(ext[2], ext[3])
        style_pca_axes(ax, title=title, legend=False)
        return n_ok

    gx, gy, ext = setup_pca_grid(xy, pad=pad)
    sg, _ = smooth_field(xy, arr, ok, gx, gy)
    ax.imshow(sg, extent=ext, origin='lower', aspect='auto', cmap=_display_cmap(cmap),
              vmin=vmin, vmax=vmax, interpolation='bilinear', alpha=0.7)
    plot_pca_value_overlay(
        ax,
        xy,
        ok,
        arr,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=point_size,
        edgecolors='none',
        zorder=3,
    )
    style_pca_axes(ax, title=title, legend=False)
    if add_colorbar:
        add_scalar_colorbar(ax.figure, [ax], cmap=cmap, vmin=vmin, vmax=vmax,
                            label=(colorbar_label or ''), shrink=colorbar_shrink)
    return n_ok

def plot_mean_sem_trace(
    ax,
    x,
    values,
    *,
    color,
    label,
    marker='o',
    linestyle='-',
    ms=4,
    lw=1.2,
    fill_alpha=0.12,
    min_points=2,
):
    """Plot mean ± SEM trajectory across boutons or trials."""
    values = np.asarray(values, dtype=float)
    nv = np.sum(np.isfinite(values), axis=0)
    mn = np.nanmean(values, axis=0)
    se = np.nanstd(values, axis=0) / np.sqrt(np.clip(nv, 1, None))
    mask = (nv >= min_points) & np.isfinite(mn)
    if not np.any(mask):
        return 0
    n0 = int(np.sum(np.isfinite(values[:, 0]))) if values.ndim == 2 else int(np.sum(np.isfinite(values)))
    ax.plot(np.asarray(x)[mask], mn[mask], marker=marker, ls=linestyle, color=color, ms=ms, lw=lw,
            label=label.format(n=n0))
    ax.fill_between(np.asarray(x)[mask], mn[mask] - se[mask], mn[mask] + se[mask],
                    alpha=fill_alpha, color=color)
    return n0

def _data_axes_for_layout(fig):
    axes = []
    for ax in getattr(fig, 'axes', []):
        if str(getattr(ax, 'get_label', lambda: '')()) == '<colorbar>':
            continue
        axes.append(ax)
    return axes

def _legend_entries(ax):
    legend = ax.get_legend()
    if legend is not None:
        handles = getattr(legend, 'legend_handles', None)
        if handles is None:
            handles = getattr(legend, 'legendHandles', None)
        labels = [txt.get_text() for txt in legend.get_texts()]
        if handles is not None and len(labels) == len(handles):
            pairs = [(h, l) for h, l in zip(handles, labels) if l and not str(l).startswith('_')]
            if pairs:
                return [h for h, _ in pairs], [l for _, l in pairs]
    handles, labels = ax.get_legend_handles_labels()
    pairs = [(h, l) for h, l in zip(handles, labels) if l and not str(l).startswith('_')]
    return [h for h, _ in pairs], [l for _, l in pairs]

def _dedupe_legend_entries(handles, labels):
    seen = set()
    out_h, out_l = [], []
    for h, l in zip(handles, labels):
        key = str(l)
        if key in seen:
            continue
        seen.add(key)
        out_h.append(h)
        out_l.append(l)
    return out_h, out_l

MOJIBAKE_REPLACEMENTS = {
    'Ca²âº': 'Ca2+',
    'Ca²⁺': 'Ca2+',
    '⁺': '+',
    'Aâ‚': 'A1',
    'A₁': 'A1',
    '₀': '0',
    '₁': '1',
    'ΔF/Fâ': 'ΔF/F0',
    'ΔF/F₀': 'ΔF/F0',
    'â•': '',
}

def _replace_mojibake_text(text):
    s = '' if text is None else str(text)
    for bad, good in MOJIBAKE_REPLACEMENTS.items():
        s = s.replace(bad, good)
    return s

def _compact_legend_label(text):
    s = _replace_mojibake_text(text)
    s = re.sub(r'\bCluster\s+(\d+)\b', r'C\1', s, flags=re.I)
    return s

def _strip_result_numbers(text):
    s = _replace_mojibake_text(text)
    if not s:
        return s
    s = re.sub(r'\n(?:paired\s+)?(?:Wilcoxon|Mann-Whitney|Welch|Bootstrap|95% CI|CI|p\s*=|r\s*=|n\s*=|Mean\b|Median\b).*$', '', s, flags=re.I)
    s = re.sub(r'\s*\((?=[^)]*(?:n\s*=|σ\s*=|p\s*=|r\s*=|mean|median|RRP|P0|refill|κ|U\s*=|t\s*=|CI)).*?\)\s*$', '', s, flags=re.I)
    s = re.sub(r'\n\((?=[^)]*(?:n\s*=|σ\s*=|p\s*=|r\s*=|mean|median|RRP|P0|refill|κ|U\s*=|t\s*=|CI)).*?\)\s*$', '', s, flags=re.I)
    s = re.sub(r'\s*\(n\s*=\s*[^)]*\)', '', s, flags=re.I)
    s = re.sub(r'(avg Q|mean|med|median|RRP|P0|refill|κ|U|t|p|r)\s*=\s*[^,;\n]+', r'\1', s, flags=re.I)
    s = re.sub(r'(95% CI|CI)\s*:\s*\[[^\]]+\]', r'\1', s, flags=re.I)
    s = re.sub(r'(Mean|Median)\s*:\s*[-+]?\d*\.?\d+%?', r'\1', s)
    s = re.sub(r'\s{2,}', ' ', s).strip(' ,;')
    return s

def _is_result_annotation_text(text):
    s = _replace_mojibake_text(text).strip()
    if not s:
        return False
    if re.search(r'too few|missing|no data|available', s, flags=re.I):
        return False
    if re.fullmatch(r'[+\-]?\d+(?:\.\d+)?%?', s):
        return True
    if re.search(r'\d+(?:\.\d+)?%', s):
        return True
    if re.fullmatch(r'\*+|ns', s, flags=re.I):
        return True
    patterns = [
        r'\bp\s*=', r'\br\s*=', r'\bn\s*=', r'RRP\s*=', r'P0\s*=', r'refill', r'Stable \(', r'Unstable \(',
        r'mean\s*=', r'median\s*=', r'κ\s*=', r'\bU\s*=', r'\bt\s*=', r'CI', r'Peak response'
    ]
    return any(re.search(pat, s, flags=re.I) for pat in patterns)

def sanitize_figure_text(fig):
    """Normalize text encoding and remove result annotations from figure interiors."""
    if fig is None:
        return None
    removed_notes = []
    if getattr(fig, '_suptitle', None) is not None:
        fig._suptitle.set_text(_strip_result_numbers(fig._suptitle.get_text()))
    for ax in getattr(fig, 'axes', []):
        ax.set_title(_strip_result_numbers(ax.get_title()))
        ax.set_xlabel(_replace_mojibake_text(ax.get_xlabel()))
        ax.set_ylabel(_replace_mojibake_text(ax.get_ylabel()))
        for txt in list(getattr(ax, 'texts', [])):
            if _is_result_annotation_text(txt.get_text()):
                removed_notes.append(_replace_mojibake_text(txt.get_text()))
                try:
                    txt.remove()
                except Exception:
                    pass
            else:
                txt.set_text(_replace_mojibake_text(txt.get_text()))
        lg = ax.get_legend()
        if lg is not None:
            for txt in lg.get_texts():
                txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
            if lg.get_title() is not None:
                lg.get_title().set_text(_replace_mojibake_text(lg.get_title().get_text()))
    for lg in list(getattr(fig, 'legends', [])):
        for txt in lg.get_texts():
            txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
        if lg.get_title() is not None:
            lg.get_title().set_text(_replace_mojibake_text(lg.get_title().get_text()))
    if removed_notes and not getattr(fig, '_sanitized_result_notes_printed', False):
        uniq = []
        seen = set()
        for note in removed_notes:
            key = str(note)
            if key in seen:
                continue
            seen.add(key)
            uniq.append(note)
        print('Figure annotations moved out of panel:')
        for note in uniq:
            print(f'  {note}')
        fig._sanitized_result_notes_printed = True
    return fig

def apply_external_legend(fig, *, max_in_axes_items=3, min_repeated_panels=2):
    """Move dense or repeated legends outside the panel grid, with optional per-axis override."""
    if fig is None:
        return None
    legend_specs = []
    for ax in _data_axes_for_layout(fig):
        handles, labels = _legend_entries(ax)
        if labels:
            labels = [_compact_legend_label(_strip_result_numbers(lbl)) for lbl in labels]
            legend_specs.append({
                'ax': ax,
                'handles': handles,
                'labels': labels,
                'force_outside': bool(getattr(ax, '_legend_force_outside', False)),
                'force_inside': bool(getattr(ax, '_legend_force_inside', False)),
            })
    if not legend_specs:
        return None

    candidate_specs = [spec for spec in legend_specs if not spec['force_inside']]
    if not candidate_specs:
        return None
    has_forced_outside = any(spec['force_outside'] for spec in candidate_specs)

    repeated_counts = {}
    for spec in candidate_specs:
        sig = tuple(spec['labels'])
        repeated_counts[sig] = repeated_counts.get(sig, 0) + 1
    has_repeated = any(len(sig) > 0 and count >= min_repeated_panels for sig, count in repeated_counts.items())
    data_axes = _data_axes_for_layout(fig)
    has_large = any(len(spec['labels']) > max_in_axes_items for spec in candidate_specs)
    if not has_forced_outside and not has_repeated and not has_large:
        return None

    all_handles, all_labels = [], []
    for spec in candidate_specs:
        all_handles.extend(spec['handles'])
        all_labels.extend(spec['labels'])
    all_handles, all_labels = _dedupe_legend_entries(all_handles, all_labels)
    if not all_labels:
        return None

    for spec in candidate_specs:
        lg_ax = spec['ax'].get_legend()
        if lg_ax is not None:
            lg_ax.remove()
    for lg in list(getattr(fig, 'legends', [])):
        try:
            lg.remove()
        except Exception:
            pass
    if data_axes:
        if 'grid_figsize' in globals() and '_figure_grid_shape' in globals():
            nrows, ncols = _figure_grid_shape(fig)
            w, h = grid_figsize(
                nrows=nrows,
                ncols=ncols,
                panel_kind=getattr(fig, '_grid_panel_kind', 'simple'),
                has_legend=True,
                has_suptitle=getattr(fig, '_suptitle', None) is not None,
            )
            fig.set_size_inches(w, h, forward=True)
        rect = _display_layout_rect(fig, has_external_legend=True) if '_display_layout_rect' in globals() else [0.03, 0.03, 0.84, 0.96]
        x_anchor = min(0.985, rect[2] + 0.008)
        y_anchor = 0.5 * (min(float(ax.get_position().y0) for ax in data_axes) + max(float(ax.get_position().y1) for ax in data_axes))
    else:
        x_anchor, y_anchor = 0.985, 0.5
    fig._external_legend_labels = tuple(all_labels)
    lg = fig.legend(
        all_handles,
        all_labels,
        loc='center left',
        bbox_to_anchor=(x_anchor, y_anchor),
        bbox_transform=fig.transFigure,
        borderaxespad=0.0,
        frameon=False,
        fontsize=clamp_fontsize(7.0),
    )
    try:
        lg.set_in_layout(True)
    except Exception:
        pass
    try:
        lg.set_clip_on(False)
    except Exception:
        pass
    return lg

def add_legend(ax, *args, **kw):
    kw = dict(kw)
    outside = kw.pop('outside', None)
    kw.setdefault('frameon', False)
    kw.setdefault('fontsize', clamp_fontsize(7.0))
    if outside is True:
        setattr(ax, '_legend_force_outside', True)
        setattr(ax, '_legend_force_inside', False)
    elif outside is False:
        setattr(ax, '_legend_force_outside', False)
        setattr(ax, '_legend_force_inside', True)
    lg = ax.legend(*args, **kw)
    if lg is not None:
        for txt in lg.get_texts():
            txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
    if outside is True and 'apply_external_legend' in globals():
        try:
            apply_external_legend(ax.figure, max_in_axes_items=0, min_repeated_panels=1)
        except Exception:
            pass
    return lg


### 1.6 Assemble the Bouton Feature Matrix

Bouton-level scalar descriptors are gathered here from the processed feature sheets. These variables form the basis of the WT reference PCA, clustering, and the direct genotype or perturbation comparisons shown later in the notebook.


In [ ]:
## Load PCA features from multi-sheet Excel file. AMP1/strength, PCA, Feature correlations

# Load PCA features from multi-sheet Excel file
pca_features_file = BASE_DIR / PPR_FILENAME
excel_data        = pd.ExcelFile(pca_features_file)

# Only process conditions that have corresponding feature sheets
available_conditions = [cond for cond in experimental_conditions if cond in excel_data.sheet_names]
CONDITIONS           = available_conditions

feature_dataframes = []
for condition_name in CONDITIONS:
    condition_features = pd.read_excel(pca_features_file, sheet_name=condition_name)
    
    # Standardize ID column name (handle various naming conventions)
    id_column_names = ['id', 'bouton', 'bouton_id', 'name']
    for column in condition_features.columns:
        if str(column).strip().lower() in id_column_names:
            condition_features = condition_features.rename(columns={column: 'ID'})
            break
    
    # Clean bouton IDs and add condition label
    condition_features['ID'] = condition_features['ID'].apply(
        lambda x: clean_bouton_id(str(x)) if pd.notnull(x) else x
    )
    condition_features['Condition'] = condition_name
    feature_dataframes.append(condition_features)

excel_data.close()
FEATURES_DATAFRAME = pd.concat(feature_dataframes, ignore_index=True)

# Identify and drop boutons with missing critical amplitude/PPR values
def _is_missing_critical(value):
    if isinstance(value, str):
        return value.strip() == ''
    return pd.isna(value)


def _is_critical_column(column_name: str) -> bool:
    column_name = str(column_name)
    if column_name.startswith('AMP') and column_name[3:].isdigit():
        return True
    if column_name.startswith('PPR') and '/1' in column_name:
        numerator = column_name[3:].split('/')[0]
        return numerator.isdigit()
    return False


critical_feature_columns = [
    col for col in FEATURES_DATAFRAME.columns if _is_critical_column(col)
]
invalid_feature_indices = []
invalid_feature_ids = set()

for row_idx, feature_row in FEATURES_DATAFRAME.iterrows():
    missing_columns = [
        col for col in critical_feature_columns
        if _is_missing_critical(feature_row.get(col))
    ]
    if missing_columns:
        raw_id = feature_row.get("ID")
        bouton_id = (
            clean_bouton_id(str(raw_id))
            if pd.notnull(raw_id) else f"row_{row_idx}"
        )
        condition_label = feature_row.get("Condition", "Unknown")
        print(
            f"! Skipping ID {bouton_id} (Condition {condition_label}) "
            f"due to missing values in {', '.join(missing_columns)}"
        )
        invalid_feature_indices.append(row_idx)
        if pd.notnull(raw_id):
            invalid_feature_ids.add(bouton_id)

if invalid_feature_indices:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(
        index=invalid_feature_indices
    ).reset_index(drop=True)

    if invalid_feature_ids and "RAW_TRACES_DF" in globals():
        before_trace_count = len(RAW_TRACES_DF)
        RAW_TRACES_DF = (
            RAW_TRACES_DF[~RAW_TRACES_DF['ID'].isin(invalid_feature_ids)]
            .reset_index(drop=True)
        )
        dropped_traces = before_trace_count - len(RAW_TRACES_DF)
        if dropped_traces:
            print(f"! Ignoring {dropped_traces} raw trace files linked to invalid IDs")

        if "raw_traces_data" in globals():
            kept_ids = set(RAW_TRACES_DF['ID'])
            raw_traces_data = [
                entry for entry in raw_traces_data
                if entry['ID'] in kept_ids
            ]

    elif invalid_feature_ids and "raw_traces_data" in globals():
        before_trace_count = len(raw_traces_data)
        raw_traces_data = [
            entry for entry in raw_traces_data
            if entry['ID'] not in invalid_feature_ids
        ]
        dropped_traces = before_trace_count - len(raw_traces_data)
        if dropped_traces:
            print(f"! Ignoring {dropped_traces} raw trace files linked to invalid IDs")
        RAW_TRACES_DF = pd.DataFrame(raw_traces_data)



# Remove specified amplitude columns from analysis
amplitude_columns_to_drop = [col for col in PCA_DROP_COLS[:8] if col in FEATURES_DATAFRAME.columns]
if amplitude_columns_to_drop:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(columns=amplitude_columns_to_drop)

In [ ]:
# Adjust problematic recordings manually

# Check if manual 50Hz fix file exists
manual_fix_file = BASE_DIR / 'Manual_50Hz_fix_last_event.xlsx'

if manual_fix_file.exists():
    print(f"Loading manual 50Hz fixes from {manual_fix_file.name}...")
    
    # Load the manual fix file
    manual_fixes = pd.read_excel(manual_fix_file)
    
    # Column E contains IDs (0-indexed: column index 4), Column D contains corrected AMP10 (index 3)
    # Assuming column E is the 5th column (index 4) and column D is the 4th column (index 3)
    fixes_df = manual_fixes.iloc[:, [4, 3]].copy()
    fixes_df.columns = ['ID', 'AMP10_corrected']
    
    # Remove any NaN rows
    fixes_df = fixes_df.dropna()
    
    # Convert IDs to string and strip whitespace for matching
    fixes_df['ID'] = fixes_df['ID'].astype(str).str.strip()
    
    # Track updates
    updated_count = 0
    
    # Apply corrections to FEATURES_DATAFRAME
    for idx, fix_row in fixes_df.iterrows():
        fix_id = fix_row['ID']
        new_amp10 = fix_row['AMP10_corrected']
        
        # Find matching rows in FEATURES_DATAFRAME
        mask = FEATURES_DATAFRAME['ID'] == fix_id
        
        if mask.any():
            # Update AMP10
            FEATURES_DATAFRAME.loc[mask, 'AMP10'] = new_amp10
            
            # Recalculate PPR10/1 = AMP10 / AMP1
            amp1_value = FEATURES_DATAFRAME.loc[mask, 'AMP1'].values[0]
            if amp1_value != 0:
                new_ppr10_1 = new_amp10 / amp1_value
                FEATURES_DATAFRAME.loc[mask, 'PPR10/1'] = new_ppr10_1
                
                updated_count += 1
                print(f"  ✓ Updated ID {fix_id}: AMP10={new_amp10:.4f}, PPR10/1={new_ppr10_1:.4f}")
            else:
                print(f"  ⚠ Warning: ID {fix_id} has AMP1=0, cannot calculate PPR10/1")
        else:
            print(f"  ✗ Warning: ID {fix_id} not found in FEATURES_DATAFRAME")
    
    print(f"\nManual corrections applied: {updated_count}/{len(fixes_df)} recordings updated")
else:
    print(f"No manual fix file found at {manual_fix_file.name}, skipping manual corrections")


### 1.7 Target Identity Annotation

Target labels derived from the L7-tdTomato experiments are merged here so that WT bouton classes can later be tested against postsynaptic identity without redefining the WT state space. The target labels are therefore treated as an overlay on the WT reference rather than as primary clustering features.


In [ ]:
## Load bouton target identity mapping (PC/IN/UN classification). Extracellular Ca²+, Temporal traces

# Load bouton target identity mapping (PC/IN/UN classification)
target_mapping_file          = BASE_DIR / TARGET_MAP_FILENAME
target_identity_data         = pd.read_excel(target_mapping_file).iloc[:, :2].copy()
target_identity_data.columns = ['ID', 'Target']

# Clean and standardize target data
target_identity_data['ID'] = target_identity_data['ID'].map(lambda x: _normalize_bouton_id(str(x).strip()))
target_identity_data['Target'] = target_identity_data['Target'].astype(str).str.strip().str.upper()

# Set invalid targets to 'UN' (undefined)
valid_targets = ['PC', 'IN', 'UN']
target_identity_data['Target'] = target_identity_data['Target'].where(
    target_identity_data['Target'].isin(valid_targets), 'UN'
)

# Merge target identities into feature data
FEATURES_DATAFRAME['ID'] = FEATURES_DATAFRAME['ID'].astype(str).str.strip()
FEATURES_DATAFRAME = FEATURES_DATAFRAME.merge(target_identity_data, on='ID', how='left')
FEATURES_DATAFRAME['Target'] = FEATURES_DATAFRAME['Target'].fillna('UN')  # Missing targets → undefined

# Enrich features and raw traces with reusable metadata columns
FEATURES_DATAFRAME = enrich_features_dataframe(FEATURES_DATAFRAME)
RAW_TRACES_DF = enrich_traces_dataframe(RAW_TRACES_DF)

# Attach canonical eventwise trial-derived failure rates without altering %Fail used for PCA
FEATURES_DATAFRAME, TEMP_FAILRATE_RECOMP_DF = attach_event_failure_rates_to_features(
    FEATURES_DATAFRAME,
    max_pulse_number=10,
)

# Keep explicit raw snapshots for later reuse
FEATURES_DATAFRAME_RAW = FEATURES_DATAFRAME.copy()
RAW_TRACES_DF_RAW = RAW_TRACES_DF.copy()

# Cached indexes for fast lookup (avoid repeated per-row filtering)
FEATURES_BY_COND_ID = FEATURES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
FEATURES_BY_ID = FEATURES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)

# Data loading summary
print("\n=== DATA LOADING SUMMARY ===")
print(f"Conditions: {len(CONDITIONS)} ({', '.join(CONDITIONS)})")
print(f"Traces: {len(RAW_TRACES_DF)} | Features: {len(FEATURES_DATAFRAME)} | Target mappings: {len(target_identity_data)}")
target_counts = FEATURES_DATAFRAME['Target'].value_counts().sort_index()
print('Target counts: ' + '  '.join(f"{k}={int(v)}" for k, v in target_counts.items()))
print(f"✓ Successfully processed {len(raw_traces_data)} bouton files")


### 1.8 Organize Traces by Experimental Condition

Trace tables are reorganized by condition so that mean responses, SEM envelopes, and single-fiber examples can be generated consistently. This step also harmonizes the time base across recordings, which is critical when comparing train responses across calcium levels, frequencies, or genotypes.


In [ ]:
## Organize traces by condition and resample to COMMON_TIME if needed

# Update COMMON_TIME based on loaded trace time vectors
if len(raw_traces_data) > 0:
    # Use the time vector from the first trace as reference
    first_time = np.array(raw_traces_data[0]['Time'])
    if len(first_time) > 0:
        COMMON_TIME = first_time
        N_SAMPLES = len(COMMON_TIME)
        CROP_END = float(COMMON_TIME[-1])
        print(f"Updated COMMON_TIME from traces: {N_SAMPLES} samples, {CROP_END:.2f}s duration")

# Per-condition ISI mapping (for visualization/annotation)
CONDITION_ISI = {
    c: 0.02 for c in (
        get_calcium_conditions('1.5mM', '50Hz')
        + get_calcium_conditions('2.5mM', '50Hz')
        + get_calcium_conditions('4mM', '50Hz')
    )
}
DEFAULT_ISI = 0.05
DEFAULT_TRAIN_START = 1.0

# Baseline offset for exceptional conditions (0.5s baseline instead of 1s)
EXCEPTIONAL_BASELINE_OFFSET = 0.5

# Build traces dictionary organized by condition
traces_by_condition = {}
all_processed_traces = []

for _, row in RAW_TRACES_DF.iterrows():
    condition = row['Condition']
    avg_trace = np.array(row['Avg'])
    time_vec = np.array(row['Time'])
    
    # Check if this is an exceptional condition with short baseline
    is_exceptional = condition in EXCEPTIONAL_CONDITIONS
    
    if is_exceptional:
        # For exceptional conditions: shift time by 0.5s and prepend NaN values
        # Original traces start at 0s with 0.5s baseline, we shift them to start at 0.5s
        shifted_time = time_vec + EXCEPTIONAL_BASELINE_OFFSET
        
        # Create NaN-padded trace aligned to COMMON_TIME
        # Points before 0.5s get NaN, points after get interpolated from shifted trace
        padded_trace = np.full_like(COMMON_TIME, np.nan)
        
        # Find indices in COMMON_TIME that are >= 0.5s (where we have actual data)
        valid_mask = COMMON_TIME >= EXCEPTIONAL_BASELINE_OFFSET
        
        # Interpolate the original trace values onto the valid portion of COMMON_TIME
        valid_times = COMMON_TIME[valid_mask]
        padded_trace[valid_mask] = np.interp(valid_times, shifted_time, avg_trace)
        
        avg_trace = padded_trace
    else:
        # Standard case: resample to COMMON_TIME if needed
        if len(time_vec) != len(COMMON_TIME) or not np.allclose(time_vec, COMMON_TIME):
            avg_trace = np.interp(COMMON_TIME, time_vec, avg_trace)
    
    processed = {
        'ID': row['ID'],
        'Condition': condition,
        'Time': COMMON_TIME,
        'Avg': avg_trace,
    }
    all_processed_traces.append(processed)
    
    # Clean ID by removing '_traces_converted' suffix if present
    if processed['ID'].endswith('_traces_converted'):
        processed['ID'] = processed['ID'][:-len('_traces_converted')]

    if condition not in traces_by_condition:
        traces_by_condition[condition] = []
    traces_by_condition[condition].append(avg_trace)

print(f"✓ Organized {len(all_processed_traces)} traces across {len(traces_by_condition)} conditions")
exceptional_count = sum(1 for t in all_processed_traces if t['Condition'] in EXCEPTIONAL_CONDITIONS)
if exceptional_count > 0:
    print(f"  → {exceptional_count} traces from exceptional conditions (0.5s baseline, NaN-padded 0-0.5s)")

### 1.9 Build the Normalized Trace Table

The normalized trace table is the common time-domain representation used for the trace figures. It links each bouton ID to a standardized trace, enabling later cluster-wise, fiber-wise, and condition-wise averaging without rebuilding the trace structure each time.


In [ ]:
## Create normalized traces dataframe from loaded traces

# Print summary per condition
for condition_name, condition_traces in traces_by_condition.items():
    print(f"Condition {condition_name}: {len(condition_traces)} traces")

# Create final processed traces dataframe
NORM_TRACES_DATAFRAME = pd.DataFrame(all_processed_traces)
NORM_TRACES_DATAFRAME = enrich_traces_dataframe(NORM_TRACES_DATAFRAME)

# Cached trace indexes/lookups to avoid repeated recomputation
TRACE_BY_COND_ID = NORM_TRACES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
TRACE_BY_ID = NORM_TRACES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)
lookup_source = str(TRACE_SINGLE_SOURCE).strip().lower()
lookup_df = NORM_TRACES_DATAFRAME if lookup_source == 'normalized' else RAW_TRACES_DF
resampled_trace_lookup = {}
for _, trace_row in lookup_df.iterrows():
    time_values = np.asarray(trace_row['Time'], dtype=float)
    if lookup_source != 'normalized' and trace_row['Condition'] in EXCEPTIONAL_CONDITIONS:
        time_values = time_values + EXCEPTIONAL_BASELINE_OFFSET
    resampled_trace_lookup[str(trace_row['ID'])] = {'Time': time_values, 'Avg': trace_row['Avg']}

print(f"\n✓ Created NORM_TRACES_DATAFRAME with {len(NORM_TRACES_DATAFRAME)} traces")


### 1.11 Exclude the Outlier 50 Hz Recording

The outlier exclusion is kept explicit because the 50 Hz analyses are sensitive to rare recordings with atypical final events or partial saturation. Keeping this decision visible in the notebook makes the downstream high-frequency comparisons easier to interpret.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cond_4_50hz = get_calcium_conditions('4mM', '50Hz')[0]
cond_2_5_50hz = get_calcium_conditions('2.5mM', '50Hz')[0]

# Filter traces, times, and IDs for each condition
traces_4_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_4_50hz]['Avg']
times_4_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_4_50hz]['Time']
trace_ids_4_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_4_50hz]['ID']

traces_2_5_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_2_5_50hz]['Avg']
times_2_5_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_2_5_50hz]['Time']
trace_ids_2_5_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_2_5_50hz]['ID']

# Function to get time corresponding to the maximum
def get_max_and_time(traces, times):
    max_values = []
    max_times = []
    for trace, time in zip(traces, times):
        max_val = np.nanmax(trace)
        max_idx = np.nanargmax(trace)
        max_values.append(max_val)
        max_times.append(time[max_idx])
    return max_values, max_times

# Compute maxima and corresponding times
max_values_4_50Hz, max_times_4_50Hz = get_max_and_time(traces_4_50Hz, times_4_50Hz)
max_values_2_5_50Hz, max_times_2_5_50Hz = get_max_and_time(traces_2_5_50Hz, times_2_5_50Hz)

# Define threshold for outlier exclusion
threshold = 10

# Identify traces above threshold for each condition
above_threshold_4_50Hz = [id for id, val in zip(trace_ids_4_50Hz, max_values_4_50Hz) if val > threshold]
above_threshold_2_5_50Hz = [id for id, val in zip(trace_ids_2_5_50Hz, max_values_2_5_50Hz) if val > threshold]

# Display IDs of traces above threshold
print(f"IDs of traces above the dF/F0 = 4.5 threshold for {cond_4_50hz} :", above_threshold_4_50Hz)
print(f"IDs of traces above the dF/F0 = 4.5 threshold for {cond_2_5_50hz} :", above_threshold_2_5_50Hz)

OUTLIER_IDS_50HZ = sorted(set(above_threshold_4_50Hz + above_threshold_2_5_50Hz))
OUTLIER_IDS_STABILITY = list(globals().get('STABILITY_OUTLIER_IDS', []))

# Keep explicit pre-filter snapshots before exclusion
NORM_TRACES_DATAFRAME_PRE_50HZ_FILTER = NORM_TRACES_DATAFRAME.copy()
FEATURES_DATAFRAME_PRE_50HZ_FILTER = FEATURES_DATAFRAME.copy()
if 'FEATURES_DATAFRAME_RAW' in globals():
    FEATURES_DATAFRAME_RAW_PRE_50HZ_FILTER = FEATURES_DATAFRAME_RAW.copy()
if 'RAW_TRACES_DF' in globals():
    RAW_TRACES_DF_PRE_50HZ_FILTER = RAW_TRACES_DF.copy()
if 'RAW_TRACES_DF_RAW' in globals():
    RAW_TRACES_DF_RAW_PRE_50HZ_FILTER = RAW_TRACES_DF_RAW.copy()


def _exclude_ids(df, excluded_ids):
    if df is None or len(excluded_ids) == 0:
        return df
    if 'ID' not in df.columns:
        return df
    excluded_norm = {_normalize_bouton_id(x) for x in excluded_ids}
    keep_mask = ~df['ID'].map(_normalize_bouton_id).isin(excluded_norm)
    return df[keep_mask].reset_index(drop=True)


# Apply 50Hz outlier exclusion globally; keep stability-pair exclusions local to stability-only views
NORM_TRACES_DATAFRAME = _exclude_ids(NORM_TRACES_DATAFRAME, OUTLIER_IDS_50HZ)
FEATURES_DATAFRAME = _exclude_ids(FEATURES_DATAFRAME, OUTLIER_IDS_50HZ)

if 'FEATURES_DATAFRAME_RAW' in globals():
    FEATURES_DATAFRAME_RAW = _exclude_ids(FEATURES_DATAFRAME_RAW, OUTLIER_IDS_50HZ)
if 'RAW_TRACES_DF' in globals():
    RAW_TRACES_DF = _exclude_ids(RAW_TRACES_DF, OUTLIER_IDS_50HZ)
if 'RAW_TRACES_DF_RAW' in globals():
    RAW_TRACES_DF_RAW = _exclude_ids(RAW_TRACES_DF_RAW, OUTLIER_IDS_50HZ)

# Refresh cached indexes/lookups after filtering
TRACE_BY_COND_ID = NORM_TRACES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
TRACE_BY_ID = NORM_TRACES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)
resampled_trace_lookup = build_trace_lookup_from_source(TRACE_SINGLE_SOURCE)

FEATURES_BY_COND_ID = FEATURES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
FEATURES_BY_ID = FEATURES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)

print(f"Excluded {len(OUTLIER_IDS_50HZ)} 50Hz outlier IDs from reusable dataframes; stability-pair exclusions remain local to stability-only views ({len(OUTLIER_IDS_STABILITY)} IDs)")


### 1.12 Pool Conditions for the Main Comparisons

Related acquisition conditions are pooled here to define the core WT, calcium, frequency, SynII, and stability datasets used later in the notebook. The pooled WT dataset is the reference anchor for the PCA and hierarchical clustering analyses.


In [ ]:
## Pool related experimental conditions for analysis. Extracellular Ca²+, Stability/plasticity, Synapsin-II / genotype

# Build pooled dataframe from cleaned RAW features (do not overwrite raw snapshot semantics)
FEATURES_DATAFRAME = make_pooled_features_dataframe(FEATURES_DATAFRAME_RAW, CONDITION_POOLS)
FEATURES_DATAFRAME = enrich_features_dataframe(FEATURES_DATAFRAME)

# Refresh feature caches after pooling
FEATURES_BY_COND_ID = FEATURES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
FEATURES_BY_ID = FEATURES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)
FEATURES_VIEWS = build_condition_views(FEATURES_DATAFRAME)

# Also keep per-condition trace views from cleaned normalized traces
NORM_TRACES_VIEWS = build_condition_views(NORM_TRACES_DATAFRAME)

# Extract condition-specific dataframes (single source, no repeated ad-hoc filtering)
PCA_Data_WT_Pooled = FEATURES_VIEWS.get('WT_pooled', pd.DataFrame()).copy()
PCA_Data_WT_Theo = FEATURES_VIEWS.get('WT_Theo', pd.DataFrame()).copy()
PCA_Data_WT_Anthime = FEATURES_VIEWS.get('WT_Anthime', pd.DataFrame()).copy()
PCA_Data_SynII = filter_df_by_conditions(FEATURES_DATAFRAME, get_synapsin_conditions())
PCA_Data_WT_Low_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '20Hz')
PCA_Data_WT_High_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '20Hz')
PCA_Data_Stability_Before = _exclude_ids(FEATURES_VIEWS.get('stability_before', pd.DataFrame()).copy(), OUTLIER_IDS_STABILITY)
PCA_Data_Stability_After = _exclude_ids(FEATURES_VIEWS.get('stability_after', pd.DataFrame()).copy(), OUTLIER_IDS_STABILITY)
PCA_Data_50Hz_1_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '50Hz')
PCA_Data_50Hz_4_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '50Hz')
PCA_Data_50Hz_2_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '2.5mM', '50Hz')

# Global reusable feature lists
PPR_COLS_GLOBAL = get_ppr_columns(FEATURES_DATAFRAME)

print(f"✓ RAW features (cleaned): {len(FEATURES_DATAFRAME_RAW)} boutons")
print(f"✓ POOLED features: {len(FEATURES_DATAFRAME)} boutons across {len(FEATURES_DATAFRAME['Condition'].unique())} conditions")
print(f"✓ Normalized traces (cleaned): {len(NORM_TRACES_DATAFRAME)}")
print(f"Key datasets: WT_pooled({len(PCA_Data_WT_Pooled)}), SynII({len(PCA_Data_SynII)}), 1.5Ca({len(PCA_Data_WT_Low_Ca)}), 4Ca({len(PCA_Data_WT_High_Ca)})")


Only the variables used for the WT reference PCA and clustering are retained downstream: AMP1, AMP2, PPR2/1-PPR10/1, and the first two failure-rate terms. This keeps the reference space anchored to release strength, apparent reliability, and short-term plasticity.


## 2. WT Reference PCA Space

WT pooled boutons define the reference low-dimensional space used throughout the notebook. The analyses below establish how the selected release variables project onto the principal components and provide the reference coordinates used for later overlays.

In the manuscript, this section corresponds to the idea that bouton-to-bouton diversity can be summarized by a small number of orthogonal release dimensions. The PCA is therefore not just a visualization step: it defines the coordinate system used to interpret calcium, frequency, and SynII perturbations.


### 2.1 Sanitize the WT Feature Matrix

Only the variables directly linked to glutamate release amplitude, failures, and train dynamics are retained for the WT PCA. This keeps the reduced space focused on synaptic physiology rather than metadata or acquisition descriptors.


In [ ]:
# Prepare datasets for PCA analysis by removing metadata columns, keeping numeric features, and scaling

def prepare_pca_features(dataframe, reference_columns=None):
    """Return PCA-ready numeric features with optional reference-column alignment."""
    features = dataframe.drop(
        columns=[col for col in PCA_DROP_COLS if col in dataframe.columns],
        errors='ignore'
    ).copy()

    # Keep only numeric columns so IDs/labels never enter the scaler
    features = features.select_dtypes(include=[np.number])

    if reference_columns is None:
        # Remove empty numeric columns from reference fit set
        return features.loc[:, features.notna().any(axis=0)]

    missing_columns = [col for col in reference_columns if col not in features.columns]
    if missing_columns:
        raise ValueError(
            f"{len(missing_columns)} PCA feature columns missing in dataset: {missing_columns}"
        )

    return features[reference_columns]


# Prepare reference dataset for PCA
WT_pooled_for_pca = prepare_pca_features(PCA_Data_WT_Pooled)
PCA_FEATURE_COLUMNS = list(WT_pooled_for_pca.columns)

forbidden_pca_cols = {'Ca_mM', 'Freq_Hz'}
forbidden_present = [col for col in PCA_FEATURE_COLUMNS if col in forbidden_pca_cols]
if forbidden_present:
    raise ValueError(f"Forbidden metadata columns included in PCA features: {forbidden_present}")

if len(PCA_FEATURE_COLUMNS) == 0:
    raise ValueError("No numeric PCA features found in PCA_Data_WT_Pooled after metadata filtering")

# Fit StandardScaler on WT_pooled reference dataset
scaler = StandardScaler()
scaled_data = {'WT_pooled': scaler.fit_transform(WT_pooled_for_pca)}

# Transform all other datasets using WT_pooled scaling parameters
datasets_to_scale = {
    'WT_Theo': PCA_Data_WT_Theo,
    'WT_Anthime': PCA_Data_WT_Anthime,
    'SynII': PCA_Data_SynII,
    'WT_1_5Ca': PCA_Data_WT_Low_Ca,
    'WT_4Ca': PCA_Data_WT_High_Ca,
    'stab_before': PCA_Data_Stability_Before,
    'stab_after': PCA_Data_Stability_After,
    '50Hz_1_5Ca': PCA_Data_50Hz_1_5_Ca,
    '50Hz_4Ca': PCA_Data_50Hz_4_Ca,
    '50Hz_2_5Ca': PCA_Data_50Hz_2_5_Ca,
}

for dataset_name, dataset_df in datasets_to_scale.items():
    pca_features = prepare_pca_features(dataset_df, reference_columns=PCA_FEATURE_COLUMNS)
    scaled_data[dataset_name] = scaler.transform(pca_features)

print(f"✓ PCA feature columns ({len(PCA_FEATURE_COLUMNS)}): {PCA_FEATURE_COLUMNS}")
print(f"✓ Removed metadata columns and non-numeric columns before scaling")
print(f"✓ Scaled {len(scaled_data)} datasets using WT_pooled reference parameters")


### 2.2 Fit the WT Reference PCA

The PCA is fit on the WT pooled dataset so that subsequent conditions can be projected as additional observations rather than mixed into the reference space definition. This preserves a WT-centered interpretation of the axes.


In [ ]:
# Fit PCA on WT_pooled reference dataset and transform all conditions

# Fit PCA model using WT_pooled as reference
pca                   = PCA(n_components=2)
pca_data              = {}
pca_data['WT_pooled'] = pca.fit_transform(scaled_data['WT_pooled'])

# Transform all other datasets using same PCA axes from WT_pooled
for dataset_name in datasets_to_scale.keys():
    pca_data[dataset_name] = pca.transform(scaled_data[dataset_name])

# Display PCA results summary
variance_pc1, variance_pc2 = pca.explained_variance_ratio_
print(f"✓ PCA transformation complete")
print(f"✓ PC1 explains {variance_pc1:.1%} of variance, PC2 explains {variance_pc2:.1%}")
print(f"✓ Total variance explained: {variance_pc1 + variance_pc2:.1%}")
print(f"✓ Transformed {len(pca_data)} datasets using WT_pooled PCA axes")

In [ ]:
# Fit PCA model with all possible components
pca_full = PCA(n_components=None)
pca_full.fit(scaled_data['WT_pooled'])

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
n_components = len(explained_variance)
component_idx = np.arange(1, n_components + 1)

fig, axes = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
ax_var, ax_cum = axes[0, 0], axes[0, 1]

ax_var.bar(component_idx, explained_variance, alpha=0.7, align='center', color='steelblue', label='Individual variance')
ax_var.set_xlabel('Number of principal components')
ax_var.set_ylabel('Explained variance')
ax_var.set_title('Elbow Plot: Individual variance')
ax_var.spines['top'].set_visible(False)
ax_var.spines['right'].set_visible(False)
add_legend(ax_var)
ax_var.grid(False)

ax_cum.plot(component_idx, cumulative_variance, 'r-', linewidth=2, marker='o', ms=4, label='Cumulative variance')
ax_cum.set_xlabel('Number of principal components')
ax_cum.set_ylabel('Explained variance')
ax_cum.set_title('Elbow Plot: Cumulative variance')
ax_cum.spines['top'].set_visible(False)
ax_cum.spines['right'].set_visible(False)
add_legend(ax_cum)
ax_cum.grid(False)

fig.tight_layout()
output_file = OUTPUT_DIR / "02_01_wt_pca_variance_summary.pdf"
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved PCA variance summary to {output_file}")
print(f"✓ Variance explained by PC1 + PC2: {cumulative_variance[1]:.1%}")
print(f"✓ Number of components shown: {n_components}")


### 2.3 Build PCA Coordinate Tables

The PCA coordinates are stored together with metadata so that each bouton keeps its identity, condition, and later target assignment within the reduced space. This merged representation is used throughout the notebook for overlays and cluster analyses.


In [ ]:
# Convert PCA coordinates to DataFrames for analysis and plotting

# Create DataFrames for PCA-transformed coordinates
principal_component_columns = ['PC1', 'PC2']
pca_dfs = {}

for dataset_name, dataset_coords in pca_data.items():
    pca_dfs[f'transformed_{dataset_name}'] = pd.DataFrame(dataset_coords, columns=principal_component_columns)

# Metadata-enriched PCA tables for downstream stats/plots
pca_source_map = {
    'WT_pooled': PCA_Data_WT_Pooled,
    'WT_Theo': PCA_Data_WT_Theo,
    'WT_Anthime': PCA_Data_WT_Anthime,
    'SynII': PCA_Data_SynII,
    'WT_1_5Ca': PCA_Data_WT_Low_Ca,
    'WT_4Ca': PCA_Data_WT_High_Ca,
    'stab_before': PCA_Data_Stability_Before,
    'stab_after': PCA_Data_Stability_After,
    '50Hz_1_5Ca': PCA_Data_50Hz_1_5_Ca,
    '50Hz_4Ca': PCA_Data_50Hz_4_Ca,
    '50Hz_2_5Ca': PCA_Data_50Hz_2_5_Ca,
}

PCA_METADATA_DFS = {}
for key, coord_df in pca_dfs.items():
    short_name = key.replace('transformed_', '')
    source_df = pca_source_map.get(short_name)
    if source_df is not None and len(source_df) == len(coord_df):
        PCA_METADATA_DFS[short_name] = pd.concat([
            coord_df.reset_index(drop=True),
            source_df.reset_index(drop=True)
        ], axis=1)

# Create PCA components table showing feature contributions to each PC
feature_names = list(WT_pooled_for_pca.columns)
df_components = pd.DataFrame(pca.components_, columns=feature_names, index=['PC1', 'PC2'])

# Display top feature contributors for each principal component
print("PCA Components Analysis (top 3 contributors per PC):")
print("-" * 50)
for pc_name in ['PC1', 'PC2']:
    top_contributing_features = df_components.loc[pc_name].abs().nlargest(3)
    feature_contributions = [f'{feature_name}({contribution:.3f})' for feature_name, contribution in top_contributing_features.items()]
    print(f"  {pc_name}: {', '.join(feature_contributions)}")

print(f"\n✓ Created {len(pca_dfs)} PCA coordinate DataFrames")
print(f"✓ Created {len(PCA_METADATA_DFS)} metadata-enriched PCA DataFrames")
print(f"✓ PCA components table shape: {df_components.shape}")


### 2.4 Relate Original Features to PCA Axes

Correlations between the original release variables and the first two components help interpret the WT reference space in terms of release strength, failures, and short-term plasticity.


In [ ]:
# Combine PCA coordinates with original features for correlation analysis

# Create combined dataset: PCA coordinates + original features
combined_pca_FEATURES_DATAFRAME = pd.concat([
    pca_dfs['transformed_WT_pooled'].reset_index(drop=True),
    PCA_Data_WT_Pooled.reset_index(drop=True)
], axis=1)

# Calculate correlation matrix between principal components and original features
full_correlation_matrix = combined_pca_FEATURES_DATAFRAME.corr(numeric_only=True)
pc_feature_correlations = full_correlation_matrix.iloc[:2, 2:]  # Extract PC1,PC2 vs features

# Display strongest correlations for interpretability
print("Strongest PC-Feature Correlations:")
print("-" * 40)
for pc_name in ['PC1', 'PC2']:
    strongest_correlations = pc_feature_correlations.loc[pc_name].abs().nlargest(3)
    correlation_strings    = [f'{feature_name}({correlation_value:.3f})' 
                          for feature_name, correlation_value in strongest_correlations.items()]
    print(f"  {pc_name}: {', '.join(correlation_strings)}")

# Store comprehensive PCA results for downstream analysis
PCA_RESULTS = {
    'pca_model'         : pca,                           # Fitted PCA transformer
    'scaler'            : scaler,                        # Fitted StandardScaler
    'pca_dataframes'    : pca_dfs,                       # PCA coordinates for all datasets
    'components'        : df_components,                 # Feature contributions to PCs
    'correlations'      : pc_feature_correlations,       # PC-feature correlation matrix
    'explained_variance': pca.explained_variance_ratio_  # Variance explained by each PC
}

print(f"\n✓ Correlation analysis complete")
print(f"✓ PCA results stored in PCA_RESULTS dictionary with {len(PCA_RESULTS)} components")

### 2.5 PCA Correlation Circle

The correlation circle summarizes how each feature loads on the first two WT components and provides a compact visual interpretation of the reference axes.


In [ ]:
# Create PCA correlation circle (biplot) showing feature contributions to principal components

feature_pc_correlations = PCA_RESULTS['correlations'].T.values
scaling_factor = 1.0
correlation_vectors = feature_pc_correlations * scaling_factor

fig, ax = make_figure_grid(figsize=(8, 8))

ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
unit_circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='-', alpha=0.5)
ax.add_patch(unit_circle)

feature_names = list(WT_pooled_for_pca.columns)

for feature_idx, feature_name in enumerate(feature_names):
    pc1_correlation = correlation_vectors[feature_idx, 0]
    pc2_correlation = correlation_vectors[feature_idx, 1]

    ax.annotate(
        '',
        xy=(pc1_correlation, pc2_correlation),
        xytext=(0, 0),
        arrowprops=dict(
            arrowstyle='-|>',
            color='darkred',
            lw=3.0,
            mutation_scale=14,
            alpha=0.9,
            shrinkA=0,
            shrinkB=0
        )
    )

    label_x_position = pc1_correlation * 1.1
    label_y_position = pc2_correlation * 1.1
    ax.text(
        label_x_position,
        label_y_position,
        feature_name,
        ha='center',
        va='center',
        fontsize=10,
        weight='bold'
    )

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
total_variance = pc1_variance + pc2_variance

ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.set_title(f'PCA Correlation Circle\n({total_variance:.1%} total variance explained)')
ax.grid(False)

plt.tight_layout()

output_file = OUTPUT_DIR / "02_02_wt_pca_correlation_circle.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print("Strongest Feature-PC Correlations:")
print("=" * 45)
pc_feature_correlations = PCA_RESULTS['correlations']

for pc_name in ['PC1', 'PC2']:
    strongest_features = pc_feature_correlations.loc[pc_name].abs().nlargest(5)
    print(f"\n{pc_name} (strongest contributors):")
    for feature_name, correlation_magnitude in strongest_features.items():
        correlation_value = pc_feature_correlations.loc[pc_name, feature_name]
        correlation_direction = "+" if correlation_value > 0 else "-"
        print(f"  {correlation_direction} {feature_name}: {correlation_magnitude:.3f}")

print(f"\n✓ Saved correlation circle to {output_file}")


In [ ]:
# PCA smoothing/grid helpers used by the WT PCA map cells
SMOOTH_ROBUST = True
SMOOTH_SIGMA_FACTOR = 2.0
SMOOTH_SUPPORT_RADIUS = 3.0
SMOOTH_CLIP_PCT = 5
GRID_RES = 80

def smooth_field(xy, vals, mask, gx, gy):
    """Gaussian-kernel smoothing with support masking."""
    pts, v = xy[mask], vals[mask]
    Xg, Yg = np.meshgrid(gx, gy)
    if len(pts) < 3:
        return np.full(Xg.shape, np.nan), np.nan
    if SMOOTH_ROBUST:
        lo, hi = np.nanpercentile(v, [SMOOTH_CLIP_PCT, 100 - SMOOTH_CLIP_PCT])
        v = np.clip(v, lo, hi)
    d = cdist(pts, pts)
    np.fill_diagonal(d, np.inf)
    sig = np.median(np.min(d, axis=1)) * SMOOTH_SIGMA_FACTOR
    gp = np.column_stack([Xg.ravel(), Yg.ravel()])
    dist_gp = cdist(gp, pts)
    w = np.exp(-0.5 * (dist_gp / sig) ** 2)
    ws = w.sum(1)
    sg = np.full(len(gp), np.nan)
    ok = ws > 1e-12
    support = np.min(dist_gp, axis=1) <= (SMOOTH_SUPPORT_RADIUS * sig)
    ok = ok & support
    sg[ok] = (w[ok] * v).sum(1) / ws[ok]
    return sg.reshape(Xg.shape), sig

def setup_pca_grid(coords, pad=0.5, at_least_reference=True):
    coords = np.asarray(coords, float)
    xmn, xmx = coords[:, 0].min() - pad, coords[:, 0].max() + pad
    ymn, ymx = coords[:, 1].min() - pad, coords[:, 1].max() + pad
    if at_least_reference and 'get_reference_pca_limits' in globals():
        ref_xlim, ref_ylim = get_reference_pca_limits(pad=pad)
        xmn = min(float(xmn), float(ref_xlim[0]))
        xmx = max(float(xmx), float(ref_xlim[1]))
        ymn = min(float(ymn), float(ref_ylim[0]))
        ymx = max(float(ymx), float(ref_ylim[1]))
    gx = np.linspace(xmn, xmx, GRID_RES)
    gy = np.linspace(ymn, ymx, GRID_RES)
    ext = [xmn, xmx, ymn, ymx]
    return gx, gy, ext


In [ ]:
# %% FIG A2b : WT PCA maps of parameter values directly

if 'scaled_data' not in globals():
    raise RuntimeError('Run the PCA preparation cells first.')

wt_coords = pca_data['WT_pooled']
wt_feature_names = list(WT_pooled_for_pca.columns)
wt_scaled = np.asarray(scaled_data['WT_pooled'], float)

n_features = len(wt_feature_names)
ncols = 4
nrows = int(np.ceil(n_features / ncols))

fig, axes = make_figure_grid(nrows, ncols, panel_kind='pca', squeeze=False)
axes_flat = axes.ravel()

for i, feature_name in enumerate(wt_feature_names):
    ax = axes_flat[i]

    feature_idx = wt_feature_names.index(feature_name)
    feature_vals = wt_scaled[:, feature_idx]
    ok = np.isfinite(feature_vals)

    if ok.sum() < 3:
        ax.text(0.5, 0.5, f'{feature_name}\nnot enough data', ha='center', va='center',
                transform=ax.transAxes, fontsize=9)
        style_pca_axes(ax, title=feature_name, legend=False)
        continue

    gx, gy, ext = setup_pca_grid(wt_coords[ok], pad=0.5)
    sg, _ = smooth_field(wt_coords[ok], feature_vals[ok], np.ones(ok.sum(), dtype=bool), gx, gy)

    vmax = np.nanmax(np.abs(feature_vals[ok]))
    vmin = -vmax

    ax.imshow(
        sg,
        extent=ext,
        origin='lower',
        aspect='equal',
        cmap='coolwarm',
        vmin=vmin,
        vmax=vmax,
        interpolation='bilinear',
        alpha=0.75,
    )

    plot_pca_value_overlay(
        ax,
        wt_coords[ok],
        np.ones(ok.sum(), dtype=bool),
        feature_vals[ok],
        cmap='coolwarm',
        vmin=vmin,
        vmax=vmax,
        s=16,
        edgecolors='none',
        zorder=3,
    )

    sm = plt.cm.ScalarMappable(
        cmap='coolwarm',
        norm=plt.Normalize(vmin, vmax)
    )
    plt.colorbar(sm, ax=ax, shrink=0.75)

    style_pca_axes(ax, title=feature_name, legend=False)
    ax.set_aspect('equal', adjustable='box')

for j in range(n_features, len(axes_flat)):
    axes_flat[j].axis('off')

fig.suptitle('WT PCA maps of standardized parameter values', fontsize=12, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.97])

output_file = OUTPUT_DIR / '02_02b_wt_parameter_value_maps.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved to {output_file}")
print(f"Mapped parameters: {', '.join(wt_feature_names)}")


### 2.6 WT Reference Check

These controls verify that the pooled WT reference remains coherent when compared with the contributing WT subsets. The goal is to confirm that the reference space is biologically interpretable rather than driven by one acquisition subset.


In [ ]:
# Create 3 boxplots comparing WT pooled parameters + paired stats
from scipy.stats import wilcoxon

fig, axes = make_figure_grid(1, 3, figsize=(15, 5))

# Prepare data for comparison
wt_data = PCA_Data_WT_Pooled.copy()

def paired_wilcoxon(df, col_a, col_b):
    sub = df[[col_a, col_b]].replace([np.inf, -np.inf], np.nan).dropna()
    n = len(sub)
    if n < 3:
        return {'n': n, 'W': np.nan, 'p': np.nan, 'mean_diff': np.nan, 'median_diff': np.nan}
    d = sub[col_b] - sub[col_a]
    try:
        W, p = wilcoxon(sub[col_a], sub[col_b], alternative='two-sided', zero_method='wilcox')
    except ValueError:
        W, p = np.nan, np.nan
    return {
        'n': n,
        'W': W,
        'p': p,
        'mean_diff': d.mean(),
        'median_diff': d.median()
    }

# Plot 1: AMP1 vs AMP2
data_amp = pd.concat([
    pd.DataFrame({'Value': wt_data['AMP1'], 'Amplitude': 'AMP1'}),
    pd.DataFrame({'Value': wt_data['AMP2'], 'Amplitude': 'AMP2'})
])
sns.boxplot(data=data_amp, x='Amplitude', y='Value', ax=axes[0])
sns.stripplot(data=data_amp, x='Amplitude', y='Value', ax=axes[0], color='black', size=3, alpha=0.4)
axes[0].set_title('AMP1 vs AMP2')
axes[0].set_ylabel('Amplitude (ΔF/F)')
axes[0].grid(False)

amp_stats = paired_wilcoxon(wt_data, 'AMP1', 'AMP2')
if np.isfinite(amp_stats['p']):
    axes[0].text(
        0.03, 0.97, f"paired Wilcoxon p={amp_stats['p']:.3g}\nn={amp_stats['n']}",
        transform=axes[0].transAxes, va='top', fontsize=9
    )

# Plot 2: %Fail1 vs %Fail2
fail_stats = None
if '%Fail1' in wt_data.columns and '%Fail2' in wt_data.columns:
    data_fail = pd.concat([
        pd.DataFrame({'Value': wt_data['%Fail1'], 'Failure Rate': '%Fail1'}),
        pd.DataFrame({'Value': wt_data['%Fail2'], 'Failure Rate': '%Fail2'})
    ])
    sns.boxplot(data=data_fail, x='Failure Rate', y='Value', ax=axes[1])
    sns.stripplot(data=data_fail, x='Failure Rate', y='Value', ax=axes[1], color='black', size=3, alpha=0.4)
    axes[1].set_title('%Fail1 vs %Fail2')
    axes[1].set_ylabel('Failure Rate (%)')
    axes[1].grid(False)

    fail_stats = paired_wilcoxon(wt_data, '%Fail1', '%Fail2')
    if np.isfinite(fail_stats['p']):
        axes[1].text(
            0.03, 0.97, f"paired Wilcoxon p={fail_stats['p']:.3g}\nn={fail_stats['n']}",
            transform=axes[1].transAxes, va='top', fontsize=9
        )
else:
    axes[1].axis('off')
    axes[1].text(0.5, 0.5, 'Missing %Fail1/%Fail2 columns', ha='center', va='center')

# Plot 3: PPR2/1 vs PPR3/1
data_ppr = pd.concat([
    pd.DataFrame({'Value': wt_data['PPR2/1'], 'PPR': 'PPR2/1'}),
    pd.DataFrame({'Value': wt_data['PPR3/1'], 'PPR': 'PPR3/1'})
])
sns.boxplot(data=data_ppr, x='PPR', y='Value', ax=axes[2])
sns.stripplot(data=data_ppr, x='PPR', y='Value', ax=axes[2], color='black', size=3, alpha=0.4)
axes[2].axhline(1.0, color='gray', linestyle='--', alpha=0.7, linewidth=1)
axes[2].set_title('PPR2/1 vs PPR3/1')
axes[2].set_ylabel('PPR (A_n/A_1)')
axes[2].grid(False)

ppr_stats = paired_wilcoxon(wt_data, 'PPR2/1', 'PPR3/1')
if np.isfinite(ppr_stats['p']):
    axes[2].text(
        0.03, 0.97, f"paired Wilcoxon p={ppr_stats['p']:.3g}\nn={ppr_stats['n']}",
        transform=axes[2].transAxes, va='top', fontsize=9
    )

plt.tight_layout()
plt.suptitle('WT Pooled Parameter Comparisons', fontsize=14, fontweight='bold', y=1.02)

output_file = OUTPUT_DIR / "02_03_wt_reference_parameter_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved parameter comparison boxplots to {output_file}")

# Print summary statistics
print("\n=== WT Pooled Parameter Statistics ===")
print(f"AMP1: mean={wt_data['AMP1'].mean():.4f}, median={wt_data['AMP1'].median():.4f}")
print(f"AMP2: mean={wt_data['AMP2'].mean():.4f}, median={wt_data['AMP2'].median():.4f}")
if '%Fail1' in wt_data.columns and '%Fail2' in wt_data.columns:
    print(f"%Fail1: mean={wt_data['%Fail1'].mean():.4f}, median={wt_data['%Fail1'].median():.4f}")
    print(f"%Fail2: mean={wt_data['%Fail2'].mean():.4f}, median={wt_data['%Fail2'].median():.4f}")
print(f"PPR2/1: mean={wt_data['PPR2/1'].mean():.4f}, median={wt_data['PPR2/1'].median():.4f}")
print(f"PPR3/1: mean={wt_data['PPR3/1'].mean():.4f}, median={wt_data['PPR3/1'].median():.4f}")

# Paired stats report
print("\n=== Paired Statistics (Wilcoxon signed-rank) ===")
print(f"AMP2-AMP1: n={amp_stats['n']}, W={amp_stats['W']:.3g}, p={amp_stats['p']:.3g}, "
      f"meanΔ={amp_stats['mean_diff']:.4f}, medianΔ={amp_stats['median_diff']:.4f}")
if fail_stats is not None:
    print(f"%Fail2-%Fail1: n={fail_stats['n']}, W={fail_stats['W']:.3g}, p={fail_stats['p']:.3g}, "
          f"meanΔ={fail_stats['mean_diff']:.4f}, medianΔ={fail_stats['median_diff']:.4f}")
print(f"PPR3/1-PPR2/1: n={ppr_stats['n']}, W={ppr_stats['W']:.3g}, p={ppr_stats['p']:.3g}, "
      f"meanΔ={ppr_stats['mean_diff']:.4f}, medianΔ={ppr_stats['median_diff']:.4f}")


In [ ]:
# Compare WT Pooled and WT Anthime datasets in PCA space

# Extract PCA coordinates for comparison datasets
wt_pooled_coordinates  = np.asarray(pca_data['WT_pooled'])
wt_anthime_coordinates = np.asarray(pca_data['WT_Anthime'])

# Create PCA comparison plot
make_figure(figsize=(8, 6))

# Plot the WT pooled reference cloud without relying on cluster assignments
plot_pca_overlay_points(
    plt.gca(),
    wt_pooled_coordinates,
    c='lightgray',
    edgecolors='dimgray',
    linewidths=0.3,
    alpha=0.45,
    label=f'WT pooled (n={len(wt_pooled_coordinates)})',
)


# Plot WT Anthime dataset  
plot_pca_overlay_points(plt.gca(), wt_anthime_coordinates[:], 
           marker='d', edgecolors='black', linewidths=0.6,  c=WT_ANTHIME_COLOR,
           label=f'WT Anthime (n={len(wt_anthime_coordinates)})')

# Format plot
style_pca_axes(plt.gca(), title='PCA Projection: WT Pooled vs WT Anthime',  legend=False)
add_legend(plt.gca(), frameon=False)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "02_04_wt_pca_reference_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ PCA comparison plot saved to {output_file}")
print(f"✓ WT Pooled: {len(wt_pooled_coordinates)} samples")
print(f"✓ WT Anthime: {len(wt_anthime_coordinates)} samples")


## 3. WT Clustering and Release Phenotypes

Once the WT reference space is defined, hierarchical clustering is used to separate boutons with distinct combinations of amplitude, reliability, and train dynamics. These cells describe the structure of the WT bouton classes and the trace phenotypes associated with them.


### 3.1 WT Cluster Visualizations

These first visualizations inspect how extreme values of amplitude, failures, and PPR populate the WT PCA space before the formal cluster summaries. They provide an intuitive link between individual physiological variables and the later bouton classes.


In [ ]:
# Perform hierarchical clustering on PCA-transformed WT_pooled data

# Extract PCA coordinates for clustering
pca_coordinates = pca_data['WT_pooled']  # Shape: (n_samples, 2)

# Perform hierarchical clustering using Ward linkage method
linkage_matrix      = linkage(pca_coordinates, method='ward')
cluster_assignments_raw = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')

# Compute mean AMP1 for each cluster and reorder labels accordingly
amp1_values = PCA_Data_WT_Pooled['AMP1'].values
cluster_amp1_means = {}
for cid in range(1, N_CLUSTERS + 1):
    mask = cluster_assignments_raw == cid
    cluster_amp1_means[cid] = np.nanmean(amp1_values[mask])

# Sort clusters by ascending mean AMP1
sorted_clusters = sorted(cluster_amp1_means.keys(), key=lambda c: cluster_amp1_means[c])
old_to_new = {old: new for new, old in enumerate(sorted_clusters, start=1)}
cluster_assignments = np.array([old_to_new[c] for c in cluster_assignments_raw])

# Add cluster labels to original dataframe
PCA_Data_WT_Pooled_clustered               = PCA_Data_WT_Pooled.copy()
PCA_Data_WT_Pooled_clustered['HC_Cluster'] = cluster_assignments
# Save clustered data to Excel
clustered_output_file = OUTPUT_DIR / f"PCA_Data_WT_Pooled_clustered_{N_CLUSTERS}.xlsx"
PCA_Data_WT_Pooled_clustered.to_excel(clustered_output_file, index=False)
print(f"✓ Saved clustered data to {clustered_output_file}")


# Visualize clusters in PCA space using plot_pca_nice
pc1_variance = PCA_RESULTS["explained_variance"][0]
pc2_variance = PCA_RESULTS["explained_variance"][1]

# Generate consistent Set1 colors for clusters
set1_colors          = Set1(np.linspace(0, 1, N_CLUSTERS))
cluster_rgba_colors  = [tuple(color) for color in set1_colors]
cluster_hex_colors   = [to_hex(color) for color in cluster_rgba_colors]
cluster_palette      = ListedColormap(cluster_rgba_colors, name='cluster_palette')
cluster_color_lookup = {cid: cluster_rgba_colors[cid - 1] for cid in range(1, N_CLUSTERS + 1)}

if 'get_cluster_color' not in globals() or 'style_pca_axes' not in globals() or 'plot_pca_background' not in globals():
    raise RuntimeError('Run the main shared helper cell near the beginning of the notebook first.')



In [ ]:
# find the id of the two values with PC1 > 8 across all clusters, from pca_coordinates[mask, 0] > 8
high_pc1_ids = []
for cluster_id in range(1, N_CLUSTERS + 1):
    mask = cluster_assignments == cluster_id
    high_pc1_mask = (pca_coordinates[mask, 0] > 8)
    cluster_indices = np.where(mask)[0]
    high_pc1_indices = cluster_indices[high_pc1_mask]
    high_pc1_ids.extend(PCA_Data_WT_Pooled.iloc[high_pc1_indices]['ID'].tolist())
print("IDs of samples with PC1 > 8 across all clusters:")
for sample_id in high_pc1_ids:
    print(f"  {sample_id}")



In [ ]:
# Plot PCA with top 10% AND bottom 10% highlighted for AMP1, PPR2/1, PPR5/1, and %Fail1
fig, axes = make_figure_grid(2, 2, figsize=(14, 12))
axes = axes.flatten()
metrics = ['AMP1', 'PPR2/1', 'PPR5/1', '%Fail1']
titles = ['Top/Bottom 10% AMP1', 'Top/Bottom 10% PPR2/1', 'Top/Bottom 10% PPR5/1', 'Top/Bottom 10% %Fail1']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx]
    
    # Plot all WT pooled with cluster colors (background)
    plot_pca_background(ax, alpha=0.3,  label='WT pooled')
    
    # Get top 10% and bottom 10% for this metric
    if metric in PCA_Data_WT_Pooled.columns:
        top_threshold = PCA_Data_WT_Pooled[metric].quantile(0.90)
        bottom_threshold = PCA_Data_WT_Pooled[metric].quantile(0.10)

        top_mask = PCA_Data_WT_Pooled[metric] >= top_threshold
        bottom_mask = PCA_Data_WT_Pooled[metric] <= bottom_threshold

        top_coords = pca_coordinates[top_mask.values]
        bottom_coords = pca_coordinates[bottom_mask.values]

        n_top = len(top_coords)
        n_bottom = len(bottom_coords)
        
        # Plot bottom 10% as orange stars (drawn first, behind top 10%)
        plot_pca_overlay_points(ax, bottom_coords[:],
                   marker='*',  c='orange', alpha=0.9,
                   edgecolors='black', linewidths=0.5,
                   label=f'Bottom 10% {metric} (n={n_bottom})')

        # Plot top 10% as purple stars (drawn on top)
        plot_pca_overlay_points(ax, top_coords[:],
                   marker='*',  c='purple', alpha=0.9,
                   edgecolors='black', linewidths=0.5,
                   label=f'Top 10% {metric} (n={n_top})')
    
    # Format subplot
    style_pca_axes(ax, title=title,  legend=False)
    add_legend(ax, loc='upper right', fontsize=9, frameon=False)

plt.tight_layout()
plt.suptitle('PCA: Top & Bottom 10% by Metric', fontsize=14, fontweight='bold', y=1.02)
output_file = OUTPUT_DIR / "03_01_wt_cluster_metric_extremes.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved to {output_file}")

for metric in metrics:
    if metric in PCA_Data_WT_Pooled.columns:
        top_threshold = PCA_Data_WT_Pooled[metric].quantile(0.90)
        bottom_threshold = PCA_Data_WT_Pooled[metric].quantile(0.10)
        print(f"  {metric}: bottom threshold = {bottom_threshold:.4f} | top threshold = {top_threshold:.4f}")


In [ ]:
# Plot PCA without cluster colors - all points in gray

make_figure(figsize=(8, 6))

# Plot all WT pooled points in gray
plot_pca_background(plt.gca(), use_cluster_colors=False, color='gray', alpha=1,  edgecolors='black', linewidths=0.6, label=None)

# Format plot
style_pca_axes(plt.gca(), title='PCA Projection: WT Pooled (unclustered)',  legend=False)
plt.tight_layout()



# Save and display
output_file = OUTPUT_DIR / "03_02_wt_pca_unclustered_scatter.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved unclustered PCA plot to {output_file}")
print(f"✓ Total samples: {len(pca_coordinates)}")


### 3.2 Hierarchical Clustering Dendrogram

The dendrogram shows how WT boutons partition when the reduced WT representation is submitted to Ward hierarchical clustering. It is the structural bridge between the continuous PCA space and the discrete class definition used later in the notebook.


In [ ]:
# Create dendrogram visualization of hierarchical clustering

from matplotlib.lines import Line2D

# Reuse consistent cluster colors for dendrogram branches
if 'cluster_hex_colors' not in globals():
    dendrogram_colormap = plt.get_cmap('Set1')
    cluster_hex_colors  = [to_hex(dendrogram_colormap(i)) for i in range(N_CLUSTERS)]
set_link_color_palette(cluster_hex_colors)

# Create sample labels for dendrogram leaves
try:
    sample_labels = PCA_Data_WT_Pooled.index.tolist()
except AttributeError:
    sample_labels = [f'Sample_{i+1}' for i in range(len(pca_coordinates))]

# Calculate clustering threshold for specified number of clusters
clustering_threshold = linkage_matrix[-N_CLUSTERS+1, 2]

# Generate dendrogram plot
fig, ax = make_figure_grid(figsize=(12, 6))
dendrogram(
    linkage_matrix,
    color_threshold=clustering_threshold,
    labels=sample_labels,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
)

cluster_handles = [
    Line2D([0], [0], color=cluster_hex_colors[cid - 1], lw=2.5, label=f'C{cid}')
    for cid in range(1, N_CLUSTERS + 1)
]
cluster_labels = [f'C{cid}' for cid in range(1, N_CLUSTERS + 1)]

ax.set_title(f'Hierarchical Clustering Dendrogram (k={N_CLUSTERS})')
ax.set_xlabel('Samples')
ax.set_ylabel('Ward Distance')
add_legend(ax, handles=cluster_handles, labels=cluster_labels, loc='upper right')
fig.tight_layout()

# Save dendrogram
output_file = OUTPUT_DIR / "03_03_wt_hcpc_dendrogram.pdf"
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display clustering information
print(f"Clustering threshold for {N_CLUSTERS} clusters: {clustering_threshold:.3f}")
print(f"Dendrogram branches colored by cluster membership")
print(f"✓ Saved dendrogram to {output_file}")


In [ ]:
# ===== PCA with Ellipses: Hard Edge + Tolerance Zone =====
# knobs
ELLIPSE_ALPHA = 0.8        # hard edge level
ELLIPSE_TOL   = 1.0       # inflate ellipse threshold by (1 + ELLIPSE_TOL)

import numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# --- build models ---
def _ellipse_models(X, y, n_clusters, ridge=1e-6):
    models=[]
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts)>2:
            mu=pts.mean(axis=0); cov=np.cov(pts.T)+np.eye(2)*ridge; inv=np.linalg.pinv(cov)
            models.append({'cluster':k,'center':mu,'cov':cov,'inv_cov':inv})
    return models
def _md2(x, m): 
    d=x-m['center']; return float(d.T @ m['inv_cov'] @ d)

ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard       = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol        = thr_hard*(1.0 + ELLIPSE_TOL)
print(f"Thresholds - Hard: {thr_hard:.2f}, Tolerance: {thr_tol:.2f}")

# --- plot PCA with hard + tolerance ---
fig, ax = make_figure_grid(figsize=(8,6))
plot_pca_background(ax, alpha=0.9, marker='o', edgecolors='black', linewidths=0.6, label=None)
uniq = np.unique(cluster_assignments)

def _draw_ellipse(ax, mean, cov, thr, color, fa, lw, ls='-'):
    U,s,_=np.linalg.svd(cov); ang=np.degrees(np.arctan2(U[1,0],U[0,0])); w,h=2*np.sqrt(thr*s)
    ax.add_patch(Ellipse(mean, w, h, angle=ang, facecolor=color, edgecolor=color, alpha=fa, lw=lw, ls=ls))

for cid in uniq:
    pts = pca_coordinates[cluster_assignments==cid]
    if len(pts)<3:
        continue
    m     = [mm for mm in ellipse_models if mm['cluster']==cid][0]
    color = get_cluster_color(cid)
    _draw_ellipse(ax, m['center'], m['cov'], thr_tol,  color, 0.10, 0.0)
    _draw_ellipse(ax, m['center'], m['cov'], thr_hard, color, 0.00, 2.0)

cluster_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=get_cluster_color(cid),
           markeredgecolor='black', markeredgewidth=0.5, markersize=7, label=f'C{cid}')
    for cid in uniq
]
cluster_labels = [f'C{int(cid)}' for cid in uniq]

style_pca_axes(ax, title=f'PCA with Ellipses: hard α={ELLIPSE_ALPHA:.2f}, tol +{ELLIPSE_TOL*100:.0f}%', legend=False)
add_legend(ax, handles=cluster_handles, labels=cluster_labels, frameon=False, loc='upper right')
fig.tight_layout()
out = OUTPUT_DIR / "03_04_wt_cluster_ellipses.pdf"
fig.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print(f"[Ellipses] PCA plot saved: {out}")


### 3.3 WT Release Property Survey

Once clusters are defined, the next figures examine how each class differs in train profile, first-response amplitude, failure rate, and mean trace shape. Together these panels establish the physiological identity of the WT bouton classes that underlie the rest of the manuscript.


#### 3.3.1 Cluster PPR Trajectories

The normalized train profiles make the short-term plasticity phenotype of each WT class explicit. These are the profiles that motivate the interpretation of classes in terms of facilitation, depression, and mixed modes of release regulation.


In [ ]:
# PPR profiles by hierarchical cluster

if 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining ppr_profile_stats).')

ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
clusters = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].unique())

fig, ax = make_figure_grid(figsize=(10, 6))

for cluster in clusters:
    cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster]
    pulse_numbers, means, sems, n_cluster = ppr_profile_stats(cluster_data, ppr_column_names=ppr_cols)
    color = get_cluster_color(cluster)

    plot_ppr_mean_sem(
        ax,
        pulse_numbers,
        means,
        sems,
        color=color,
        marker='o',
        label=f'Cluster {cluster} (n={n_cluster})',
        linewidth=2,
        sem_alpha=0.2,
    )

finalize_ppr_axis(
    ax,
    pulse_numbers,
    title='PPR Profiles by Hierarchical Cluster',
    ylabel='Mean PPR (A_n/A_1)',
    legend=True,
    legend_loc='upper right',
)
plt.tight_layout()

output_file = OUTPUT_DIR / '03_05_wt_cluster_ppr_profiles.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()


#### 3.3.2 Cluster Metric Utility

The cluster-wise boxplot utility is used repeatedly because the manuscript relies on comparing the same scalar descriptors across classes. Keeping the statistics and plotting logic together reduces the risk of inconsistent summary figures.


In [ ]:
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
from statsmodels.stats.multitest import multipletests

def cluster_boxplot_analysis(data, column, title, output_prefix):
    'Create boxplot by cluster with pairwise Mann-Whitney U tests and Benjamini-Hochberg (FDR) correction.'

    # Boxplot creation (unchanged)
    make_figure(figsize=(8, 5))
    clusters = sorted(data['HC_Cluster'].unique())
    ax = sns.boxplot(x='HC_Cluster', y=column, hue='HC_Cluster', data=data, palette=cluster_hex_colors, legend=False)
    sns.stripplot(x='HC_Cluster', y=column, data=data, color='k', size=3, alpha=0.5, ax=ax)

    if 'PPR' in column:
        plt.axhline(1, color='gray', linestyle='dotted', linewidth=2)

    plt.ylabel(column)
    plt.title(title)

    # Style cleanup (unchanged)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.get_xaxis().set_visible(False)

    # Legend (unchanged)
    handles = [plt.Line2D([0], [0], color=get_cluster_color(cluster_id), lw=4) for cluster_id in clusters]
    labels = [f'Cluster {cluster_id}' for cluster_id in clusters]
    add_legend(plt.gca(), handles, labels, loc='upper right')

    # Statistical tests
    data_per_cluster = {c: data.loc[data['HC_Cluster'] == c, column].dropna().values for c in clusters}

    try:
        kw_stat, kw_p = kruskal(*data_per_cluster.values())
    except ValueError:
        kw_stat, kw_p = float('nan'), float('nan')

    pairs = list(combinations(clusters, 2))
    results = []
    p_values = []
    for a, b in pairs:
        x, y = data_per_cluster[a], data_per_cluster[b]
        try:
            stat, p = mannwhitneyu(x, y, alternative='two-sided')
        except ValueError:
            stat, p = float('nan'), float('nan')
        results.append({
            'Cluster_A': a,
            'Cluster_B': b,
            'n_A': len(x),
            'n_B': len(y),
            'U_statistic': stat,
            'p_value_raw': p,
            'p_value_corrected': float('nan')  # Placeholder, filled after FDR correction
        })
        p_values.append(p)

    # Apply Benjamini-Hochberg correction (FDR)
    _, corrected_p_values, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

    # Update corrected p-values in the results
    for idx, result in enumerate(results):
        result['p_value_corrected'] = corrected_p_values[idx] if not pd.isna(p_values[idx]) else pd.NA

    # Sort results by corrected p-value
    results.sort(key=lambda d: d['p_value_corrected'] if not pd.isna(d['p_value_corrected']) else 1)

    # Create a DataFrame for the results
    results_df = pd.DataFrame(results)

    # Add Kruskal-Wallis test results
    kw_results = pd.DataFrame({
        'Test': ['Kruskal-Wallis'],
        'Statistic': [kw_stat],
        'p_value': [kw_p],
        'Number_of_comparisons': [len(pairs)]
    })

    # Export vers Excel
    with pd.ExcelWriter(OUTPUT_DIR / f"{output_prefix}_statistics.xlsx") as writer:
        results_df.to_excel(writer, sheet_name='Pairwise_Comparisons', index=False)
        kw_results.to_excel(writer, sheet_name='Kruskal_Wallis', index=False)

    # Save boxplot
    plt.tight_layout()
    output_file = OUTPUT_DIR / f"{output_prefix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Analysis saved to {output_file} and {OUTPUT_DIR / f'{output_prefix}_statistics.xlsx'}.")


#### 3.3.3 Cluster-Level Parameter Comparisons

These comparisons quantify how amplitude, failure rate, and early PPR differ from one WT class to another. They support the idea that bouton diversity is not reducible to a single monotonic axis of synaptic strength.


In [ ]:
# AMP1 analysis  
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'AMP1', 'AMP1 by Cluster', '03_06_wt_cluster_amp1_boxplot')

# PPR2/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR2/1', 'PPR2/1 by Cluster', '03_07_wt_cluster_ppr2_1_boxplot')

# PPR3/1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, 'PPR3/1', 'PPR3/1 by Cluster', '03_08_wt_cluster_ppr3_1_boxplot')

# %Fail1 analysis
cluster_boxplot_analysis(PCA_Data_WT_Pooled_clustered, '%Fail1', '%Fail1 by Cluster', '03_09_wt_cluster_fail1_boxplot')

#### 3.3.4 Mean Traces by WT Cluster

The mean-trace panels show the time-domain signature of each WT class and make the cluster interpretation easier to relate to the original fluorescence data. They are especially useful when comparing classes that overlap partly in one scalar metric but diverge across the full train.


In [ ]:
# Plot mean traces for each cluster with SEM shading

fig, axes = make_figure_grid(2, 3, figsize=(15, 8))
axes = axes.flatten()
cluster_trace_ids = {cid: [] for cid in range(1, N_CLUSTERS + 1)}
for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    cluster_trace_ids[row['HC_Cluster']].append(row['ID'])
cluster_stats = {cid: compute_trace_stats(trace_ids=cluster_trace_ids[cid], source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS) for cid in range(1, N_CLUSTERS + 1) if cluster_trace_ids[cid]}
all_means = [np.asarray(stats['average'], float) for stats in cluster_stats.values()]
all_sems = [np.asarray(stats['sem'], float) for stats in cluster_stats.values()]
all_upper = [m + s for m, s in zip(all_means, all_sems)]
all_lower = [m - s for m, s in zip(all_means, all_sems)]
y_min_global = np.nanmin([np.nanmin(l) for l in all_lower])
y_max_global = np.nanmax([np.nanmax(u) for u in all_upper])
y_margin = (y_max_global - y_min_global) * 0.05
y_lim = (y_min_global - y_margin, y_max_global + y_margin)
stim_times = [1.0 + i * 0.05 for i in range(10)]
for idx, cid in enumerate(range(1, N_CLUSTERS + 1)):
    ax = axes[idx]
    stats = cluster_stats[cid]
    color = get_cluster_color(cid)
    plot_traces(ax=ax, rows=stats['rows'], source=TRACE_MEAN_SOURCE, tol_factor=1, aggregation='mean', alignment='nearest', color=color, show_average=True, show_sem=True, xlim=(0.8, 1.6), ylim=y_lim, xlabel='Time (s)', ylabel='ΔF/F', title=f'Cluster {cid} (n={stats["n"]})', stim_times=stim_times, stim_kwargs={'mode': 'fixed', 'y_span': (y_lim[1] * 0.92, y_lim[1] * 0.97), 'linewidth': 1.5})
plt.tight_layout()
plt.suptitle('Mean Traces by Cluster', fontsize=14, fontweight='bold', y=1.02)
output_file = OUTPUT_DIR / "03_10_wt_cluster_mean_traces.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved cluster mean traces to {output_file}")

print("Traces per cluster:")
for cid in range(1, N_CLUSTERS + 1):
    print(f"  Cluster {cid}: {cluster_stats[cid]['n']} traces")


#### 3.3.5 Selected-Cluster Trace Overlay

Direct overlay of selected cluster averages highlights that boutons with related train dynamics can still differ in absolute glutamate output, consistent with partially dissociable control of short-term plasticity and synaptic weight.


In [ ]:
# Overlay mean traces for Cluster 3 and Cluster 4

clusterA_ids = []
clusterB_ids = []
clusterA_id = 1
clusterB_id = 6
for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    if row['HC_Cluster'] == clusterA_id:
        clusterA_ids.append(row['ID'])
    elif row['HC_Cluster'] == clusterB_id:
        clusterB_ids.append(row['ID'])
clusterA_stats = compute_trace_stats(trace_ids=clusterA_ids, source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
clusterB_stats = compute_trace_stats(trace_ids=clusterB_ids, source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
fig, ax = make_figure_grid(figsize=(10, 6))
plot_traces(ax=ax, rows=clusterB_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_cluster_color(clusterB_id), label=f'C{clusterB_id} (n={clusterB_stats["n"]})', show_average=True, show_sem=True, style_axis=False)
plot_traces(ax=ax, rows=clusterA_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_cluster_color(clusterA_id), label=f'C{clusterA_id} (n={clusterA_stats["n"]})', show_average=True, show_sem=True, zero_line=True, event_time=1.0, event_kwargs={'color': 'red', 'linestyle': '--', 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, xlabel='Time (s)', ylabel='ΔF/F', title=f'Mean Traces: C{clusterA_id} vs C{clusterB_id}', legend=True)
plt.tight_layout()
output_file = OUTPUT_DIR / "03_11_wt_selected_cluster_mean_trace_overlay.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved to {output_file}")
print(f"  C{clusterA_id}: {clusterA_stats['n']} traces")
print(f"  C{clusterB_id}: {clusterB_stats['n']} traces")


## 4. WT Quantal, N, P, and Refilling Analyses

Trial-level responses are normalized to an empirical single-vesicle estimate to interpret WT bouton responses in terms of quantal size, effective release-site occupancy, effective release probability, and refilling dynamics. This section provides a mechanistic bridge between the WT clustering results and the calcium/frequency perturbations.

Several panels in this block extend beyond the wording of the current manuscript draft and should be read as mechanistic elaborations of the same N-Pr framework. The guiding idea is unchanged: bouton classes differ not only in apparent release probability, but also in the occupancy and rapid reuse of release sites during trains.

Methodological note: quantal calibration is anchored to successes at 1.5 mM Ca²⁺, cumulative release is summarized with Schneggenburger-Neher style plots, and N/P trajectories are estimated with binomial or bootstrap-based procedures. The more recent refilling and occupancy figures are included here because they sharpen the interpretation of sustained facilitation at high release drive.


In [ ]:


# %% ###############################################################
# CELL 0 : Shared utilities, style, smoothing engine
# Assumes upstream: trials_all, PCA_Data_WT_Pooled_clustered,
#   pca_coordinates, cluster_assignments, sorted_clusters,
#   cluster_color_lookup, FEATURES_DATAFRAME, strip_calcium,
#   ppr_cols, num_pulses, OUTPUT_DIR, BASE_DIR, PPR_TRIALS_FILENAME
# ###################################################################

import re, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import linregress, norm, spearmanr, kruskal, pearsonr
from scipy.special import gammaln
from scipy.spatial.distance import cdist
from collections import defaultdict, OrderedDict
from pathlib import Path

# Global figure defaults (PNAS-aligned): no grid, no top/right spines, no legend box
if 'clamp_fontsize' not in globals():
    def clamp_fontsize(value):
        return float(np.clip(float(value), 6.0, 12.0))
if 'clamp_linewidth' not in globals():
    def clamp_linewidth(value):
        return float(np.clip(float(value), 0.25, 1.5))
if 'clamp_markersize' not in globals():
    def clamp_markersize(value):
        return float(np.clip(float(value), 3.0, 9.0))
if '_enforce_pnas_artist_style' not in globals():
    def _enforce_pnas_artist_style(ax):
        for line in ax.lines:
            line.set_linewidth(clamp_linewidth(line.get_linewidth()))
            line.set_markersize(clamp_markersize(line.get_markersize()))
        for collection in ax.collections:
            if hasattr(collection, 'get_linewidths'):
                widths = collection.get_linewidths()
                if widths is not None and len(widths) > 0:
                    collection.set_linewidths(np.clip(widths, 0.25, 1.5))

plt.rcParams['axes.grid'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['legend.frameon'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.size'] = clamp_fontsize(8.0)
plt.rcParams['axes.labelsize'] = clamp_fontsize(8.0)
plt.rcParams['axes.titlesize'] = clamp_fontsize(9.0)
plt.rcParams['xtick.labelsize'] = clamp_fontsize(7.0)
plt.rcParams['ytick.labelsize'] = clamp_fontsize(7.0)
plt.rcParams['legend.fontsize'] = clamp_fontsize(7.0)
plt.rcParams['contour.linewidth'] = clamp_linewidth(0.8)
if '_install_pnas_matplotlib_sizers' in globals():
    _install_pnas_matplotlib_sizers()

warnings.filterwarnings('ignore', category=RuntimeWarning)

# All conditions recorded at 2.5 mM Ca, 20 Hz (for quantal size estimation AND normalization)
conditions_2_5_20Hz = get_friendly_conditions('2.5mM_20Hz_WT')

# All conditions recorded at 2.5 mM Ca (any frequency): for grouping later
conditions_2_5_all = get_calcium_conditions('2.5mM', 'all')

# Map each condition to its [Ca] and frequency for later grouping
condition_meta = {}
for cond in get_calcium_conditions('1.5mM', '20Hz'):
    condition_meta[cond] = {'Ca': 1.5, 'Freq': 20}
for cond in get_calcium_conditions('1.5mM', '50Hz'):
    condition_meta[cond] = {'Ca': 1.5, 'Freq': 50}
for cond in get_calcium_conditions('2.5mM', '20Hz'):
    condition_meta[cond] = {'Ca': 2.5, 'Freq': 20}
for cond in get_calcium_conditions('2.5mM', '50Hz'):
    condition_meta[cond] = {'Ca': 2.5, 'Freq': 50}
for cond in get_calcium_conditions('4mM', '20Hz'):
    condition_meta[cond] = {'Ca': 4.0, 'Freq': 20}
for cond in get_calcium_conditions('4mM', '50Hz'):
    condition_meta[cond] = {'Ca': 4.0, 'Freq': 50}


# == Upstream dependency guard ====================================
# These should already be defined in your notebook from earlier cells.
# Uncomment / adjust if running this file standalone.
# OUTPUT_DIR = Path('figures')
# BASE_DIR   = Path('data')
# PPR_TRIALS_FILENAME = 'PPR_trials.xlsx'
if 'ppr_cols' not in dir():
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11)]
if 'num_pulses' not in dir():
    num_pulses = 10

# == strip_calcium: normalise bouton IDs across Ca²âº conditions ==
# Removes calcium/frequency suffixes so the same bouton recorded
# under different conditions maps to a single base ID.
_CA_SUFFIXES = re.compile(
    r'[_\s]*(1[._]5[_\s]?Ca|4[_\s]?Ca|2[._]5[_\s]?Ca'
    r'|1[._]5[_\s]?50Hz|2[._]5[_\s]?50Hz|4[_\s]?50Hz'
    r'|50Hz|20Hz)\s*$', re.IGNORECASE)

def strip_calcium(name: str) -> str:
    """Remove trailing Ca²âº / frequency tags from a bouton file-ID."""
    s = str(name).strip()
    prev = None
    while s != prev:          # iterative : handles stacked suffixes
        prev = s
        s = _CA_SUFFIXES.sub('', s).rstrip('_ ')
    return s

# == Publication style ============================================
def style_ax(ax, xlabel=None, ylabel=None, title=None):
    """Clean axes style; enforce unified limits when plotting PCA space."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(direction='out', length=4, width=clamp_linewidth(0.8))
    ax.grid(False)
    show_x, show_y = (_outer_axis_label_visibility(ax) if '_outer_axis_label_visibility' in globals() else (True, True))

    if xlabel == 'PC1' and 'style_pca_axes' in globals():
        xlim = (-7, 10)
        ylim = (-6, 6)
        if len(ax.images) > 0:
            ext = ax.images[0].get_extent()
            if ext is not None and len(ext) == 4:
                xlim = (float(min(ext[0], ext[1])), float(max(ext[0], ext[1])))
                ylim = (float(min(ext[2], ext[3])), float(max(ext[2], ext[3])))

        style_pca_axes(
            ax,
            title=title,
            xlim=xlim,
            ylim=ylim,
legend=False,
            show_xlabel=show_x,
            show_ylabel=(show_y and bool(ylabel)),
            extend_limits=True,
            align_to_background=True,
        )
        if '_enforce_pnas_artist_style' in globals():
            _enforce_pnas_artist_style(ax)
        return

    if xlabel and show_x:
        ax.set_xlabel(xlabel)
    elif xlabel is not None:
        ax.set_xlabel('')

    if ylabel and show_y:
        ax.set_ylabel(ylabel)
    elif ylabel is not None:
        ax.set_ylabel('')

    if title:
        ax.set_title(title, fontsize=clamp_fontsize(9.0), fontweight='bold')

    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)

def add_legend(ax, *args, **kw):
    kw = dict(kw)
    outside = kw.pop('outside', None)
    kw.setdefault('frameon', False)
    kw.setdefault('fontsize', clamp_fontsize(7.0))
    if outside is True:
        setattr(ax, '_legend_force_outside', True)
        setattr(ax, '_legend_force_inside', False)
    elif outside is False:
        setattr(ax, '_legend_force_outside', False)
        setattr(ax, '_legend_force_inside', True)
    lg = ax.legend(*args, **kw)
    if lg is not None:
        for txt in lg.get_texts():
            txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
    if outside is True and 'apply_external_legend' in globals():
        try:
            apply_external_legend(ax.figure, max_in_axes_items=0, min_repeated_panels=1)
        except Exception:
            pass
    return lg


def grid_size(nrows=1, ncols=1, panel_kind='simple'):
    """Notebook-local bridge to the global grid sizing helper."""
    nrows = max(1, int(nrows))
    ncols = max(1, int(ncols))
    if "recommended_panel_figsize" in globals():
        return recommended_panel_figsize(nrows=nrows, ncols=ncols, panel_kind=panel_kind)
    base_w = 8.7 / 2.54
    base_h = base_w * 0.78
    return base_w * ncols, base_h * nrows

# == Smoothing engine (flags set in Cell C, used by C:E) ==========
# Defaults : overridden in Cell C
SMOOTH_ROBUST      = True    # clip outliers before smoothing
SMOOTH_SIGMA_FACTOR   = 2.0    # sigma = median_NN_dist × factor
SMOOTH_SUPPORT_RADIUS = 3.0    # keep grid points within this many sigma of a datapoint
SMOOTH_CLIP_PCT       = 5      # percentile for robust clipping
GRID_RES              = 80     # grid resolution for heatmaps

def smooth_field(xy, vals, mask, gx, gy):
    """Gaussian-kernel smoothing with support masking.
    Grid points farther than one smoothing sigma from every datapoint are set to NaN.
    Returns (grid_image, sigma_used).
    """
    pts, v = xy[mask], vals[mask]
    Xg, Yg = np.meshgrid(gx, gy)
    if len(pts) < 3:
        return np.full(Xg.shape, np.nan), np.nan
    if SMOOTH_ROBUST:
        lo, hi = np.nanpercentile(v, [SMOOTH_CLIP_PCT, 100-SMOOTH_CLIP_PCT])
        v = np.clip(v, lo, hi)
    d = cdist(pts, pts); np.fill_diagonal(d, np.inf)
    sig = np.median(np.min(d, axis=1)) * SMOOTH_SIGMA_FACTOR
    gp = np.column_stack([Xg.ravel(), Yg.ravel()])
    dist_gp = cdist(gp, pts)
    w = np.exp(-0.5 * (dist_gp / sig)**2)
    ws = w.sum(1)
    sg = np.full(len(gp), np.nan)
    ok = ws > 1e-12
    support = np.min(dist_gp, axis=1) <= (SMOOTH_SUPPORT_RADIUS * sig)
    ok = ok & support
    sg[ok] = (w[ok] * v).sum(1) / ws[ok]
    return sg.reshape(Xg.shape), sig

# == PCA grid helpers (set once, reused everywhere) ===============
def setup_pca_grid(coords, pad=0.5, at_least_reference=True):
    coords = np.asarray(coords, float)
    xmn, xmx = coords[:,0].min()-pad, coords[:,0].max()+pad
    ymn, ymx = coords[:,1].min()-pad, coords[:,1].max()+pad
    if at_least_reference and 'get_reference_pca_limits' in globals():
        ref_xlim, ref_ylim = get_reference_pca_limits(pad=pad)
        xmn = min(float(xmn), float(ref_xlim[0]))
        xmx = max(float(xmx), float(ref_xlim[1]))
        ymn = min(float(ymn), float(ref_ylim[0]))
        ymx = max(float(ymx), float(ref_ylim[1]))
    gx = np.linspace(xmn, xmx, GRID_RES)
    gy = np.linspace(ymn, ymx, GRID_RES)
    ext = [xmn, xmx, ymn, ymx]
    return gx, gy, ext

# == Trial amplitude extractor ====================================
_amp_corr   = ['AMP1_UNCORR'] + [f'AMP{k}_CORR'   for k in range(2, 11)]
_amp_uncorr = ['AMP1_UNCORR'] + [f'AMP{k}_UNCORR' for k in range(2, 11)]
N_STIM = 10

def get_amp_row(row):
    """Extract 10-pulse amplitude vector; CORR<UNCORR → use UNCORR."""
    v = np.full(N_STIM, np.nan)
    for k in range(N_STIM):
        cc, uc = _amp_corr[k], _amp_uncorr[k]
        if cc not in row.index: continue
        vc = row[cc]
        if k == 0:
            v[k] = row[uc] if uc in row.index else vc
        else:
            if uc in row.index:
                vu = row[uc]
                if np.isfinite(vc) and np.isfinite(vu) and vc < vu:
                    vc = vu
            v[k] = vc
    return v

THR_SHARED_COL_MLE = 'thr_shared'

def get_thr_shared_value(row, threshold_col=THR_SHARED_COL_MLE):
    """Return the trial-wise shared threshold computed from the pre-train baseline."""
    if threshold_col not in row.index:
        return np.nan
    thr = pd.to_numeric(pd.Series([row[threshold_col]]), errors='coerce').iloc[0]
    return float(thr) if np.isfinite(thr) else np.nan

def get_failure_mask_row(row, threshold_col=THR_SHARED_COL_MLE):
    """Return per-event failures using the trial-wise thr_shared threshold."""
    fail_mask = np.full(N_STIM, np.nan)
    thr = get_thr_shared_value(row, threshold_col=threshold_col)
    if not np.isfinite(thr):
        return fail_mask, np.nan
    for k in range(N_STIM):
        cc, uc = _amp_corr[k], _amp_uncorr[k]
        vals = []
        if cc in row.index and np.isfinite(row[cc]):
            vals.append(float(row[cc]))
        if uc in row.index and np.isfinite(row[uc]):
            vals.append(float(row[uc]))
        if len(vals) == 0:
            continue
        is_failure = np.min(vals) < thr
        fail_mask[k] = 1.0 if is_failure else 0.0
    return fail_mask, thr

def get_count_row_with_failures(row, q_value, threshold_col=THR_SHARED_COL_MLE):
    """Convert amplitudes to quantal counts using thr_shared-defined failures."""
    counts = np.full(N_STIM, np.nan)
    amps = get_amp_row(row)
    fail_mask, _ = get_failure_mask_row(row, threshold_col=threshold_col)
    q_use = q_value if np.isfinite(q_value) and q_value > 0 else average_Q
    for k in range(N_STIM):
        if not np.isfinite(amps[k]) or not np.isfinite(fail_mask[k]):
            continue
        if fail_mask[k] > 0:
            counts[k] = 0
        else:
            counts[k] = max(1, int(round(amps[k] / q_use)))
    return counts

def build_trial_dict(trials_df, conditions, min_trials=5, extra_conds=None):
    """bouton_id → list of amplitude vectors.  Optionally top-up from extra_conds."""
    td = defaultdict(list)
    for _, r in trials_df[trials_df['condition'].isin(conditions)].iterrows():
        td[strip_calcium(str(r['file']).strip())].append(get_amp_row(r))
    if extra_conds:
        for _, r in trials_df[trials_df['condition'].isin(extra_conds)].iterrows():
            bid = strip_calcium(str(r['file']).strip())
            if len(td[bid]) < min_trials:
                td[bid].append(get_amp_row(r))
    return td

# == Binomial MLE engine =========================================
def log_binom_pmf(k, N, P):
    k = np.asarray(k, float)
    return (gammaln(N+1) - gammaln(k+1) - gammaln(N-k+1)
            + k * (np.log(P) if P > 0 else -np.inf)
            + (N-k) * (np.log(1-P) if P < 1 else -np.inf))

def fit_binom_mle(counts, N_max=20):
    """MLE for Binomial(N,P).  Returns (N, P, loglik)."""
    counts = np.asarray(counts, int)
    if len(counts) < 3: return np.nan, np.nan, -np.inf
    k_max, k_mean = counts.max(), counts.mean()
    if k_mean < 1e-10: return np.nan, np.nan, -np.inf
    best_N, best_P, best_ll = np.nan, np.nan, -np.inf
    for N in range(max(k_max, 1), N_max+1):
        P = k_mean / N
        if P <= 0 or P >= 1: continue
        ll = np.sum(log_binom_pmf(counts, N, P))
        if np.isfinite(ll) and ll > best_ll:
            best_ll, best_N, best_P = ll, float(N), P
    return best_N, best_P, best_ll

# == SN cumulative fit ============================================
def sn_cumulative(amps_matrix, n_fit_last=4):
    """Schneggenburger-Neher fit on (n_boutons × n_pulses) array.
    Returns dict with cum_mean, cum_sem, RRP, P0, slope, r2, n."""
    nb, npuls = amps_matrix.shape
    cum = np.cumsum(amps_matrix, axis=1)
    cm = np.nanmean(cum, axis=0)
    cs = np.nanstd(cum, axis=0) / np.sqrt(nb)
    px = np.arange(npuls)
    sl, ic, r, _, _ = linregress(px[-n_fit_last:], cm[-n_fit_last:])
    p0 = np.nanmean(amps_matrix[:,0]) / ic if ic > 0 else np.nan
    return dict(cum_mean=cm, cum_sem=cs, RRP=ic, P0=p0,
                slope=sl, r2=r**2, n=nb,
                x_fit=px, y_fit=sl*px+ic)

def reconstruct_amps_qnorm(df, Q_dict, Q_fallback):
    """Summary dataframe → (n_boutons, num_pulses) Q-normalised amplitudes."""
    ids = df['ID'].astype(str).str.strip().values
    q = np.array([Q_dict.get(strip_calcium(b), Q_fallback) for b in ids])
    q[q <= 0] = Q_fallback
    a1 = df['AMP1'].values / q
    return np.column_stack([a1] + [df[c].values * a1 for c in ppr_cols])

def extract_base_name(bouton_id):
    """Extract base name (date + linescan/fibre + bouton) for pairing
    across conditions.  Handles BOTH naming conventions:

      20210721_linescan1_50Hz_10pulses_4mMCa_bouton1_traces_converted
        → 20210721_linescan1_bouton1

      250113_Fibre2_Bouton_4
        → 250113_Fibre2_Bouton_4   (unchanged : no Hz/Ca to strip)
    """
    bid = str(bouton_id).strip()
    bid = re.sub(r'\.xlsx?$', '', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_traces_converted$', '', bid)
    bid = re.sub(r'_\d+Hz_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+pulses_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+\.?\d*mMCa_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+[_.]?\d*Ca_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_set\d+', '', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_+$', '', bid)
    bid = re.sub(r'_+', '_', bid)
    return bid

def find_valid_pairs(ids_1, ids_2, coords_1=None, coords_2=None):
    """Find paired boutons between two ID sets using normalized base names.

    Returns tuple: (indices_in_first, indices_in_second, base_names).
    """
    base_1 = {i: extract_base_name(bid) for i, bid in enumerate(ids_1)}
    base_2 = {i: extract_base_name(bid) for i, bid in enumerate(ids_2)}

    lookup_2 = {}
    for i, base in base_2.items():
        if base not in lookup_2:
            lookup_2[base] = i

    valid_1, valid_2, names = [], [], []
    for i1, base in base_1.items():
        if base in lookup_2:
            valid_1.append(i1)
            valid_2.append(lookup_2[base])
            names.append(base)

    return valid_1, valid_2, names


In [ ]:
# %% ###############################################################
# CELL A : Q extraction + figures
# A1: Empirical Bayes Q from 1.5 mM successes + histogram
# A2: Q mapped onto PCA axes (regression vs PC1, PC2)
# ###################################################################

USE_EMPIRICAL_BAYES = False
MIN_TRIALS_FOR_RAW  = 1

# == Condition→Ca mapping (SynII excluded) ========================
CA_1_5_CONDITIONS = get_calcium_conditions('1.5mM', 'all')
CA_4_CONDITIONS   = get_calcium_conditions('4mM', 'all')
CA_2_5_CONDITIONS = get_calcium_conditions('2.5mM', 'all')
ALL_Q_CONDITIONS = CA_1_5_CONDITIONS + CA_4_CONDITIONS + CA_2_5_CONDITIONS
ca_map = {**{c:'1.5Ca' for c in CA_1_5_CONDITIONS},
          **{c:'4Ca'   for c in CA_4_CONDITIONS},
          **{c:'2.5Ca' for c in CA_2_5_CONDITIONS}}

# == Load & filter trials =========================================
trials_for_q = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
trials_for_q = trials_for_q[trials_for_q['condition'].isin(ALL_Q_CONDITIONS)].copy()
trials_for_q['CaGroup'] = trials_for_q['condition'].map(ca_map)
trials_for_q['base_id'] = trials_for_q['file'].apply(strip_calcium)
trials_for_q['ID'] = trials_for_q['file']
status_norm = trials_for_q['status'].astype(str).str.strip().str.lower()
t15_succ = trials_for_q[(trials_for_q['CaGroup']=='1.5Ca') & (status_norm=='success')]

# == Raw Q per bouton (median of successes) =======================
Q_raw, Q_n_trials = {}, {}
for bid, grp in t15_succ.groupby('base_id'):
    amps = grp['AMP1_UNCORR'].dropna()
    if len(amps) >= MIN_TRIALS_FOR_RAW:
        Q_raw[bid] = float(amps.median())
        Q_n_trials[bid] = len(amps)

# == Empirical Bayes shrinkage ====================================
if USE_EMPIRICAL_BAYES and len(Q_raw) > 2:
    q_raw_vals  = np.array(list(Q_raw.values()))
    n_trials_vals = np.array([Q_n_trials[b] for b in Q_raw])
    Q_global_raw = float(np.nanmean(q_raw_vals))

    # Pooled within-bouton variance
    pooled_ss, pooled_df = 0.0, 0
    for bid, grp in t15_succ.groupby('base_id'):
        a = grp['AMP1_UNCORR'].dropna()
        if len(a) >= 2:
            pooled_ss += float(np.sum((a - a.mean())**2))
            pooled_df += len(a) - 1
    sigma2_obs = pooled_ss / pooled_df if pooled_df > 0 else float(np.var(q_raw_vals))

    sigma2_total = float(np.nanvar(q_raw_vals))
    avg_noise    = float(np.mean(sigma2_obs / n_trials_vals))
    tau2         = max(sigma2_total - avg_noise, 1e-10)
    kappa        = sigma2_obs / tau2

    Q_estimates = {}
    for bid, qr in Q_raw.items():
        ni = Q_n_trials[bid]
        w  = ni / (ni + kappa)
        Q_estimates[bid] = w * qr + (1-w) * Q_global_raw
else:
    Q_estimates = Q_raw.copy()
    kappa = np.nan

average_Q = float(np.nanmean(list(Q_estimates.values())))
median_Q  = float(np.nanmedian(list(Q_estimates.values())))
print(f"Q: {len(Q_estimates)} boutons | mean={average_Q:.4f} | median={median_Q:.4f}"
      + (f" | κ={kappa:.2f}" if np.isfinite(kappa) else ""))

# == Build global trials_all with Ca/Q columns ====================
trials_all = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
trials_all = trials_all[trials_all['condition'].isin(ALL_Q_CONDITIONS)].copy()
trials_all['CaGroup']  = trials_all['condition'].map(ca_map)
trials_all['base_id']  = trials_all['file'].apply(strip_calcium)
trials_all['Q_bouton'] = trials_all['base_id'].map(Q_estimates).fillna(average_Q)
trials_all['amp_norm'] = trials_all['AMP1_UNCORR'] / trials_all['Q_bouton']

# Enrich trial-level table once (avoid repeated recomputation downstream)
trials_all = add_condition_metadata(trials_all.rename(columns={'condition': 'Condition'}), condition_col='Condition').rename(columns={'Condition': 'condition'})
TRIALS_DATAFRAME = trials_all.copy()

# == FIX: Build Q_by_ebn HERE so norm_summary can use it =========
Q_by_ebn = {}
for k, v in Q_estimates.items():
    Q_by_ebn.setdefault(extract_base_name(k), v)

# == Normalize summary dataframes =================================
def norm_summary(df, paired):
    """Add AMP1_norm, AMP2_norm columns."""
    d = df.copy()
    d['base_id'] = d['ID'].apply(extract_base_name)
    if paired:
        q = d['base_id'].map(Q_by_ebn).fillna(average_Q)
    else:
        q = pd.Series(average_Q, index=d.index)
    d['Q'] = q; d['AMP1_norm'] = d['AMP1']/q; d['AMP2_norm'] = d['AMP2']/q
    return d

# Apply to all summary dataframes that exist
for vn, paired in [('PCA_Data_WT_Low_Ca',True),('PCA_Data_50Hz_1_5_Ca',True),
                    ('PCA_Data_WT_High_Ca',True),('PCA_Data_50Hz_4_Ca',True),
                    ('PCA_Data_WT_Pooled',False),('PCA_Data_WT_Theo',False),
                    ('PCA_Data_WT_Anthime',False),('PCA_Data_Stability_Before',False),
                    ('PCA_Data_Stability_After',False),('PCA_Data_50Hz_2_5_Ca',False)]:
    if vn in globals():
        globals()[vn] = norm_summary(globals()[vn], paired)


In [ ]:
# ##################################################################
# FIG A0a : Binomial grid search space
# color = median(K | K>=1), capped at 10
# ##################################################################

from math import comb
from matplotlib.colors import ListedColormap, BoundaryNorm

N_vals = np.arange(1, 16)                # 1 .. 15
P_vals = np.linspace(0.01, 0.50, 200)    # 0.01 .. 0.50

median_success_grid = np.full((len(P_vals), len(N_vals)), np.nan)

for i, p in enumerate(P_vals):
    for j, N in enumerate(N_vals):
        k = np.arange(0, N + 1)
        pmf = np.array([comb(N, kk) * (p ** kk) * ((1 - p) ** (N - kk)) for kk in k], float)

        succ_mask = k >= 1
        succ_k = k[succ_mask]
        succ_pmf = pmf[succ_mask]

        if succ_pmf.sum() <= 0:
            continue

        succ_pmf = succ_pmf / succ_pmf.sum()
        succ_cdf = np.cumsum(succ_pmf)
        median_success_k = succ_k[np.searchsorted(succ_cdf, 0.5)]

        median_success_grid[i, j] = min(median_success_k, 10)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

x_edges = np.arange(N_vals.min() - 0.5, N_vals.max() + 1.5, 1.0)
p_step = P_vals[1] - P_vals[0]
y_edges = np.r_[P_vals - 0.5 * p_step, P_vals[-1] + 0.5 * p_step]

colors = [
    '#2e7d32',  # 1
    '#66bb6a',  # 2
    '#c0ca33',  # 3
    '#fdd835',  # 4
    '#ffb300',  # 5
    '#fb8c00',  # 6
    '#f4511e',  # 7
    '#e53935',  # 8
    '#8e24aa',  # 9
    '#546e7a',  # 10
]
cmap = ListedColormap(colors)
bounds = np.arange(0.5, 10.6, 1.0)
norm = BoundaryNorm(bounds, cmap.N)

mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    median_success_grid,
    cmap=cmap,
    norm=norm,
    shading='auto'
)

cbar = plt.colorbar(mesh, ax=ax, ticks=np.arange(1, 11))
cbar.set_label('Median success count | K≥1')

style_ax(
    ax,
    'N',
    'Release probability p',
    'Median success count conditioned on success'
)

ax.set_xlim(N_vals.min() - 0.5, N_vals.max() + 0.5)
ax.set_ylim(0, 0.5)

plt.tight_layout()
plt.show()

fig.savefig(OUTPUT_DIR / '04_00a_binomial_grid_search_space.pdf', dpi=300, bbox_inches='tight')


In [ ]:
# ##################################################################
# FIG A0 - Panels 1 to 4
# DISPLAY_STIMS:
#   - 'stim1' : pulse 1 only
#   - 'all'   : all 10 pulses
# ##################################################################

import json
import os
import subprocess
import sys
from pathlib import Path

from scipy.signal import savgol_filter
from scipy.stats import norm

panel1_condition = 'Theo_1_5Ca'
panel1_file = '20200909_linescan1_20Hz_10pulses_1.5mMCa_bouton4_traces_converted'
panel1_trial_col = '0'
panel1_trial_input_col_1based = int(panel1_trial_col) + 1

DISPLAY_STIMS = 'stim1'   # 'stim1' or 'all'

def _sanitize_token(value):
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(value))

def _amp_min_from_row(row, k0):
    vals = []
    cc = _amp_corr[k0]
    uc = _amp_uncorr[k0]
    if cc in row.index and np.isfinite(row[cc]):
        vals.append(float(row[cc]))
    if uc in row.index and np.isfinite(row[uc]):
        vals.append(float(row[uc]))
    return float(np.min(vals)) if vals else np.nan

def _amp_uncorr_from_row(row, k0):
    uc = _amp_uncorr[k0]
    if uc in row.index and np.isfinite(row[uc]):
        return float(row[uc])
    cc = _amp_corr[k0]
    if cc in row.index and np.isfinite(row[cc]):
        return float(row[cc])
    return np.nan

REPO_ROOT = Path.cwd()
helper_script = REPO_ROOT / 'Feature_extraction' / 'tmp_selected_recording_nnls_overlay.py'
overlay_npz = REPO_ROOT / (
    f"_tmp_nnls_overlay__{_sanitize_token(panel1_condition)}__"
    f"{_sanitize_token(panel1_file)}.npz"
)

def _overlay_npz_matches(npz_path, condition, file_stem):
    if not npz_path.exists():
        return False
    try:
        d = np.load(npz_path, allow_pickle=True)
        required = {
            'condition', 'file_stem', 'time_s',
            'all_trial_input_col_1based', 'all_trial_yproc', 'all_trial_yhat',
            'all_trial_null_amps_nnls',
            'average_yproc', 'average_yhat',
        }
        ok = (
            required.issubset(set(d.files))
            and str(d['condition']) == str(condition)
            and str(d['file_stem']) == str(file_stem)
        )
        d.close()
        return ok
    except Exception:
        return False

if not _overlay_npz_matches(overlay_npz, panel1_condition, panel1_file):
    env = dict(os.environ)
    env['PYTHONIOENCODING'] = 'utf-8'
    subprocess.run(
        [
            sys.executable,
            str(helper_script),
            '--condition', panel1_condition,
            '--file', f'{panel1_file}.xlsx',
            '--trial', '1',
            '--out', str(overlay_npz),
        ],
        check=True,
        cwd=str(REPO_ROOT),
        env=env,
    )

overlay_data = np.load(overlay_npz, allow_pickle=True)
overlay_time = np.asarray(overlay_data['time_s'], float)
all_trial_input_col_1based = np.asarray(overlay_data['all_trial_input_col_1based'], int)
all_trial_yproc = np.asarray(overlay_data['all_trial_yproc'], float)
all_trial_yhat = np.asarray(overlay_data['all_trial_yhat'], float)
all_trial_null_amps_nnls = np.asarray(overlay_data['all_trial_null_amps_nnls'], dtype=object)
panel1_average_yproc = np.asarray(overlay_data['average_yproc'], float)
panel1_average_yhat = np.asarray(overlay_data['average_yhat'], float)
overlay_data.close()

trial_yproc_map = {
    int(k): np.asarray(v, float)
    for k, v in zip(all_trial_input_col_1based, all_trial_yproc)
}
trial_yhat_map = {
    int(k): np.asarray(v, float)
    for k, v in zip(all_trial_input_col_1based, all_trial_yhat)
}

panel1_trial_yproc = trial_yproc_map[panel1_trial_input_col_1based]
panel1_trial_yhat = trial_yhat_map[panel1_trial_input_col_1based]
panel1_trial_sg = savgol_filter(panel1_trial_yproc, 9, 2, mode='interp')

panel1_path = BASE_DIR / panel1_condition / f'{panel1_file}.xlsx'
panel1_df = pd.read_excel(panel1_path, sheet_name='Traces DF_F0')
panel1_time_raw = panel1_df['Time'].to_numpy(float)
panel1_trace_raw = panel1_df[panel1_trial_col].to_numpy(float)

bouton_trials = (
    trials_for_q[
        (trials_for_q['condition'] == panel1_condition) &
        (trials_for_q['file'] == panel1_file)
    ]
    .copy()
    .sort_values('trial_input_col_1based')
)

trial_row = bouton_trials[
    bouton_trials['trial_input_col_1based'] == panel1_trial_input_col_1based
].iloc[0]

thr_selected = float(trial_row['thr_shared'])
bouton_thr = bouton_trials['thr_shared'].to_numpy(float)
bouton_thr = bouton_thr[np.isfinite(bouton_thr)]
thr_bouton = float(np.nanmedian(bouton_thr)) if bouton_thr.size else np.nan

stim_times = 0.5 + 0.05 * np.arange(N_STIM)
pulse_indices = [0] if str(DISPLAY_STIMS).lower() == 'stim1' else list(range(N_STIM))
title_suffix = 'stim 1 only' if pulse_indices == [0] else 'all stimuli'
pool_label = 'stim 1' if pulse_indices == [0] else 'all events'

# Selected trial exact event classification
selected_fail_mask, _ = get_failure_mask_row(trial_row)
selected_event_fail = np.full(N_STIM, False)
for k in range(N_STIM):
    selected_event_fail[k] = bool(np.isfinite(selected_fail_mask[k]) and (selected_fail_mask[k] > 0))

# All-trial structures
bouton_trial_traces_sg = []
pooled_amp = []
pooled_is_fail = []
success_pct_by_pulse = np.full(N_STIM, np.nan)

all_fail_masks = []
for _, row in bouton_trials.iterrows():
    trial_1based = int(row['trial_input_col_1based'])
    if trial_1based not in trial_yproc_map:
        continue

    trace_proc = trial_yproc_map[trial_1based]
    trace_sg = savgol_filter(trace_proc, 9, 2, mode='interp')
    fail_mask, _ = get_failure_mask_row(row)

    bouton_trial_traces_sg.append(trace_sg)
    all_fail_masks.append(np.asarray(fail_mask, float))

    for k in pulse_indices:
        amp_k = _amp_min_from_row(row, k)
        is_fail_k = bool(np.isfinite(fail_mask[k]) and (fail_mask[k] > 0)) if len(fail_mask) > k else False
        if np.isfinite(amp_k):
            pooled_amp.append(amp_k)
            pooled_is_fail.append(is_fail_k)

bouton_trial_traces_sg = np.asarray(bouton_trial_traces_sg, float)
pooled_amp = np.asarray(pooled_amp, float)
pooled_is_fail = np.asarray(pooled_is_fail, bool)

if len(all_fail_masks):
    all_fail_masks = np.asarray(all_fail_masks, float)
    for k in range(N_STIM):
        valid = np.isfinite(all_fail_masks[:, k])
        if np.any(valid):
            success_pct_by_pulse[k] = 100.0 * np.mean(all_fail_masks[valid, k] == 0)

success_amps = pooled_amp[np.isfinite(pooled_amp) & (~pooled_is_fail)]
failure_amps = pooled_amp[np.isfinite(pooled_amp) & pooled_is_fail]
median_success_amp = float(np.nanmedian(success_amps)) if success_amps.size else np.nan

# Pooled NNLS null from all trials of the same bouton workbook
panel2_null = []
for arr in all_trial_null_amps_nnls:
    arr = np.asarray(arr, float)
    arr = arr[np.isfinite(arr)]
    if arr.size:
        panel2_null.append(arr)
panel2_null = np.concatenate(panel2_null) if panel2_null else np.array([], float)

# Panel 4 source rebuilt directly from trials
if pulse_indices == [0]:
    q_source_df = trials_for_q[
        trials_for_q['condition'].isin(get_calcium_conditions('1.5mM', 'all'))
    ].copy()
    q_source_df = q_source_df[q_source_df['status'].astype(str).str.strip().str.lower() == 'success']
    q_vals = q_source_df['AMP1_UNCORR'].dropna().to_numpy(float)
    q_title = f'Q from 1.5 mM stim 1 successes : n={len(q_vals)}'
    fit_cap = 0.75
else:
    q_vals = []
    q_source_df = trials_for_q[
        trials_for_q['condition'].isin(get_calcium_conditions('1.5mM', 'all'))
    ].copy()

    for _, row in q_source_df.iterrows():
        fail_mask, _ = get_failure_mask_row(row)
        for k in range(N_STIM):
            if k >= len(fail_mask):
                continue
            if not np.isfinite(fail_mask[k]) or fail_mask[k] > 0:
                continue
            amp_k = _amp_uncorr_from_row(row, k)
            if np.isfinite(amp_k) and amp_k > 0:
                q_vals.append(float(amp_k))

    q_vals = np.asarray(q_vals, float)
    q_title = f'Aggregated successful 1.5 mM events : n={len(q_vals)}'
    fit_cap = float(np.nanpercentile(q_vals, 95)) if q_vals.size else np.nan

q_vals = q_vals[np.isfinite(q_vals) & (q_vals > 0)]
q_mean_panel = float(np.nanmean(q_vals)) if q_vals.size else np.nan
q_median_panel = float(np.nanmedian(q_vals)) if q_vals.size else np.nan

fig = plt.figure(figsize=(10.8, 7.8))
gs = fig.add_gridspec(2, 2)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

baseline_kw = dict(color='black', lw=1.3, linestyle=(0, (1.2, 2.0)), alpha=1.0, zorder=20)
thr_kw = dict(color='#c62828', lw=1.5, linestyle=(0, (1.2, 2.0)), alpha=1.0, zorder=21)

# Panel 1: selected trial
ax1.plot(panel1_time_raw, panel1_trace_raw, color='0.78', lw=0.9, alpha=0.95, zorder=1)
ax1.plot(overlay_time, panel1_trial_sg, color='black', lw=1.1, zorder=2)
ax1.plot(overlay_time, panel1_trial_yhat, color='#f28e2b', lw=1.8, zorder=8)
ax1.axvspan(0.4, 0.5, color='0.94', zorder=0)

ymin1 = np.nanmin(np.r_[panel1_trace_raw, panel1_trial_sg, panel1_trial_yhat])
ymax1 = np.nanmax(np.r_[panel1_trace_raw, panel1_trial_sg, panel1_trial_yhat])
ypad1 = 0.08 * (ymax1 - ymin1 if ymax1 > ymin1 else 1.0)
ax1.set_ylim(ymin1 - ypad1, ymax1 + ypad1)
ax1.set_xlim(0.4, 1.0)

tick_base1 = ax1.get_ylim()[0] + 0.04 * (ax1.get_ylim()[1] - ax1.get_ylim()[0])
tick_top1 = ax1.get_ylim()[0] + 0.10 * (ax1.get_ylim()[1] - ax1.get_ylim()[0])
for stim_time in stim_times:
    ax1.plot([stim_time, stim_time], [tick_base1, tick_top1], color='black', lw=1.0, solid_capstyle='butt', zorder=5)

event_pre_s = 0.003
event_post_s = 0.030
for k in pulse_indices:
    stim_time = stim_times[k]
    mask = (overlay_time >= stim_time - event_pre_s) & (overlay_time <= stim_time + event_post_s)
    if not np.any(mask):
        continue
    color = '#c62828' if selected_event_fail[k] else '#2e7d32'
    ax1.plot(overlay_time[mask], panel1_trial_sg[mask], color=color, lw=2.2, zorder=7)

style_ax(ax1, 'Time (s)', 'ΔF/F0', f'Single trial with NNLS fit (trial {panel1_trial_input_col_1based}, {title_suffix})')
ax1.hlines(0.0, xmin=0.4, xmax=1.0, **baseline_kw)
ax1.hlines(thr_selected, xmin=0.4, xmax=1.0, **thr_kw)

# Panel 2: all trials, SavGol only + success percentages
ax2.axvspan(0.4, 0.5, color='0.94', zorder=0)
for trace_sg in bouton_trial_traces_sg:
    ax2.plot(overlay_time, trace_sg, color='0.75', lw=0.8, alpha=0.55, zorder=1)

avg_sg = savgol_filter(panel1_average_yproc, 9, 2, mode='interp')
ax2.plot(overlay_time, avg_sg, color='black', lw=1.3, zorder=6)
ax2.plot(overlay_time, panel1_average_yhat, color='#f28e2b', lw=2.0, zorder=7)

ymin2 = np.nanmin(np.r_[bouton_trial_traces_sg.ravel(), panel1_average_yhat])
ymax2 = np.nanmax(np.r_[bouton_trial_traces_sg.ravel(), panel1_average_yhat])
ypad2 = 0.08 * (ymax2 - ymin2 if ymax2 > ymin2 else 1.0)
ax2.set_ylim(ymin2 - ypad2, ymax2 + ypad2)
ax2.set_xlim(0.4, 1.0)

tick_base2 = ax2.get_ylim()[0] + 0.04 * (ax2.get_ylim()[1] - ax2.get_ylim()[0])
tick_top2 = ax2.get_ylim()[0] + 0.10 * (ax2.get_ylim()[1] - ax2.get_ylim()[0])
for stim_time in stim_times:
    ax2.plot([stim_time, stim_time], [tick_base2, tick_top2], color='black', lw=1.0, solid_capstyle='butt', zorder=5)

for k in pulse_indices:
    if not np.isfinite(success_pct_by_pulse[k]):
        continue
    win = (overlay_time >= stim_times[k] - event_pre_s) & (overlay_time <= stim_times[k] + event_post_s)
    if not np.any(win):
        continue
    y_peak = float(np.nanmax(panel1_average_yhat[win]))
    ax2.annotate(
        f'{success_pct_by_pulse[k]:.0f}%',
        xy=(stim_times[k], y_peak),
        xytext=(0, 6),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=6,
        color='black',
        zorder=9,
        clip_on=False,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=0.2),
    )

style_ax(ax2, 'Time (s)', 'ΔF/F0', f'All trials from same bouton ({title_suffix})')
ax2.hlines(0.0, xmin=0.4, xmax=1.0, **baseline_kw)
if np.isfinite(thr_bouton):
    ax2.hlines(thr_bouton, xmin=0.4, xmax=1.0, **thr_kw)

# Panel 3: pooled amplitudes
hist_vals = np.r_[panel2_null, pooled_amp[np.isfinite(pooled_amp)]]
xmax3 = max(
    np.nanmax(hist_vals) if hist_vals.size else 1.0,
    thr_bouton if np.isfinite(thr_bouton) else 0.0,
    median_success_amp if np.isfinite(median_success_amp) else 0.0,
) * 1.08
bins3 = np.linspace(0.0, xmax3, 24)

if panel2_null.size:
    ax3.hist(panel2_null, bins=bins3, density=True, color='0.75', edgecolor='none', alpha=0.70,
             label=f'NNLS null all trials (n={len(panel2_null)})')
if failure_amps.size:
    ax3.hist(failure_amps, bins=bins3, density=True, color='#c62828', edgecolor='none', alpha=0.45,
             label=f'{pool_label} failures (n={len(failure_amps)})')
if success_amps.size:
    ax3.hist(success_amps, bins=bins3, density=True, color='#2e7d32', edgecolor='none', alpha=0.45,
             label=f'{pool_label} successes (n={len(success_amps)})')

style_ax(ax3, 'Amplitude (ΔF/F0)', 'Density', f'Pooled bouton null and {pool_label} amplitudes')
ax3.set_xlim(-0.01, xmax3)
ax3.vlines(0.0, ymin=0, ymax=ax3.get_ylim()[1], **baseline_kw)
if np.isfinite(thr_bouton):
    ax3.vlines(thr_bouton, ymin=0, ymax=ax3.get_ylim()[1], **thr_kw)
if np.isfinite(median_success_amp):
    ax3.vlines(
        median_success_amp,
        ymin=0,
        ymax=ax3.get_ylim()[1],
        color='#1b5e20',
        lw=1.7,
        linestyle='-',
        alpha=1.0,
        zorder=22,
    )
add_legend(ax3)

# Panel 4: Q distribution rebuilt directly from trials
if q_vals.size:
    ax4.hist(q_vals, bins=15, color='steelblue', edgecolor='none', alpha=0.8, density=True)

fit_data = q_vals[q_vals <= fit_cap] if np.isfinite(fit_cap) else np.array([], float)
if len(fit_data) > 1:
    mu, std = norm.fit(fit_data)
    xf = np.linspace(0, q_vals.max() * 1.2, 200)
    ax4.plot(
        xf,
        norm.pdf(xf, mu, std),
        color='#0b4f8a',
        lw=2.0,
        alpha=1.0,
        zorder=10,
        solid_capstyle='round',
        label=f'Gauss μ={mu:.4f} σ={std:.4f}',
    )

if np.isfinite(q_mean_panel):
    ax4.axvline(q_mean_panel, color='red', lw=1.2, ls='--', label=f'mean={q_mean_panel:.4f}')
if np.isfinite(q_median_panel):
    ax4.axvline(q_median_panel, color='orange', lw=1.2, ls='-.', label=f'med={q_median_panel:.4f}')
if q_vals.size:
    ax4.plot(q_vals, -0.12 * np.ones_like(q_vals), '|k', ms=6, mew=0.8)
    ax4.set_xlim(0, q_vals.max() * 1.2)

style_ax(ax4, None, 'Density', q_title)
ax4.set_xlabel('Quantal size Q (ΔF/F₀)')
ax4.set_ylim(bottom=0)
add_legend(ax4)
if 'style_hist_axis' in globals():
    style_hist_axis(ax4)

# Re-assert the Gaussian style after global styling helpers
for line in ax4.lines:
    if 'Gauss μ=' in str(line.get_label()):
        line.set_color('#0b4f8a')
        line.set_alpha(1.0)
        line.set_linewidth(2.0)
        line.set_zorder(10)
        break


plt.tight_layout()
fig.savefig(OUTPUT_DIR / '04_00b_q_failure_method_technical.pdf', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# ##################################################################
# FIG A1 : Q distribution histogram
# ##################################################################
q_vals = np.array(list(Q_estimates.values()))
fig_a1, ax = make_figure_grid(figsize=(5, 3.5))
ax.hist(q_vals, bins=15, color='steelblue', edgecolor='none', alpha=0.8, density=True)
# Gaussian fit (exclude outliers > 0.75)
fit_data = q_vals[q_vals <= 0.75]
if len(fit_data) > 1:
    mu, std = norm.fit(fit_data)
    xf = np.linspace(0, q_vals.max()*1.2, 200)
    ax.plot(xf, norm.pdf(xf, mu, std), color='darkred', lw=1.5,
            label=f'Gauss μ={mu:.4f} σ={std:.4f}')
ax.axvline(average_Q, color='red',    lw=1.2, ls='--', label=f'mean={average_Q:.4f}')
ax.axvline(median_Q,  color='orange',  lw=1.2, ls='-.', label=f'med={median_Q:.4f}')
ax.plot(q_vals, -0.12*np.ones_like(q_vals), '|k', ms=6, mew=0.8)
ax.set_xlim(0, q_vals.max()*1.2)
style_ax(ax, 'Quantal size Q (ΔF/F₀)', 'Density',
         f'Q from 1.5 mM {"(EB)" if USE_EMPIRICAL_BAYES else "(raw)"} : n={len(q_vals)}')
add_legend(ax)
if 'style_hist_axis' in globals():
    style_hist_axis(ax)
fig_a1.tight_layout()
fig_a1.savefig(OUTPUT_DIR / '04_01_quantal_size_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# %% FIG A2 : Q maps + Q vs PCA axes (1.5 Ca projected, Q matched via extract_base_name)

# Get coordinates + match Q per bouton
low_ca_coords = pca_data['WT_1_5Ca']
low_ca_ids    = PCA_Data_WT_Low_Ca['ID'].astype(str).str.strip().values
q_vals_a2     = np.array([Q_by_ebn.get(extract_base_name(s), np.nan) for s in low_ca_ids])
has_q         = np.isfinite(q_vals_a2)

print(f"[A2] Matched: {has_q.sum()}/{len(low_ca_ids)}")
print(f"     Q range: {np.nanmin(q_vals_a2[has_q]):.4f} : {np.nanmax(q_vals_a2[has_q]):.4f}")

shared_ylim = (-2, 2)

fig_a2, axes_a2 = plt.subplots(1, 3, figsize=(15, 4))
ax0, ax1, ax2 = axes_a2

# Panel 0: PCA map of Q
ok = has_q
if ok.sum() >= 3:
    gx_a2, gy_a2, ext_a2 = setup_pca_grid(low_ca_coords[ok], pad=0.5)
    sg_a2, _ = smooth_field(low_ca_coords[ok], q_vals_a2[ok], np.ones(ok.sum(), dtype=bool), gx_a2, gy_a2)
    ax0.imshow(
        sg_a2,
        extent=ext_a2,
        origin='lower',
        aspect='auto',
        cmap='viridis',
        interpolation='bilinear',
        alpha=0.7,
    )
    plot_pca_value_overlay(
        ax0,
        low_ca_coords[ok],
        np.ones(ok.sum(), dtype=bool),
        q_vals_a2[ok],
        cmap='viridis',
        vmin=np.nanmin(q_vals_a2[ok]),
        vmax=np.nanmax(q_vals_a2[ok]),
        s=18,
        edgecolors='none',
        zorder=3,
    )
    sm = plt.cm.ScalarMappable(
        cmap='viridis',
        norm=plt.Normalize(np.nanmin(q_vals_a2[ok]), np.nanmax(q_vals_a2[ok]))
    )
    plt.colorbar(sm, ax=ax0, shrink=0.8, label='Q')
else:
    ax0.text(0.5, 0.5, 'Too few Q-matched boutons', ha='center', va='center', transform=ax0.transAxes)

style_pca_axes(ax0, title=f'Q map (n={ok.sum()})', legend=False)

# Panels 1-2: Q vs PC1 / PC2
for ax, pc_idx, pc_lab in [(ax1, 0, 'PC1'), (ax2, 1, 'PC2')]:
    x = low_ca_coords[has_q, pc_idx]
    y = q_vals_a2[has_q]

    ax.scatter(x, y, s=20, alpha=0.6, c='steelblue', edgecolors='none')

    if len(x) >= 2:
        z = np.polyfit(x, y, 1)
        xl = np.linspace(x.min(), x.max(), 200)
        ax.plot(xl, np.polyval(z, xl), 'r--', lw=1.2)

    ax.axhline(average_Q, color='orange', ls=':', lw=1, label=f'avg Q={average_Q:.4f}')

    style_ax(ax, pc_lab, 'Q (ΔF/F₀)')
    add_legend(ax)

    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(*shared_ylim)

fig_a2.suptitle(f'Q in PCA space and Q vs PCA axes : 1.5 mM Ca projected (n={has_q.sum()})',
                fontsize=11, fontweight='bold')
fig_a2.tight_layout()
fig_a2.savefig(OUTPUT_DIR / '04_02_quantal_size_projection_on_wt_pca.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# %% ###############################################################
# CELL B : Shared cumulative-release helpers
# WT figure kept here: B3 (20 Hz, 2.5 mM, per cluster)
# Calcium and frequency cumulative-release figures are shown later
# in sections 8 and 9 using the same helper state.
# ###################################################################

N_FIT_LAST = 4

# == Condition groups =============================================
COND_GROUPS_20 = OrderedDict([
    ('1.5 mM', {'conds': get_calcium_conditions('1.5mM', '20Hz'), 'color': get_wt_ca_color('1.5mM')}),
    ('2.5 mM', {'conds': get_calcium_conditions('2.5mM', '20Hz'), 'color': get_wt_ca_color('2.5mM')}),
    ('4.0 mM', {'conds': get_calcium_conditions('4mM', '20Hz'), 'color': get_wt_ca_color('4mM')}),
])
COND_GROUPS_50 = OrderedDict([
    ('1.5 mM', {'conds': get_calcium_conditions('1.5mM', '50Hz'), 'color': get_50hz_ca_color('1.5mM')}),
    ('2.5 mM', {'conds': get_calcium_conditions('2.5mM', '50Hz'), 'color': get_50hz_ca_color('2.5mM')}),
    ('4.0 mM', {'conds': get_calcium_conditions('4mM', '50Hz'), 'color': get_50hz_ca_color('4mM')}),
])

# Get summary DataFrames per condition
for groups in [COND_GROUPS_20, COND_GROUPS_50]:
    for lab, info in groups.items():
        mask = FEATURES_DATAFRAME['Condition'].isin(info['conds'])
        info['df'] = FEATURES_DATAFRAME[mask].copy()


In [ ]:
# ##################################################################
# FIG B3 : SN 20 Hz 2.5 mM per cluster
# ##################################################################
ncl = len(sorted_clusters)
px = np.arange(num_pulses)
fig_b3, ax = make_figure_grid(figsize=(8, 5))
clust_sn = {}

for cid in sorted_clusters:
    mask = PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cid
    df_cl = PCA_Data_WT_Pooled_clustered[mask]
    if len(df_cl) < 2:
        continue
    amps = reconstruct_amps_qnorm(df_cl, Q_estimates, average_Q)
    r = sn_cumulative(amps, N_FIT_LAST)
    clust_sn[cid] = r
    c = cluster_color_lookup.get(cid, 'gray')
    ax.errorbar(px, r['cum_mean'], yerr=r['cum_sem'], fmt='o-',
                color=c, capsize=2, ms=3, lw=1,
                label=f"C{cid} (n={r['n']})")
    ax.plot(r['x_fit'], r['y_fit'], '--', color=c, alpha=0.4, lw=1)

ax.axhline(0, color='gray', lw=0.5, ls=':')
style_ax(ax, 'Stimulus', 'Cumulative release (quanta)', 'SN 20 Hz 2.5 mM : per cluster')
add_legend(ax, loc='upper left')
fig_b3.tight_layout()
fig_b3.savefig(OUTPUT_DIR / '04_05_sn_cumulative_per_wt_cluster.pdf', dpi=300, bbox_inches='tight')
plt.show()

print('\n== SN Summary ==')
print(f"{'Label':>16} {'n':>4} {'RRP':>7} {'P0':>6} {'slope':>8} {'r2':>6}")
for cid in sorted_clusters:
    if cid in clust_sn:
        r = clust_sn[cid]
        print(f"{'C'+str(cid):>16} {r['n']:>4} {r['RRP']:>7.2f} {r['P0']:>6.3f} {r['slope']:>8.3f} {r['r2']:>6.3f}")


In [ ]:
# %% ###############################################################
# CELL C : Smoothing flags + PCA maps of refilling rate
# C1: cluster dots + smoothed refilling slope background
# C2: slope/A1 normalised + isoline at 1.0
# ###################################################################

# ╔##############################################################╗
# ║ SMOOTHING & MAP FLAGS : apply to ALL downstream maps (C:E) ║
# ╚##############################################################â•
SMOOTH_ROBUST       = True     # clip outlier percentiles before smoothing
SMOOTH_SIGMA_FACTOR = 2.0      # sigma = median_NN_distance × this
SMOOTH_CLIP_PCT     = 5        # percentile for clipping (each tail)
GRID_RES            = 80       # heatmap grid resolution

# == Per-bouton SN slope (no rejection : ALL boutons) =============
_ppr_cols = [c for c in [f'PPR{i}/1' for i in range(2, 11)]
             if c in PCA_Data_WT_Pooled_clustered.columns]
_npulses = 1 + len(_ppr_cols)
_df = PCA_Data_WT_Pooled_clustered.copy()
_ids = _df['ID'].astype(str).str.strip().values
_q = np.array([Q_estimates.get(strip_calcium(b), average_Q) for b in _ids])
_q[_q <= 0] = average_Q
_a1 = _df['AMP1'].values / _q
_amps = np.column_stack([_a1] + [_df[c].values * _a1 for c in _ppr_cols])
_nb = len(_amps)
_px_map = np.arange(1, _npulses+1)
_nfit = min(N_FIT_LAST, _npulses)

_slopes = np.full(_nb, np.nan)
for i in range(_nb):
    ar = _amps[i]
    if np.any(~np.isfinite(ar)): continue
    cum = np.cumsum(ar)
    sl, ic, r, _, _ = linregress(_px_map[-_nfit:], cum[-_nfit:])
    if np.isfinite(sl):
        _slopes[i] = sl

_slope_over_a1 = np.where(np.isfinite(_slopes) & (_a1 > 0),
                           _slopes / _a1, np.nan)
_coords = pca_coordinates.copy()
_clust  = cluster_assignments.copy()
gx, gy, ext = setup_pca_grid(_coords)

In [ ]:
# ##################################################################
# FIG C1 : Refilling slope map (raw dots + smoothed background)
# ##################################################################
ok_s = np.isfinite(_slopes)
v1, v2 = np.nanpercentile(_slopes[ok_s], [2, 98])

fig_c1, ax = make_figure_grid(figsize=grid_size(1, 1, 'pca'))
sg, sig = smooth_field(_coords, _slopes, ok_s, gx, gy)
ax.imshow(sg, extent=ext, origin='lower', aspect='auto', cmap=_display_cmap('viridis'),
          vmin=v1, vmax=v2, interpolation='bilinear', zorder=1, alpha=0.8)
# Cluster-coloured dots on top
for cid in sorted_clusters:
    m = (_clust == cid) & ok_s
    c = cluster_color_lookup.get(cid, 'gray')
    plot_pca_overlay_points(ax, _coords[m], c=[c]*m.sum(),
              s=18, edgecolors='none', zorder=3, label=f'C{cid}')
sm = plt.cm.ScalarMappable(cmap=_display_cmap('viridis'), norm=plt.Normalize(v1, v2))
plt.colorbar(sm, ax=ax, shrink=0.7, label='slope (q/stim)')
style_pca_axes(ax, title=f'Refilling slope (σ={sig:.2f}, n={ok_s.sum()})', legend=False)
add_legend(ax, loc='upper right', markerscale=1.5)
fig_c1.tight_layout()
fig_c1.savefig(OUTPUT_DIR / '04_06_refilling_slope_map.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ##################################################################
# FIG C2 : Slope/A1 normalised + isoline at 1.0
# ##################################################################
ok_sa = np.isfinite(_slope_over_a1)
v3, v4 = np.nanpercentile(_slope_over_a1[ok_sa], [2, 98])

fig_c2, ax = make_figure_grid(figsize=grid_size(1, 1, 'pca'))
sg2, sig2 = smooth_field(_coords, _slope_over_a1, ok_sa, gx, gy)
Xg, Yg = np.meshgrid(gx, gy)
im = ax.imshow(sg2, extent=ext, origin='lower', aspect='auto', cmap=_display_cmap('RdBu_r'),
               vmin=v3, vmax=v4, interpolation='bilinear', zorder=1, alpha=0.8)
# Isoline at norm_rate = 1
if np.any(np.isfinite(sg2)):
    ax.contour(Xg, Yg, sg2, levels=[1.0], colors='black', linewidths=1.5, zorder=2)
for cid in sorted_clusters:
    m = (_clust == cid) & ok_sa
    c = cluster_color_lookup.get(cid, 'gray')
    plot_pca_overlay_points(ax, _coords[m], c=[c]*m.sum(),
              s=18, edgecolors='none', zorder=3, label=f'C{cid}')
plt.colorbar(im, ax=ax, shrink=0.7, label='slope / Aâ‚')
style_pca_axes(ax, title=f'Slope/Aâ‚ (σ={sig2:.2f}, n={ok_sa.sum()})', legend=False)
add_legend(ax, loc='upper right', markerscale=1.5)
fig_c2.tight_layout()
fig_c2.savefig(OUTPUT_DIR / '04_07_refilling_slope_norm_map.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# %% ###############################################################
# CELL D : Binomial MLE N/P with explicit failure calls
# D1: N maps at stim 1,2,5,10
# D2: P maps at stim 1,2,5,10
# + cluster trajectories
# ###################################################################

# ╔##############################################################╗
# ║                     CONTROL FLAGS                           ║
# ╠##############################################################╣
USE_BOOTSTRAP = True          # bootstrap for cleaner estimates
N_BOOT        = 200           # number of resamples
N_MAX_D       = 20            # max N in grid search
MIN_TRIALS_D  = 5
STIM_SHOW     = [1, 2, 5, 10]  # stimulus positions to map
# Override colour axis (None = auto from data)
N_VLIM        = (1, 8)       # (vmin, vmax) for N maps
P_VLIM        = (0, 0.8)     # (vmin, vmax) for P maps
# ╚##############################################################â•

Q_D = average_Q

_pca_ids = PCA_Data_WT_Pooled_clustered['ID'].astype(str).str.strip().values

# == Build count dict keyed by extract_base_name ====================
# Event failures are defined per pulse from the lower of corrected /
# uncorrected amplitudes compared against the row-specific thr_shared.
# Non-failure amplitudes are then rounded to quantal counts.
td_D = defaultdict(list)
for _, r in trials_all[trials_all['condition'].isin(conditions_2_5_20Hz)].iterrows():
    td_D[extract_base_name(str(r['file']).strip())].append(get_count_row_with_failures(r, Q_D))

# == Precompute valid count matrices ==============================
_pca_ebn_D = [extract_base_name(s) for s in _pca_ids]
n_pca = len(_pca_ids)
bouton_counts_D = {}
for i in range(n_pca):
    bid = _pca_ebn_D[i]
    if bid not in td_D: continue
    qc = np.array(td_D[bid])
    ok = np.all(np.isfinite(qc), axis=1)
    qc = qc[ok]
    if len(qc) >= MIN_TRIALS_D:
        bouton_counts_D[i] = qc.astype(int)

print(f"Boutons with ≥{MIN_TRIALS_D} trials: {len(bouton_counts_D)}")

# == Fit: point or bootstrap =====================================
D_N = np.full((n_pca, N_STIM), np.nan)
D_P = np.full((n_pca, N_STIM), np.nan)

if USE_BOOTSTRAP:
    rng = np.random.default_rng(42)
    for i, counts in bouton_counts_D.items():
        ntr = len(counts)
        boot_N = np.full((N_STIM, N_BOOT), np.nan)
        boot_P = np.full((N_STIM, N_BOOT), np.nan)
        for b in range(N_BOOT):
            qc_b = counts[rng.integers(0, ntr, ntr)]
            for k in range(N_STIM):
                qc = qc_b[:, k].astype(int)
                Nf, Pf, _ = fit_binom_mle(qc, N_MAX_D)
                if np.isfinite(Nf) and Nf < N_MAX_D:
                    boot_N[k, b] = Nf; boot_P[k, b] = Pf
        for k in range(N_STIM):
            if np.isfinite(boot_N[k]).sum() >= N_BOOT*0.5:
                D_N[i, k] = np.nanmedian(boot_N[k])
                D_P[i, k] = np.nanmedian(boot_P[k])
else:
    for i, counts in bouton_counts_D.items():
        for k in range(N_STIM):
            qc = counts[:, k].astype(int)
            Nf, Pf, _ = fit_binom_mle(qc, N_MAX_D)
            if np.isfinite(Nf) and Nf < N_MAX_D:
                D_N[i, k] = Nf; D_P[i, k] = Pf

ok_D = np.isfinite(D_N[:, 0])
print(f"Valid (stim 1): {ok_D.sum()} | N med={np.nanmedian(D_N[ok_D,0]):.1f} | P med={np.nanmedian(D_P[ok_D,0]):.3f}")
print(f"MLE failure rule: per event, min(corrected, uncorrected) < {THR_SHARED_COL_MLE} -> count 0")

In [ ]:
# ##################################################################
# FIG D1/D2 : N and P maps at selected stimuli
# ##################################################################
_coords_D = pca_coordinates.copy()
_clust_D  = cluster_assignments.copy()
gx_D, gy_D, ext_D = setup_pca_grid(_coords_D)

n_show = len(STIM_SHOW)
fig_d, axes_d = make_figure_grid(2, n_show, panel_kind='pca')

for col, sk in enumerate(STIM_SHOW):
    k = sk - 1
    # == N map ==
    vals = D_N[:, k]; ok = np.isfinite(vals); ax = axes_d[0, col]
    nv, nxv = N_VLIM if N_VLIM else (np.nanpercentile(vals[ok], 2), np.nanpercentile(vals[ok], 98))
    n_ok = render_pca_scalar_panel(
        ax,
        _coords_D,
        vals,
        cmap='Spectral_r',
        vmin=nv,
        vmax=nxv,
        title=f'N : stim {sk} (n={ok.sum()})',
        point_size=14,
        min_points=3,
        empty_label=f'n={ok.sum()} (too few)',
    )
    if col == n_show-1:
        add_scalar_colorbar(fig_d, axes_d[0, :], cmap='Spectral_r', vmin=nv, vmax=nxv, label='N', shrink=0.7)

    # == P map ==
    vals = D_P[:, k]; ok = np.isfinite(vals); ax = axes_d[1, col]
    pv, pxv = P_VLIM if P_VLIM else (0, 0.8)
    p_ok = render_pca_scalar_panel(
        ax,
        _coords_D,
        vals,
        cmap='coolwarm',
        vmin=pv,
        vmax=pxv,
        title=f'P : stim {sk} (n={ok.sum()})',
        point_size=14,
        min_points=3,
        empty_label=f'n={ok.sum()} (too few)',
    )
    if col == n_show-1:
        add_scalar_colorbar(fig_d, axes_d[1, :], cmap='coolwarm', vmin=pv, vmax=pxv, label='P', shrink=0.7)

meth = 'bootstrap MLE (explicit failures)' if USE_BOOTSTRAP else 'point MLE (explicit failures)'
finalize_figure(
    fig_d,
    title=f'Binomial {meth} : N and P maps',
    rect=[0, 0, 0.95, 0.95],
    save_path=OUTPUT_DIR / '04_08_np_maps_selected_stimuli.pdf',
)

# == Cluster trajectories ========================================
x_stim = np.arange(1, N_STIM+1)
fig_dt, axes_dt = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
ax_p, ax_n = axes_dt[0, 0], axes_dt[0, 1]
for cid in sorted_clusters:
    idx = np.where(_clust_D == cid)[0]
    c = cluster_color_lookup.get(cid, 'gray')
    for vals, ax_t, yl in [(D_P, ax_p, '$P_k$'), (D_N, ax_n, '$N_k$')]:
        vv = vals[idx]
        plot_mean_sem_trace(ax_t, x_stim, vv, color=c, label=f'C{cid} '+'({n})', marker='o', linestyle='-', ms=3, lw=1, fill_alpha=0.12)
ax_p.set_ylim(0, 1)
style_ax(ax_p, 'Stimulus', '$P_k$', 'P trajectory'); add_legend(ax_p)
style_ax(ax_n, 'Stimulus', '$N_k$', 'N trajectory'); add_legend(ax_n)
finalize_figure(
    fig_dt,
    title=f'Binomial {meth} : per cluster',
    save_path=OUTPUT_DIR / '04_09_np_condition_trajectories.pdf',
)

# Print table
print(f"\n{'Cl':>4} {'n':>4} {'P1':>6} {'P2':>6} {'P5':>6} {'P10':>6} "
      f"{'N1':>6} {'N2':>6} {'N5':>6} {'N10':>6}")
for cid in sorted_clusters:
    idx = np.where(_clust_D == cid)[0]
    ok = np.isfinite(D_N[idx, 0])
    if ok.sum() == 0: continue
    ii = idx[ok]
    vals = [np.nanmean(D_P[ii, k-1]) for k in STIM_SHOW] + \
           [np.nanmean(D_N[ii, k-1]) for k in STIM_SHOW]
    print(f"C{cid:>3} {ok.sum():>4} " + " ".join(f"{v:>6.3f}" if v<2 else f"{v:>6.1f}" for v in vals))


## 5. Diversity Along Individual Parallel Fibers

The following analyses address whether heterogeneous bouton phenotypes coexist along individual WT fibers. Fiber-level diversity is quantified first at the population level and then illustrated with representative axons.

This part of the notebook supports the central claim that diversity is expressed at the bouton level along the same axon, not only across unrelated fibers. Both the population summary and the worked examples are therefore kept together.


In [ ]:
# Extract fiber IDs from WT pooled data (precomputed FiberID when available)
if 'FiberID' not in PCA_Data_WT_Pooled.columns:
    PCA_Data_WT_Pooled['FiberID'] = PCA_Data_WT_Pooled['ID'].map(_extract_fiber_id)

fiber_ids = PCA_Data_WT_Pooled['FiberID'].dropna().unique()

# Filter fibers with more than 2 boutons
fiber_bouton_counts = PCA_Data_WT_Pooled.groupby('FiberID').size().to_dict()
eligible_fibers = [fiber for fiber, count in fiber_bouton_counts.items() if count > 2]

print(f"Fibers with >2 boutons: {len(eligible_fibers)} out of {len(fiber_ids)} total fibers")

# Calculate grid dimensions for all eligible fibers
n_fibers = len(eligible_fibers)
n_cols = 3
n_rows = int(np.ceil(n_fibers / n_cols))

# Create figure with subplots for all eligible fibers
fig, axes = make_figure_grid(n_rows, n_cols, figsize=(18, 6 * n_rows))
axes = axes.flatten()

# Color palette for fibers
set1_cmap = plt.get_cmap('Set1')

# Plot each eligible fiber in its own subplot
for idx, fiber_prefix in enumerate(eligible_fibers):
    ax = axes[idx]
    
    # Plot WT pooled with cluster colors (background) - lighter
    plot_pca_background(ax, alpha=0.15, s=20, label=None)
    
    # Find boutons belonging to this fiber
    fiber_mask = PCA_Data_WT_Pooled['FiberID'] == fiber_prefix
    fiber_indices = np.where(fiber_mask)[0]
    
    if len(fiber_indices) > 0:
        # Use consistent color for this fiber
        fiber_color = set1_cmap(idx % 9)  # Cycle through Set1 colors
        
        # Plot all boutons from this fiber with the same color
        plot_pca_overlay_points(ax, pca_coordinates[fiber_indices],
                   color=fiber_color,  marker='o', 
                   edgecolors='black', linewidths=1.5, alpha=0.9,
                   label=f'{fiber_prefix}\n(n={len(fiber_indices)} boutons)')
    
    # Format subplot
    style_pca_axes(ax, title=f'Fiber: {fiber_prefix}',  legend=False)
    add_legend(ax, loc='upper right', fontsize=8, frameon=False)

# Hide unused subplots
for idx in range(n_fibers, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle(f'PCA: All Fibers from WT Pooled with >2 boutons ({n_fibers} fibers)', 
             fontsize=16, fontweight='bold', y=1.002)
plt.show()

# Print fiber statistics
print(f"\n=== ALL FIBERS STATISTICS (>2 boutons) ===")
print(f"Total unique fibers in WT pooled: {len(fiber_ids)}")
print(f"Fibers with >2 boutons: {len(eligible_fibers)}")
print("-" * 60)
for fiber_prefix in eligible_fibers:
    fiber_mask = PCA_Data_WT_Pooled['FiberID'] == fiber_prefix
    n_boutons = fiber_mask.sum()
    
    # Get cluster distribution for this fiber
    fiber_clusters = cluster_assignments[fiber_mask]
    cluster_counts = {i: np.sum(fiber_clusters == i) for i in range(1, N_CLUSTERS + 1) if np.sum(fiber_clusters == i) > 0}
    
    print(f"  {fiber_prefix}: {n_boutons} boutons")
    print('    ' + '  '.join(f"C{int(k)}={int(v)}" for k, v in pd.Series(cluster_counts).sort_index().items()))

# Fiber area distribution
from scipy.spatial import ConvexHull
from scipy.stats import mannwhitneyu
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

# Analyze PCA area coverage for individual fibers with 4+ boutons


def calculate_convex_hull_area(points):
    """Calculate area of convex hull for a set of 2D points."""
    if len(points) < 3:
        return 0.0
    try:
        hull = ConvexHull(points)
        return hull.volume  # In 2D, volume is area
    except Exception:
        return 0.0

# Extract fiber IDs from WT pooled data (use precomputed FiberID)
fiber_ids = PCA_Data_WT_Pooled['FiberID'].dropna().unique()

# Filter fibers with 4+ boutons
fiber_bouton_counts = PCA_Data_WT_Pooled['FiberID'].value_counts().to_dict()
eligible_fibers = {fiber: count for fiber, count in fiber_bouton_counts.items() if count >= 4}

print(f"=== FIBER AREA ANALYSIS ===")
print(f"Total unique fibers in WT pooled: {len(fiber_ids)}")
print(f"Fibers with 4+ boutons: {len(eligible_fibers)}")

# Calculate real fiber areas
real_fiber_areas = []
fiber_size_distribution = {}  # Track how many boutons per fiber

for fiber_prefix, n_boutons in eligible_fibers.items():
    fiber_mask = PCA_Data_WT_Pooled['FiberID'] == fiber_prefix
    fiber_indices = np.where(fiber_mask)[0]
    fiber_coords = pca_coordinates[fiber_indices]
    
    area = calculate_convex_hull_area(fiber_coords)
    real_fiber_areas.append(area)
    
    # Track size distribution
    if n_boutons not in fiber_size_distribution:
        fiber_size_distribution[n_boutons] = 0
    fiber_size_distribution[n_boutons] += 1

print(f"\nFiber size distribution:")
for size in sorted(fiber_size_distribution.keys()):
    count = fiber_size_distribution[size]
    print(f"  {size} boutons: {count} fibers ({100*count/len(eligible_fibers):.1f}%)")

# Generate null distribution by random sampling
n_permutations = 10000
null_areas = []

print(f"\nGenerating null distribution with {n_permutations} permutations...")

# Get all WT pooled coordinates
all_wt_coords = pca_coordinates.copy()
n_total_boutons = len(all_wt_coords)

def generate_null_areas_for_permutation(args):
    """Generate null areas for a single permutation."""
    fiber_size_distribution, all_wt_coords, seed = args
    np.random.seed(seed)
    n_total_boutons = len(all_wt_coords)
    perm_areas = []
    
    for fiber_size, n_fibers in fiber_size_distribution.items():
        for _ in range(n_fibers):
            random_indices = np.random.choice(n_total_boutons, size=fiber_size, replace=False)
            random_coords = all_wt_coords[random_indices]
            area = calculate_convex_hull_area(random_coords)
            perm_areas.append(area)
    
    return perm_areas

# Prepare arguments for parallel execution
seeds = np.random.randint(0, 2**31, size=n_permutations)
args_list = [(fiber_size_distribution, all_wt_coords, seed) for seed in seeds]

# Use multiprocessing to parallelize
# Use threading to parallelize (ProcessPoolExecutor fails in notebooks due to pickling issues)
n_workers = min(multiprocessing.cpu_count(), 8)
print(f"Using {n_workers} workers for parallel computation...")

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    results = list(executor.map(generate_null_areas_for_permutation, args_list))
for perm_areas in results:
    null_areas.extend(perm_areas)

print(f"  Completed {n_permutations} permutations")

# Statistical comparison
real_areas_array = np.array(real_fiber_areas)
null_areas_array = np.array(null_areas)

# Mann-Whitney U test
u_stat, p_value = mannwhitneyu(real_areas_array, null_areas_array, alternative='two-sided')

# Calculate summary statistics
real_median = np.median(real_areas_array)
real_mean = np.mean(real_areas_array)
null_median = np.median(null_areas_array)
null_mean = np.mean(null_areas_array)

print(f"\n=== STATISTICAL RESULTS ===")
print(f"Real fibers (n={len(real_areas_array)}):")
print(f"  Mean area: {real_mean:.3f}")
print(f"  Median area: {real_median:.3f}")
print(f"  Range: [{np.min(real_areas_array):.3f}, {np.max(real_areas_array):.3f}]")

print(f"\nNull distribution (n={len(null_areas_array)}):")
print(f"  Mean area: {null_mean:.3f}")
print(f"  Median area: {null_median:.3f}")
print(f"  Range: [{np.min(null_areas_array):.3f}, {np.max(null_areas_array):.3f}]")

print(f"\nMann-Whitney U test:")
print(f"  U-statistic: {u_stat:.2f}")
print(f"  p-value: {p_value:.6g}")
print(f"  Effect: Real fibers {'smaller' if real_median < null_median else 'larger'} than random")

# Visualization: side-by-side histograms
fig, (ax1, ax2) = make_figure_grid(1, 2, figsize=(14, 5))

# Determine common bins for fair comparison
all_areas = np.concatenate([real_areas_array, null_areas_array])
common_bins = np.linspace(0, np.percentile(all_areas, 99), 30)

# Left panel: Real fiber areas
ax1.hist(real_areas_array, bins=common_bins, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(real_median, color='red', linestyle='--', linewidth=2, label=f'Median: {real_median:.3f}')
ax1.set_xlabel('Convex Hull Area (PCA units²)')
ax1.set_ylabel('Frequency')
ax1.set_title(f'Real Fibers (n={len(real_areas_array)})')
add_legend(ax1, frameon=False)
if 'style_hist_axis' in globals():
    style_hist_axis(ax1)
ax1.grid(False)

# Right panel: Null distribution
ax2.hist(null_areas_array, bins=common_bins, alpha=0.7, color='lightcoral', edgecolor='black')
ax2.axvline(null_median, color='darkred', linestyle='--', linewidth=2, label=f'Median: {null_median:.3f}')
ax2.axvline(real_median, color='steelblue', linestyle=':', linewidth=2, label=f'Real median (ref)')
ax2.set_xlabel('Convex Hull Area (PCA units²)')
ax2.set_ylabel('Frequency')
ax2.set_title(f'Null Distribution (n={len(null_areas_array)})')
add_legend(ax2, frameon=False)
if 'style_hist_axis' in globals():
    style_hist_axis(ax2)
ax2.grid(False)

plt.suptitle(f'Fiber Area Distribution: Real vs Random\nMann-Whitney p={p_value:.6g}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

output_file = OUTPUT_DIR / "05_01_fiber_area_real_vs_null.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Additional visualization: Overlay with transparency
make_figure(figsize=(10, 6))

plt.hist(null_areas_array, bins=common_bins, alpha=0.5, color='lightcoral', 
         edgecolor='black', label=f'Null (n={len(null_areas_array)})', density=True)
plt.hist(real_areas_array, bins=common_bins, alpha=0.7, color='steelblue', 
         edgecolor='black', label=f'Real fibers (n={len(real_areas_array)})', density=True)

plt.axvline(null_median, color='darkred', linestyle='--', linewidth=2, 
            label=f'Null median: {null_median:.3f}')
plt.axvline(real_median, color='darkblue', linestyle='--', linewidth=2, 
            label=f'Real median: {real_median:.3f}')

plt.xlabel('Convex Hull Area (PCA units²)')
plt.ylabel('Probability Density')
plt.title(f'Fiber Area Distribution Comparison\nMann-Whitney U={u_stat:.0f}, p={p_value:.6g}')
add_legend(plt.gca(), frameon=False)
if 'style_hist_axis' in globals():
    style_hist_axis(plt.gca())
plt.grid(False)
plt.tight_layout()

output_file_overlay = OUTPUT_DIR / "05_02_fiber_area_overlay_comparison.pdf"
plt.savefig(output_file_overlay, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved area distribution plots:")
print(f"  {output_file}")
print(f"  {output_file_overlay}")

# Save detailed results to text file
stats_output = OUTPUT_DIR / "fiber_area_statistics.txt"
with open(stats_output, 'w') as f:
    f.write("FIBER AREA ANALYSIS: REAL vs NULL DISTRIBUTION\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Dataset: WT pooled (fibers with 4+ boutons)\n")
    f.write(f"Total fibers analyzed: {len(eligible_fibers)}\n\n")
    
    f.write("FIBER SIZE DISTRIBUTION:\n")
    for size in sorted(fiber_size_distribution.keys()):
        count = fiber_size_distribution[size]
        f.write(f"  {size} boutons: {count} fibers ({100*count/len(eligible_fibers):.1f}%)\n")
    
    f.write(f"\nREAL FIBER AREAS (n={len(real_areas_array)}):\n")
    f.write(f"  Mean:   {real_mean:.6f}\n")
    f.write(f"  Median: {real_median:.6f}\n")
    f.write(f"  SD:     {np.std(real_areas_array):.6f}\n")
    f.write(f"  Min:    {np.min(real_areas_array):.6f}\n")
    f.write(f"  Max:    {np.max(real_areas_array):.6f}\n")
    
    f.write(f"\nNULL DISTRIBUTION (n={len(null_areas_array)}):\n")
    f.write(f"  Permutations: {n_permutations}\n")
    f.write(f"  Mean:   {null_mean:.6f}\n")
    f.write(f"  Median: {null_median:.6f}\n")
    f.write(f"  SD:     {np.std(null_areas_array):.6f}\n")
    f.write(f"  Min:    {np.min(null_areas_array):.6f}\n")
    f.write(f"  Max:    {np.max(null_areas_array):.6f}\n")
    
    f.write(f"\nSTATISTICAL TEST:\n")
    f.write(f"  Test: Mann-Whitney U (two-sided)\n")
    f.write(f"  U-statistic: {u_stat:.6f}\n")
    f.write(f"  p-value: {p_value:.10f}\n")
    f.write(f"  Effect: Real fibers are {'smaller' if real_median < null_median else 'larger'} than random\n")
    f.write(f"  Median difference: {real_median - null_median:.6f}\n")

print(f"✓ Saved detailed statistics to {stats_output}")


### 5.1 Fiber-Level Diversity

Fiber-level convex-hull and class-count analyses test whether boutons from the same PF occupy restricted subregions or sample several WT bouton classes. The observed mixture of classes within single fibers argues against a purely fiber-specific organization of short-term plasticity.


In [ ]:
# Analyze cluster diversity within individual fibers (boutons from same axon)

def extract_fiber_id(bouton_id):
    """Extract fiber ID from bouton ID for both WT linescan and SynII Fibre naming."""
    if '_extract_fiber_id' in globals():
        return _extract_fiber_id(bouton_id)
    s = str(bouton_id).strip().replace('_traces_converted', '')
    m = re.match(r'^(.*?)(?:[_\s]*Bouton[_\s]*\d+.*|[_\s]*bouton\d+.*)$', s, flags=re.IGNORECASE)
    return m.group(1).rstrip('_ ') if m else s

def analyze_fiber_diversity(min_boutons_per_fiber=4):
    """Analyze how clusters are distributed within individual fibers."""

    fiber_data = PCA_Data_WT_Pooled_clustered.copy()
    if 'FiberID' not in fiber_data.columns:
        fiber_data['FiberID'] = fiber_data['ID'].apply(extract_fiber_id)
    fiber_data['Fiber_ID'] = fiber_data['FiberID']

    fiber_bouton_counts = fiber_data.groupby('Fiber_ID').size()

    valid_fibers  = fiber_bouton_counts[fiber_bouton_counts >= min_boutons_per_fiber].index
    filtered_data = fiber_data[fiber_data['Fiber_ID'].isin(valid_fibers)]

    print(f"Fiber analysis: {len(valid_fibers)} fibers with {min_boutons_per_fiber}+ boutons")
    print(f"Total boutons analyzed: {len(filtered_data)}")

    return filtered_data, valid_fibers

def calculate_cluster_diversity(filtered_data):
    """Calculate number of different clusters per fiber."""
    fiber_diversity = {}

    for fiber_id in filtered_data['Fiber_ID'].unique():
        fiber_boutons        = filtered_data[filtered_data['Fiber_ID'] == fiber_id]
        unique_clusters      = fiber_boutons['HC_Cluster'].nunique()
        total_boutons        = len(fiber_boutons)
        cluster_distribution = fiber_boutons['HC_Cluster'].value_counts(normalize=True)

        fiber_diversity[fiber_id] = {
            'num_cluster_types': unique_clusters,
            'total_boutons': total_boutons,
            'cluster_props': cluster_distribution.to_dict()
        }

    return fiber_diversity

filtered_data, valid_fibers = analyze_fiber_diversity(min_boutons_per_fiber=4)
fiber_diversity             = calculate_cluster_diversity(filtered_data)

diversity_categories = {'1': [], '2': [], '3': [], '4+': []}
for fiber_id, info in fiber_diversity.items():
    num_types = info['num_cluster_types']
    category  = str(num_types) if num_types <= 3 else '4+'
    diversity_categories[category].append(info)

diversity_means  = {}
diversity_counts = {}
diversity_order  = ['1', '2', '3', '4+']

for category, fiber_list in diversity_categories.items():
    diversity_counts[category] = len(fiber_list)
    if len(fiber_list) > 0:
        cluster_means = {}
        for cluster_id in range(1, N_CLUSTERS + 1):
            proportions               = [fiber['cluster_props'].get(cluster_id, 0) for fiber in fiber_list]
            cluster_means[cluster_id] = np.mean(proportions)
        diversity_means[category] = cluster_means
    else:
        diversity_means[category] = {i: 0 for i in range(1, N_CLUSTERS + 1)}

fig, ax = make_figure_grid(figsize=(10, 6))

x_positions   = np.arange(len(diversity_order))
bottom_values = np.zeros(len(diversity_order))
total_fibers  = len(valid_fibers)

for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_heights = []
    for category in diversity_order:
        fiber_percentage   = (diversity_counts[category] / total_fibers) * 100 if total_fibers else 0.0
        cluster_proportion = diversity_means[category][cluster_id]
        height             = fiber_percentage * cluster_proportion
        cluster_heights.append(height)

    ax.bar(
        x_positions,
        cluster_heights,
        bottom=bottom_values,
        color=get_cluster_color(cluster_id),
        label=f'C{cluster_id}',
        alpha=0.8,
    )
    bottom_values += cluster_heights

ax.set_xlabel('Number of cluster types per fiber')
ax.set_ylabel('Percentage of fibers (%)')
ax.set_title('Fiber cluster diversity (fibers with ≥4 boutons)')
ax.set_xticks(x_positions)
ax.set_xticklabels(diversity_order)
add_legend(ax, title='Clusters', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(False)
ax.set_ylim(0, 105)

plt.tight_layout()

output_file = OUTPUT_DIR / "05_03_fiber_cluster_diversity_summary.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved to {output_file}")

print('\nFiber cluster diversity summary:')
print(f"{'Types/fiber':<12} {'n fibers':>8} {'% fibers':>8} {'Mean cluster composition':<60}")
for category in diversity_order:
    count = diversity_counts[category]
    pct = (100.0 * count / total_fibers) if total_fibers else np.nan
    comp = '  '.join(
        f"C{cluster_id}={100 * diversity_means[category][cluster_id]:.1f}%"
        for cluster_id in range(1, N_CLUSTERS + 1)
        if diversity_means[category][cluster_id] > 0
    )
    if not comp:
        comp = '-'
    print(f"{category:<12} {count:>8} {pct:>7.1f}%  {comp}")

print(f"\nTotal fibers: {total_fibers}")
print(f"Total boutons: {len(filtered_data)}")


### 5.2 Same-Fiber Control Context

The examples below emphasize that heterogeneous bouton phenotypes coexist within single fibers rather than being restricted to separate axons. This point is central for interpreting diversity as bouton-specific rather than fiber-specific.


### 5.3 Representative Fiber Dynamics

The representative-fiber panels translate the population-level diversity result back into the raw traces and bouton-wise PPR profiles. They show directly that neighboring boutons on one axon can display distinct glutamate-release dynamics during the same stimulation train.


In [ ]:
# Build trace lookup from normalized traces
if 'build_trace_lookup' in globals():
    resampled_trace_lookup = build_trace_lookup_from_source(TRACE_SINGLE_SOURCE)
else:
    resampled_trace_lookup = {}
    for _, trace_row in NORM_TRACES_DATAFRAME.iterrows():
        resampled_trace_lookup[trace_row['ID']] = {'Time': trace_row['Time'], 'Avg': trace_row['Avg']}

from scipy.signal import savgol_filter


def analyze_single_fiber(fiber_prefix, window_length=9, poly_order=2):
    """Analyze all boutons from a specific fiber and return shifted raw/smoothed traces."""
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(fiber_prefix)]

    if len(fiber_boutons) == 0:
        print(f"No boutons found with prefix '{fiber_prefix}'")
        return None

    print(f"Found {len(fiber_boutons)} boutons from fiber '{fiber_prefix}'")

    fiber_trace_rows = select_traces(trace_ids=fiber_boutons['ID'], condition_names=WT_2_5_20HZ_CONDITIONS, source=TRACE_SINGLE_SOURCE)
    fiber_trace_by_base = {
        str(trace_row['BaseID']): trace_row
        for _, trace_row in fiber_trace_rows.iterrows()
    }

    fiber_traces = {}
    for _, bouton in fiber_boutons.iterrows():
        bouton_id = str(bouton['ID'])
        bouton_base = _normalize_bouton_id(bouton_id)
        trace_row = fiber_trace_by_base.get(str(bouton_base))
        if trace_row is None:
            continue

        trace_time = np.asarray(trace_row['Time'], dtype=float)
        if TRACE_SINGLE_SOURCE != 'normalized' and trace_row['Condition'] in EXCEPTIONAL_CONDITIONS:
            trace_time = trace_time + EXCEPTIONAL_BASELINE_OFFSET

        fiber_traces[bouton_id] = {
            'time': trace_time,
            'raw': np.asarray(trace_row['Avg'], dtype=float),
            'cluster': bouton['HC_Cluster'],
            'condition': trace_row['Condition'],
        }

    if not fiber_traces:
        print(f"No trace data found for fiber '{fiber_prefix}'")
        return None

    for bouton_id in fiber_traces:
        raw_trace = np.asarray(fiber_traces[bouton_id]['raw'], dtype=float)

        if np.all(np.isnan(raw_trace)):
            fiber_traces[bouton_id]['smoothed'] = raw_trace.copy()
            continue

        trace_for_smoothing = raw_trace.copy()
        nan_mask = np.isnan(trace_for_smoothing)

        if np.any(nan_mask):
            valid_indices = np.where(~nan_mask)[0]
            valid_values = trace_for_smoothing[~nan_mask]
            if len(valid_indices) < 3:
                fiber_traces[bouton_id]['smoothed'] = raw_trace.copy()
                continue
            nan_indices = np.where(nan_mask)[0]
            trace_for_smoothing[nan_mask] = np.interp(nan_indices, valid_indices, valid_values)

        win_len = min(window_length, len(trace_for_smoothing))
        if win_len % 2 == 0:
            win_len -= 1
        if win_len < 3:
            smoothed_trace = trace_for_smoothing.copy()
        else:
            smoothed_trace = savgol_filter(
                trace_for_smoothing,
                window_length=win_len,
                polyorder=min(poly_order, win_len - 1),
                mode='interp',
            )

        if np.any(nan_mask):
            smoothed_trace[nan_mask] = np.nan

        fiber_traces[bouton_id]['smoothed'] = smoothed_trace

    return fiber_traces


def plot_fiber_smoothed_traces_and_ppr(fiber_boutons, fiber_traces, fiber_prefix, figure_tag='05_06'):
    """Plot smoothed traces and bouton-wise colored PPR profiles side by side."""
    if 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'finalize_ppr_axis' not in globals():
        raise RuntimeError('Run the shared PPR helper cell first (cell defining ppr_profile_stats).')
    if fiber_traces is None or len(fiber_boutons) == 0:
        raise ValueError(f"No fiber data available for '{fiber_prefix}'")

    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in fiber_boutons.columns]
    if not ppr_cols:
        raise ValueError('No PPR columns found for this fiber selection')

    ordered_boutons = fiber_boutons.copy()
    n_boutons = len(ordered_boutons)
    fig, axes = make_figure_grid(n_boutons, 2, figsize=(12, max(4.0, 2.6 * n_boutons)), sharex='col')
    axes = np.asarray(axes)
    if n_boutons == 1:
        axes = axes.reshape(1, 2)

    all_trace_vals = []
    for bouton_id in ordered_boutons['ID']:
        if bouton_id not in fiber_traces:
            raise ValueError(f"Missing trace data for bouton '{bouton_id}'")
        trace_vals = np.asarray(fiber_traces[bouton_id]['smoothed'], dtype=float)
        all_trace_vals.extend(trace_vals[np.isfinite(trace_vals)])

    if not all_trace_vals:
        raise ValueError(f"No valid smoothed trace values found for '{fiber_prefix}'")

    y_min, y_max = np.min(all_trace_vals), np.max(all_trace_vals)
    y_padding = max((y_max - y_min) * 0.08, 0.05)
    y_lims = (y_min - y_padding, y_max + y_padding)
    y_tick = y_lims[1] - 0.06 * (y_lims[1] - y_lims[0])
    stim_times = [1.0 + i * 0.05 for i in range(10)]

    all_ppr_vals = [1.0]
    for _, bouton in ordered_boutons.iterrows():
        for col in ppr_cols:
            if pd.notna(bouton[col]):
                all_ppr_vals.append(float(bouton[col]))
    ppr_y_max = max(1.2, np.nanmax(all_ppr_vals) * 1.1)

    for idx, (_, bouton) in enumerate(ordered_boutons.iterrows()):
        bouton_id = str(bouton['ID'])
        data = fiber_traces[bouton_id]
        cluster_id = data['cluster']
        color = get_cluster_color(cluster_id)

        ax_trace = axes[idx, 0]
        ax_trace.plot(data['time'], data['smoothed'], color=color, linewidth=1.5)
        ax_trace.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_trace.axvline(1.0, color='red', linestyle='--', alpha=0.5)
        for stim_t in stim_times:
            ax_trace.plot(stim_t, y_tick, marker='|', color='black', markersize=7, markeredgewidth=1.4)
        ax_trace.set_xlim(*TRACE_XLIM_20HZ)
        ax_trace.set_ylim(y_lims)
        ax_trace.set_ylabel('ΔF/F')
        ax_trace.set_title(f'{bouton_id} (Cluster {cluster_id})', fontsize=9, loc='left')
        if 'style_trace_axis' in globals():
            style_trace_axis(ax_trace)

        ax_ppr = axes[idx, 1]
        pulse_numbers, means, sems, _ = ppr_profile_stats(bouton.to_frame().T, ppr_column_names=ppr_cols)
        plot_ppr_mean_sem(
            ax_ppr,
            pulse_numbers,
            means,
            sems,
            color=color,
            marker='o',
            label=f'Cluster {cluster_id}',
            linewidth=2,
            markersize=5,
            sem_alpha=0.2,
        )
        finalize_ppr_axis(
            ax_ppr,
            pulse_numbers,
            ylabel='PPR (A_n/A_1)',
            ylim=(0, ppr_y_max),
            unity_kwargs={'color': 'gray', 'linestyle': '--', 'linewidth': 1},
            legend=False,
        )
        ax_ppr.set_title('PPR profile', fontsize=9, loc='left')
        if 'style_trace_axis' in globals():
            style_trace_axis(ax_ppr)
        if ax_ppr.get_legend() is not None:
            ax_ppr.get_legend().remove()

    axes[-1, 0].set_xlabel('Time (s)')
    finalize_ppr_axis(axes[-1, 1], None, xlabel='Pulse Number', ylabel=None, unity_line=False, legend=False)

    fig.subplots_adjust(top=0.97, hspace=0.45, wspace=0.28)
    plt.suptitle(f'Fiber {fiber_prefix} (n={n_boutons} boutons)', fontsize=12, fontweight='bold', y=0.995)

    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"{figure_tag}_fiber_{safe_prefix}_smoothed_traces_ppr.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    return output_file


### 5.4 Representative Fiber Example

Set `EXAMPLE_FIBER_PREFIX` at the top of the next cell to inspect any representative WT fiber. The figure combines smoothed single-bouton traces on the left with the corresponding colored PPR profiles on the right.


In [ ]:
# Combined representative fiber example

if 'analyze_single_fiber' not in globals() or 'plot_fiber_smoothed_traces_and_ppr' not in globals():
    raise RuntimeError('Run the shared fiber helper cell first (the cell defining analyze_single_fiber).')

EXAMPLE_FIBER_PREFIX = '20220425_linescan1'

fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(EXAMPLE_FIBER_PREFIX)].copy()
fiber_traces = analyze_single_fiber(EXAMPLE_FIBER_PREFIX)

if fiber_traces is None:
    available_prefixes = PCA_Data_WT_Pooled_clustered['ID'].str[:25].value_counts()
    print(f"No analysis possible for fiber '{EXAMPLE_FIBER_PREFIX}'")
    print('\nAvailable fiber prefixes (showing top 10):')
    for prefix, count in available_prefixes.head(10).items():
        if count >= 3:
            print(f"  '{prefix}': {count} boutons")
else:
    output_file = plot_fiber_smoothed_traces_and_ppr(
        fiber_boutons=fiber_boutons,
        fiber_traces=fiber_traces,
        fiber_prefix=EXAMPLE_FIBER_PREFIX,
        figure_tag='05_06',
    )

    print(f"✓ Plotted {len(fiber_boutons)} boutons from fiber '{EXAMPLE_FIBER_PREFIX}'")
    print(f'✓ Saved to {output_file}')

    cluster_distribution = fiber_boutons['HC_Cluster'].value_counts().sort_index()
    print('\nFiber cluster composition:')
    for cluster_id, count in cluster_distribution.items():
        print(f'  Cluster {cluster_id}: {count} boutons')


### 5.5 Switching Fibers

To inspect another representative axon, change `EXAMPLE_FIBER_PREFIX` in the previous cell and rerun it. The old duplicate example cell was removed so this section stays single-source.


In [ ]:
# Representative fiber plotting is controlled by EXAMPLE_FIBER_PREFIX in the previous cell.


## 6. Postsynaptic Target Identity

These cells test whether bouton diversity can be reduced to the identity of the postsynaptic target. WT PCA classes are kept fixed while PC- and interneuron-associated boutons are projected onto the same reference space.

This section addresses the manuscript result that target identity can influence apparent release probability or failures, yet does not explain the full diversity of STP profiles along single fibers.


### 6.1 Project PC and Interneuron Boutons into WT PCA Space

Target-identified boutons are overlaid onto the WT reference PCA to test whether they segregate into distinct regions of state space. The analysis is intentionally performed as an overlay rather than a re-clustering step so that the WT classes remain the reference axis of interpretation.


In [ ]:
# Scatter plot: WT pooled points with PC (dark green) and IN (purple) overlays
# Assumes pca_coordinates (np.ndarray), PCA_Data_WT_Pooled (DataFrame) and PCA_RESULTS are available

make_figure(figsize=(9, 7))

# Plot original PCA
plot_pca_background(plt.gca(), alpha=0.4,  marker='o', linewidths=0.6, label='WT pooled')


# Target masks
targets = PCA_Data_WT_Pooled['Target'].values
pc_mask = targets == 'PC'
in_mask = targets == 'IN'

# Calculate medians for PC and IN groups
pc_coordinates = pca_coordinates[pc_mask]
in_coordinates = pca_coordinates[in_mask]

pc_median = np.median(pc_coordinates, axis=0) if len(pc_coordinates) > 0 else None
in_median = np.median(in_coordinates, axis=0) if len(in_coordinates) > 0 else None

# Calculate distance between medians
if pc_median is not None and in_median is not None:
    median_distance = np.linalg.norm(pc_median - in_median)
else:
    median_distance = None

# Plot Purkinje Cells (PC)
if np.any(pc_mask):
    plot_pca_overlay_points(plt.gca(), pca_coordinates[pc_mask],
                c=PC_TARGET_COLOR, marker='o',  
                edgecolors='black', linewidths=0.6,
                label='PC')

# Plot Interneurons (IN)
if np.any(in_mask):
    plot_pca_overlay_points(plt.gca(), pca_coordinates[in_mask],
                c=IN_TARGET_COLOR, marker='^',  
                edgecolors='black', linewidths=0.6,
                label='IN')
    
# Optional visual cue for group medians (kept out of legend/text for minimalism)
if pc_median is not None and in_median is not None:
    plot_pca_overlay_points(plt.gca(), np.array([pc_median]), c=PC_TARGET_MEDIAN_COLOR, marker='X',
               label='_nolegend_', edgecolors='black', linewidth=1.2, s=35, alpha=0.9)
    plot_pca_overlay_points(plt.gca(), np.array([in_median]), c=IN_TARGET_MEDIAN_COLOR, marker='X',
               label='_nolegend_', edgecolors='black', linewidth=1.2, s=35, alpha=0.9)
    plt.plot([pc_median[0], in_median[0]], [pc_median[1], in_median[1]],
             color='black', linestyle='--', linewidth=1.0, alpha=0.5,
             label='_nolegend_')


# Axis labels with explained variance
style_pca_axes(plt.gca(), title='PCA: WT pooled colored by Target (PC vs IN)',  legend=False)
add_legend(plt.gca(), loc='best', fontsize=9, frameon=False)
plt.tight_layout()

# Save and show
output_path = OUTPUT_DIR / "06_01_target_overlay_pca_scatter.pdf"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved PCA PC/IN scatter to {output_path}")

# Summary statistics
print("=== PC vs IN SEPARATION ANALYSIS ===")
print(f"Purkinje Cells (PC): {np.sum(pc_mask):2d} samples")
print(f"Interneurons (IN):   {np.sum(in_mask):2d} samples")




if pc_median is not None and in_median is not None:
    print(f"PC median coordinates:     ({pc_median[0]:.3f}, {pc_median[1]:.3f})")
    print(f"IN median coordinates:     ({in_median[0]:.3f}, {in_median[1]:.3f})")
    print(f"Median-to-median distance: {median_distance:.3f}")
else:
    print("Cannot calculate median distance - insufficient data")


In [ ]:
# Paired within-fiber PC vs IN comparison on WT PCA coordinates
from scipy.stats import wilcoxon

paired_target_df = PCA_Data_WT_Pooled.copy().reset_index(drop=True)
paired_target_xy = np.asarray(pca_coordinates, float)
n_pair = min(len(paired_target_df), len(paired_target_xy))
paired_target_df = paired_target_df.iloc[:n_pair].copy()
paired_target_xy = paired_target_xy[:n_pair, :2]
paired_target_df['PC1'] = paired_target_xy[:, 0]
paired_target_df['PC2'] = paired_target_xy[:, 1]
if 'FiberID' not in paired_target_df.columns:
    paired_target_df['FiberID'] = paired_target_df['ID'].map(_extract_fiber_id)
paired_target_df['Target'] = paired_target_df['Target'].astype(str).str.strip().str.upper()
paired_target_df = paired_target_df[paired_target_df['Target'].isin(['PC', 'IN'])].copy()

fiber_target_summary = (
    paired_target_df.groupby(['FiberID', 'Target'])[['PC1', 'PC2']]
    .mean()
    .reset_index()
)
fiber_target_counts = paired_target_df.groupby(['FiberID', 'Target']).size().unstack(fill_value=0)
paired_fibers = fiber_target_counts[(fiber_target_counts.get('PC', 0) > 0) & (fiber_target_counts.get('IN', 0) > 0)].index.tolist()
paired_mean_wide = (
    fiber_target_summary[fiber_target_summary['FiberID'].isin(paired_fibers)]
    .pivot(index='FiberID', columns='Target', values=['PC1', 'PC2'])
    .sort_index()
)
paired_counts_wide = fiber_target_counts.loc[paired_fibers].sort_index() if len(paired_fibers) else fiber_target_counts.iloc[:0].copy()

def _paired_target_stat(pc_vals, in_vals):
    mask = np.isfinite(pc_vals) & np.isfinite(in_vals)
    pc_vals = np.asarray(pc_vals)[mask]
    in_vals = np.asarray(in_vals)[mask]
    if len(pc_vals) == 0:
        return dict(n=0, stat=np.nan, p=np.nan)
    stat, p = wilcoxon(pc_vals, in_vals, alternative='two-sided')
    return dict(n=len(pc_vals), stat=stat, p=p)

if len(paired_fibers) == 0:
    print('No fibers contain both PC and IN boutons; paired within-fiber comparison skipped.')
else:
    pc1_pc = paired_mean_wide[('PC1', 'PC')].to_numpy(float)
    pc1_in = paired_mean_wide[('PC1', 'IN')].to_numpy(float)
    pc2_pc = paired_mean_wide[('PC2', 'PC')].to_numpy(float)
    pc2_in = paired_mean_wide[('PC2', 'IN')].to_numpy(float)

    stat_pc1 = _paired_target_stat(pc1_pc, pc1_in)
    stat_pc2 = _paired_target_stat(pc2_pc, pc2_in)

    fig, axes = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
    ax_pc1, ax_pc2 = axes[0, 0], axes[0, 1]

    def _plot_paired_axis(ax, vals_pc, vals_in, ylabel, title, stat_res):
        bp = ax.boxplot([vals_pc, vals_in], positions=[0, 1], widths=0.55, patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], [PC_TARGET_COLOR, IN_TARGET_COLOR]):
            patch.set_facecolor(color)
            patch.set_alpha(0.40)
        for a, b in zip(vals_pc, vals_in):
            if np.isfinite(a) and np.isfinite(b):
                ax.plot([0, 1], [a, b], color='0.55', lw=0.8, alpha=0.7, zorder=1)
        ax.scatter(np.random.uniform(-0.06, 0.06, len(vals_pc)), vals_pc, c=PC_TARGET_COLOR, s=28, alpha=0.85, edgecolors='black', linewidths=0.3, zorder=3)
        ax.scatter(1 + np.random.uniform(-0.06, 0.06, len(vals_in)), vals_in, c=IN_TARGET_COLOR, s=28, alpha=0.85, edgecolors='black', linewidths=0.3, zorder=3)
        y_all = np.r_[vals_pc, vals_in]
        y_all = y_all[np.isfinite(y_all)]
        if len(y_all):
            y_top = np.nanmax(y_all)
            y_bot = np.nanmin(y_all)
            y_pad = 0.08 * (y_top - y_bot) if y_top > y_bot else 0.15
            ax.plot([0, 1], [y_top + y_pad, y_top + y_pad], color='black', lw=1)
            ax.text(0.5, y_top + 1.15 * y_pad, f'p={stat_res["p"]:.2g}', ha='center', va='bottom', fontsize=8)
            ax.set_ylim(y_bot - 0.12 * max(y_top - y_bot, 1.0), y_top + 0.28 * max(y_top - y_bot, 1.0))
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['PC', 'IN'])
        style_ax(ax, 'Target', ylabel, title)

    _plot_paired_axis(ax_pc1, pc1_pc, pc1_in, 'Mean within-fiber PC1', 'Within-fiber paired target mean: PC1', stat_pc1)
    _plot_paired_axis(ax_pc2, pc2_pc, pc2_in, 'Mean within-fiber PC2', 'Within-fiber paired target mean: PC2', stat_pc2)

    output_path = OUTPUT_DIR / '06_01b_target_within_fiber_paired_pca_means.pdf'
    finalize_figure(fig, title=f'Within-fiber paired PC vs IN means ({len(paired_fibers)} fibers)', save_path=output_path)

    print('=== WITHIN-FIBER PAIRED PC vs IN PCA MEANS ===')
    print(f'Paired fibers: {len(paired_fibers)}')
    print(f'PC1: PC median={np.nanmedian(pc1_pc):.3f} | IN median={np.nanmedian(pc1_in):.3f} | Wilcoxon p={stat_pc1["p"]:.4g}')
    print(f'PC2: PC median={np.nanmedian(pc2_pc):.3f} | IN median={np.nanmedian(pc2_in):.3f} | Wilcoxon p={stat_pc2["p"]:.4g}')
    paired_print = pd.DataFrame({
        'FiberID': paired_fibers,
        'n_PC': paired_counts_wide['PC'].to_numpy(int),
        'n_IN': paired_counts_wide['IN'].to_numpy(int),
        'PC1_PC': pc1_pc,
        'PC1_IN': pc1_in,
        'PC2_PC': pc2_pc,
        'PC2_IN': pc2_in,
    })
    print(paired_print.to_string(index=False))
    print(f'✓ Saved paired within-fiber PC/IN PCA means to {output_path}')


### 6.2 Target-Aligned Trace Summaries

These traces compare boutons that fall inside or outside the target-defined regions of the WT space. The goal is to connect the anatomical classification to the original glutamate transients rather than only to PCA coordinates.


### 6.3 Target Composition Across WT Clusters

Cluster composition is summarized here to test whether particular WT bouton classes are enriched at one target type. In the manuscript logic, a weak or partial enrichment supports target modulation of some parameters without making target identity the primary determinant of WT class identity.


In [ ]:
# Compare cluster distributions between Purkinje Cells, Interneurons, and overall WT
target_cell_types = PCA_Data_WT_Pooled_clustered['Target'].values
pc_mask = target_cell_types == 'PC'
in_mask = target_cell_types == 'IN'

# Get cluster assignments for each target type
pc_cluster_assignments = cluster_assignments[pc_mask]
in_cluster_assignments = cluster_assignments[in_mask]
wt_cluster_assignments = cluster_assignments  # All WT pooled

# Count clusters for each group
pc_cluster_counts = pd.Series(pc_cluster_assignments).value_counts().sort_index()
in_cluster_counts = pd.Series(in_cluster_assignments).value_counts().sort_index()
wt_cluster_counts = pd.Series(wt_cluster_assignments).value_counts().sort_index()

# Ensure all clusters represented
all_clusters = sorted(range(1, N_CLUSTERS + 1))
pc_complete = pd.Series([pc_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
in_complete = pd.Series([in_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
wt_complete = pd.Series([wt_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
pc_percentages = 100 * pc_complete / len(pc_cluster_assignments)
in_percentages = 100 * in_complete / len(in_cluster_assignments)
wt_percentages = 100 * wt_complete / len(wt_cluster_assignments)

# Stacked bar plot with 3 bars
fig, ax = make_figure_grid(figsize=(10, 6))
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

bottom_pc = bottom_in = bottom_wt = 0
for cluster_id in all_clusters:
    color = get_cluster_color(cluster_id)
    pc_pct = pc_percentages.iloc[cluster_id - 1]
    in_pct = in_percentages.iloc[cluster_id - 1]
    wt_pct = wt_percentages.iloc[cluster_id - 1]

    # PC bar
    ax.bar(0, pc_pct, bar_width, bottom=bottom_pc, color=color,
           label=f'Cluster {cluster_id}' if cluster_id == 1 else None)
    bottom_pc += pc_pct

    # IN bar
    ax.bar(1, in_pct, bar_width, bottom=bottom_in, color=color)
    bottom_in += in_pct
    
    # WT bar
    ax.bar(2, wt_pct, bar_width, bottom=bottom_wt, color=color)
    bottom_wt += wt_pct

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution: PC vs IN vs WT Pooled')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Purkinje Cells', 'Interneurons', 'WT Pooled'])
ax.set_ylim(0, 110)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
add_legend(ax, handles, [f'Cluster {c}' for c in all_clusters],
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
output_file = OUTPUT_DIR / "06_03_target_cluster_distribution.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster distribution to {output_file}")

if '_format_cluster_count_summary' not in globals():
    def _format_cluster_count_summary(label, counts):
        items = [f"C{int(k)}={int(v)}" for k, v in pd.Series(counts).sort_index().items()]
        joined = '  '.join(items) if items else '-'
        print(f"{label}: {joined}")

print("Target cluster counts:")
_format_cluster_count_summary('PC', pc_cluster_counts)
_format_cluster_count_summary('IN', in_cluster_counts)
_format_cluster_count_summary('WT', wt_cluster_counts)
print(f"n: PC={len(pc_cluster_assignments)}  IN={len(in_cluster_assignments)}  WT={len(wt_cluster_assignments)}")


## 7. Additional WT Descriptive Analyses

These WT-only summaries are kept after the main mechanistic and anatomical sections because they contextualize the reference dataset without redefining the central PCA and clustering results. They are useful for figure assembly, supplementary interpretation, and cohort-level quality control.


### 7.1 WT Population PPR Profiles

Population-average PPR trajectories summarize how the WT reference dataset behaves before class-specific decomposition. They provide a compact benchmark for later comparisons with perturbations and alternative cohorts.


In [ ]:
# Plot mean PPR profile for WT_pooled with individual profiles in background

if 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'ppr_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining ppr_profile_stats).')

wt_profiles = ppr_profiles_matrix(PCA_Data_WT_Pooled)

fig, ax = make_figure_grid(figsize=(10, 6))

# Individual overlays via shared helper (mean hidden for this pass)
plot_ppr_profiles_overlay(
    ax,
    wt_profiles,
    color='gray',
    label='_nolegend_',
    individual_alpha=0.05,
    individual_lw=3.0,
    mean_lw=0.0,
    sem_alpha=0.0,
)

# Mean ± SEM via shared helper
pulse_numbers, ppr_means, ppr_sems, n_wt = ppr_profile_stats(PCA_Data_WT_Pooled)
plot_ppr_mean_sem(
    ax,
    pulse_numbers,
    ppr_means,
    ppr_sems,
    color='black',
    marker='o',
    label=f'WT pooled mean (n={n_wt})',
    linewidth=2.5,
    markersize=6,
    sem_alpha=0.25,
)

finalize_ppr_axis(
    ax,
    pulse_numbers,
    title='PPR Profile: WT Pooled (Individual + Mean)',
    ylim=(0, 3.5),
    unity_kwargs={'color': 'gray', 'linestyle': '--', 'linewidth': 3, 'label': 'No facilitation'},
    legend=True,
    legend_loc='best',
)
plt.tight_layout()

output_file = OUTPUT_DIR / '07_01_wt_ppr_profile_individuals.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Saved to {output_file}')
print(f'✓ Plotted {n_wt} individual profiles with mean ± SEM')


### 7.2 Sex Distribution Control in WT Data

Sex is examined here as a descriptive control to verify that the WT reference structure is not trivially explained by cohort composition.


In [ ]:
# # PCA colored by sex (M=blue, F=red), excluding UN, with pie chart

# fig, axes = make_figure_grid(1, 2, figsize=(14, 6))

# # Get sex data from WT pooled
# sex_data = PCA_Data_WT_Pooled['Sexe'].values

# # Create masks for M and F (exclude UN)
# male_mask = sex_data == 'M'
# female_mask = sex_data == 'F'

# # Left panel: PCA scatter plot
# ax_pca = axes[0]

# # Plot males in blue
# if np.any(male_mask):
#     ax_pca.scatter(pca_coordinates[male_mask, 0], pca_coordinates[male_mask, 1],
#                c='blue',  marker='o', edgecolors='black', linewidths=0.6,
#                alpha=0.7, label=f'Male (n={np.sum(male_mask)})')

# # Plot females in red
# if np.any(female_mask):
#     ax_pca.scatter(pca_coordinates[female_mask, 0], pca_coordinates[female_mask, 1],
#                c='red',  marker='o', edgecolors='black', linewidths=0.6,
#                alpha=0.7, label=f'Female (n={np.sum(female_mask)})')

# # Format PCA plot
# pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
# ax_pca.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
# ax_pca.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
# ax_pca.set_xlim(-7, 10)
# ax_pca.set_ylim(-6, 6)
# ax_pca.set_title('PCA: WT Pooled by Sex (M vs F)')
# add_legend(ax_pca, loc='upper right')
# ax_pca.grid(True, alpha=0.3)

# # Right panel: Pie chart
# ax_pie = axes[1]
# n_male = np.sum(male_mask)
# n_female = np.sum(female_mask)
# n_un = np.sum(sex_data == 'UN')

# sizes = [n_male, n_female]
# labels = [f'Male\n(n={n_male})', f'Female\n(n={n_female})']
# colors = ['blue', 'red']
# explode = (0.02, 0.02)

# ax_pie.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
#            shadow=False, startangle=90, textprops={'fontsize': 11})
# ax_pie.set_title(f'Sex Distribution\n(Excluded UN: n={n_un})')

# plt.tight_layout()

# output_file = OUTPUT_DIR / "pca_wt_pooled_by_sex.pdf"
# plt.savefig(output_file, dpi=300, bbox_inches='tight')
# plt.show()

# print(f"✓ Saved to {output_file}")
# print(f"Male: {n_male}, Female: {n_female}, Excluded (UN): {n_un}")

## 8. Extracellular Calcium Perturbations

Changes in extracellular calcium are used to probe how release strength, apparent failures, and train dynamics shift relative to the WT reference organization. These analyses provide the main physiological perturbation of the WT bouton state space.

In the manuscript, the calcium manipulations are interpreted as a way to test whether bouton trajectories follow only a release-probability axis or whether they also recruit changes consistent with altered release-site occupancy and refilling.


### 8.1 Calcium Modulation in WT PCA Space

Low- and high-calcium boutons are projected onto the WT reference PCA to visualize how each bouton moves in the state space when release drive is changed. This is the geometric summary of the calcium manipulation. 


In [ ]:
# Overlay all 20Hz PPR profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Use a shared y-axis max across all three plots
# Note: WT_pooled is the 20Hz / 2.5mM reference in this notebook

if 'ppr_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (the cell defining ppr_profiles_matrix).')

profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_WT_Low_Ca, get_wt_ca_color('1.5mM'), '08_01a_calcium_20hz_ppr_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_WT_Pooled, get_wt_ca_color('2.5mM'), '08_01b_calcium_20hz_ppr_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_WT_High_Ca, get_wt_ca_color('4mM'), '08_01c_calcium_20hz_ppr_profiles_4p0mM.pdf'),
]

prepared_profiles = []
global_ymax = 0

for title, df_cond, color, save_name in profiles_by_condition:
    profiles = ppr_profiles_matrix(df_cond)

    if profiles.size == 0:
        prepared_profiles.append((title, profiles, color, save_name))
        continue

    mean_profile = np.nanmean(profiles, axis=0)
    sem_profile = np.nanstd(profiles, axis=0) / np.sqrt(profiles.shape[0])
    cond_ymax = np.nanmax(np.vstack([profiles, mean_profile + sem_profile]))
    global_ymax = max(global_ymax, cond_ymax)

    prepared_profiles.append((title, profiles, color, save_name))

global_ylim = (0, global_ymax * 1.05 if global_ymax > 0 else 1)

for title, profiles, color, save_name in prepared_profiles:
    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles,
        color=color,
        label='Mean',
        individual_alpha=0.25,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid PPR profiles')
        plt.close(fig)
        continue

    pulse_numbers, _, _ = result
    finalize_ppr_axis(
        ax,
        pulse_numbers,
        title=(f'20Hz PPR Profiles - {title}\n' f'(n={profiles.shape[0]} boutons)'),
        ylabel='PPR (A_n/A_1)',
        ylim=global_ylim,
        unity_kwargs={'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1.5},
        legend=True,
        legend_loc='best',
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')


In [ ]:
# Overlay all 20Hz failure-rate profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Explicit failures are defined per event from min(AMP_CORR, AMP_UNCORR) < thr_shared.

if 'failure_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared failure/PPR helper cell first (the cell defining failure_profiles_matrix).')

failure_profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_WT_Low_Ca, get_wt_ca_color('1.5mM'), '08_01d_calcium_20hz_failure_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_WT_Pooled, get_wt_ca_color('2.5mM'), '08_01e_calcium_20hz_failure_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_WT_High_Ca, get_wt_ca_color('4mM'), '08_01f_calcium_20hz_failure_profiles_4p0mM.pdf'),
]

for title, df_cond, color, save_name in failure_profiles_by_condition:
    profiles = failure_profiles_matrix(df_cond, trials_all, min_trials=5)
    n_valid = int(np.sum(np.isfinite(profiles).any(axis=1))) if profiles.size else 0

    if profiles.size == 0 or n_valid == 0:
        print(f'Skipping {title}: no valid failure-rate profiles')
        continue

    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles[np.isfinite(profiles).any(axis=1)],
        color=color,
        label='Mean',
        individual_alpha=0.20,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid failure-rate profiles')
        plt.close(fig)
        continue

    pulse_numbers, mean_profile, sem_profile = result
    finalize_ppr_axis(
        ax,
        pulse_numbers,
        title=f'20Hz Failure Profiles - {title} (n={n_valid} boutons)',
        ylabel='Failure rate',
        ylim=(0, 1),
        unity_line=False,
        legend=True,
        legend_loc='best',
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')
    print(
        f"{title}: n={n_valid} | "
        f"Fail1={mean_profile[0]:.3f}±{sem_profile[0]:.3f} | "
        f"Fail2={mean_profile[1]:.3f}±{sem_profile[1]:.3f} | "
        f"Fail10={mean_profile[-1]:.3f}±{sem_profile[-1]:.3f}"
    )


In [ ]:
# Analyze calcium concentration effects on bouton properties in PCA space

def plot_calcium_trajectories():
    """Plot how calcium concentration changes affect PCA positioning."""
    
    make_figure(figsize=(10, 8))
    
    # Background: WT pooled (2.5mM Ca standard condition)
    plot_pca_background(plt.gca(), alpha=0.4,  label='WT pooled (2.5mM Ca)')
    
    # Low calcium (1.5mM) - blue triangles pointing down
    low_ca_coords = pca_data['WT_1_5Ca']
    plot_pca_overlay_points(plt.gca(), low_ca_coords[:], 
               marker='v',  c=get_wt_ca_color('1.5mM'), alpha=0.8, edgecolors='black', linewidth=0.5,
               label=f'1.5mM Ca (n={len(low_ca_coords)})')
    
    # High calcium (4mM) - red triangles pointing up  
    high_ca_coords = pca_data['WT_4Ca']
    plot_pca_overlay_points(plt.gca(), high_ca_coords[:], 
               marker='^',  c=get_wt_ca_color('4mM'), alpha=0.8, edgecolors='black', linewidth=0.5,
               label=f'4mM Ca (n={len(high_ca_coords)})')
    
    # Calculate centroids
    center_pooled = np.mean(pca_coordinates, axis=0)
    center_low_ca = np.mean(low_ca_coords, axis=0)
    center_high_ca = np.mean(high_ca_coords, axis=0)
    
    # Plot centroids
    plot_pca_overlay_points(plt.gca(), np.array([center_low_ca]), marker='X',  c=get_wt_ca_color('1.5mM'), 
               edgecolor='black', linewidth=2, label='1.5mM centroid')
    plot_pca_overlay_points(plt.gca(), np.array([center_high_ca]), marker='X',  c=get_wt_ca_color('4mM'), 
               edgecolor='black', linewidth=2, label='4mM centroid')
    
    # Draw single arrow from low calcium to high calcium centroid
    plt.arrow(center_low_ca[0], center_low_ca[1],
              center_high_ca[0] - center_low_ca[0], center_high_ca[1] - center_low_ca[1],
              color='purple', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.9)
        
    # Connect paired boutons between conditions (assuming matched order)
    n_pairs = min(len(low_ca_coords), len(high_ca_coords))
    if n_pairs > 0:
        for i in range(n_pairs):
            plt.plot([low_ca_coords[i, 0], high_ca_coords[i, 0]],
                     [low_ca_coords[i, 1], high_ca_coords[i, 1]],
                     color='gray', alpha=0.4, linewidth=1)
        print(f"Connected {n_pairs} bouton pairs between calcium conditions")
    
    # Format plot
    style_pca_axes(plt.gca(), title='Calcium Concentration Effects on Bouton Properties',  legend=False)
    add_legend(plt.gca(), frameon=False)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "08_02_calcium_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_low_ca, center_high_ca, n_pairs

def calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs):
    """Calculate movement statistics for calcium concentration changes."""
    
    # Distances from standard condition to each calcium level
    dist_to_low  = np.linalg.norm(center_low_ca - center_pooled)
    dist_to_high = np.linalg.norm(center_high_ca - center_pooled)
    
    # Individual bouton movements
    low_ca_coords  = pca_data['WT_1_5Ca']
    high_ca_coords = pca_data['WT_4Ca']
    
    # Movement from standard to low calcium
    movements_to_low  = []
    n_low_comparisons = min(len(pca_coordinates), len(low_ca_coords))
    for i in range(n_low_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - low_ca_coords[i])
        movements_to_low.append(dist)
    
    # Movement from standard to high calcium
    movements_to_high  = []
    n_high_comparisons = min(len(pca_coordinates), len(high_ca_coords))
    for i in range(n_high_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - high_ca_coords[i])
        movements_to_high.append(dist)
    
    # Movement between calcium conditions (paired boutons)
    calcium_range_movements = []
    for i in range(n_pairs):
        dist = np.linalg.norm(low_ca_coords[i] - high_ca_coords[i])
        calcium_range_movements.append(dist)
    
    return {
        'centroid_distances': {'low': dist_to_low, 'high': dist_to_high},
        'individual_movements': {
            'to_low': movements_to_low,
            'to_high': movements_to_high,
            'between_ca': calcium_range_movements
        }
    }

# Run analysis
center_pooled, center_low_ca, center_high_ca, n_pairs = plot_calcium_trajectories()
movement_stats = calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs)

# Display results
print(f"\n=== CALCIUM CONCENTRATION ANALYSIS ===")
print(f"Centroid coordinates:")
print(f"  WT pooled (2.5mM): ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  1.5mM Ca:          ({center_low_ca[0]:.3f}, {center_low_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['low']:.3f}")
print(f"  4mM Ca:            ({center_high_ca[0]:.3f}, {center_high_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['high']:.3f}")

print(f"\nIndividual bouton movements in PCA space:")
if movement_stats['individual_movements']['to_low']:
    low_moves = movement_stats['individual_movements']['to_low']
    print(f"2.5mM → 1.5mM Ca (n={len(low_moves)}): {np.mean(low_moves):.3f} ± {np.std(low_moves):.3f}")

if movement_stats['individual_movements']['to_high']:
    high_moves = movement_stats['individual_movements']['to_high']
    print(f"2.5mM → 4mM Ca (n={len(high_moves)}): {np.mean(high_moves):.3f} ± {np.std(high_moves):.3f}")

if movement_stats['individual_movements']['between_ca']:
    range_moves = movement_stats['individual_movements']['between_ca']
    print(f"1.5mM ↔ 4mM Ca (n={len(range_moves)}): {np.mean(range_moves):.3f} ± {np.std(range_moves):.3f}")

print(f"\n✓ Calcium trajectory analysis complete")


### 8.2 Calcium Dependence of the First Response

The first transient amplitude is examined directly because it provides the most immediate readout of initial synaptic weight at the start of the train. Its change with Ca²⁺ is later interpreted jointly with failures and PPR.


In [ ]:
# Compare AMP1 distributions between calcium concentrations
AMP1_HIST_BIN_WIDTH = 0.1


def plot_calcium_amp1_comparison(bin_width=AMP1_HIST_BIN_WIDTH):
    """Compare AMP1 distributions between 1.5mM, 2.5mM and 4.0mM calcium."""
    
    # Get AMP1 data for all three conditions
    amp1_standard = PCA_Data_WT_Pooled['AMP1'].dropna()
    amp1_low_ca   = PCA_Data_WT_Low_Ca['AMP1'].dropna()
    amp1_high_ca  = PCA_Data_WT_High_Ca['AMP1'].dropna()
    
    # Calculate common bins for fair comparison
    all_amp1_values = pd.concat([amp1_standard, amp1_low_ca, amp1_high_ca])
    amp1_min = float(all_amp1_values.min())
    amp1_max = float(all_amp1_values.max())
    bin_edges = np.arange(amp1_min, amp1_max + bin_width, bin_width, dtype=float)
    if bin_edges.size < 2:
        bin_edges = np.array([amp1_min - bin_width / 2.0, amp1_max + bin_width / 2.0], dtype=float)
    elif bin_edges[-1] < amp1_max:
        bin_edges = np.append(bin_edges, bin_edges[-1] + bin_width)
    
    # Create figure
    make_figure(figsize=(10, 6))
    
    # Calculate weights for percentage display
    weights_standard = np.ones(len(amp1_standard)) * (100.0 / len(amp1_standard))
    weights_low_ca   = np.ones(len(amp1_low_ca)) * (100.0 / len(amp1_low_ca))
    weights_high_ca  = np.ones(len(amp1_high_ca)) * (100.0 / len(amp1_high_ca))
    
    # Plot histograms
    plt.hist(amp1_standard, bins=bin_edges, alpha=0.4, color=get_wt_ca_color('2.5mM'), 
             weights=weights_standard, edgecolor='black', linewidth=0.8,
             label=f'WT 2.5mM Ca (n={len(amp1_standard)})')
    plt.hist(amp1_low_ca, bins=bin_edges, alpha=0.4, color=get_wt_ca_color('1.5mM'), 
             weights=weights_low_ca, edgecolor='black', linewidth=0.8,
             label=f'WT 1.5mM Ca (n={len(amp1_low_ca)})')
    plt.hist(amp1_high_ca, bins=bin_edges, alpha=0.35, color=get_wt_ca_color('4mM'), 
             weights=weights_high_ca, edgecolor='black', linewidth=0.8,
             label=f'WT 4.0mM Ca (n={len(amp1_high_ca)})')
    
    # Add vertical lines for means
    mean_standard = amp1_standard.mean()
    mean_low_ca = amp1_low_ca.mean()
    mean_high_ca = amp1_high_ca.mean()
    
    plt.axvline(mean_standard, color=get_wt_ca_color('2.5mM'), linestyle='--', linewidth=1.8, alpha=0.9,
                label=f'Mean 2.5mM: {mean_standard:.3f}')
    plt.axvline(mean_low_ca, color=get_wt_ca_color('1.5mM'), linestyle='--', linewidth=1.8, alpha=0.9,
                label=f'Mean 1.5mM: {mean_low_ca:.3f}')
    plt.axvline(mean_high_ca, color=get_wt_ca_color('4mM'), linestyle='--', linewidth=1.8, alpha=0.9,
                label=f'Mean 4.0mM: {mean_high_ca:.3f}')
    
    # Format plot
    plt.xlabel('AMP1 (Amplitude)')
    plt.ylabel('Proportion (%)')
    plt.title('AMP1 Distribution: 1.5mM vs 2.5mM vs 4.0mM Calcium')
    add_legend(plt.gca(), frameon=False, outside=True)
    if 'style_hist_axis' in globals():
        style_hist_axis(plt.gca())
    plt.grid(False)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "08_03_calcium_amp1_histogram_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return amp1_standard, amp1_low_ca, amp1_high_ca, output_file

# Run analysis
amp1_standard, amp1_low_ca, amp1_high_ca, output_file = plot_calcium_amp1_comparison()

# Statistical comparison
from scipy.stats import mannwhitneyu

# Summary statistics
print(f"=== AMP1 CALCIUM COMPARISON ===")
print(f"2.5mM Ca (standard): {amp1_standard.mean():.3f} ± {amp1_standard.std():.3f} (n={len(amp1_standard)})")
print(f"1.5mM Ca (low):      {amp1_low_ca.mean():.3f} ± {amp1_low_ca.std():.3f} (n={len(amp1_low_ca)})")
print(f"4.0mM Ca (high):     {amp1_high_ca.mean():.3f} ± {amp1_high_ca.std():.3f} (n={len(amp1_high_ca)})")

print(f"\nPairwise Mann-Whitney tests:")
for pair_label, a, b in [
    ('1.5mM vs 2.5mM', amp1_low_ca, amp1_standard),
    ('2.5mM vs 4.0mM', amp1_standard, amp1_high_ca),
    ('1.5mM vs 4.0mM', amp1_low_ca, amp1_high_ca),
]:
    mw_stat, mw_p = mannwhitneyu(a, b, alternative='two-sided')
    print(f"{pair_label:<15}: U={mw_stat:.1f}, p={mw_p:.4g}")

print(f"\n✓ Saved comparison to {output_file}")


### 8.3 Calcium Dependence of Failure Rates

Failure-rate comparisons test whether increasing extracellular calcium primarily reduces apparent release failures, as expected if initial release probability rises. This provides one of the simplest physiological anchors for interpreting the PCA shifts. 


In [ ]:
# Compare failure rates between calcium concentrations
FAIL1_HIST_BIN_WIDTH = 10.0

fail1_standard = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_low_ca   = PCA_Data_WT_Low_Ca['%Fail1'].dropna()
fail1_high_ca  = PCA_Data_WT_High_Ca['%Fail1'].dropna()

# Plot histograms
all_fail1 = pd.concat([fail1_standard, fail1_low_ca, fail1_high_ca])
fail1_min = float(all_fail1.min())
fail1_max = float(all_fail1.max())
bins = np.arange(fail1_min, fail1_max + FAIL1_HIST_BIN_WIDTH, FAIL1_HIST_BIN_WIDTH, dtype=float)
if bins.size < 2:
    bins = np.array([fail1_min - FAIL1_HIST_BIN_WIDTH / 2.0, fail1_max + FAIL1_HIST_BIN_WIDTH / 2.0], dtype=float)
elif bins[-1] < fail1_max:
    bins = np.append(bins, bins[-1] + FAIL1_HIST_BIN_WIDTH)

make_figure(figsize=(10, 6))
weights_standard = np.ones(len(fail1_standard)) / len(fail1_standard) * 100
weights_low_ca   = np.ones(len(fail1_low_ca)) / len(fail1_low_ca) * 100
weights_high_ca  = np.ones(len(fail1_high_ca)) / len(fail1_high_ca) * 100

plt.hist(fail1_standard, bins=bins, alpha=0.4, color=get_wt_ca_color('2.5mM'), weights=weights_standard, 
         edgecolor='black', label=f'WT 2.5mM Ca (n={len(fail1_standard)})')
plt.hist(fail1_low_ca, bins=bins, alpha=0.4, color=get_wt_ca_color('1.5mM'), weights=weights_low_ca, 
         edgecolor='black', label=f'WT 1.5mM Ca (n={len(fail1_low_ca)})')
plt.hist(fail1_high_ca, bins=bins, alpha=0.35, color=get_wt_ca_color('4mM'), weights=weights_high_ca, 
         edgecolor='black', label=f'WT 4.0mM Ca (n={len(fail1_high_ca)})')

plt.axvline(fail1_standard.mean(), color=get_wt_ca_color('2.5mM'), linestyle='--', linewidth=1.8)
plt.axvline(fail1_low_ca.mean(), color=get_wt_ca_color('1.5mM'), linestyle='--', linewidth=1.8)
plt.axvline(fail1_high_ca.mean(), color=get_wt_ca_color('4mM'), linestyle='--', linewidth=1.8)

plt.xlabel('%Fail1')
plt.ylabel('Proportion (%)')
plt.title('Failure Rate: 1.5mM vs 2.5mM vs 4.0mM Calcium')
add_legend(plt.gca(), frameon=False, outside=True)
if 'style_hist_axis' in globals():
    style_hist_axis(plt.gca())
plt.grid(False)
plt.tight_layout()

output_file = OUTPUT_DIR / "08_04_calcium_fail1_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Stats
from scipy.stats import mannwhitneyu
print(f"1.5mM Ca: {fail1_low_ca.mean():.1f}% ± {fail1_low_ca.std():.1f}% (n={len(fail1_low_ca)})")
print(f"2.5mM Ca: {fail1_standard.mean():.1f}% ± {fail1_standard.std():.1f}% (n={len(fail1_standard)})")
print(f"4.0mM Ca: {fail1_high_ca.mean():.1f}% ± {fail1_high_ca.std():.1f}% (n={len(fail1_high_ca)})")
for pair_label, a, b in [
    ('1.5mM vs 2.5mM', fail1_low_ca, fail1_standard),
    ('2.5mM vs 4.0mM', fail1_standard, fail1_high_ca),
    ('1.5mM vs 4.0mM', fail1_low_ca, fail1_high_ca),
]:
    _, p_value = mannwhitneyu(a, b, alternative='two-sided')
    print(f"{pair_label:<15}: p = {p_value:.4g}")


### 8.4 Calcium Effects on Summary Release Metrics

These grouped comparisons combine amplitude, failures, and plasticity ratios to show that calcium changes do not map onto a single scalar descriptor. The joint presentation is important because the manuscript interprets calcium as affecting both Pr and apparent N. 


In [ ]:
from scipy.stats import wilcoxon

# Prepare data for plotting (unchanged)
amp1_comparison = pd.DataFrame({
    'AMP1': pd.concat([PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

fail1_comparison = pd.DataFrame({
    '%Fail1': pd.concat([PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

ppr2_comparison = pd.DataFrame({
    'PPR2/1': pd.concat([PCA_Data_WT_Low_Ca['PPR2/1'], PCA_Data_WT_High_Ca['PPR2/1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

# Create plots
fig, (ax1, ax2, ax3) = make_figure_grid(1, 3, figsize=(15, 6))

# Comparaison AMP1
sns.boxplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1,
           palette=[get_wt_ca_color('1.5mM'), get_wt_ca_color('4mM')], fliersize=0)
sns.stripplot(data=amp1_comparison, x='Condition', y='AMP1', ax=ax1,
             color='black', size=3, alpha=0.6)
ax1.set_title('AMP1: Low vs High Calcium')
ax1.set_ylim(bottom=0)

# Comparaison %Fail1
sns.boxplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
           palette=[get_wt_ca_color('1.5mM'), get_wt_ca_color('4mM')], fliersize=0)
sns.stripplot(data=fail1_comparison, x='Condition', y='%Fail1', ax=ax2,
             color='black', size=3, alpha=0.6)
ax2.set_title('%Fail1: Low vs High Calcium')

# Comparaison PPR2/1
sns.boxplot(data=ppr2_comparison, x='Condition', y='PPR2/1', ax=ax3,
           palette=[get_wt_ca_color('1.5mM'), get_wt_ca_color('4mM')], fliersize=0)
sns.stripplot(data=ppr2_comparison, x='Condition', y='PPR2/1', ax=ax3,
             color='black', size=3, alpha=0.6)
ax3.axhline(1, color='gray', linestyle='dotted', linewidth=2)
ax3.set_title('PPR2/1: Low vs High Calcium')
ax3.set_ylim(0.5, 2)

# Statistical tests with Wilcoxon
amp1_stat, amp1_p = wilcoxon(PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1'])
fail1_stat, fail1_p = wilcoxon(PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1'])
ppr2_stat, ppr2_p = wilcoxon(PCA_Data_WT_Low_Ca['PPR2/1'].dropna(), PCA_Data_WT_High_Ca['PPR2/1'].dropna())

# Add p-values on plots
def add_p_value(ax, p_value, y_pos_ratio=0.9):
    ax.text(0.5, y_pos_ratio, f"p = {p_value:.4g}", ha='center', va='center', transform=ax.transAxes, fontsize=10)

add_p_value(ax1, amp1_p)
add_p_value(ax2, fail1_p)
add_p_value(ax3, ppr2_p)

plt.tight_layout()

# Save plots
output_file = OUTPUT_DIR / "08_05_calcium_direct_comparison_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display results
print("1.5mM vs 4mM Calcium Comparison:")
print("-" * 40)
print(f"AMP1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_Low_Ca['AMP1'].std():.3f}")
print(f"  4mM:   {PCA_Data_WT_High_Ca['AMP1'].mean():.3f} ± {PCA_Data_WT_High_Ca['AMP1'].std():.3f}")
print(f"  p = {amp1_p:.4g}")

print(f"%Fail1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_Low_Ca['%Fail1'].std():.1f}%")
print(f"  4mM:   {PCA_Data_WT_High_Ca['%Fail1'].mean():.1f}% ± {PCA_Data_WT_High_Ca['%Fail1'].std():.1f}%")
print(f"  p = {fail1_p:.4g}")

print(f"PPR2/1:")
print(f"  1.5mM: {PCA_Data_WT_Low_Ca['PPR2/1'].mean():.3f} ± {PCA_Data_WT_Low_Ca['PPR2/1'].std():.3f}")
print(f"  4mM:   {PCA_Data_WT_High_Ca['PPR2/1'].mean():.3f} ± {PCA_Data_WT_High_Ca['PPR2/1'].std():.3f}")
print(f"  p = {ppr2_p:.4g}")

# Save statistics
stats_file = OUTPUT_DIR / "FIG4DEF_calcium_comparison_statistics.txt"
with open(stats_file, 'w') as f:
    f.write("1.5mM vs 4mM Calcium Statistical Comparison\n")
    f.write("=" * 45 + "\n\n")
    f.write(f"AMP1: Wilcoxon stat={amp1_stat:.1f}, p={amp1_p:.6g}\n")
    f.write(f"%Fail1: Wilcoxon stat={fail1_stat:.1f}, p={fail1_p:.6g}\n")
    f.write(f"PPR2/1: Wilcoxon stat={ppr2_stat:.1f}, p={ppr2_p:.6g}\n")

print(f"Saved to {output_file} and {stats_file}")


In [ ]:
# ##################################################################
# FIG B1 : SN cumulative, 3 Ca²⁺ at 20 Hz
# ##################################################################
fig_b1, ax = make_figure_grid(figsize=(7, 5))
px = np.arange(num_pulses)
res_b1 = {}

for lab, info in COND_GROUPS_20.items():
    df = info['df']
    if len(df) < 2:
        continue
    amps = reconstruct_amps_qnorm(df, Q_estimates, average_Q)
    r = sn_cumulative(amps, N_FIT_LAST)
    res_b1[lab] = r
    ax.errorbar(px, r['cum_mean'], yerr=r['cum_sem'], fmt='o-',
                color=info['color'], capsize=3, ms=4, lw=1.2,
                label=f"{lab} (n={r['n']})")
    ax.plot(r['x_fit'], r['y_fit'], '--', color=info['color'], alpha=0.5, lw=1)
    ax.plot(0, r['RRP'], 's', color=info['color'], ms=6, zorder=5)

ax.axhline(0, color='gray', lw=0.5, ls=':')
style_ax(ax, 'Stimulus number', 'Cumulative release (quanta)',
         'SN cumulative : 20 Hz, per-bouton Q')
add_legend(ax, loc='upper left')

fig_b1.tight_layout()
fig_b1.savefig(OUTPUT_DIR / '08_06_calcium_sn_cumulative_three_conditions_20hz.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\nSN cumulative summary (20 Hz):")
for lb in res_b1:
    print(f"{lb}: RRP={res_b1[lb]['RRP']:.2f}q  "
          f"P0={res_b1[lb]['P0']:.3f}  "
          f"refill={res_b1[lb]['slope']:.3f}q/stim  "
          f"r2={res_b1[lb]['r2']:.3f}")


### 8.5 Calcium Effects on Paired-Pulse Facilitation

Paired-pulse behavior is interpreted jointly with the scalar comparisons and mean-trace views below, rather than as a separate standalone block.


### 8.6 Calcium Effects on Mean Trace Shape

The mean-trace overlays preserve the temporal structure of the calcium effects across the full train. They make it easier to see whether increased release remains sustained after the first response or collapses into rapid depression.


In [ ]:
# Compare mean traces between calcium concentrations (0.5-2.0s window)

low_ca_20hz_conditions = set(get_calcium_conditions('1.5mM', '20Hz'))
high_ca_20hz_conditions = set(get_calcium_conditions('4mM', '20Hz'))
low_ca_stats = compute_trace_stats(condition_names=low_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
high_ca_stats = compute_trace_stats(condition_names=high_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
low_ca_rows = select_traces(condition_names=low_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
high_ca_rows = select_traces(condition_names=high_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
low_raw_stats = compute_trace_stats(rows=low_ca_rows, source=TRACE_MEAN_SOURCE)
high_raw_stats = compute_trace_stats(rows=high_ca_rows, source=TRACE_MEAN_SOURCE)
idx_low_1s = np.argmin(np.abs(low_raw_stats['time'] - 1.0))
idx_high_1s = np.argmin(np.abs(high_raw_stats['time'] - 1.0))
low_dt = np.diff(np.asarray(low_raw_stats['time'], float)); low_dt = low_dt[np.isfinite(low_dt) & (low_dt > 0)]
high_dt = np.diff(np.asarray(high_raw_stats['time'], float)); high_dt = high_dt[np.isfinite(high_dt) & (high_dt > 0)]
low_window_samples = max(1, int(np.ceil(0.05 / np.nanmedian(low_dt)))) if low_dt.size else 1
high_window_samples = max(1, int(np.ceil(0.05 / np.nanmedian(high_dt)))) if high_dt.size else 1
low_ca_peak_1s = np.nanmax(low_raw_stats['average'][idx_low_1s:idx_low_1s + low_window_samples])
high_ca_peak_1s = np.nanmax(high_raw_stats['average'][idx_high_1s:idx_high_1s + high_window_samples])
low_ca_rows_norm = low_ca_rows.copy(); high_ca_rows_norm = high_ca_rows.copy()
low_ca_rows_norm['Avg'] = [np.asarray(trace, float) / low_ca_peak_1s for trace in low_ca_rows_norm['Avg']]
high_ca_rows_norm['Avg'] = [np.asarray(trace, float) / high_ca_peak_1s for trace in high_ca_rows_norm['Avg']]
low_norm_stats = compute_trace_stats(rows=low_ca_rows_norm, source=TRACE_MEAN_SOURCE)
high_norm_stats = compute_trace_stats(rows=high_ca_rows_norm, source=TRACE_MEAN_SOURCE)
raw_bounds = np.concatenate([
    low_ca_stats['average'] - low_ca_stats['sem'],
    low_ca_stats['average'] + low_ca_stats['sem'],
    high_ca_stats['average'] - high_ca_stats['sem'],
    high_ca_stats['average'] + high_ca_stats['sem'],
])
raw_y_min, raw_y_max = np.nanmin(raw_bounds), np.nanmax(raw_bounds)
raw_y_pad = 0.05 * (raw_y_max - raw_y_min)
raw_y_lim = (raw_y_min - raw_y_pad, raw_y_max + raw_y_pad)
norm_bounds = np.concatenate([
    low_norm_stats['average'] - low_norm_stats['sem'],
    low_norm_stats['average'] + low_norm_stats['sem'],
    high_norm_stats['average'] - high_norm_stats['sem'],
    high_norm_stats['average'] + high_norm_stats['sem'],
    np.array([1.0]),
])
norm_y_min, norm_y_max = np.nanmin(norm_bounds), np.nanmax(norm_bounds)
norm_y_pad = 0.05 * (norm_y_max - norm_y_min)
norm_y_lim = (norm_y_min - norm_y_pad, norm_y_max + norm_y_pad)
stim_times = [t for t in [1.0 + 0.05 * i for i in range(10)] if t <= 2.0]
fig, (ax_raw, ax_norm) = make_figure_grid(1, 2, figsize=(12, 5), sharex=True)
plot_traces(ax=ax_raw, rows=low_ca_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('1.5mM'), label=f'1.5mM Ca (n={low_ca_stats["n"]})', show_average=True, show_sem=True, style_axis=False)
plot_traces(ax=ax_raw, rows=high_ca_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('4mM'), label=f'4mM Ca (n={high_ca_stats["n"]})', show_average=True, show_sem=True, stim_times=stim_times, stim_kwargs={'mode': 'top', 'linewidth': 1.5}, zero_line=True, xlim=TRACE_XLIM_20HZ, ylim=raw_y_lim, xlabel='Time (s)', ylabel='ΔF/F', title='Raw', legend=True)
plot_traces(ax=ax_norm, rows=low_norm_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('1.5mM'), label=f'1.5mM Ca (n={low_norm_stats["n"]})', show_average=True, show_sem=True, style_axis=False)
plot_traces(ax=ax_norm, rows=high_norm_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('4mM'), label=f'4mM Ca (n={high_norm_stats["n"]})', show_average=True, show_sem=True, stim_times=stim_times, stim_kwargs={'mode': 'top', 'linewidth': 1.5}, zero_line=True, hlines=[1], hline_kwargs={'color': 'black', 'linestyle': ':', 'linewidth': 1, 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, ylim=norm_y_lim, xlabel='Time (s)', ylabel='ΔF/F (normalized)', title='Normalized to first event', legend=True)
fig.suptitle('Calcium Trace Comparison: 1.5mM vs 4mM', fontsize=12, fontweight='bold')
plt.tight_layout()
output_file = OUTPUT_DIR / "08_07_08_calcium_mean_traces_combined.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Calcium trace comparison: 1.5mM (n={low_ca_stats['n']}) vs 4mM (n={high_ca_stats['n']})")
print(f"✓ Calcium trace comparison (normalized to 1st peak = 1):")
print(f"  1.5mM Ca: n={low_norm_stats['n']}, peak at 1s={low_ca_peak_1s:.3f}")
print(f"  4mM Ca:   n={high_norm_stats['n']}, peak at 1s={high_ca_peak_1s:.3f}")
print(f"✓ Saved to {output_file}")



### 8.7 Calcium Correlation Structure

We correlate paired-pulse facilitation with amplitude and failure metrics under each calcium condition to test whether changes in release strength and apparent reliability are aligned with changes in short-term plasticity. These scatterplots help determine whether calcium effects can be explained by a single release-probability axis or instead require an additional occupancy/refilling component.


In [ ]:
# Correlation analysis: AMP1 and %Fail1 vs PPR2/1 across calcium conditions
from scipy.stats import pearsonr, t

def calcium_scatter_analysis(x_param, nbr, y_param='PPR2/1'):
    """Create scatter plot with regression analysis for calcium conditions."""

    # Extract data for both conditions
    low_ca_data  = PCA_Data_WT_Low_Ca[[x_param, y_param]].dropna()
    high_ca_data = PCA_Data_WT_High_Ca[[x_param, y_param]].dropna()

    x_low, y_low   = low_ca_data[x_param].values, low_ca_data[y_param].values
    x_high, y_high = high_ca_data[x_param].values, high_ca_data[y_param].values

    # Calculate separate correlations
    r_low, p_low   = pearsonr(x_low, y_low) if len(x_low) > 1 else (float('nan'), float('nan'))
    r_high, p_high = pearsonr(x_high, y_high) if len(x_high) > 1 else (float('nan'), float('nan'))

    # Pooled analysis
    x_pool = np.concatenate([x_low, x_high])
    y_pool = np.concatenate([y_low, y_high])

    slope = intercept = margin = None
    if len(x_pool) > 2:
        slope, intercept = np.polyfit(x_pool, y_pool, 1)
        r_pool, p_pool = pearsonr(x_pool, y_pool)

        # Simplified CI calculation
        residuals = y_pool - (intercept + slope * x_pool)
        mse = np.sum(residuals**2) / (len(x_pool) - 2)
        se = np.sqrt(mse)

        t_crit = t.ppf(0.975, len(x_pool) - 2)
        margin = t_crit * se
    else:
        r_pool = p_pool = float('nan')

    # Create plot
    make_figure(figsize=(7, 5))
    plt.scatter(x_low, y_low, c=get_wt_ca_color('1.5mM'), alpha=0.7, edgecolor='black',
                label=f'1.5mM Ca (n={len(x_low)})')
    plt.scatter(x_high, y_high, c=get_wt_ca_color('4mM'), alpha=0.7, edgecolor='black',
                label=f'4mM Ca (n={len(x_high)})')

    # Set axis limits first so the fit spans the whole plotted range
    if len(x_pool) > 0:
        if x_param == 'AMP1':
            x_min, x_max = 0, max(3, x_pool.max() * 1.1)
        else:
            x_span = x_pool.max() - x_pool.min()
            pad = max(0.1 * x_span, 0.05)
            x_min, x_max = x_pool.min() - pad, x_pool.max() + pad
        y_max = max(3, y_pool.max() * 1.1)
        plt.xlim(x_min, x_max)
        plt.ylim(0, y_max)

    # Add regression line and confidence band across the full plot width
    if slope is not None:
        x_grid = np.linspace(*plt.gca().get_xlim(), 200)
        y_fit = intercept + slope * x_grid
        plt.plot(x_grid, y_fit, color='black', linewidth=2, label='Pooled regression')
        plt.fill_between(x_grid, y_fit - margin, y_fit + margin,
                         color='black', alpha=0.15, label='95% CI')

    # Format plot
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.title(f'{x_param} vs {y_param}: Calcium Comparison')
    plt.grid(False)
    add_legend(plt.gca(), )
    plt.tight_layout()

    # Save results
    safe_param = x_param.replace('%', 'pct').replace('/', '_')
    output_file = OUTPUT_DIR / f"08_09_calcium_{nbr}{safe_param}_vs_ppr2_1_calcium.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    # Save statistics
    stats_file = OUTPUT_DIR / f"08_07_calcium_{nbr}{safe_param}_correlation_vs_ppr2_1_stats.txt"
    with open(stats_file, 'w') as f:
        f.write(f"Correlation: {x_param} vs {y_param}\n")
        f.write(f"1.5mM Ca: r={r_low:.4f}, p={p_low:.6g}, n={len(x_low)}\n")
        f.write(f"4mM Ca: r={r_high:.4f}, p={p_high:.6g}, n={len(x_high)}\n")
        f.write(f"Pooled: r={r_pool:.4f}, p={p_pool:.6g}, n={len(x_pool)}\n")

    # Print statistics instead of writing them inside the figure
    print(f"\n{x_param} vs {y_param}")
    print(f"  1.5mM Ca: r={r_low:.2f}, p={p_low:.2g}, n={len(x_low)}")
    print(f"  4mM Ca:   r={r_high:.2f}, p={p_high:.2g}, n={len(x_high)}")
    print(f"  Pooled:   r={r_pool:.2f}, p={p_pool:.2g}, n={len(x_pool)}")
    print(f"  Saved figure: {output_file}")
    print(f"  Saved stats:  {stats_file}")

    return output_file

# Run both analyses
amp1_output = calcium_scatter_analysis('AMP1', '40_')
fail1_output = calcium_scatter_analysis('%Fail1', '41_')

print(f"\nScatter analyses complete:")
print(f"  AMP1 vs PPR2/1: {amp1_output}")
print(f"  %Fail1 vs PPR2/1: {fail1_output}")


## 9. High-Frequency Stimulation (50 Hz)

The 50 Hz experiments test how bouton properties shift when stimulation frequency is increased. These analyses are kept separate from the calcium block so that frequency-dependent trajectories can be interpreted explicitly against the WT 20 Hz reference.

In the manuscript logic, frequency mainly probes how boutons respond to stronger temporal demand. The key question is whether higher frequency pushes boutons along the same axis as calcium or instead produces a distinct trajectory in the WT reference space.


### 9.1 PCA Trajectories at 50 Hz

The 50 Hz trajectory panels compare the frequency-driven displacement of bouton centroids and, where available, paired boutons across calcium conditions. They provide the compact summary used to test whether high frequency predominantly affects apparent Pr rather than the same N-like axis engaged by calcium.


In [ ]:
# Cell 1
# Figure 1: 2.5mM Ca2+ change between 20Hz and 50Hz (single arrow)

wt_20_2_5_coords = np.asarray(pca_data['WT_pooled'])
hz50_2_5_coords = np.asarray(pca_data['50Hz_2_5Ca'])

center_20_2_5 = wt_20_2_5_coords.mean(axis=0)
center_50_2_5 = hz50_2_5_coords.mean(axis=0)

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']

fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    wt_20_2_5_coords[:, 0], wt_20_2_5_coords[:, 1],
    s=12, c='lightgray', alpha=0.35,
    label=f'20Hz 2.5mM / WT pooled (n={len(wt_20_2_5_coords)})'
)

ax.scatter(
    hz50_2_5_coords[:, 0], hz50_2_5_coords[:, 1],
    s=24, marker='D', c=get_50hz_ca_color('2.5mM'), alpha=0.80,
    edgecolors='#c9894f', linewidths=0.4,
    label=f'50Hz 2.5mM (n={len(hz50_2_5_coords)})'
)

ax.scatter(
    center_20_2_5[0], center_20_2_5[1],
    s=110, marker='o', c='white',
    edgecolors='black', linewidths=1.4, zorder=5,
    label='20Hz 2.5mM centroid'
)

ax.scatter(
    center_50_2_5[0], center_50_2_5[1],
    s=110, marker='X', c=get_50hz_ca_color('2.5mM'),
    edgecolors='black', linewidths=1.0, zorder=6,
    label='50Hz 2.5mM centroid'
)

ax.annotate(
    '',
    xy=(center_50_2_5[0], center_50_2_5[1]),
    xytext=(center_20_2_5[0], center_20_2_5[1]),
    arrowprops=dict(arrowstyle='->', color=get_50hz_ca_color('2.5mM'), lw=2.5, mutation_scale=18),
    zorder=7
)

ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_title('2.5mM Ca2+ PCA Shift: 20Hz → 50Hz')
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()

output_file = OUTPUT_DIR / '09_01_50hz_pca_shift_2p5mM.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'20Hz 2.5mM centroid: ({center_20_2_5[0]:.3f}, {center_20_2_5[1]:.3f})')
print(f'50Hz 2.5mM centroid: ({center_50_2_5[0]:.3f}, {center_50_2_5[1]:.3f})')
print(f'Centroid shift: {np.linalg.norm(center_50_2_5 - center_20_2_5):.3f}')
print(f'Saved to {output_file}')


In [ ]:
# Cell 2
# Figure 2: frequency-driven PCA shift at each Ca2+ level (3 arrows)

conditions = [
    ('1.5mM', 'WT_1_5Ca',  '50Hz_1_5Ca', get_wt_ca_color('1.5mM'), get_50hz_ca_color('1.5mM'), 'v'),
    ('2.5mM', 'WT_pooled', '50Hz_2_5Ca', get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM'), 'D'),
    ('4mM',   'WT_4Ca',    '50Hz_4Ca',   get_wt_ca_color('4mM'),   get_50hz_ca_color('4mM'),   '^'),
]

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']

fig, ax = plt.subplots(figsize=(10, 8))

for label, key20, key50, color20, color50, marker50 in conditions:
    coords20 = np.asarray(pca_data[key20])
    coords50 = np.asarray(pca_data[key50])

    center20 = coords20.mean(axis=0)
    center50 = coords50.mean(axis=0)

    ax.scatter(
        coords20[:, 0], coords20[:, 1],
        s=12, marker='o', facecolors='none', edgecolors=color20,
        alpha=0.35, linewidths=0.7,
        label=f'{label} 20Hz (n={len(coords20)})'
    )

    ax.scatter(
        coords50[:, 0], coords50[:, 1],
        s=24, marker=marker50, c=color50,
        alpha=0.80, edgecolors=color50, linewidths=0.4,
        label=f'{label} 50Hz (n={len(coords50)})'
    )

    ax.scatter(
        center20[0], center20[1],
        s=95, marker='o', c='white',
        edgecolors=color20, linewidths=1.5, zorder=5
    )

    ax.scatter(
        center50[0], center50[1],
        s=110, marker='X', c=color50,
        edgecolors='black', linewidths=0.9, zorder=6
    )

    ax.annotate(
        '',
        xy=(center50[0], center50[1]),
        xytext=(center20[0], center20[1]),
        arrowprops=dict(arrowstyle='->', color=color50, lw=2.3, mutation_scale=16),
        zorder=7
    )



ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_title('PCA Shift from 20Hz to 50Hz at Each Ca2+ Level')
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=8, ncol=2, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

output_file = OUTPUT_DIR / '09_02_50hz_pca_shift_all_calcium.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

for label, key20, key50, _, _, _ in conditions:
    center20 = np.asarray(pca_data[key20]).mean(axis=0)
    center50 = np.asarray(pca_data[key50]).mean(axis=0)
    print(f'{label}: 20Hz -> 50Hz centroid shift = {np.linalg.norm(center50 - center20):.3f}')

print(f'Saved to {output_file}')


In [ ]:
# Overlay all 50Hz PPR profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Use a shared y-axis max across all three plots

if 'ppr_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (the cell defining ppr_profiles_matrix).')

profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_50Hz_1_5_Ca, get_50hz_ca_color('1.5mM'), '09_03a_50hz_ppr_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_50Hz_2_5_Ca, get_50hz_ca_color('2.5mM'), '09_03b_50hz_ppr_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_50Hz_4_Ca, get_50hz_ca_color('4mM'), '09_03c_50hz_ppr_profiles_4p0mM.pdf'),
]

prepared_profiles = []
global_ymax = 0

for title, df_cond, color, save_name in profiles_by_condition:
    profiles = ppr_profiles_matrix(df_cond)

    if profiles.size == 0:
        prepared_profiles.append((title, profiles, color, save_name))
        continue

    mean_profile = np.nanmean(profiles, axis=0)
    sem_profile = np.nanstd(profiles, axis=0) / np.sqrt(profiles.shape[0])
    cond_ymax = np.nanmax(np.vstack([profiles, mean_profile + sem_profile]))
    global_ymax = max(global_ymax, cond_ymax)

    prepared_profiles.append((title, profiles, color, save_name))

global_ylim = (0, global_ymax * 1.05 if global_ymax > 0 else 1)

for title, profiles, color, save_name in prepared_profiles:
    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles,
        color=color,
        label='Mean',
        individual_alpha=0.25,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid PPR profiles')
        plt.close(fig)
        continue

    pulse_numbers, _, _ = result
    finalize_ppr_axis(
        ax,
        pulse_numbers,
        title=(f'50Hz PPR Profiles - {title}\n' f'(n={profiles.shape[0]} boutons)'),
        ylabel='PPR (A_n/A_1)',
        ylim=global_ylim,
        unity_kwargs={'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1.5},
        legend=True,
        legend_loc='best',
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')


In [ ]:
# Overlay all 50Hz failure-rate profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Explicit failures are defined per event from min(AMP_CORR, AMP_UNCORR) < thr_shared.

if 'failure_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared failure/PPR helper cell first (the cell defining failure_profiles_matrix).')

failure_profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_50Hz_1_5_Ca, get_50hz_ca_color('1.5mM'), '09_03d_50hz_failure_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_50Hz_2_5_Ca, get_50hz_ca_color('2.5mM'), '09_03e_50hz_failure_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_50Hz_4_Ca, get_50hz_ca_color('4mM'), '09_03f_50hz_failure_profiles_4p0mM.pdf'),
]

for title, df_cond, color, save_name in failure_profiles_by_condition:
    profiles = failure_profiles_matrix(df_cond, trials_all, min_trials=5)
    n_valid = int(np.sum(np.isfinite(profiles).any(axis=1))) if profiles.size else 0

    if profiles.size == 0 or n_valid == 0:
        print(f'Skipping {title}: no valid failure-rate profiles')
        continue

    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles[np.isfinite(profiles).any(axis=1)],
        color=color,
        label='Mean',
        individual_alpha=0.20,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid failure-rate profiles')
        plt.close(fig)
        continue

    pulse_numbers, mean_profile, sem_profile = result
    finalize_ppr_axis(
        ax,
        pulse_numbers,
        title=f'50Hz Failure Profiles - {title} (n={n_valid} boutons)',
        ylabel='Failure rate',
        ylim=(0, 1),
        unity_line=False,
        legend=True,
        legend_loc='best',
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')
    print(
        f"{title}: n={n_valid} | "
        f"Fail1={mean_profile[0]:.3f}±{sem_profile[0]:.3f} | "
        f"Fail2={mean_profile[1]:.3f}±{sem_profile[1]:.3f} | "
        f"Fail10={mean_profile[-1]:.3f}±{sem_profile[-1]:.3f}"
    )


### 9.2 Cluster Composition at 50 Hz

The class-composition view asks whether raising frequency redistributes boutons across the WT classes or mainly shifts their continuous coordinates without a strong class reassignment. This complements the centroid-trajectory analysis with a discrete summary. 


In [ ]:
# Reuse shared helper: get_text_color

from sklearn.neighbors import KNeighborsClassifier

# Compare cluster distributions between 20Hz WT and 50Hz for each calcium concentration
# Three separate figures: 1.5mM Ca, 2.5mM Ca, 4mM Ca


# Get WT cluster counts (20Hz baseline)
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
all_clusters = sorted(wt_cluster_counts.index)

# Train kNN classifier on WT pooled data
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(pca_coordinates, cluster_assignments)

# Project 50Hz conditions and assign to clusters
hz50_1_5_assignments = knn.predict(pca_data['50Hz_1_5Ca'])
hz50_2_5_assignments = knn.predict(pca_data['50Hz_2_5Ca'])
hz50_4_assignments   = knn.predict(pca_data['50Hz_4Ca'])

hz50_1_5_counts = pd.Series(hz50_1_5_assignments).value_counts()
hz50_2_5_counts = pd.Series(hz50_2_5_assignments).value_counts()
hz50_4_counts   = pd.Series(hz50_4_assignments).value_counts()

#Project 20Hz calcium condition data and assign to clusters
hz20_1_5_assignments = knn.predict(pca_data['WT_1_5Ca'])
hz20_4_assignments   = knn.predict(pca_data['WT_4Ca'])

hz20_1_5_counts = pd.Series(hz20_1_5_assignments).value_counts()
hz20_4_counts   = pd.Series(hz20_4_assignments).value_counts()

# Ensure all clusters represented
hz50_1_5_complete = pd.Series([hz50_1_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_2_5_complete = pd.Series([hz50_2_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_4_complete   = pd.Series([hz50_4_counts.get(c, 0) for c in all_clusters], index=all_clusters)

hz20_1_5_complete = pd.Series([hz20_1_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz20_4_complete   = pd.Series([hz20_4_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)

hz50_1_5_percentages = 100 * hz50_1_5_complete / len(hz50_1_5_assignments)
hz50_2_5_percentages = 100 * hz50_2_5_complete / len(hz50_2_5_assignments)
hz50_4_percentages   = 100 * hz50_4_complete / len(hz50_4_assignments)

hz20_1_5_percentages = 100 * hz20_1_5_complete / len(hz20_1_5_assignments)
hz20_4_percentages   = 100 * hz20_4_complete / len(hz20_4_assignments)



### 9.3 PPR Profiles at 50 Hz

Frequency-dependent PPR trajectories test whether facilitation is strengthened when the inter-stimulus interval is shortened. These panels connect the 50 Hz PCA shifts back to the train-normalized readout most directly linked to STP.


In [ ]:
# Compare PPR profiles across WT and 50Hz conditions at different calcium concentrations
if 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (the cell defining ppr_profile_stats).')

comparison_panels = [
    {
        'title': '1.5mM Ca: 20Hz vs 50Hz',
        'series': [
            ('20Hz 1.5mM', PCA_Data_WT_Low_Ca, get_wt_ca_color('1.5mM'), 'o'),
            ('50Hz 1.5mM', PCA_Data_50Hz_1_5_Ca, get_50hz_ca_color('1.5mM'), 'v'),
        ],
    },
    {
        'title': '2.5mM Ca: 20Hz vs 50Hz',
        'series': [
            ('20Hz 2.5mM', PCA_Data_WT_Pooled, get_wt_ca_color('2.5mM'), 'o'),
            ('50Hz 2.5mM', PCA_Data_50Hz_2_5_Ca, get_50hz_ca_color('2.5mM'), 'D'),
        ],
    },
    {
        'title': '4mM Ca: 20Hz vs 50Hz',
        'series': [
            ('20Hz 4mM', PCA_Data_WT_High_Ca, get_wt_ca_color('4mM'), 'o'),
            ('50Hz 4mM', PCA_Data_50Hz_4_Ca, get_50hz_ca_color('4mM'), '^'),
        ],
    },
]

fig, axes = make_figure_grid(1, 3, figsize=(18, 5))
summary_rows = []

for ax, panel in zip(axes, comparison_panels):
    for label, df_cond, color, marker in panel['series']:
        pulse_numbers, means, sems, n = ppr_profile_stats(df_cond)
        plot_ppr_mean_sem(
            ax,
            pulse_numbers,
            means,
            sems,
            color=color,
            marker=marker,
            label=f'{label} (n={n})',
            linewidth=2.5,
            markersize=5,
            sem_alpha=0.2,
        )
        summary_rows.append((label, n, means[1] if len(means) > 1 else np.nan, means[-1] if len(means) > 1 else np.nan,
                             sems[1] if len(sems) > 1 else np.nan, sems[-1] if len(sems) > 1 else np.nan))

    finalize_ppr_axis(
        ax,
        pulse_numbers,
        title=panel['title'],
        ylabel='PPR (A_n/A_1)',
        ylim=(0.9, 3.0),
        unity_kwargs={'color': 'gray', 'linestyle': 'dotted', 'linewidth': 2},
        legend=True,
        legend_loc='best',
        legend_fontsize=9,
xlabel_fontsize=11,
        ylabel_fontsize=11,
        title_fontsize=12,
        title_fontweight='bold',
    )

plt.tight_layout()
output_fig = OUTPUT_DIR / '09_04_50hz_ppr_20hz_vs_50hz_calcium_comparison.pdf'
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

print('=== PPR PROFILE COMPARISON: 20Hz vs 50Hz ===')
for label, n, ppr2, ppr10, sem2, sem10 in summary_rows:
    print(f"{label:12} (n={n:3d}): PPR2/1={ppr2:.3f}±{sem2:.3f}, PPR10/1={ppr10:.3f}±{sem10:.3f}")

print(f'\n✓ Saved to {output_fig}')


### 9.4 Mean Traces at 50 Hz

Mean traces at 50 Hz keep the original temporal structure of the high-frequency responses visible, which is useful for checking whether the fluorescence signal remains physiologically interpretable under rapid stimulation. 


In [ ]:
# Plot mean traces for all three 50Hz conditions on the same graph

condition_specs = [
    ('50Hz 1.5mM Ca', get_calcium_conditions('1.5mM', '50Hz')[0], get_50hz_ca_color('1.5mM')),
    ('50Hz 2.5mM Ca', get_calcium_conditions('2.5mM', '50Hz')[0], get_50hz_ca_color('2.5mM')),
    ('50Hz 4mM Ca', get_calcium_conditions('4mM', '50Hz')[0], get_50hz_ca_color('4mM')),
]
trace_rows_by_label = {label: select_traces(condition_names=condition_name, source=TRACE_MEAN_SOURCE) for label, condition_name, _ in condition_specs}
fig, ax = make_figure_grid(figsize=(12, 6))
trace_stats = {}
for idx, (label, _, color) in enumerate(condition_specs):
    stats = compute_trace_stats(rows=trace_rows_by_label[label], source=TRACE_MEAN_SOURCE)
    trace_stats[label] = stats
    plot_traces(ax=ax, rows=stats['rows'], source=TRACE_MEAN_SOURCE, color=color, label=f'{label} (n={stats["n"]})', show_average=True, show_sem=True, style_axis=(idx == len(condition_specs) - 1), stim_times=[t for t in [1.0 + 0.02 * i for i in range(10)] if t <= 2.0] if idx == len(condition_specs) - 1 else None, stim_kwargs={'mode': 'fixed', 'y_span': (-0.15, -0.10), 'linewidth': 2} if idx == len(condition_specs) - 1 else None, zero_line=(idx == len(condition_specs) - 1), xlim=(0.898, 1.4) if idx == len(condition_specs) - 1 else None, xlabel='Time (s)' if idx == len(condition_specs) - 1 else None, ylabel='ΔF/F' if idx == len(condition_specs) - 1 else None, title='Mean Traces: 50Hz Stimulation at Different Calcium Concentrations' if idx == len(condition_specs) - 1 else None, legend=(idx == len(condition_specs) - 1), legend_kwargs={'fontsize': 10})
plt.tight_layout()
output_file = OUTPUT_DIR / '09_05_50hz_mean_traces_all_calcium.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print('=== 50Hz MEAN TRACE COMPARISON ===')
for label in [s[0] for s in condition_specs]:
    stats = trace_stats[label]
    peak_idx = int(np.nanargmax(stats['average'])) if np.any(np.isfinite(stats['average'])) else None
    if peak_idx is None:
        print(f"{label}: n={stats['n']}, peak response unavailable")
    else:
        print(f"{label}: n={stats['n']}")
        print(f"  Peak response: {stats['average'][peak_idx]:.3f} ΔF/F at t={stats['time'][peak_idx]:.2f}s")
print(f'\n✓ Saved mean traces to {output_file}')


In [ ]:
# Plot mean traces for all three 50Hz conditions from raw traces,
# using the shared nearest-sample raw alignment path.

condition_specs = [
    ('50Hz 1.5mM Ca', get_calcium_conditions('1.5mM', '50Hz')[0], get_50hz_ca_color('1.5mM')),
    ('50Hz 2.5mM Ca', get_calcium_conditions('2.5mM', '50Hz')[0], get_50hz_ca_color('2.5mM')),
    ('50Hz 4mM Ca', get_calcium_conditions('4mM', '50Hz')[0], get_50hz_ca_color('4mM')),
]
raw_stats = {}
fig, ax = make_figure_grid(figsize=(12, 6))
for idx, (label, condition_name, color) in enumerate(condition_specs):
    stats = plot_traces(ax=ax, condition_names=condition_name, source='raw_nearest', color=color, label=f'{label} (n={len(select_traces(condition_names=condition_name, source="raw"))})', show_average=True, show_sem=True, zero_line=(idx == len(condition_specs) - 1), event_time=1.0 if idx == len(condition_specs) - 1 else None, event_kwargs={'color': 'gray', 'linestyle': '--', 'linewidth': 1} if idx == len(condition_specs) - 1 else None, stim_times=[t for t in [1.0 + 0.02 * i for i in range(10)] if t <= 1.4] if idx == len(condition_specs) - 1 else None, stim_kwargs={'mode': 'fixed', 'y_span': (-0.15, -0.10), 'linewidth': 2} if idx == len(condition_specs) - 1 else None, xlim=(0.898, 1.4) if idx == len(condition_specs) - 1 else None, xlabel='Time (s)' if idx == len(condition_specs) - 1 else None, ylabel='ΔF/F' if idx == len(condition_specs) - 1 else None, title='Mean Traces: 50Hz Stimulation at Different Calcium Concentrations (raw, stim-aligned, nearest-sample)' if idx == len(condition_specs) - 1 else None, legend=(idx == len(condition_specs) - 1), legend_kwargs={'fontsize': 10}, style_axis=(idx == len(condition_specs) - 1), return_data=True)
    raw_stats[label] = stats
plt.tight_layout()
output_file = OUTPUT_DIR / '09_05b_50hz_mean_traces_all_calcium_raw_nearest.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print('=== 50Hz RAW TRACE COMPARISON (nearest-sample average) ===')
for label in [s[0] for s in condition_specs]:
    stats = raw_stats[label]
    peak_idx = int(np.nanargmax(stats['average'])) if np.any(np.isfinite(stats['average'])) else None
    if peak_idx is None:
        print(f"{label}: n={stats['n']}, peak response unavailable")
    else:
        print(f"{label}: n={stats['n']}")
        print(f"  Peak response: {stats['average'][peak_idx]:.3f} ΔF/F at t={stats['time'][peak_idx]:.2f}s")
print(f'\n✓ Saved raw mean traces to {output_file}')


### 9.5 Scalar Comparisons Between 20 Hz and 50 Hz

Scalar boxplots and paired comparisons summarize how amplitude and plasticity ratios change when frequency is increased. Together with the PCA projection, they support the interpretation that frequency and calcium do not perturb the WT state space in the same way.


In [ ]:
from scipy.stats import mannwhitneyu

# Boxplot comparing AMP1 between 20Hz and 50Hz at 2.5mM Ca

# Extract data
amp1_20hz = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50hz = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()

# Create figure
fig, ax = make_figure_grid(figsize=(6, 6))

# Prepare data for boxplot
data = [amp1_20hz.values, amp1_50hz.values]
positions = [0, 1]
colors = [get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [f'AMP1 20Hz\n(n={len(amp1_20hz)})', f'AMP1 50Hz\n(n={len(amp1_50hz)})']

# Create boxplots
bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add strip points
for i, (vals, pos, color) in enumerate(zip(data, positions, colors)):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=25, alpha=0.5, edgecolors='black', linewidths=0.3)

# Statistical test
_, p_amp1 = mannwhitneyu(amp1_20hz, amp1_50hz, alternative='two-sided')

def get_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

# Add significance bar
y_max = max([d.max() for d in data]) * 1.1
ax.plot([0, 1], [y_max, y_max], 'k-', lw=1.5)
ax.text(0.5, y_max * 1.02, f'{get_stars(p_amp1)} (p={p_amp1:.2g})', ha='center', fontsize=10)

# Formatting
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Amplitude (ΔF/F)', fontsize=11)
ax.set_title('AMP1 Comparison: 20Hz vs 50Hz at 2.5mM Ca', fontsize=12, fontweight='bold')
ax.grid(False)
ax.set_ylim(top=y_max * 1.15)

plt.tight_layout()
output_file = OUTPUT_DIR / "09_06_50hz_amp1_20hz_vs_50hz_boxplot.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== AMP1 COMPARISON: 20Hz vs 50Hz @ 2.5mM Ca ===")
print(f"AMP1 20Hz: {amp1_20hz.mean():.3f} ± {amp1_20hz.std():.3f} (n={len(amp1_20hz)})")
print(f"AMP1 50Hz: {amp1_50hz.mean():.3f} ± {amp1_50hz.std():.3f} (n={len(amp1_50hz)})")
print(f"  Mann-Whitney p = {p_amp1:.4g}")
print(f"\n✓ Saved to {output_file}")


In [ ]:
# Boxplots: AMP1 20Hz vs 50Hz for 1.5mM, 2.5mM, and 4mM
def _get_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

def _plot_amp1_comparison(ax, data20, data50, title, colors=('lightgray', 'gray'), ylims=None):
    data = [data20.values, data50.values]
    positions = [0, 1]
    bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    # strip points
    for vals, pos, color in zip(data, positions, colors):
        x_jit = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
        ax.scatter(x_jit, vals, c=color, s=25, alpha=0.5, edgecolors='black', linewidths=0.3)
    # stats
    p = np.nan
    if len(data20) > 0 and len(data50) > 0:
        _, p = mannwhitneyu(data20, data50, alternative='two-sided')
    y_top = max([d.max() if len(d) else 0 for d in data], default=0) * 1.1 + 1e-9
    ax.plot([positions[0], positions[1]], [y_top, y_top], 'k-', lw=1.5)
    if not np.isnan(p):
        ax.text(0.5, y_top * 1.02, f'{_get_stars(p)} (p={p:.2g})', ha='center', fontsize=10)
    else:
        ax.text(0.5, y_top * 1.02, 'ns (insufficient data)', ha='center', fontsize=10)
    ax.set_xticks(positions)
    ax.set_xticklabels([f'20Hz (n={len(data20)})', f'50Hz (n={len(data50)})'], fontsize=9)
    ax.set_ylabel('AMP1 (ΔF/F)', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(False)
    if ylims is not None:
        ax.set_ylim(ylims)

# Extract AMP1 data
amp1_20_15 = PCA_Data_WT_Low_Ca['AMP1'].dropna()
amp1_50_15 = PCA_Data_50Hz_1_5_Ca['AMP1'].dropna()
amp1_20_25 = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50_25 = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()
amp1_20_4 = PCA_Data_WT_High_Ca['AMP1'].dropna()
amp1_50_4 = PCA_Data_50Hz_4_Ca['AMP1'].dropna()

# Compute global y-limits for all boxplots
all_amp1 = pd.concat([amp1_20_15, amp1_50_15, amp1_20_25, amp1_50_25, amp1_20_4, amp1_50_4])
ymin = all_amp1.min()
ymax = all_amp1.max()
yrange = ymax - ymin
ylims = (ymin - 0.1 * yrange, ymax + 0.1 * yrange)

fig, axes = make_figure_grid(1, 3, figsize=(16, 6))

_plot_amp1_comparison(axes[0], amp1_20_15, amp1_50_15, 'AMP1: 1.5mM (20Hz vs 50Hz)', colors=(get_wt_ca_color('1.5mM'), get_50hz_ca_color('1.5mM')), ylims=ylims)
_plot_amp1_comparison(axes[1], amp1_20_25, amp1_50_25, 'AMP1: 2.5mM (20Hz vs 50Hz)', colors=(get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')), ylims=ylims)
_plot_amp1_comparison(axes[2], amp1_20_4,  amp1_50_4,  'AMP1: 4mM (20Hz vs 50Hz)', colors=(get_wt_ca_color('4mM'), get_50hz_ca_color('4mM')), ylims=ylims)

plt.tight_layout()
out = OUTPUT_DIR / "09_07_50hz_amp1_20hz_vs_50hz_all_calcium.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()

# Summary
def _summary(name, d20, d50):
    if len(d20) and len(d50):
        _, p = mannwhitneyu(d20, d50, alternative='two-sided')
        print(f"{name}: 20Hz {d20.mean():.3f}±{d20.std():.3f} (n={len(d20)}), "
              f"50Hz {d50.mean():.3f}±{d50.std():.3f} (n={len(d50)}) | p={p:.4g}")
    else:
        print(f"{name}: insufficient data")

print("=== AMP1 20Hz vs 50Hz by Ca ===")
_summary("1.5mM", amp1_20_15, amp1_50_15)
_summary("2.5mM", amp1_20_25, amp1_50_25)
_summary("4mM",   amp1_20_4,  amp1_50_4)
print(f"✓ Saved to {out}")

In [ ]:
# Reuse shared helper: _get_stars
# Boxplots: PPR2/1 20Hz vs 50Hz for 1.5mM, 2.5mM, 4mM
print(f"✓ Saved to {out}")

In [ ]:
# Reuse shared helper: add_p_value

from scipy.stats import mannwhitneyu
from matplotlib.patches import Patch

# Compare pooled AMP1 against AMP2 at 20 Hz and 50 Hz (2.5 mM Ca)
amp1_20hz = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50hz = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()
amp1_combined = pd.concat([amp1_20hz, amp1_50hz], ignore_index=True)

amp2_20hz = PCA_Data_WT_Pooled['AMP2'].dropna()
amp2_50hz = PCA_Data_50Hz_2_5_Ca['AMP2'].dropna()

fig, ax = make_figure_grid(figsize=(8, 6))

data = [amp1_combined.values, amp2_20hz.values, amp2_50hz.values]
positions = [0, 1, 2]
colors = ['gray', get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [
    f'AMP1\n20Hz+50Hz\n(n={len(amp1_combined)})',
    f'AMP2\n20Hz\n(n={len(amp2_20hz)})',
    f'AMP2\n50Hz\n(n={len(amp2_50hz)})',
]

bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for vals, pos, color in zip(data, positions, colors):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=20, alpha=0.4, edgecolors='black', linewidths=0.3)

def _add_p_bar(ax, x1, x2, y, p, h):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='black', lw=1)
    txt = '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < 5e-2 else f'ns (p={p:.2g})'
    ax.text((x1 + x2) / 2, y + h, txt, ha='center', va='bottom', fontsize=8)

_, p_amp1_vs_amp2_20 = mannwhitneyu(amp1_combined, amp2_20hz, alternative='two-sided')
_, p_amp1_vs_amp2_50 = mannwhitneyu(amp1_combined, amp2_50hz, alternative='two-sided')
_, p_amp2_20_vs_50 = mannwhitneyu(amp2_20hz, amp2_50hz, alternative='two-sided')

ymax = max(np.nanmax(vals) for vals in data)
ymin = min(np.nanmin(vals) for vals in data)
yrange = ymax - ymin if ymax > ymin else 1.0
_add_p_bar(ax, 0, 1, ymax + 0.06 * yrange, p_amp1_vs_amp2_20, 0.02 * yrange)
_add_p_bar(ax, 0, 2, ymax + 0.15 * yrange, p_amp1_vs_amp2_50, 0.02 * yrange)
_add_p_bar(ax, 1, 2, ymax + 0.24 * yrange, p_amp2_20_vs_50, 0.02 * yrange)

ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Amplitude (ΔF/F)')
ax.set_title('AMP2 versus pooled AMP1 at 2.5 mM Ca')
ax.set_ylim(ymin - 0.08 * yrange, ymax + 0.35 * yrange)
add_legend(
    ax,
    handles=[Patch(facecolor=c, edgecolor='black', alpha=0.7) for c in colors],
    labels=['AMP1 pooled', 'AMP2 20 Hz', 'AMP2 50 Hz'],
    loc='upper right',
)
ax.grid(False)

out = OUTPUT_DIR / '09_07b_50hz_amp2_vs_amp1_pooled.pdf'
plt.tight_layout()
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()

print('=== AMP2 versus pooled AMP1 @ 2.5 mM Ca ===')
print(f"AMP1 pooled: mean={amp1_combined.mean():.3f} ± {amp1_combined.std():.3f} (n={len(amp1_combined)})")
print(f"AMP2 20 Hz:  mean={amp2_20hz.mean():.3f} ± {amp2_20hz.std():.3f} (n={len(amp2_20hz)}) | p vs pooled AMP1={p_amp1_vs_amp2_20:.4g}")
print(f"AMP2 50 Hz:  mean={amp2_50hz.mean():.3f} ± {amp2_50hz.std():.3f} (n={len(amp2_50hz)}) | p vs pooled AMP1={p_amp1_vs_amp2_50:.4g}")
print(f"20 Hz vs 50 Hz AMP2: p={p_amp2_20_vs_50:.4g}")
print(f"✓ Saved to {out}")


In [ ]:
# Reuse shared helper: _get_stars

from scipy.stats import mannwhitneyu
from matplotlib.patches import Patch

# Compare %Fail1 between 20 Hz and 50 Hz at 2.5 mM Ca
fail1_20hz = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_50hz = PCA_Data_50Hz_2_5_Ca['%Fail1'].dropna()

fig, ax = make_figure_grid(figsize=(6, 6))

data = [fail1_20hz.values, fail1_50hz.values]
positions = [0, 1]
colors = [get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [f'%Fail1 20Hz\n(n={len(fail1_20hz)})', f'%Fail1 50Hz\n(n={len(fail1_50hz)})']

bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for vals, pos, color in zip(data, positions, colors):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=25, alpha=0.5, edgecolors='black', linewidths=0.3)

_, p_fail1 = mannwhitneyu(fail1_20hz, fail1_50hz, alternative='two-sided')
ymax = max(np.nanmax(vals) for vals in data)
ymin = min(np.nanmin(vals) for vals in data)
yrange = ymax - ymin if ymax > ymin else 1.0
ax.plot([0, 0, 1, 1], [ymax + 0.05 * yrange, ymax + 0.08 * yrange, ymax + 0.08 * yrange, ymax + 0.05 * yrange], color='black', lw=1)
ax.text(0.5, ymax + 0.09 * yrange, _get_stars(p_fail1) + f' (p={p_fail1:.2g})', ha='center', va='bottom', fontsize=8)

ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Failure rate (%)')
ax.set_title('%Fail1: 20 Hz versus 50 Hz at 2.5 mM Ca')
ax.set_ylim(ymin - 0.08 * yrange, ymax + 0.18 * yrange)
add_legend(
    ax,
    handles=[Patch(facecolor=c, edgecolor='black', alpha=0.7) for c in colors],
    labels=['20 Hz', '50 Hz'],
    loc='upper right',
)
ax.grid(False)

output_file = OUTPUT_DIR / '09_07c_50hz_fail1_20hz_vs_50hz.pdf'
plt.tight_layout()
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print('=== %Fail1 comparison: 20 Hz versus 50 Hz @ 2.5 mM Ca ===')
print(f"20 Hz: {fail1_20hz.mean():.2f} ± {fail1_20hz.std():.2f} (n={len(fail1_20hz)})")
print(f"50 Hz: {fail1_50hz.mean():.2f} ± {fail1_50hz.std():.2f} (n={len(fail1_50hz)})")
print(f"Mann-Whitney p = {p_fail1:.4g}")
print(f"✓ Saved to {output_file}")


In [ ]:
# Reuse shared helper: add_p_value

from scipy.stats import mannwhitneyu
from matplotlib.patches import Patch

# Compare pooled %Fail1 against %Fail2 at 20 Hz and 50 Hz (2.5 mM Ca)
fail1_20hz = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_50hz = PCA_Data_50Hz_2_5_Ca['%Fail1'].dropna()
fail1_combined = pd.concat([fail1_20hz, fail1_50hz], ignore_index=True)
fail2_20hz = PCA_Data_WT_Pooled['%Fail2'].dropna()
fail2_50hz = PCA_Data_50Hz_2_5_Ca['%Fail2'].dropna()

fig, ax = make_figure_grid(figsize=(8, 6))

data = [fail1_combined.values, fail2_20hz.values, fail2_50hz.values]
positions = [0, 1, 2]
colors = ['gray', get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [
    f'%Fail1\n20Hz+50Hz\n(n={len(fail1_combined)})',
    f'%Fail2\n20Hz\n(n={len(fail2_20hz)})',
    f'%Fail2\n50Hz\n(n={len(fail2_50hz)})',
]

bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for vals, pos, color in zip(data, positions, colors):
    x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
    ax.scatter(x_jitter, vals, c=color, s=20, alpha=0.4, edgecolors='black', linewidths=0.3)

def _add_p_bar(ax, x1, x2, y, p, h):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='black', lw=1)
    txt = '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < 5e-2 else f'ns (p={p:.2g})'
    ax.text((x1 + x2) / 2, y + h, txt, ha='center', va='bottom', fontsize=8)

_, p_fail1_vs_fail2_20 = mannwhitneyu(fail1_combined, fail2_20hz, alternative='two-sided')
_, p_fail1_vs_fail2_50 = mannwhitneyu(fail1_combined, fail2_50hz, alternative='two-sided')
_, p_fail2_20_vs_50 = mannwhitneyu(fail2_20hz, fail2_50hz, alternative='two-sided')

ymax = max(np.nanmax(vals) for vals in data)
ymin = min(np.nanmin(vals) for vals in data)
yrange = ymax - ymin if ymax > ymin else 1.0
_add_p_bar(ax, 0, 1, ymax + 0.06 * yrange, p_fail1_vs_fail2_20, 0.02 * yrange)
_add_p_bar(ax, 0, 2, ymax + 0.15 * yrange, p_fail1_vs_fail2_50, 0.02 * yrange)
_add_p_bar(ax, 1, 2, ymax + 0.24 * yrange, p_fail2_20_vs_50, 0.02 * yrange)

ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Failure rate (%)')
ax.set_title('%Fail2 versus pooled %Fail1 at 2.5 mM Ca')
ax.set_ylim(ymin - 0.08 * yrange, ymax + 0.35 * yrange)
add_legend(
    ax,
    handles=[Patch(facecolor=c, edgecolor='black', alpha=0.7) for c in colors],
    labels=['%Fail1 pooled', '%Fail2 20 Hz', '%Fail2 50 Hz'],
    loc='upper right',
)
ax.grid(False)

output_file = OUTPUT_DIR / '09_07d_50hz_fail2_vs_fail1_pooled.pdf'
plt.tight_layout()
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print('=== %Fail2 versus pooled %Fail1 @ 2.5 mM Ca ===')
print(f"%Fail1 pooled: mean={fail1_combined.mean():.2f} ± {fail1_combined.std():.2f} (n={len(fail1_combined)})")
print(f"%Fail2 20 Hz:  mean={fail2_20hz.mean():.2f} ± {fail2_20hz.std():.2f} (n={len(fail2_20hz)}) | p vs pooled %Fail1={p_fail1_vs_fail2_20:.4g}")
print(f"%Fail2 50 Hz:  mean={fail2_50hz.mean():.2f} ± {fail2_50hz.std():.2f} (n={len(fail2_50hz)}) | p vs pooled %Fail1={p_fail1_vs_fail2_50:.4g}")
print(f"20 Hz versus 50 Hz %Fail2: p={p_fail2_20_vs_50:.4g}")
print(f"✓ Saved to {output_file}")


In [ ]:
# Extract PPR2/1 and PPR3/1 data for 20Hz WT and 50Hz WT at 2.5mM Ca
ppr2_1_20hz = PCA_Data_WT_Pooled['PPR2/1'].dropna()
ppr2_1_50hz = PCA_Data_50Hz_2_5_Ca['PPR2/1'].dropna()

ppr3_1_20hz = PCA_Data_WT_Pooled['PPR3/1'].dropna()
ppr3_1_50hz = PCA_Data_50Hz_2_5_Ca['PPR3/1'].dropna()

# Create figure with two subplots
fig, (ax1, ax2) = make_figure_grid(1, 2, figsize=(12, 5))

def plot_boxplot(ax, data_20hz, data_50hz, title, ylabel):
    """Helper function to plot boxplot on given axis."""
    data = [data_20hz.values, data_50hz.values]
    positions = [0, 1]
    colors = [get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
    labels = [f'{ylabel} 20Hz\n(n={len(data_20hz)})', 
              f'{ylabel} 50Hz\n(n={len(data_50hz)})']
    
    bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add strip points
    for i, (vals, pos, color) in enumerate(zip(data, positions, colors)):
        x_jitter = np.ones(len(vals)) * pos + np.random.uniform(-0.15, 0.15, len(vals))
        ax.scatter(x_jitter, vals, c=color, s=25, alpha=0.5, edgecolors='black', linewidths=0.3)
    
    # Statistical test
    _, p_val = mannwhitneyu(data_20hz, data_50hz, alternative='two-sided')
    
    def get_stars(p):
        return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    
    # Add significance bar
    y_max = max([d.max() for d in data]) * 1.1
    ax.plot([0, 1], [y_max, y_max], 'k-', lw=1.5)
    ax.text(0.5, y_max * 1.02, f'{get_stars(p_val)} (p={p_val:.2g})', ha='center', fontsize=10)
    
    # Formatting
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('Ratio', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(False)
    ax.set_ylim(top=y_max * 1.15)
    
    return p_val

# Plot PPR2/1
p_ppr2_1 = plot_boxplot(ax1, ppr2_1_20hz, ppr2_1_50hz, 
                        'PPR2/1 Comparison: 20Hz vs 50Hz @ 2.5mM Ca', 'PPR2/1')

# Plot PPR3/1
p_ppr3_1 = plot_boxplot(ax2, ppr3_1_20hz, ppr3_1_50hz, 
                        'PPR3/1 Comparison: 20Hz vs 50Hz @ 2.5mM Ca', 'PPR3/1')

plt.tight_layout()
output_file = OUTPUT_DIR / "09_08_50hz_ppr2_1_ppr3_1_20hz_vs_50hz.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== PPR RATIO COMPARISON: 20Hz vs 50Hz @ 2.5mM Ca ===")
print(f"\nPPR2/1:")
print(f"  20Hz: {ppr2_1_20hz.mean():.3f} ± {ppr2_1_20hz.std():.3f} (n={len(ppr2_1_20hz)})")
print(f"  50Hz: {ppr2_1_50hz.mean():.3f} ± {ppr2_1_50hz.std():.3f} (n={len(ppr2_1_50hz)})")
print(f"  Mann-Whitney p = {p_ppr2_1:.4g}")

print(f"\nPPR3/1:")
print(f"  20Hz: {ppr3_1_20hz.mean():.3f} ± {ppr3_1_20hz.std():.3f} (n={len(ppr3_1_20hz)})")
print(f"  50Hz: {ppr3_1_50hz.mean():.3f} ± {ppr3_1_50hz.std():.3f} (n={len(ppr3_1_50hz)})")
print(f"  Mann-Whitney p = {p_ppr3_1:.4g}")

print(f"\n✓ Saved to {output_file}")

In [ ]:
# ##################################################################
# FIG B2 : 20 Hz vs 50 Hz per Ca2+ (3 panels)
# ##################################################################
fig_b2, axes = make_figure_grid(1, 3, figsize=(16, 4.5), sharey=True)
ca_labels = ['1.5 mM', '2.5 mM', '4.0 mM']
res_b2 = {}

for ax, ca_lab in zip(axes, ca_labels):
    res_b2[ca_lab] = {}
    for freq_lab, groups, ls in [('20 Hz', COND_GROUPS_20, 'o-'), ('50 Hz', COND_GROUPS_50, 's--')]:
        if ca_lab not in groups:
            continue
        info = groups[ca_lab]
        df = info['df']
        if len(df) < 2:
            continue
        amps = reconstruct_amps_qnorm(df, Q_estimates, average_Q)
        r = sn_cumulative(amps, N_FIT_LAST)
        res_b2[ca_lab][freq_lab] = r
        ax.errorbar(px, r['cum_mean'], yerr=r['cum_sem'], fmt=ls,
                    color=info['color'], capsize=2, ms=3, lw=1,
                    label=f"{freq_lab} (n={r['n']})")
        ax.plot(r['x_fit'], r['y_fit'], '--', color=info['color'], alpha=0.3, lw=1)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    style_ax(ax, 'Stimulus', 'Cumul. release (q)' if ax is axes[0] else None, ca_lab)
    add_legend(ax)

fig_b2.suptitle('20 Hz vs 50 Hz per [Ca2+]', fontsize=11, fontweight='bold')
out_b2 = OUTPUT_DIR / '09_09_50hz_sn_cumulative_20hz_vs_50hz.pdf'
fig_b2.tight_layout()
fig_b2.savefig(out_b2, dpi=300, bbox_inches='tight')
plt.show()

print('=== SN cumulative: 20 Hz versus 50 Hz by Ca2+ ===')
print(f"{'Ca':<8} {'Freq':<6} {'n':>4} {'RRP':>7} {'P0':>6} {'slope':>8} {'r2':>6}")
for ca_lab in ca_labels:
    for freq_lab in ['20 Hz', '50 Hz']:
        if freq_lab not in res_b2.get(ca_lab, {}):
            continue
        r = res_b2[ca_lab][freq_lab]
        print(f"{ca_lab:<8} {freq_lab:<6} {r['n']:>4} {r['RRP']:>7.2f} {r['P0']:>6.3f} {r['slope']:>8.3f} {r['r2']:>6.3f}")
print(f"✓ Saved to {out_b2}")


### 9.6 Sensor Saturation Control

The saturation-control panel checks that the 50 Hz interpretation is not driven by sensor ceiling effects, especially at high calcium. This is essential because the mechanistic conclusions depend on changes in fluorescence remaining proportional to changes in glutamate release.


In [ ]:
# Plot the pre-filter 50 Hz saturation diagnostic after the main 50 Hz analyses
trace_source_df = globals().get('NORM_TRACES_DATAFRAME_PRE_50HZ_FILTER', NORM_TRACES_DATAFRAME)
cond_4_50hz = get_calcium_conditions('4mM', '50Hz')[0]
cond_2_5_50hz = get_calcium_conditions('2.5mM', '50Hz')[0]

traces_4_50Hz = trace_source_df[trace_source_df['Condition'] == cond_4_50hz]['Avg']
times_4_50Hz = trace_source_df[trace_source_df['Condition'] == cond_4_50hz]['Time']
traces_2_5_50Hz = trace_source_df[trace_source_df['Condition'] == cond_2_5_50hz]['Avg']
times_2_5_50Hz = trace_source_df[trace_source_df['Condition'] == cond_2_5_50hz]['Time']

max_values_4_50Hz, max_times_4_50Hz = get_max_and_time(traces_4_50Hz, times_4_50Hz)
max_values_2_5_50Hz, max_times_2_5_50Hz = get_max_and_time(traces_2_5_50Hz, times_2_5_50Hz)

all_max_values = max_values_4_50Hz + max_values_2_5_50Hz
bins_max_values = np.linspace(min(all_max_values), max(all_max_values), 50)
all_max_times = max_times_4_50Hz + max_times_2_5_50Hz
bins_max_times = np.linspace(min(all_max_times), max(all_max_times), 50)

fig, axs = make_figure_grid(2, 1)

axs[0].hist(max_values_4_50Hz, bins=bins_max_values, color=get_50hz_ca_color('4mM'), alpha=0.6, edgecolor='black', label=cond_4_50hz)
axs[0].hist(max_values_2_5_50Hz, bins=bins_max_values, color=get_50hz_ca_color('2.5mM'), alpha=0.6, edgecolor='black', label=cond_2_5_50hz)
axs[0].set_title('Histogram of maxima (overlaid)')
axs[0].set_xlabel('Maximum value')
axs[0].set_ylabel('Frequency')
add_legend(axs[0])
if 'style_hist_axis' in globals():
    style_hist_axis(axs[0])

axs[1].hist(max_times_4_50Hz, bins=bins_max_times, color=get_50hz_ca_color('4mM'), alpha=0.6, edgecolor='black', label=cond_4_50hz)
axs[1].hist(max_times_2_5_50Hz, bins=bins_max_times, color=get_50hz_ca_color('2.5mM'), alpha=0.6, edgecolor='black', label=cond_2_5_50hz)
axs[1].set_title('Histogram of max-time values (overlaid)')
axs[1].set_xlabel('Time')
axs[1].set_ylabel('Frequency')
add_legend(axs[1])
if 'style_hist_axis' in globals():
    style_hist_axis(axs[1])

fig.tight_layout()
output_file = OUTPUT_DIR / '09_10_50hz_saturation_diagnostic_histograms.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {output_file}')


In [ ]:
# Retrieve traces and times for the 4mM 50Hz condition
cond_4_50hz = get_calcium_conditions('4mM', '50Hz')[0]
traces_theo_4_50hz = select_traces(condition_names=cond_4_50hz, source=TRACE_SINGLE_SOURCE)
if TRACE_SINGLE_SOURCE != 'normalized' and cond_4_50hz in EXCEPTIONAL_CONDITIONS:
    traces_theo_4_50hz['Time'] = [np.asarray(t, dtype=float) + EXCEPTIONAL_BASELINE_OFFSET for t in traces_theo_4_50hz['Time']]
traces = traces_theo_4_50hz['Avg']
times = traces_theo_4_50hz['Time']
num_traces = len(traces)
num_cols = 5
num_rows = int(np.ceil(num_traces / num_cols))
fig, axes = make_figure_grid(num_rows, num_cols, figsize=(20, 4 * num_rows))
axes = axes.flatten()
for i, (ax, trace, time) in enumerate(zip(axes, traces, times)):
    ax.plot(time, trace, color=get_50hz_ca_color('4mM'))
    ax.set_xlim(0.8, 1.5)
    ax.set_title(f'Trace {i+1}')
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
for j in range(i + 1, len(axes)):
    axes[j].axis('off')
fig.tight_layout()
output_file = OUTPUT_DIR / '09_11_50hz_4mM_raw_trace_grid.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {output_file}')


In [ ]:
# %% ###############################################################
# CELL E : Multi-condition MLE (calcium and frequency perturbations)
#
# Each condition has:
#   - trial data in trials_all (keyed by condition)
#   - a summary DF (row-aligned with projected PCA coordinates)
#   - projected PCA coordinates in pca_data[key]
#
# We match trial files → summary DF rows WITHIN each condition
# projected coords. This is the retained N/P method for calcium and
# frequency comparison figures shown after the WT-only section.
# ###################################################################

COND_DEFS_E = [
    # label              trial conditions                               colour     pca_key        summary_df_var
    ('1.5 mM 20 Hz',   ['Theo_1_5Ca'],                                 get_wt_ca_color('1.5mM'), 'WT_1_5Ca',    'PCA_Data_WT_Low_Ca'),
    ('4.0 mM 20 Hz',   ['Theo_4Ca'],                                   get_wt_ca_color('4mM'),   'WT_4Ca',      'PCA_Data_WT_High_Ca'),
    ('1.5 mM 50 Hz',   ['Theo_1_5_50Hz'],                              get_50hz_ca_color('1.5mM'), '50Hz_1_5Ca',  'PCA_Data_50Hz_1_5_Ca'),
    ('2.5 mM 50 Hz',   ['Theo_2_5_50Hz'],                              get_50hz_ca_color('2.5mM'), '50Hz_2_5Ca',  'PCA_Data_50Hz_2_5_Ca'),
    ('4.0 mM 50 Hz',   ['Theo_4_50Hz'],                                get_50hz_ca_color('4mM'),   '50Hz_4Ca',    'PCA_Data_50Hz_4_Ca'),
    ('2.5 mM 20 Hz',   get_calcium_conditions('2.5mM', '20Hz'),        get_wt_ca_color('2.5mM'), None,          'PCA_Data_WT_Pooled_clustered'),
]

# == Re-key Q with extract_base_name (same as A2 fix) ============
Q_by_ebn = {}
for k, v in Q_estimates.items():
    Q_by_ebn.setdefault(extract_base_name(k), v)

cond_results_E = {}

for (cond_label, cond_list, color, pca_key, sdf_var) in COND_DEFS_E:

    # == 1. Get projected coordinates =============================
    if pca_key is None:
        proj_xy = pca_coordinates.copy()
    else:
        if pca_key not in pca_data:
            print(f"  [{cond_label}] pca_data['{pca_key}'] missing : skip")
            continue
        proj_xy = np.array(pca_data[pca_key])

    if sdf_var not in globals():
        print(f"  [{cond_label}] {sdf_var} not defined : skip")
        continue
    sdf = globals()[sdf_var]
    n_pts = min(len(proj_xy), len(sdf))

    # == 2. Build count dict (keyed by extract_base_name) =========
    tg = defaultdict(list)
    sub = trials_all[trials_all['condition'].isin(cond_list)]
    for _, r in sub.iterrows():
        key = extract_base_name(str(r['file']).strip())
        tg[key].append(get_count_row_with_failures(r, Q_D))

    # == 3. Match summary DF rows → counts, run MLE ===============
    sdf_ids = sdf['ID'].astype(str).str.strip().values

    E_N = np.full((n_pts, N_STIM), np.nan)
    E_P = np.full((n_pts, N_STIM), np.nan)
    E_fail = np.full((n_pts, N_STIM), np.nan)
    n_matched, n_fit = 0, 0

    for i in range(n_pts):
        key = extract_base_name(sdf_ids[i])
        if key not in tg:
            continue
        qc = np.array(tg[key])
        ok_rows = np.all(np.isfinite(qc), axis=1)
        qc = qc[ok_rows].astype(int)
        if len(qc) < MIN_TRIALS_D:
            continue
        n_matched += 1
        E_fail[i, :] = np.mean(qc == 0, axis=0)

        if USE_BOOTSTRAP:
            rng_e = np.random.default_rng(42 + i)
            ntr = len(qc)
            bN = np.full((N_STIM, N_BOOT), np.nan)
            bP = np.full((N_STIM, N_BOOT), np.nan)
            for b in range(N_BOOT):
                qc_b = qc[rng_e.integers(0, ntr, ntr)]
                for k in range(N_STIM):
                    qk = qc_b[:, k].astype(int)
                    Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
                    if np.isfinite(Nf) and Nf < N_MAX_D:
                        bN[k, b] = Nf
                        bP[k, b] = Pf
            for k in range(N_STIM):
                if np.isfinite(bN[k]).sum() >= N_BOOT * 0.5:
                    E_N[i, k] = np.nanmedian(bN[k])
                    E_P[i, k] = np.nanmedian(bP[k])
        else:
            for k in range(N_STIM):
                qk = qc[:, k].astype(int)
                Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
                if np.isfinite(Nf) and Nf < N_MAX_D:
                    E_N[i, k] = Nf
                    E_P[i, k] = Pf

        if np.isfinite(E_N[i, 0]):
            n_fit += 1

    # Debug on zero matches
    if n_matched == 0 and len(tg) > 0:
        print(f"  [{cond_label:>16}]  !! 0 matches")
        print(f"     trial keys sample : {sorted(tg.keys())[:3]}")
        print(f"     sdf keys sample   : {[extract_base_name(s) for s in sdf_ids[:3]]}")
    else:
        print(f"  [{cond_label:>16}]  pts={n_pts}  matched={n_matched}  fit={n_fit}")

    cond_results_E[cond_label] = dict(
        E_N=E_N, E_P=E_P, E_fail=E_fail,
        proj_xy=proj_xy[:n_pts],
        color=color, n_fit=n_fit,
        n_pts=n_pts, n_matched=n_matched,
    )


In [ ]:
# ##################################################################
# FIG E1 : Trajectories (all conditions overlaid)
# ##################################################################
import re

def _base_label(lbl):
    return re.sub(r"\s*\(\d+\)\s*$", "", lbl).strip()

def _cond_sort_key(base):
    mm = float(re.search(r"(\d+(?:\.\d+)?)\s*mM", base).group(1))
    hz = float(re.search(r"(\d+(?:\.\d+)?)\s*Hz", base).group(1))
    return (hz, mm)

def _force_ordered_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    by_base = {}
    for h, lbl in zip(handles, labels):
        if not lbl:
            continue
        base = _base_label(lbl)
        if base not in by_base:
            by_base[base] = (h, lbl)
    ordered_bases = sorted(by_base.keys(), key=_cond_sort_key)
    ordered_handles = [by_base[base][0] for base in ordered_bases]
    ordered_labels = [by_base[base][1] for base in ordered_bases]
    if ordered_labels:
        ax.legend(ordered_handles, ordered_labels, frameon=False)

fig_e1, axes_e1 = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
axp_e, axn_e = axes_e1[0, 0], axes_e1[0, 1]
summary_rows_e1 = []

for (cond_label, _, color, _, _) in COND_DEFS_E:
    if cond_label not in cond_results_E:
        continue

    res = cond_results_E[cond_label]
    vals_P = np.asarray(res['E_P'])
    vals_N = np.asarray(res['E_N'])
    n_fit = int(np.isfinite(vals_N[:, 0]).sum()) if vals_N.ndim == 2 and vals_N.shape[1] else 0

    plot_mean_sem_trace(axp_e, x_stim, vals_P, color=color, label=f'{cond_label} ({n_fit})', marker='o', linestyle='-', ms=4, lw=1, fill_alpha=0.10)
    plot_mean_sem_trace(axn_e, x_stim, vals_N, color=color, label=f'{cond_label} ({n_fit})', marker='o', linestyle='-', ms=4, lw=1, fill_alpha=0.10)

    def _med(arr, k):
        ok = np.isfinite(arr[:, k])
        return np.nanmedian(arr[ok, k]) if ok.sum() else np.nan

    summary_rows_e1.append((
        cond_label,
        n_fit,
        _med(vals_P, 0), _med(vals_P, 1), _med(vals_P, 9),
        _med(vals_N, 0), _med(vals_N, 1), _med(vals_N, 9),
    ))

axp_e.set_ylim(0, 1)
style_ax(axp_e, 'Stimulus', '$P_k$', 'P per condition')
style_ax(axn_e, 'Stimulus', '$N_k$', 'N per condition')
_force_ordered_legend(axp_e)
_force_ordered_legend(axn_e)

meth = 'bootstrap MLE (explicit failures)' if USE_BOOTSTRAP else 'point MLE (explicit failures)'
save_path_e1 = OUTPUT_DIR / '09_12_bootstrap_mle_trajectories.pdf'
finalize_figure(
    fig_e1,
    title=f'{meth} trajectories : {len(cond_results_E)} conditions',
    save_path=save_path_e1,
)

print('=== Multi-condition MLE trajectories ===')
print(f"{'Condition':<18} {'n':>4} {'P1':>6} {'P2':>6} {'P10':>6} {'N1':>6} {'N2':>6} {'N10':>6}")
for row in summary_rows_e1:
    cond_label, n_fit, p1, p2, p10, n1, n2, n10 = row
    vals = [p1, p2, p10, n1, n2, n10]
    formatted = ' '.join(f"{v:>6.3f}" if np.isfinite(v) and v < 2 else (f"{v:>6.1f}" if np.isfinite(v) else '   nan') for v in vals)
    print(f"{cond_label:<18} {n_fit:>4} {formatted}")
print(f"✓ Saved to {save_path_e1}")


In [ ]:
# ##################################################################
# FIG E2 : PCA maps per condition, split into TWO figures
#   Figure E2a = N maps at stim 1, 2, 10
#   Figure E2b = P maps at stim 1, 2, 10
#   Each condition in its OWN projected coordinate space
# ##################################################################
import re

E2_STIM = [1, 2, 10]

def _cond_sort_key(label):
    mm = float(re.search(r"(\d+(?:\.\d+)?)\s*mM", label).group(1))
    hz = float(re.search(r"(\d+(?:\.\d+)?)\s*Hz", label).group(1))
    return (hz, mm)

active_conds = sorted([c[0] for c in COND_DEFS_E if c[0] in cond_results_E], key=_cond_sort_key)
n_cond_plot = len(active_conds)
n_stim_e2 = len(E2_STIM)

def make_condition_map_figure(value_key, cmap, vlim, value_name, save_name):
    vmin_e, vmax_e = vlim
    fig_e2, axes_e2 = make_figure_grid(n_cond_plot, n_stim_e2, panel_kind='pca')

    for row_i, cond_label in enumerate(active_conds):
        res = cond_results_E[cond_label]
        xy = res['proj_xy']
        src = res[value_key]

        for si, sk in enumerate(E2_STIM):
            kidx = sk - 1
            arr = src[:, kidx]
            ok = np.isfinite(arr)
            ax = axes_e2[row_i, si]

            render_pca_scalar_panel(
                ax,
                xy,
                arr,
                cmap=cmap,
                vmin=vmin_e,
                vmax=vmax_e,
                title=f'{cond_label} : {value_name}$_{{{sk}}}$ (n={ok.sum()})',
                point_size=14,
                min_points=3,
                empty_label=f'n={ok.sum()} (too few)',
                add_colorbar=True,
                colorbar_label=value_name,
                colorbar_shrink=0.7,
            )

    save_path = OUTPUT_DIR / save_name
    finalize_figure(
        fig_e2,
        title=f'Per-condition {value_name} maps at stim 1, 2, 10 (explicit-failure MLE)',
        rect=[0, 0, 1, 0.97],
        save_path=save_path,
    )
    print(f'✓ Saved {value_name} maps to {save_path}')
    return save_path

save_path_n = make_condition_map_figure('E_N', 'Spectral_r', N_VLIM, 'N', '09_13_bootstrap_mle_condition_maps_N.pdf')
save_path_p = make_condition_map_figure('E_P', 'coolwarm', P_VLIM, 'P', '09_14_bootstrap_mle_condition_maps_P.pdf')

print(f"\n{'Condition':<18} {'n':>4} {'P1':>6} {'P2':>6} {'P10':>6} {'N1':>6} {'N2':>6} {'N10':>6}")
for cond_label in active_conds:
    res = cond_results_E[cond_label]
    N, P = res['E_N'], res['E_P']
    n = np.isfinite(N[:, 0]).sum()
    vals = []
    for k in [0, 1, 9]:
        ok_k = np.isfinite(P[:, k])
        vals.append(np.nanmedian(P[ok_k, k]) if ok_k.sum() else np.nan)
    for k in [0, 1, 9]:
        ok_k = np.isfinite(N[:, k])
        vals.append(np.nanmedian(N[ok_k, k]) if ok_k.sum() else np.nan)
    print(f"{cond_label:<18} {n:>4} " + ' '.join(f"{v:>6.3f}" if np.isfinite(v) and v < 2 else (f"{v:>6.1f}" if np.isfinite(v) else '   nan') for v in vals))


## 10. Synapsin II Perturbation

Synapsin II loss is examined after the WT reference, calcium, and frequency analyses so that genotype effects can be interpreted against the full WT framework. The retained SynII analyses focus on direct WT-versus-SynII comparisons and on the redistribution of SynII boutons within the WT reference space.

Several SynII panels in this notebook are more recent than the current manuscript wording and should be read as an extension of the same biological question: which subsets of bouton release properties depend on Synapsin II, and which parts of the WT state space persist without it?


### 10.1 Scope Note

SynII-only HCPC figures are dropped. The analyses below keep only direct WT-versus-SynII comparisons and projection of SynII boutons into the WT reference PCA space.


In [ ]:
# SynII-only HCPC figures removed; analyses continue directly in the WT reference space.

# Figure 1: WT pooled individual PPR profiles with SynII profiles overlaid in brown
if 'ppr_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining ppr_profile_stats).')

ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled.columns and f'PPR{i}/1' in PCA_Data_SynII.columns]

fig, ax = make_figure_grid(figsize=(10, 6))

wt_profiles = ppr_profiles_matrix(PCA_Data_WT_Pooled, ppr_column_names=ppr_cols)
synii_profiles = ppr_profiles_matrix(PCA_Data_SynII, ppr_column_names=ppr_cols)

plot_ppr_profiles_overlay(
    ax, wt_profiles,
    color='gray',
    label='_nolegend_',
    individual_alpha=0.05,
    individual_lw=2.0,
    mean_lw=0.0,
    sem_alpha=0.0,
)
plot_ppr_profiles_overlay(
    ax, synii_profiles,
    color=SYNII_COLOR,
    label='_nolegend_',
    individual_alpha=0.10,
    individual_lw=2.0,
    mean_lw=0.0,
    sem_alpha=0.0,
)

pulse_numbers, wt_means, wt_sems, n_wt = ppr_profile_stats(PCA_Data_WT_Pooled, ppr_column_names=ppr_cols)
_, syn_means, syn_sems, n_syn = ppr_profile_stats(PCA_Data_SynII, ppr_column_names=ppr_cols)
plot_ppr_mean_sem(ax, pulse_numbers, wt_means, wt_sems, color=get_wt_ca_color('2.5mM'), marker='o', label=f'WT mean (n={n_wt})', linewidth=2, sem_alpha=0.15)
plot_ppr_mean_sem(ax, pulse_numbers, syn_means, syn_sems, color=SYNII_COLOR, marker='s', label=f'SynII mean (n={n_syn})', linewidth=2, sem_alpha=0.15)

finalize_ppr_axis(
    ax,
    pulse_numbers,
    title='WT pooled individual PPR profiles with SynII overlay',
    ylabel='PPR (A_n/A_1)',
    ylim=(0, 3.5),
    unity_kwargs={'color': 'gray', 'linestyle': '--', 'linewidth': 2, 'label': 'No facilitation'},
    legend=True,
    legend_loc='best',
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

output_file1 = OUTPUT_DIR / '10_04_synii_overlay_on_wt_ppr_profiles.pdf'
plt.tight_layout()
plt.savefig(output_file1, dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Saved to {output_file1}')
print(f'✓ Plotted WT profiles: {n_wt} | SynII profiles: {n_syn}')


# Figure 2: Three histograms (PPR2/1, PPR5/1, PPR10/1) WT vs SynII
# Probability histograms: bar heights sum to 1 within each group
PPR_SELECTED_HIST_BIN_WIDTH = 0.10

target_pulses = [2, 5, 10]
target_cols = [f'PPR{p}/1' for p in target_pulses]

fig2, axes = make_figure_grid(1, 3, figsize=(15, 4.5), sharey=True)

for ax, p, col in zip(axes, target_pulses, target_cols):
    if col not in PCA_Data_WT_Pooled.columns or col not in PCA_Data_SynII.columns:
        ax.text(0.5, 0.5, f'{col}\nmissing', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Pulse {p}')
        continue

    wt_vals = PCA_Data_WT_Pooled[col].to_numpy()
    syn_vals = PCA_Data_SynII[col].to_numpy()
    wt_vals = wt_vals[np.isfinite(wt_vals)]
    syn_vals = syn_vals[np.isfinite(syn_vals)]

    if len(wt_vals) == 0 or len(syn_vals) == 0:
        ax.text(0.5, 0.5, f'{col}\nno data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Pulse {p}')
        continue

    all_vals = np.concatenate([wt_vals, syn_vals])
    vmin, vmax = np.nanpercentile(all_vals, [1, 99])
    bins = np.arange(vmin, vmax + PPR_SELECTED_HIST_BIN_WIDTH, PPR_SELECTED_HIST_BIN_WIDTH, dtype=float)
    if bins.size < 2:
        bins = np.array([vmin - PPR_SELECTED_HIST_BIN_WIDTH / 2.0, vmax + PPR_SELECTED_HIST_BIN_WIDTH / 2.0], dtype=float)
    elif bins[-1] < vmax:
        bins = np.append(bins, bins[-1] + PPR_SELECTED_HIST_BIN_WIDTH)

    # Probability histogram: sum of bar heights = 1 per group
    ax.hist(
        wt_vals, bins=bins, density=False,
        weights=np.ones(len(wt_vals)) / len(wt_vals),
        color=get_wt_ca_color('2.5mM'), alpha=0.45, label='WT'
    )
    ax.hist(
        syn_vals, bins=bins, density=False,
        weights=np.ones(len(syn_vals)) / len(syn_vals),
        color=SYNII_COLOR, alpha=0.35, label='SynII'
    )

    ax.set_title(f'{col}')
    ax.set_xlabel('PPR value')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)

axes[0].set_ylabel('Probability')
add_legend(axes[-1], fontsize=9)
if 'style_hist_axis' in globals():
    for _ax in axes:
        style_hist_axis(_ax)

fig2.suptitle('WT vs SynII Histograms at Pulses 2, 5, 10', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])

output_file2 = OUTPUT_DIR / "10_05_synii_vs_wt_ppr_histograms_selected_pulses.pdf"
plt.savefig(output_file2, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved to {output_file2}")

# === Second figure: same visualization, but filtering uses ONLY A1 window (1.000 to 1.049 s) ===

A1_START, A1_END = 1.000, 1.049

def a1_window_amp(trace_time, trace_values):
    tt = np.asarray(trace_time, dtype=float)
    tv = np.asarray(trace_values, dtype=float)
    if tt.shape[0] != tv.shape[0]:
        return np.nan
    a1_mask = (tt >= A1_START) & (tt <= A1_END)
    w = tv[a1_mask]
    if w.size == 0 or np.all(~np.isfinite(w)):
        return np.nan
    return np.nanmax(np.abs(w))

# --- Collect SynII traces + A1-window amplitudes ---
synii_trace_data_a1 = []
synii_a1_amplitudes = []

synii_rows_a1 = select_traces(condition_names=SYNAPSIN_CONDITIONS, source=TRACE_SINGLE_SOURCE)
for _, trace_row in synii_rows_a1.iterrows():
    bouton_id = trace_row['ID']
    trace_time = np.asarray(trace_row['Time'], float)
    trace_values = np.asarray(trace_row['Avg'], float)
    synii_trace_data_a1.append((bouton_id, trace_time, trace_values))
    amp_a1 = a1_window_amp(trace_time, trace_values)
    if np.isfinite(amp_a1):
        synii_a1_amplitudes.append(amp_a1)

# SynII threshold from A1 window only
synii_a1_threshold = np.percentile(synii_a1_amplitudes, 95) if len(synii_a1_amplitudes) > 0 else np.nan

# --- Collect WT traces and classify by A1-window threshold ---
wt_trace_data_a1 = []
wt_high_a1_traces = []
wt_regular_a1_traces = []

wt_rows_a1 = select_traces(trace_ids=PCA_Data_WT_Pooled_clustered['ID'], condition_names=WT_2_5_20HZ_CONDITIONS, source=TRACE_SINGLE_SOURCE)
for _, trace_row in wt_rows_a1.iterrows():
    bouton_id = trace_row['ID']
    trace_time = np.asarray(trace_row['Time'], float)
    trace_values = np.asarray(trace_row['Avg'], float)
    wt_trace_data_a1.append((bouton_id, trace_time, trace_values))
    amp_a1 = a1_window_amp(trace_time, trace_values)
    if np.isfinite(amp_a1) and np.isfinite(synii_a1_threshold):
        if amp_a1 > synii_a1_threshold:
            wt_high_a1_traces.append((bouton_id, trace_time, trace_values, amp_a1))
        else:
            wt_regular_a1_traces.append((bouton_id, trace_time, trace_values, amp_a1))

# Common y-limits
all_vals = []
for _, _, trv in synii_trace_data_a1:
    all_vals.extend(np.asarray(trv)[np.isfinite(trv)])
for _, _, trv in wt_trace_data_a1:
    all_vals.extend(np.asarray(trv)[np.isfinite(trv)])

y_min = np.min(all_vals) * 1.1 if len(all_vals) else -0.5
y_max = np.max(all_vals) * 1.1 if len(all_vals) else 1.0

# --- Plot ---
fig, (ax1, ax2) = make_figure_grid(1, 2, figsize=(16, 6))

# Panel 1: SynII
ax1.set_title(f'SynII Traces (A1-window threshold source, n={len(synii_trace_data_a1)})', fontweight='bold')

for _, trace_time, trace_values in synii_trace_data_a1:
    ax1.plot(trace_time, trace_values, color=SYNII_COLOR, alpha=0.10, linewidth=0.3)

synii_avg_stats = compute_trace_stats(condition_names='SynII', source=TRACE_SINGLE_SOURCE)
ax1.plot(synii_avg_stats['time'], synii_avg_stats['average'], color=SYNII_COLOR, linewidth=1.0, label='SynII Average')

if np.isfinite(synii_a1_threshold):
    ax1.axhline(synii_a1_threshold, color='red', linestyle='--', linewidth=2,
                label=f'A1 threshold (95th): {synii_a1_threshold:.3f}')
    ax1.axhline(-synii_a1_threshold, color='red', linestyle='--', linewidth=2)

ax1.axvspan(A1_START, A1_END, color='red', alpha=0.08, label='A1 window')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('ΔF/F')
if 'style_trace_axis' in globals():
    style_trace_axis(ax1)
ax1.set_xlim(*TRACE_XLIM_20HZ)
ax1.set_ylim(y_min, y_max)
add_legend(ax1, frameon=False)

# Panel 2: WT
ax2.set_title(f'WT Traces (A1-window filter): {len(wt_high_a1_traces)}/{len(wt_trace_data_a1)} above threshold',
              fontweight='bold')

for _, trace_time, trace_values, _ in wt_regular_a1_traces:
    ax2.plot(trace_time, trace_values, color='black', alpha=0.05, linewidth=0.25)

for _, trace_time, trace_values, _ in wt_high_a1_traces:
    ax2.plot(trace_time, trace_values, color='red', alpha=0.25, linewidth=0.3)

wt_avg_stats = compute_trace_stats(condition_names=WT_2_5_20HZ_CONDITIONS, source=TRACE_SINGLE_SOURCE)
ax2.plot(wt_avg_stats['time'], wt_avg_stats['average'], color='darkblue', linewidth=1.0, label='WT Average')

if np.isfinite(synii_a1_threshold):
    ax2.axhline(synii_a1_threshold, color=SYNII_COLOR, linestyle='--', linewidth=2, label='SynII A1 threshold')
    ax2.axhline(-synii_a1_threshold, color='red', linestyle='--', linewidth=2)

ax2.axvspan(A1_START, A1_END, color='red', alpha=0.08, label='A1 window')
ax2.plot([], [], color='black', alpha=0.6, linewidth=2, label=f'Below threshold (n={len(wt_regular_a1_traces)})')
ax2.plot([], [], color='red', alpha=0.7, linewidth=10, label=f'Above threshold (n={len(wt_high_a1_traces)})')

ax2.set_xlabel('Time (s)')
ax2.set_ylabel('ΔF/F')
if 'style_trace_axis' in globals():
    style_trace_axis(ax2)
ax2.set_xlim(*TRACE_XLIM_20HZ)
ax2.set_ylim(y_min, y_max)
ax2.axvline(1.0, color='blue', linestyle=':', alpha=0.7, linewidth=2)
add_legend(ax2, frameon=False)

plt.tight_layout()

output_file2 = OUTPUT_DIR / "10_06_synii_vs_wt_a1_window_filtering.pdf"
plt.savefig(output_file2, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved to {output_file2}")
if np.isfinite(synii_a1_threshold):
    print(f"✓ A1-window threshold (95th percentile SynII, {A1_START:.3f}-{A1_END:.3f}s): {synii_a1_threshold:.4f}")
else:
    print("⚠ Could not compute A1-window threshold (no valid SynII A1 values).")

# SynII-only HCPC cluster PPR profile figure removed.



### 10.2 SynII Projection onto the WT PCA Space

Projecting SynII boutons onto the WT PCA makes it possible to ask which parts of the WT reference manifold are lost, compressed, or overrepresented after Synapsin II deletion. This is the main state-space summary of the genotype effect.


In [ ]:
# Project SynII data onto WT-trained PCA space
make_figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors
plot_pca_background(plt.gca(), alpha=0.4,  label='WT pooled (2.5mM Ca)')

# Plot SynII projected data
synii_pca_coords = pca_data['SynII']
plot_pca_overlay_points(plt.gca(), synii_pca_coords[:],
           marker='*',  c=SYNII_COLOR, alpha=0.8,
           linewidth=1, label=f'SynII KO (n={len(synii_pca_coords)})')

# Calculate and plot SynII centroid
synii_centroid = np.mean(synii_pca_coords, axis=0)
plot_pca_overlay_points(plt.gca(), np.array([synii_centroid]),
           marker='X',  c=SYNII_COLOR, linewidth=1,
           label=f'SynII Centroid ({synii_centroid[0]:.2f}, {synii_centroid[1]:.2f})', zorder=10)

# Format plot
style_pca_axes(plt.gca(), title='SynII KO Projection onto WT PCA Space',  legend=False)
add_legend(plt.gca(), frameon=False)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "10_08_synii_projection_on_wt_pca.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space")
print(f"✓ SynII centroid: PC1={synii_centroid[0]:.3f}, PC2={synii_centroid[1]:.3f}")
print(f"✓ Saved to {output_file}")


### 10.3 Fiber-Level SynII Controls

The per-fiber SynII projections test whether the genotype effect is homogeneous across fibers or whether it remains structured at the level of individual axons. This mirrors the same-fiber logic used for the WT diversity analysis.


In [ ]:
# Project SynII data onto WT-trained PCA space with ID-based coloring
make_figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors (background)
plot_pca_background(plt.gca(), alpha=0.4,  label='WT pooled (2.5mM Ca)')

# Get SynII bouton IDs and extract first 14 characters
synii_ids = PCA_Data_SynII['ID'].values
synii_prefixes = [str(bouton_id)[:14] for bouton_id in synii_ids]

# Create unique prefix-to-color mapping with Set2 colormap
unique_prefixes = sorted(set(synii_prefixes))
set2_colormap = plt.get_cmap('Set1')
specified_colors = [set2_colormap(i) for i in range(len(unique_prefixes))]
prefix_color_map = dict(zip(unique_prefixes, specified_colors))

# Plot SynII points grouped by prefix
synii_pca_coords = pca_data['SynII']
for prefix in unique_prefixes:
    mask = np.array([p == prefix for p in synii_prefixes])
    plot_pca_overlay_points(plt.gca(), synii_pca_coords[mask],
               marker='*',  c=[prefix_color_map[prefix]], 
               linewidth=1, label=f'SynII {prefix}')

# Format plot
style_pca_axes(
    plt.gca(),
    title='SynII KO Projection onto WT PCA Space (colored by ID prefix)',
legend=False,
)
add_legend(plt.gca(), loc='best', fontsize=8, frameon=False)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "10_09_synii_projection_by_prefix.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space")
print(f"✓ Found {len(unique_prefixes)} unique ID prefixes: {unique_prefixes}")
print(f"✓ Saved to {output_file}")


### 10.4 WT Cluster Attribution for SynII Boutons

Assigning SynII boutons to WT-derived classes provides a direct way to see which WT phenotypes are depleted or enriched in the knockout. This keeps the genotype comparison grounded in the WT reference taxonomy.


In [ ]:
# Assign SynII samples to WT-derived clusters using PCA space
def assign_clusters_robust(X_existing, labels_existing, X_new, method='centroid'):
    """Assign new samples to existing clusters using specified linkage method."""
    unique_labels = np.unique(labels_existing)
    
    if method == 'centroid':
        centroids = np.vstack([X_existing[labels_existing == lbl].mean(axis=0) for lbl in unique_labels])
        distances = cdist(X_new, centroids)
        return unique_labels[np.argmin(distances, axis=1)]
    
    elif method == 'single':
        dist_matrix = np.empty((X_new.shape[0], len(unique_labels)))
        for j, lbl in enumerate(unique_labels):
            cluster_points = X_existing[labels_existing == lbl]
            dist_matrix[:, j] = np.min(cdist(X_new, cluster_points), axis=1)
        return unique_labels[np.argmin(dist_matrix, axis=1)]
    
    else:
        raise ValueError("method must be 'centroid' or 'single'")

# Assign SynII samples to WT clusters
synii_cluster_assignments = assign_clusters_robust(pca_coordinates, cluster_assignments, 
                                                   pca_data['SynII'], method='single')

print(f"✓ Assigned {len(synii_cluster_assignments)} SynII samples to WT clusters")
if '_format_cluster_count_summary' not in globals():
    def _format_cluster_count_summary(label, counts):
        items = [f"C{int(k)}={int(v)}" for k, v in pd.Series(counts).sort_index().items()]
        joined = '  '.join(items) if items else '-'
        print(f"{label}: {joined}")

_format_cluster_count_summary('SynII', pd.Series(synii_cluster_assignments).value_counts())

### 10.5 SynII Cluster Composition

The cluster-composition summaries quantify the redistribution of SynII boutons relative to WT. These panels are useful for identifying whether the genotype collapses diversity globally or only removes selected phenotypic states. 


In [ ]:
# Reuse shared helper: get_text_color
# Compare cluster distributions between WT and SynII
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
synii_cluster_counts = pd.Series(synii_cluster_assignments).value_counts()

# Ensure all clusters represented
all_clusters = sorted(wt_cluster_counts.index)
synii_complete = pd.Series([synii_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)
synii_percentages = 100 * synii_complete / len(synii_cluster_assignments)

# Stacked bar plot
fig, ax = make_figure_grid(figsize=(8, 6))
bar_width = 0.6
x_positions = np.array([0, 1], dtype=float)
labels = ['WT pooled', 'SynII']

wt_vals = np.array([wt_percentages.get(c, 0.0) for c in all_clusters], dtype=float)
syn_vals = np.array([synii_percentages.get(c, 0.0) for c in all_clusters], dtype=float)

bottom = np.zeros(2, dtype=float)
legend_handles = []
legend_labels = []
for idx, cid in enumerate(all_clusters):
    heights = np.array([wt_vals[idx], syn_vals[idx]], dtype=float)
    color = get_cluster_color(cid)
    bars = ax.bar(
        x_positions,
        heights,
        width=bar_width,
        bottom=bottom,
        color=color,
        edgecolor='white',
        linewidth=0.8,
    )
    bottom += heights
    legend_handles.append(bars[0])
    legend_labels.append(f'C{cid}')

ax.set_xticks(x_positions)
ax.set_xticklabels([f'WT pooled\n(n={len(cluster_assignments)})', f'SynII\n(n={len(synii_cluster_assignments)})'])
style_ax(ax, 'Group', 'Cluster fraction (%)', 'WT versus SynII cluster composition')
ax.set_ylim(0, 100)
add_legend(ax, handles=legend_handles, labels=legend_labels, loc='center left', bbox_to_anchor=(1.01, 0.5), outside=True)
ax.grid(False)

output_file = OUTPUT_DIR / '10_05_synii_cluster_composition_vs_wt.pdf'
fig.tight_layout()
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Saved cluster composition to {output_file}')

print('WT:    ' + '  '.join(f"C{int(c)}={int(wt_cluster_counts.get(c, 0))}" for c in all_clusters))
print('SynII: ' + '  '.join(f"C{int(c)}={int(synii_complete.get(c, 0))}" for c in all_clusters))

summary_df = pd.DataFrame({
    'Cluster': [f'C{int(c)}' for c in all_clusters],
    'WT_n': [int(wt_cluster_counts.get(c, 0)) for c in all_clusters],
    'WT_pct': [float(wt_percentages.get(c, 0.0)) for c in all_clusters],
    'SynII_n': [int(synii_complete.get(c, 0)) for c in all_clusters],
    'SynII_pct': [float(synii_percentages.get(c, 0.0)) for c in all_clusters],
})
print(summary_df.to_string(index=False, formatters={'WT_pct': '{:.1f}'.format, 'SynII_pct': '{:.1f}'.format}))

try:
    from scipy.stats import chi2_contingency
    contingency = np.vstack([
        [int(wt_cluster_counts.get(c, 0)) for c in all_clusters],
        [int(synii_complete.get(c, 0)) for c in all_clusters],
    ])
    chi2, p_val, dof, _ = chi2_contingency(contingency)
    print(f'Chi-square test: chi2={chi2:.3f}, dof={dof}, p={p_val:.4g}')
except Exception as exc:
    print(f'Chi-square test unavailable: {exc}')


### 10.6 SynII Descriptive Statistics

Simple descriptive metrics are retained here because they show how the genotype reshapes amplitude and reliability even before considering the full PCA organization. They provide the scalar counterpart to the state-space comparison. 


In [ ]:
# Calculate mean ± SD for Amp1, Amp2, and %Fail1 in SynII data
print("Mean ± SD for SynII data:")
print(f"AMP1: {PCA_Data_SynII['AMP1'].mean():.3f} ± {PCA_Data_SynII['AMP1'].std():.3f}")
print(f"AMP2: {PCA_Data_SynII['AMP2'].mean():.3f} ± {PCA_Data_SynII['AMP2'].std():.3f}")
print(f"%Fail1: {PCA_Data_SynII['%Fail1'].mean():.3f} ± {PCA_Data_SynII['%Fail1'].std():.3f}")

### 10.7 SynII vs WT PPR Profiles

The train-normalized SynII profiles test whether Synapsin II loss preferentially alters sustained facilitation, early paired-pulse behavior, or both. This directly supports the claim that Synapsin II contributes to specific subsets of STP phenotypes.


In [ ]:
from scipy.stats import f as f_dist

# Identify and analyze SynII-enriched clusters

# PPR profiles comparison
if 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining ppr_profile_stats).')

ppr_cols = [
    f'PPR{i}/1'
    for i in range(2, 11)
    if f'PPR{i}/1' in PCA_Data_WT_Pooled.columns and f'PPR{i}/1' in PCA_Data_SynII.columns
]

pulse_numbers, WT_means, WT_sems, n_wt = ppr_profile_stats(PCA_Data_WT_Pooled, ppr_column_names=ppr_cols)
_, synii_means, synii_sems, n_syn = ppr_profile_stats(PCA_Data_SynII, ppr_column_names=ppr_cols)

fig, ax = make_figure_grid(figsize=(8, 5))
plot_ppr_mean_sem(
    ax,
    pulse_numbers,
    WT_means,
    WT_sems,
    color=get_wt_ca_color('2.5mM'),
    marker='o',
    label=f'WT Pooled (n={n_wt})',
    linewidth=1,
    sem_alpha=0.2,
)
plot_ppr_mean_sem(
    ax,
    pulse_numbers,
    synii_means,
    synii_sems,
    color=SYNII_COLOR,
    marker='s',
    label=f'SynII KO (n={n_syn})',
    linewidth=1,
    sem_alpha=0.2,
)

finalize_ppr_axis(
    ax,
    pulse_numbers,
    title='PPR Profiles: WT vs SynII KO',
    ylabel='Mean PPR (A_n/A_1)',
    legend=True,
    legend_loc='best',
)
plt.tight_layout()

output_file = OUTPUT_DIR / '10_10_synii_enriched_cluster_ppr_comparison.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Saved enriched clusters comparison to {output_file}')

# === STATISTICAL ANALYSIS ===
# Prepare data in wide format for repeated measures ANOVA

# Build matrices: rows = subjects, columns = pulses
wt_matrix = PCA_Data_WT_Pooled[ppr_cols].dropna().values
synii_matrix = PCA_Data_SynII[ppr_cols].dropna().values

n_wt = wt_matrix.shape[0]
n_synii = synii_matrix.shape[0]
n_total = n_wt + n_synii
k = len(ppr_cols)  # number of pulses (within-subject levels)

# Combined matrix
data_matrix = np.vstack([wt_matrix, synii_matrix])
group_labels = np.array([0]*n_wt + [1]*n_synii)  # 0=WT, 1=SynII

# === MIXED ANOVA (Group: between, Pulse: within) ===
# Manual computation of Sum of Squares

# Grand mean
grand_mean = np.nanmean(data_matrix)

# Group means (across all pulses)
wt_group_mean = np.nanmean(wt_matrix)
synii_group_mean = np.nanmean(synii_matrix)

# Pulse means (across all subjects)
pulse_means = np.nanmean(data_matrix, axis=0)

# Subject means (across all pulses)
subject_means = np.nanmean(data_matrix, axis=1)

# Cell means (group x pulse)
wt_pulse_means = np.nanmean(wt_matrix, axis=0)
synii_pulse_means = np.nanmean(synii_matrix, axis=0)

# --- Sum of Squares ---

# SS_between (Group effect)
SS_between = k * (n_wt * (wt_group_mean - grand_mean)**2 + 
                  n_synii * (synii_group_mean - grand_mean)**2)

# SS_subjects_within_groups (error term for between-subjects)
SS_subjects_wt = k * np.sum((np.nanmean(wt_matrix, axis=1) - wt_group_mean)**2)
SS_subjects_synii = k * np.sum((np.nanmean(synii_matrix, axis=1) - synii_group_mean)**2)
SS_subjects_within = SS_subjects_wt + SS_subjects_synii

# SS_within (Pulse effect)
SS_within = n_total * np.sum((pulse_means - grand_mean)**2)

# SS_interaction (Group x Pulse)
SS_interaction = 0
for j in range(k):
    SS_interaction += n_wt * (wt_pulse_means[j] - wt_group_mean - pulse_means[j] + grand_mean)**2
    SS_interaction += n_synii * (synii_pulse_means[j] - synii_group_mean - pulse_means[j] + grand_mean)**2

# SS_error (residual for within-subjects)
SS_total = np.nansum((data_matrix - grand_mean)**2)
SS_error = SS_total - SS_between - SS_subjects_within - SS_within - SS_interaction

# --- Degrees of Freedom ---
df_between = 1  # 2 groups - 1
df_subjects_within = n_total - 2  # n - a
df_within = k - 1  # k levels - 1
df_interaction = (2 - 1) * (k - 1)  # (a-1)(k-1)
df_error = (n_total - 2) * (k - 1)  # (n-a)(k-1)

# --- Mean Squares ---
MS_between = SS_between / df_between
MS_subjects_within = SS_subjects_within / df_subjects_within
MS_within = SS_within / df_within
MS_interaction = SS_interaction / df_interaction
MS_error = SS_error / df_error

# --- F-statistics ---
F_group = MS_between / MS_subjects_within  # Between-subjects effect
F_pulse = MS_within / MS_error  # Within-subjects effect
F_interaction = MS_interaction / MS_error  # Interaction effect

# --- p-values ---
p_group = 1 - f_dist.cdf(F_group, df_between, df_subjects_within)
p_pulse = 1 - f_dist.cdf(F_pulse, df_within, df_error)
p_interaction = 1 - f_dist.cdf(F_interaction, df_interaction, df_error)

# --- Effect sizes (partial eta-squared) ---
eta2_group = SS_between / (SS_between + SS_subjects_within)
eta2_pulse = SS_within / (SS_within + SS_error)
eta2_interaction = SS_interaction / (SS_interaction + SS_error)

print("\n" + "="*70)
print("MIXED ANOVA (Group: between-subjects, Pulse: within-subjects)")
print("="*70)
print(f"\n{'Source':<25} {'SS':>12} {'df':>6} {'MS':>12} {'F':>10} {'p':>12} {'η²p':>8}")
print("-"*70)
print(f"{'Group (WT vs SynII)':<25} {SS_between:>12.3f} {df_between:>6} {MS_between:>12.3f} {F_group:>10.3f} {p_group:>12.6f} {eta2_group:>8.3f}")
print(f"{'Subjects(Group)':<25} {SS_subjects_within:>12.3f} {df_subjects_within:>6} {MS_subjects_within:>12.3f}")
print(f"{'Pulse':<25} {SS_within:>12.3f} {df_within:>6} {MS_within:>12.3f} {F_pulse:>10.3f} {p_pulse:>12.6f} {eta2_pulse:>8.3f}")
print(f"{'Group × Pulse':<25} {SS_interaction:>12.3f} {df_interaction:>6} {MS_interaction:>12.3f} {F_interaction:>10.3f} {p_interaction:>12.6f} {eta2_interaction:>8.3f}")
print(f"{'Error':<25} {SS_error:>12.3f} {df_error:>6} {MS_error:>12.3f}")
print("-"*70)

print(f"\nSignificance (α=0.05):")
print(f"  • Group effect:       {'✓ Significant' if p_group < 0.05 else '✗ Not significant'} (p={p_group:.6f})")
print(f"  • Pulse effect:       {'✓ Significant' if p_pulse < 0.05 else '✗ Not significant'} (p={p_pulse:.6f})")
print(f"  • Interaction effect: {'✓ Significant' if p_interaction < 0.05 else '✗ Not significant'} (p={p_interaction:.6f})")

# Prepare long format for post-hoc
wt_long_data = []
for idx, row in PCA_Data_WT_Pooled.iterrows():
    subject_id = f"WT_{idx}"
    for pulse_num, col in enumerate(ppr_cols, start=2):
        wt_long_data.append({
            'Subject': subject_id,
            'Group': 'WT',
            'Pulse': pulse_num,
            'PPR': row[col]
        })

synii_long_data = []
for idx, row in PCA_Data_SynII.iterrows():
    subject_id = f"SynII_{idx}"
    for pulse_num, col in enumerate(ppr_cols, start=2):
        synii_long_data.append({
            'Subject': subject_id,
            'Group': 'SynII',
            'Pulse': pulse_num,
            'PPR': row[col]
        })

long_df = pd.DataFrame(wt_long_data + synii_long_data)
long_df = long_df.dropna(subset=['PPR'])

# === POST-HOC COMPARISONS PER PULSE ===
print("\n" + "="*70)
print("POST-HOC COMPARISONS (Mann-Whitney U per Pulse, FDR-corrected)")
print("="*70)

posthoc_results = []
for pulse_num in sorted(long_df['Pulse'].unique()):
    pulse_data = long_df[long_df['Pulse'] == pulse_num]
    wt_vals = pulse_data[pulse_data['Group'] == 'WT']['PPR'].values
    synii_vals = pulse_data[pulse_data['Group'] == 'SynII']['PPR'].values
    
    if len(wt_vals) > 1 and len(synii_vals) > 1:
        stat, p = mannwhitneyu(wt_vals, synii_vals, alternative='two-sided')
        posthoc_results.append({
            'Pulse': pulse_num,
            'WT_mean': np.mean(wt_vals),
            'WT_sem': np.std(wt_vals) / np.sqrt(len(wt_vals)),
            'WT_n': len(wt_vals),
            'SynII_mean': np.mean(synii_vals),
            'SynII_sem': np.std(synii_vals) / np.sqrt(len(synii_vals)),
            'SynII_n': len(synii_vals),
            'U_statistic': stat,
            'p_value_raw': p
        })

posthoc_df = pd.DataFrame(posthoc_results)

# Apply FDR correction for multiple comparisons
if len(posthoc_df) > 0:
    _, corrected_p, _, _ = multipletests(posthoc_df['p_value_raw'], method='fdr_bh')
    posthoc_df['p_value_corrected'] = corrected_p
    posthoc_df['significant'] = posthoc_df['p_value_corrected'] < 0.05

print(posthoc_df.to_string())

# Summary statistics
summary_stats = []
for group_name, group_df in [('WT', PCA_Data_WT_Pooled), ('SynII', PCA_Data_SynII)]:
    for param in ['AMP1', 'AMP2', '%Fail1'] + ppr_cols:
        if param in group_df.columns:
            vals = group_df[param].dropna()
            summary_stats.append({
                'Group': group_name,
                'Parameter': param,
                'Mean': vals.mean(),
                'SD': vals.std(),
                'SEM': vals.sem(),
                'N': len(vals),
                'Median': vals.median(),
                'Min': vals.min(),
                'Max': vals.max()
            })

summary_df = pd.DataFrame(summary_stats)

# Save all statistics to Excel
stats_output_file = OUTPUT_DIR / "Fig6f_WT_vs_SynII_PPR_statistics.xlsx"
with pd.ExcelWriter(stats_output_file, engine='openpyxl') as writer:
    # Mixed ANOVA results
    mixed_anova_df = pd.DataFrame({
        'Source': ['Group (WT vs SynII)', 'Subjects(Group)', 'Pulse', 'Group × Pulse', 'Error'],
        'SS': [SS_between, SS_subjects_within, SS_within, SS_interaction, SS_error],
        'df': [df_between, df_subjects_within, df_within, df_interaction, df_error],
        'MS': [MS_between, MS_subjects_within, MS_within, MS_interaction, MS_error],
        'F': [F_group, np.nan, F_pulse, F_interaction, np.nan],
        'p_value': [p_group, np.nan, p_pulse, p_interaction, np.nan],
        'partial_eta2': [eta2_group, np.nan, eta2_pulse, eta2_interaction, np.nan],
        'Significant': [p_group < 0.05, np.nan, p_pulse < 0.05, p_interaction < 0.05, np.nan]
    })
    mixed_anova_df.to_excel(writer, sheet_name='Mixed_ANOVA', index=False)
    posthoc_df.to_excel(writer, sheet_name='PostHoc_PerPulse', index=False)
    summary_df.to_excel(writer, sheet_name='Summary_Statistics', index=False)

print(f"\n✓ Saved statistics to {stats_output_file}")

# SynII summary statistics
print(f"\nSynII Summary Statistics:")
for param in ['AMP1', 'AMP2', '%Fail1']:
    if param in PCA_Data_SynII.columns:
        mean_val = PCA_Data_SynII[param].mean()
        std_val  = PCA_Data_SynII[param].std()
        print(f"{param}: {mean_val:.3f} ± {std_val:.3f}")


### 10.8 SynII vs WT Mean Trace Overlays

Mean-trace overlays show the genotype effect in the original fluorescence domain rather than only in reduced coordinates or normalized profiles. They are particularly useful for relating SynII changes in PPR to changes in absolute glutamate output. 


In [ ]:
# Compare mean traces between SynII and WT with cluster assignments

if 'cluster_synII' not in PCA_Data_SynII.columns:
    PCA_Data_SynII['cluster_synII'] = synii_cluster_assignments
synii_stats = compute_trace_stats(trace_ids=PCA_Data_SynII['ID'], source=TRACE_MEAN_SOURCE, condition_names=get_synapsin_conditions())
wt_stats = compute_trace_stats(trace_ids=PCA_Data_WT_Pooled_clustered['ID'], source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
all_values = np.concatenate([synii_stats['average'] - synii_stats['sem'], synii_stats['average'] + synii_stats['sem'], wt_stats['average'] - wt_stats['sem'], wt_stats['average'] + wt_stats['sem']])
y_min, y_max = np.nanmin(all_values), np.nanmax(all_values)
y_padding = (y_max - y_min) * 0.05
y_lim = (y_min - y_padding, y_max + y_padding)
fig, (ax_synii, ax_wt, ax_overlay) = make_figure_grid(1, 3, figsize=(16, 5), sharey=True)
plot_traces(ax=ax_synii, rows=synii_stats['rows'], source=TRACE_MEAN_SOURCE, color=SYNII_COLOR, label=f'SynII KO (n={synii_stats["n"]})', show_average=True, show_sem=True, zero_line=True, event_time=1.0, event_kwargs={'color': 'red', 'linestyle': '--', 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, ylim=y_lim, xlabel='Time (s)', ylabel='ΔF/F', title='SynII KO - Average Response', legend=True)
plot_traces(ax=ax_wt, rows=wt_stats['rows'], source=TRACE_MEAN_SOURCE, color='black', label=f'WT (n={wt_stats["n"]})', show_average=True, show_sem=True, zero_line=True, event_time=1.0, event_kwargs={'color': 'red', 'linestyle': '--', 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, ylim=y_lim, xlabel='Time (s)', title='WT - Average Response', legend=True)
plot_traces(ax=ax_overlay, rows=wt_stats['rows'], source=TRACE_MEAN_SOURCE, color='black', label=f'WT (n={wt_stats["n"]})', show_average=True, show_sem=True, sem_alpha=0.2, style_axis=False)
plot_traces(ax=ax_overlay, rows=synii_stats['rows'], source=TRACE_MEAN_SOURCE, color=SYNII_COLOR, label=f'SynII KO (n={synii_stats["n"]})', show_average=True, show_sem=True, sem_alpha=0.2, zero_line=True, event_time=1.0, event_kwargs={'color': 'red', 'linestyle': '--', 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, ylim=y_lim, xlabel='Time (s)', title='Overlay - SynII vs WT', legend=True)
plt.tight_layout()
output_file = OUTPUT_DIR / "10_11_synii_vs_wt_mean_traces.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Mean trace comparison: SynII (n={synii_stats['n']}) vs WT (n={wt_stats['n']})")
print(f"✓ Time axis focused on stimulus response period {TRACE_XLIM_20HZ}")
if enriched_clusters:
    print(f'\nSynII distribution in enriched clusters {enriched_clusters}:')
    for cluster_id in enriched_clusters:
        count = np.sum(synii_cluster_assignments == cluster_id)
        percent = 100 * count / len(synii_cluster_assignments)
        print(f"  C{cluster_id}: {count} ({percent:.1f}%)")
print(f"✓ Saved to {output_file}")


### 10.9 Cluster-Level SynII Comparisons

These cluster-wise comparisons ask whether SynII effects are global or concentrated in specific sectors of the WT bouton taxonomy. This is one of the most direct ways to test the hypothesis of class-selective molecular regulation. 


In [ ]:
# Create combined WT pooled + SynII dataset with cluster labels

# Start with WT pooled clustered data
PCA_Data_WT_pooled_SynII_Clust = PCA_Data_WT_Pooled_clustered.copy()

# Add SynII data with 'SynII KO' as cluster label
synii_data_with_cluster = PCA_Data_SynII.copy()
synii_data_with_cluster['HC_Cluster'] = 7

# Concatenate both datasets
PCA_Data_WT_pooled_SynII_Clust = pd.concat([
    PCA_Data_WT_pooled_SynII_Clust,
    synii_data_with_cluster
], ignore_index=True)

print(f"✓ Created PCA_Data_WT_pooled_SynII_Clust dataset")
print(f"  WT clustered samples: {len(PCA_Data_WT_Pooled_clustered)}")
print(f"  SynII samples: {len(PCA_Data_SynII)}")
print(f"  Total samples: {len(PCA_Data_WT_pooled_SynII_Clust)}")
print(f"\nCluster distribution:")
print(PCA_Data_WT_pooled_SynII_Clust['HC_Cluster'].value_counts().sort_index())

In [ ]:
# Reuse shared helper: cluster_boxplot_analysis

cluster_boxplot_analysis(PCA_Data_WT_pooled_SynII_Clust, '%Fail1', '%Fail1 by Cluster', '10_12_synii_fail1_cluster_boxplot')

In [ ]:
from scipy.stats import mannwhitneyu

# Boxplot comparison: WT pooled vs SynII for AMP1, PPR2/1, PPR3/1


# Prepare combined data
wt_data = PCA_Data_WT_Pooled[['AMP1', 'PPR2/1', 'PPR3/1']].copy()
wt_data['Genotype'] = 'WT'

synii_data = PCA_Data_SynII[['AMP1', 'PPR2/1', 'PPR3/1']].copy()
synii_data['Genotype'] = 'SynII'

combined_data = pd.concat([wt_data, synii_data], ignore_index=True)

# Create figure with 3 subplots
fig, axes = make_figure_grid(1, 3, figsize=(12, 5))
parameters = ['AMP1', 'PPR2/1', 'PPR3/1']
colors = {'WT': WT_THEO_COLOR, 'SynII': SYNII_COLOR}

stats_results = []

for idx, param in enumerate(parameters):
    ax = axes[idx]
    
    # Boxplot
    sns.boxplot(x='Genotype', y=param, hue='Genotype', data=combined_data, 
                palette=colors, ax=ax, legend=False)
    sns.stripplot(x='Genotype', y=param, data=combined_data, 
                  color='black', size=3, alpha=0.5, ax=ax)
    
    # Add reference line for PPR
    if 'PPR' in param:
        ax.axhline(1, color='gray', linestyle='dotted', linewidth=1.5)
    
    # Statistical test
    wt_vals = combined_data.loc[combined_data['Genotype'] == 'WT', param].dropna()
    synii_vals = combined_data.loc[combined_data['Genotype'] == 'SynII', param].dropna()
    
    stat, p_val = mannwhitneyu(wt_vals, synii_vals, alternative='two-sided')
    stats_results.append({'Parameter': param, 'U_statistic': stat, 'p_value': p_val,
                          'n_WT': len(wt_vals), 'n_SynII': len(synii_vals)})
    
    # Add significance annotation
    y_max = combined_data[param].max()
    y_range = combined_data[param].max() - combined_data[param].min()
    
    if p_val < 0.001:
        sig_text = '***'
    elif p_val < 0.01:
        sig_text = '**'
    elif p_val < 0.05:
        sig_text = '*'
    else:
        sig_text = 'ns'
    
    ax.plot([0, 0, 1, 1], [y_max + 0.05*y_range, y_max + 0.1*y_range, 
                           y_max + 0.1*y_range, y_max + 0.05*y_range], 
            color='black', linewidth=1)
    ax.text(0.5, y_max + 0.12*y_range, sig_text, ha='center', va='bottom', fontsize=12)
    
    ax.set_title(f'{param}\n(p={p_val:.4g})')
    ax.set_xlabel('')
    ax.set_ylabel(param)
    
    # Clean styling
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'WT (n={len(wt_vals)}) vs SynII (n={len(synii_vals)})', fontsize=14, fontweight='bold')
plt.tight_layout()

output_file = OUTPUT_DIR / "10_12_synii_vs_wt_amp1_ppr_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Save statistics
stats_df = pd.DataFrame(stats_results)
stats_file = OUTPUT_DIR / "Fig6_WT_vs_SynII_statistics.xlsx"
stats_df.to_excel(stats_file, index=False)

print(f"✓ Saved boxplots to {output_file}")
print(f"✓ Saved statistics to {stats_file}")
print("\n=== Statistical Results (Mann-Whitney U) ===")
for res in stats_results:
    print(f"{res['Parameter']}: U={res['U_statistic']:.1f}, p={res['p_value']:.4g} (WT n={res['n_WT']}, SynII n={res['n_SynII']})")

### 10.10 High-Amplitude WT Boutons and the SynII Phenotype

The high-amplitude WT subset is separated here because it represents the upper end of the WT synaptic-weight distribution. Comparing this subset to SynII boutons helps determine whether the genotype specifically eliminates the largest-output states or broadly compresses the full distribution.


#### 10.10.1 Identify High-Amplitude WT Boutons

This thresholding step isolates the strongest WT boutons in the original fluorescence space. It provides a concrete subset for testing whether SynII loss selectively removes high-output release states. 


#### 10.10.2 Mean Trace Comparison at the High-Amplitude End

The mean-trace comparison asks whether the largest WT responses have a time-domain signature that resembles or diverges from the SynII phenotype. This adds temporal interpretation to the threshold-based subset analysis.


In [ ]:
# Extract WT pooled trace data and identify high-amplitude traces
# Resolve threshold locally so this cell is runnable even if execution order changed.

A1_START_LOCAL = globals().get('A1_START', 1.000)
A1_END_LOCAL = globals().get('A1_END', 1.049)

def _a1_window_amp_local(trace_time, trace_values):
    tt = np.asarray(trace_time, dtype=float)
    tv = np.asarray(trace_values, dtype=float)
    if tt.shape[0] != tv.shape[0]:
        return np.nan
    a1_mask_local = (tt >= A1_START_LOCAL) & (tt <= A1_END_LOCAL)
    w = tv[a1_mask_local]
    if w.size == 0 or np.all(~np.isfinite(w)):
        return np.nan
    return np.nanmax(np.abs(w))

if 'synii_a1_threshold' in globals() and np.isfinite(synii_a1_threshold):
    synii_amplitude_threshold = float(synii_a1_threshold)
else:
    synii_a1_values_local = []
    synii_rows_local = select_traces(condition_names=SYNAPSIN_CONDITIONS, source=TRACE_SINGLE_SOURCE)
    for _, trace_row in synii_rows_local.iterrows():
        amp_a1 = _a1_window_amp_local(trace_row['Time'], trace_row['Avg'])
        if np.isfinite(amp_a1):
            synii_a1_values_local.append(amp_a1)
    synii_amplitude_threshold = np.percentile(synii_a1_values_local, 95) if len(synii_a1_values_local) > 0 else np.nan
if not np.isfinite(synii_amplitude_threshold):
    raise ValueError('Could not compute SynII amplitude threshold for this cell')
wt_trace_data = []
wt_high_amplitude_traces = []
wt_regular_traces = []
wt_trace_rows = select_traces(trace_ids=PCA_Data_WT_Pooled_clustered['ID'], condition_names=WT_2_5_20HZ_CONDITIONS, source=TRACE_SINGLE_SOURCE)
for _, trace_row in wt_trace_rows.iterrows():
    bouton_id = trace_row['ID']
    bouton_base_id = _normalize_bouton_id(bouton_id)
    trace_time = np.asarray(trace_row['Time'], float)
    trace_values = np.asarray(trace_row['Avg'], float)
    wt_trace_data.append((bouton_id, trace_time, trace_values))
    matching_feature = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['BaseID'] == bouton_base_id]
    if not matching_feature.empty:
        feature_row = matching_feature.iloc[0]
        amp_columns = [f'AMP{i}' for i in range(1, 11)]
        amp_values = []
        for col in amp_columns:
            if col in matching_feature.columns:
                val = feature_row[col]
                if pd.notna(val):
                    amp_values.append(abs(val))
        if len(amp_values) > 0:
            canonical_id = feature_row['ID']
            max_amp = np.max(amp_values)
            if max_amp > synii_amplitude_threshold:
                wt_high_amplitude_traces.append((canonical_id, trace_time, trace_values, max_amp))
            else:
                wt_regular_traces.append((canonical_id, trace_time, trace_values))
if 'synii_trace_data_a1' in globals() and len(synii_trace_data_a1) > 0:
    synii_trace_data_local = synii_trace_data_a1
else:
    synii_trace_data_local = []
    synii_rows_local = select_traces(condition_names=SYNAPSIN_CONDITIONS, source=TRACE_SINGLE_SOURCE)
    for _, trace_row in synii_rows_local.iterrows():
        synii_trace_data_local.append((trace_row['ID'], trace_row['Time'], trace_row['Avg']))
wt_regular_stats = compute_trace_stats(trace_ids=[bouton_id for bouton_id, _, _ in wt_regular_traces], source=TRACE_SINGLE_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
wt_high_stats = compute_trace_stats(trace_ids=[bouton_id for bouton_id, _, _, _ in wt_high_amplitude_traces], source=TRACE_SINGLE_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
synii_stats = compute_trace_stats(condition_names=SYNAPSIN_CONDITIONS, source=TRACE_SINGLE_SOURCE)
fig, ax = make_figure_grid(figsize=(10, 6))
plot_traces(ax=ax, rows=wt_regular_stats['rows'], source=TRACE_SINGLE_SOURCE, color='black', label='WT Below Threshold Mean', show_average=True, show_sem=False, style_axis=False)
plot_traces(ax=ax, rows=wt_high_stats['rows'], source=TRACE_SINGLE_SOURCE, color='red', label='WT Above Threshold Mean', show_average=True, show_sem=False, style_axis=False)
plot_traces(ax=ax, rows=synii_stats['rows'], source=TRACE_SINGLE_SOURCE, color=SYNII_COLOR, label='SynII Mean', show_average=True, show_sem=False, xlim=TRACE_XLIM_20HZ, xlabel='Time (s)', ylabel='ΔF/F', title='Mean WT Traces (Below and Above Threshold)', legend=True)
plt.tight_layout()
output_file = OUTPUT_DIR / '10_13_wt_high_amplitude_mean_traces.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'✓ SynII threshold used in this cell: {synii_amplitude_threshold:.4f}')
print(f'✓ WT above-threshold traces: {len(wt_high_amplitude_traces)} / {len(wt_trace_data)}')


#### 10.10.3 Map High-Amplitude WT Boutons in PCA Space

Mapping the high-amplitude WT boutons back onto the PCA shows whether they occupy a distinct sector of the WT state space. This is useful when interpreting which portions of that space are absent or compressed in SynII. 


In [ ]:
# Show PCA locations of high-amplitude WT traces identified in the previous cell (ignoring target type)
if wt_high_amplitude_traces:
    high_amp_base_ids = [_normalize_bouton_id(item[0]) for item in wt_high_amplitude_traces]
    high_amp_mask = PCA_Data_WT_Pooled_clustered['BaseID'].isin(high_amp_base_ids)
    high_amp_coordinates = pca_coordinates[high_amp_mask]   

    make_figure(figsize=(8, 6)) 
# Plot WT pooled data with cluster colors
    plot_pca_background(plt.gca(), alpha=0.4,  label='WT pooled (2.5mM Ca)')
    
# Plot high-amplitude WT traces
    plot_pca_overlay_points(plt.gca(), high_amp_coordinates[:], 
               marker='p', c='red', edgecolors='black', linewidths=0.6,  label=f'High-Amplitude WT (n={len(high_amp_coordinates)})')
    
# Format plot
    style_pca_axes(plt.gca(), title='PCA Locations of High-Amplitude WT Traces',  legend=False)
    add_legend(plt.gca(), frameon=False)
    plt.tight_layout()
    output_file = OUTPUT_DIR / "10_14_wt_high_amplitude_pca_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()


### 10.11 SynII and Postsynaptic Target Identity

Target identity is revisited under SynII loss to test whether the genotype effect interacts with postsynaptic class. The logic parallels the WT target analysis while asking a more specific molecular question. 


#### 10.11.1 Confidence-Ellipse Comparison

Confidence-ellipse analyses summarize whether the SynII perturbation changes how target-identified boutons occupy the reduced state space. This offers a compact geometric counterpart to the trace-based target comparisons. 


In [ ]:
def compare_inside_outside_simplified(in_ids, in_inside_mask, pc_ids, pc_inside_mask):
    """Create clean 2-panel comparison of inside vs outside ellipse traces."""
    in_inside_ids = [in_ids[i] for i in range(len(in_ids)) if in_inside_mask[i]]
    in_outside_ids = [in_ids[i] for i in range(len(in_ids)) if not in_inside_mask[i]]
    pc_inside_ids = [pc_ids[i] for i in range(len(pc_ids)) if pc_inside_mask[i]]
    pc_outside_ids = [pc_ids[i] for i in range(len(pc_ids)) if not pc_inside_mask[i]]
    in_inside_stats = compute_trace_stats(trace_ids=in_inside_ids, source=TRACE_MEAN_SOURCE)
    in_outside_stats = compute_trace_stats(trace_ids=in_outside_ids, source=TRACE_MEAN_SOURCE)
    pc_inside_stats = compute_trace_stats(trace_ids=pc_inside_ids, source=TRACE_MEAN_SOURCE)
    pc_outside_stats = compute_trace_stats(trace_ids=pc_outside_ids, source=TRACE_MEAN_SOURCE)
    fig, (ax1, ax2) = make_figure_grid(1, 2, figsize=(14, 5))
    plot_traces(ax=ax1, rows=in_inside_stats['rows'], source=TRACE_MEAN_SOURCE, color=IN_TARGET_COLOR, label=f'Inside ellipse (n={in_inside_stats["n"]})', show_average=True, show_sem=True)
    plot_traces(ax=ax1, rows=in_outside_stats['rows'], source=TRACE_MEAN_SOURCE, color='black', label=f'Outside ellipse (n={in_outside_stats["n"]})', show_average=True, show_sem=True, xlim=TRACE_XLIM_20HZ, xlabel='Time (s)', ylabel='ΔF/F', title='IN Cells: Inside vs Outside SynII Ellipse', legend=True)
    ax1.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax1.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    plot_traces(ax=ax2, rows=pc_inside_stats['rows'], source=TRACE_MEAN_SOURCE, color=PC_TARGET_COLOR, label=f'Inside ellipse (n={pc_inside_stats["n"]})', show_average=True, show_sem=True)
    plot_traces(ax=ax2, rows=pc_outside_stats['rows'], source=TRACE_MEAN_SOURCE, color='black', label=f'Outside ellipse (n={pc_outside_stats["n"]})', show_average=True, show_sem=True, xlim=TRACE_XLIM_20HZ, xlabel='Time (s)', ylabel='ΔF/F', title='PC Cells: Inside vs Outside SynII Ellipse', legend=True)
    ax2.axhline(0, color='gray', linestyle='dotted', linewidth=1)
    ax2.axvline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    plt.tight_layout()
    output_file = OUTPUT_DIR / "10_15_synii_ellipse_inside_outside_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ IN: Inside n={in_inside_stats['n']}, Outside n={in_outside_stats['n']}")
    print(f"✓ PC: Inside n={pc_inside_stats['n']}, Outside n={pc_outside_stats['n']}")

compare_inside_outside_simplified(in_ids, in_inside_mask, pc_ids, pc_inside_mask)


#### 10.11.2 Target Composition Within the SynII Space

The target-composition summaries test whether PC- and interneuron-associated boutons remain mixed within the SynII state space or become more segregated under the perturbation. 


In [ ]:
# Four-panel comparison: IN/PC inside/outside SynII ellipse traces

def plot_group_traces(ax, trace_ids, group_name, color, show_individuals=True):
    """Plot individual traces + average for a group."""
    return plot_traces(ax=ax, trace_ids=trace_ids, source=TRACE_SINGLE_SOURCE, show_average=True, show_sem=True, show_individuals=show_individuals, alignment='grid', color=color, individual_color=color, linewidth=1.0, sem_alpha=0.3, individual_alpha=0.2, individual_lw=0.5, label=f'{group_name} avg', zero_line=True, event_time=1.0, event_kwargs={'color': 'black', 'linestyle': '--', 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, title=f'{group_name}\n(n={len(trace_ids)})', return_data=True)

in_inside_ids = [in_ids[i] for i in range(len(in_ids)) if in_inside_mask[i]]
in_outside_ids = [in_ids[i] for i in range(len(in_ids)) if not in_inside_mask[i]]
pc_inside_ids = [pc_ids[i] for i in range(len(pc_ids)) if pc_inside_mask[i]]
pc_outside_ids = [pc_ids[i] for i in range(len(pc_ids)) if not pc_inside_mask[i]]
fig, axes = make_figure_grid(2, 2, figsize=(14, 10), sharex=True, sharey=True)
plot_group_traces(axes[0, 0], in_inside_ids, 'IN Inside Ellipse', IN_TARGET_COLOR)
plot_group_traces(axes[0, 1], in_outside_ids, 'IN Outside Ellipse', IN_TARGET_MEDIAN_COLOR)
plot_group_traces(axes[1, 0], pc_inside_ids, 'PC Inside Ellipse', PC_TARGET_COLOR)
plot_group_traces(axes[1, 1], pc_outside_ids, 'PC Outside Ellipse', PC_TARGET_MEDIAN_COLOR)
for ax in axes[-1, :]:
    ax.set_xlabel('Time (s)')
for ax in axes[:, 0]:
    ax.set_ylabel('ΔF/F')
fig.suptitle(f'Trace Analysis: Inside vs Outside SynII {ELLIPSE_CONFIDENCE:.0%} Ellipse', fontsize=14, fontweight='bold')
plt.tight_layout()
output_file = OUTPUT_DIR / "10_16_synii_four_panel_ellipse_trace_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"=== ELLIPSE TRACE ANALYSIS SUMMARY ===")
print(f"Ellipse confidence level: {ELLIPSE_CONFIDENCE:.0%}")
print(f"IN inside ellipse:  {len(in_inside_ids):2d}  traces ({len(in_inside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
print(f"IN outside ellipse: {len(in_outside_ids):2d} traces ({len(in_outside_ids)/(len(in_inside_ids)+len(in_outside_ids))*100:.1f}%)")
print(f"PC inside ellipse:  {len(pc_inside_ids):2d}  traces ({len(pc_inside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
print(f"PC outside ellipse: {len(pc_outside_ids):2d} traces ({len(pc_outside_ids)/(len(pc_inside_ids)+len(pc_outside_ids))*100:.1f}%)")
print(f'\n✓ Saved four-panel comparison to {output_file}')


### 10.12 NP on SynII

The remaining SynII analyses focus on direct comparison to WT and on WT-reference-space interpretation of SynII boutons, without a separate SynII-only HCPC layer.


In [ ]:
# %% ###############################################################
# CELL F : SynII Binomial MLE N/P maps + refilling (shared-threshold failures)
#   - Failures are called per event from min(AMP_CORR[k], AMP_UNCORR[k]) < thr_shared
#   - Refilling = slope of cumulative NP over end of train (last N_FIT_LAST pulses)
#   - Normalised refilling = slope / NP1
# ###################################################################

from scipy.stats import linregress

# == Reload SynII trials from raw Excel (not in trials_all) ======
_synii_trials = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
_synii_trials = _synii_trials[_synii_trials['condition'] == 'SynII']
print(f"SynII trials loaded from Excel: {len(_synii_trials)} rows")

# == Build count dict for SynII ===================================
tg_F = defaultdict(list)
for _, r in _synii_trials.iterrows():
    key = extract_base_name(str(r['file']).strip())
    tg_F[key].append(get_count_row_with_failures(r, Q_D))

# == Match SynII summary DF rows -> counts, run MLE ===============
synii_xy_all = np.array(pca_data['SynII'])
synii_ids    = PCA_Data_SynII['ID'].astype(str).str.strip().values
n_synii_F    = min(len(synii_xy_all), len(synii_ids))
synii_xy     = synii_xy_all[:n_synii_F]

F_N = np.full((n_synii_F, N_STIM), np.nan)
F_P = np.full((n_synii_F, N_STIM), np.nan)
F_fail = np.full((n_synii_F, N_STIM), np.nan)
n_matched_F, n_fit_F = 0, 0

for i in range(n_synii_F):
    key = extract_base_name(synii_ids[i])
    if key not in tg_F:
        continue
    qc = np.array(tg_F[key])
    ok_rows = np.all(np.isfinite(qc), axis=1)
    qc = qc[ok_rows].astype(int)
    if len(qc) < MIN_TRIALS_D:
        continue
    n_matched_F += 1
    F_fail[i, :] = np.mean(qc == 0, axis=0)
    ntr = len(qc)

    if USE_BOOTSTRAP:
        rng_f = np.random.default_rng(1000 + i)
        bN = np.full((N_STIM, N_BOOT), np.nan)
        bP = np.full((N_STIM, N_BOOT), np.nan)
        for b in range(N_BOOT):
            qc_b = qc[rng_f.integers(0, ntr, ntr)]
            for k in range(N_STIM):
                qk = qc_b[:, k].astype(int)
                Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
                if np.isfinite(Nf) and Nf < N_MAX_D:
                    bN[k, b] = Nf
                    bP[k, b] = Pf
        for k in range(N_STIM):
            if np.isfinite(bN[k]).sum() >= N_BOOT * 0.5:
                F_N[i, k] = np.nanmedian(bN[k])
                F_P[i, k] = np.nanmedian(bP[k])
    else:
        for k in range(N_STIM):
            qk = qc[:, k].astype(int)
            Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
            if np.isfinite(Nf) and Nf < N_MAX_D:
                F_N[i, k] = Nf
                F_P[i, k] = Pf

    if np.isfinite(F_N[i, 0]):
        n_fit_F += 1

print(f"SynII: {n_synii_F} boutons, matched={n_matched_F}, fit={n_fit_F}")
if n_matched_F == 0 and len(tg_F) > 0:
    print(f"  trial keys sample : {sorted(tg_F.keys())[:3]}")
    print(f"  sdf keys sample   : {[extract_base_name(s) for s in synii_ids[:3]]}")

# == Derived: NP + refilling slope ================================
F_NP   = F_N * F_P
F_NP_1 = F_NP[:, 0]

F_NP_norm = np.where(
    np.isfinite(F_NP_1[:, None]) & (F_NP_1[:, None] > 0),
    F_NP / F_NP_1[:, None],
    np.nan
)

F_NFIT = int(min(globals().get('N_FIT_LAST', 4), N_STIM))
x_tail = np.arange(N_STIM - F_NFIT + 1, N_STIM + 1)

F_refill_slope = np.full(n_synii_F, np.nan)
for i in range(n_synii_F):
    tail = F_NP[i, -F_NFIT:]
    if not np.all(np.isfinite(tail)):
        continue
    cum_tail = np.cumsum(tail)
    sl, ic, r, p, se = linregress(x_tail, cum_tail)
    if np.isfinite(sl):
        F_refill_slope[i] = sl

F_refill_norm = np.where(
    np.isfinite(F_refill_slope) & np.isfinite(F_NP_1) & (F_NP_1 > 0),
    F_refill_slope / F_NP_1,
    np.nan
)

# == Reference WT 2.5 mM 20 Hz results ============================
wt_key_candidates = ['2.5 mM 20 Hz', '2.5 mM 20 Hz (WT)']
wt_key = next((k for k in wt_key_candidates if k in cond_results_E), None)
if wt_key is None:
    wt_key = next((k for k in cond_results_E.keys() if ('2.5 mM 20 Hz' in str(k)) and ('WT' in str(k))), None)
if wt_key is None:
    raise RuntimeError(f"WT 2.5 mM 20 Hz key not found in cond_results_E. Available keys: {list(cond_results_E.keys())}")

WT_E_N = cond_results_E[wt_key]['E_N']
WT_E_P = cond_results_E[wt_key]['E_P']
WT_NP = WT_E_N * WT_E_P
WT_NP1 = WT_NP[:, 0]

WT_refill_slope = np.full(WT_NP.shape[0], np.nan)
for i in range(WT_NP.shape[0]):
    tail = WT_NP[i, -F_NFIT:]
    if not np.all(np.isfinite(tail)):
        continue
    cum_tail = np.cumsum(tail)
    sl, ic, r, p, se = linregress(x_tail, cum_tail)
    if np.isfinite(sl):
        WT_refill_slope[i] = sl

WT_refill_norm = np.where(
    np.isfinite(WT_refill_slope) & np.isfinite(WT_NP1) & (WT_NP1 > 0),
    WT_refill_slope / WT_NP1,
    np.nan
)

# ##################################################################
# FIG F1 : N and P maps at stim 1, 2, 5, 10
# ##################################################################
F_STIM = STIM_SHOW
n_show_F = len(F_STIM)
nv_f, nxv_f = N_VLIM
pv_f, pxv_f = P_VLIM

gx_f, gy_f, ext_f = setup_pca_grid(synii_xy, pad=0.5)

fig_f1, axes_f1 = make_figure_grid(2, n_show_F, panel_kind='pca')
for col_f, sk in enumerate(F_STIM):
    kidx = sk - 1
    for row_f, (vals, cmap, vlim, vname) in enumerate([
            (F_N, 'Spectral_r', (nv_f, nxv_f), 'N'),
            (F_P, 'coolwarm',   (pv_f, pxv_f), 'P')]):
        ok = np.isfinite(vals[:, kidx])
        ax = axes_f1[row_f, col_f]
        vlo, vhi = vlim
        render_pca_scalar_panel(
            ax,
            synii_xy,
            vals[:, kidx],
            cmap=cmap,
            vmin=vlo,
            vmax=vhi,
            title=f'SynII {vname}: stim {sk} (n={ok.sum()})',
            point_size=18,
            min_points=3,
            empty_label=f'n={ok.sum()} (too few)',
        )
        if col_f == n_show_F - 1:
            add_scalar_colorbar(fig_f1, axes_f1[row_f, :], cmap=cmap, vmin=vlo, vmax=vhi, label=vname, shrink=0.7)

finalize_figure(
    fig_f1,
    title='Fig F1: SynII binomial MLE N and P maps',
    rect=[0, 0, 0.95, 0.95],
    save_path=OUTPUT_DIR / '10_19_synii_np_maps_sharedthr.pdf',
)

# ##################################################################
# FIG F2 : NP map (mean quantal content) at stim 1,2,5,10
# ##################################################################
_np_ok = np.isfinite(F_NP)
_np_vmin = np.nanpercentile(F_NP[_np_ok], 2) if _np_ok.any() else 0
_np_vmax = np.nanpercentile(F_NP[_np_ok], 98) if _np_ok.any() else 5

fig_f2, axes_f2 = make_figure_grid(1, n_show_F, panel_kind='pca', squeeze=False)
for col_f, sk in enumerate(F_STIM):
    kidx = sk - 1
    ok = np.isfinite(F_NP[:, kidx])
    ax = axes_f2[0, col_f]
    render_pca_scalar_panel(
        ax,
        synii_xy,
        F_NP[:, kidx],
        cmap='viridis',
        vmin=_np_vmin,
        vmax=_np_vmax,
        title=f'SynII NP: stim {sk} (n={ok.sum()})',
        point_size=18,
        min_points=3,
        empty_label=f'n={ok.sum()} (too few)',
    )
    if col_f == n_show_F - 1:
        add_scalar_colorbar(fig_f2, axes_f2[0, :], cmap='viridis', vmin=_np_vmin, vmax=_np_vmax, label='NP (quanta)', shrink=0.7)

finalize_figure(
    fig_f2,
    title='Fig F2: SynII NP map',
    rect=[0, 0, 0.93, 0.92],
    save_path=OUTPUT_DIR / '10_20_synii_refilling_sharedthr.pdf',
)

# ##################################################################
# FIG F3 : Refilling slope/NP1 map + isoline at 1.0
# ##################################################################
ok_rf = np.isfinite(F_refill_norm)
if ok_rf.sum() == 0:
    print("No finite SynII refilling slope/NP1 values to plot.")
else:
    v3, v4 = np.nanpercentile(F_refill_norm[ok_rf], [2, 98])
    v3 = 0.55
    v4 = 2.1

    fig_f3, ax = make_figure_grid(figsize=grid_size(1, 1, 'pca'))
    sg2, sig2 = smooth_field(synii_xy, F_refill_norm, ok_rf, gx_f, gy_f)
    Xg, Yg = np.meshgrid(gx_f, gy_f)

    im = ax.imshow(
        sg2,
        extent=ext_f,
        origin='lower',
        aspect='auto',
        cmap=_display_cmap('RdBu_r'),
        vmin=v3,
        vmax=v4,
        interpolation='bilinear',
        zorder=1,
        alpha=0.8,
    )

    if np.any(np.isfinite(sg2)):
        ax.contour(Xg, Yg, sg2, levels=[1.0], colors='black', linewidths=1.5, zorder=2)

    plot_pca_overlay_points(
        ax,
        synii_xy[ok_rf],
        c='gray',
        s=18,
        edgecolors='none',
        zorder=3,
        label='SynII',
    )

    plt.colorbar(im, ax=ax, shrink=0.7, label='refilling slope / NP$_1$')
    style_pca_axes(ax, title=f'SynII slope/NP$_1$ (σ={sig2:.2f}, n={ok_rf.sum()})', legend=False)

    fig_f3.tight_layout()
    fig_f3.savefig(
        OUTPUT_DIR / '10_21_synii_refilling_slope_norm_sharedthr.pdf',
        dpi=300,
        bbox_inches='tight'
    )
    plt.show()

# ##################################################################
# FIG F4 : Pooled N and P trajectories, SynII vs WT 2.5 mM 20 Hz
# ##################################################################
fig_f4, axes_f4 = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
wt_color = cond_results_E[wt_key].get('color', 'tab:orange')

plot_mean_sem_trace(
    axes_f4[0, 0], x_stim, F_P,
    color=SYNII_COLOR, label='SynII ({n})',
    marker='o', linestyle='-', ms=5, lw=1.8, fill_alpha=0.15
)
plot_mean_sem_trace(
    axes_f4[0, 0], x_stim, WT_E_P,
    color=wt_color, label='WT 2.5 mM 20 Hz ({n})',
    marker='o', linestyle='-', ms=5, lw=1.6, fill_alpha=0.12
)
style_ax(axes_f4[0, 0], 'Stimulus', '$P_k$', 'P trajectory')
axes_f4[0, 0].set_ylim(0, 1)
add_legend(axes_f4[0, 0], loc='best')

plot_mean_sem_trace(
    axes_f4[0, 1], x_stim, F_N,
    color=SYNII_COLOR, label='SynII ({n})',
    marker='o', linestyle='-', ms=5, lw=1.8, fill_alpha=0.15
)
plot_mean_sem_trace(
    axes_f4[0, 1], x_stim, WT_E_N,
    color=wt_color, label='WT 2.5 mM 20 Hz ({n})',
    marker='o', linestyle='-', ms=5, lw=1.6, fill_alpha=0.12
)
style_ax(axes_f4[0, 1], 'Stimulus', '$N_k$', 'N trajectory')
add_legend(axes_f4[0, 1], loc='best')

finalize_figure(
    fig_f4,
    title='Fig F4: SynII vs WT 2.5 mM 20 Hz binomial trajectories',
    save_path=OUTPUT_DIR / '10_22_synii_vs_wt20_np_trajectories_sharedthr.pdf',
)

# ##################################################################
# FIG F4b : Pooled per-event failure rate by condition
# ##################################################################
fig_fail, axes_fail = make_figure_grid(1, 1, panel_kind='simple', squeeze=False)
ax_fail = axes_fail[0, 0]
failure_plot_defs = [
    (cond_label, cond_results_E[cond_label]['E_fail'], cond_results_E[cond_label]['color'])
    for (cond_label, _, _, _, _) in COND_DEFS_E
    if cond_label in cond_results_E
]
failure_plot_defs.append(('SynII', F_fail, SYNII_COLOR))

for cond_label, fail_mat, color in failure_plot_defs:
    ls = '--' if '50 Hz' in cond_label else '-'
    lw = 2.0 if cond_label == 'SynII' else 1.4
    mk = 's' if cond_label == 'SynII' else 'o'
    plot_mean_sem_trace(
        ax_fail, x_stim, fail_mat, color=color,
        label=f'{cond_label} ' + '({n})',
        marker=mk, linestyle=ls, ms=4, lw=lw, fill_alpha=0.10
    )

ax_fail.set_ylim(-0.02, 1.02)
style_ax(ax_fail, 'Stimulus', 'Failure rate', 'Per-event failure rate by pooled condition')
add_legend(ax_fail, loc='best')
finalize_figure(
    fig_fail,
    save_path=OUTPUT_DIR / '10_23_synii_failure_rate_per_event_sharedthr.pdf',
)

# ##################################################################
# FIG F5 : Boxplot of normalised refilling slope
#          SynII vs WT 2.5 mM 20 Hz + statistics
# ##################################################################
from scipy.stats import mannwhitneyu

synii_refill_vals = F_refill_norm[np.isfinite(F_refill_norm)]
wt_refill_vals = WT_refill_norm[np.isfinite(WT_refill_norm)]

df_refill_box = pd.DataFrame({
    'Group': (['SynII'] * len(synii_refill_vals)) +
             (['WT 2.5mM 20Hz'] * len(wt_refill_vals)),
    'Refill_norm': np.concatenate([synii_refill_vals, wt_refill_vals])
})

if df_refill_box.empty:
    raise RuntimeError("No finite slope/NP1 values to plot.")

u_stat, p_val = mannwhitneyu(synii_refill_vals, wt_refill_vals, alternative='two-sided')

fig_f5, ax = make_figure_grid(figsize=grid_size(1, 1, 'simple'))
sns.boxplot(
    data=df_refill_box, x='Group', y='Refill_norm', ax=ax,
    showfliers=False, width=0.55
)
sns.stripplot(
    data=df_refill_box, x='Group', y='Refill_norm', ax=ax,
    color='black', alpha=0.3, size=3
)

ax.axhline(1.0, ls='--', color='gray', lw=1, alpha=0.7)

ymax = np.nanmax(df_refill_box['Refill_norm'].values)
yline = ymax * 1.08
ytext = ymax * 1.12
ax.plot([0, 0, 1, 1], [yline, ytext, ytext, yline], color='black', lw=1.2)
ax.text(
    0.5, ytext,
    f'Mann-Whitney U p = {p_val:.3e}',
    ha='center', va='bottom', fontsize=10
)

ax.set_ylim(top=max(ymax * 1.18, ytext * 1.05))
ax.set_ylabel('Normalised refilling slope (slope / NP$_1$)')
ax.set_xlabel('')
ax.set_title('SynII vs WT 2.5 mM 20 Hz: refilling slope')
ax.grid(False)

fig_f5.tight_layout()
out_f5 = OUTPUT_DIR / '10_24_synii_refilling_slope_boxplot_sharedthr.pdf'
fig_f5.savefig(out_f5, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved to {out_f5}")
print(f"SynII n={len(synii_refill_vals)}, WT n={len(wt_refill_vals)}")
print(f"Mann-Whitney U test: U={u_stat:.3f}, p={p_val:.3e}")
print(f"Median slope/NP1: SynII={np.nanmedian(synii_refill_vals):.3f}, WT={np.nanmedian(wt_refill_vals):.3f}")


In [ ]:
from scipy.stats import ttest_ind

# Reuse boxplot dataframe if present, else rebuild from arrays
if 'df_refill_box' in globals():
    syn = df_refill_box.loc[df_refill_box['Group'] == 'SynII', 'Refill_norm'].dropna().values
    wt  = df_refill_box.loc[df_refill_box['Group'] == 'WT 2.5mM 20Hz', 'Refill_norm'].dropna().values
else:
    syn = F_refill_norm[np.isfinite(F_refill_norm)]
    wt  = WT_refill_norm[np.isfinite(WT_refill_norm)]

if len(syn) < 3 or len(wt) < 3:
    raise RuntimeError(f"Not enough values for stats (SynII n={len(syn)}, WT n={len(wt)}).")

# Non-parametric (recommended)
u_stat, p_mwu = mannwhitneyu(syn, wt, alternative='two-sided')
rbc = (2 * u_stat) / (len(syn) * len(wt)) - 1  # rank-biserial correlation

# Parametric (optional)
t_stat, p_t = ttest_ind(syn, wt, equal_var=False, nan_policy='omit')

print("=== Refilling slope/NP1: SynII vs WT 2.5mM 20Hz ===")
print(f"n SynII={len(syn)}, n WT={len(wt)}")
print(f"median SynII={np.median(syn):.4f}, median WT={np.median(wt):.4f}")
print(f"Mann-Whitney U={u_stat:.2f}, p={p_mwu:.4g}, rank-biserial={rbc:.3f}")
print(f"Welch t-test: t={t_stat:.3f}, p={p_t:.4g}")

In [ ]:
# FIG / TABLE : Within-fiber PCA spread in SynII vs WT
#   Compare SynII within-fiber spread against a size-matched WT reference.
from scipy.spatial import ConvexHull
from scipy.spatial.distance import cdist
from scipy.stats import mannwhitneyu

MIN_BOUTONS_PER_FIBER_SPREAD = 3
N_WT_SIZE_MATCH_SHUFFLES = 5000
WT_SIZE_MATCH_SEED = 42
SPREAD_METRIC = 'HullArea'
SPREAD_LABEL = 'Convex hull area'
WT_SHUFFLE_MEDIAN_BIN_WIDTH = 0.01


def _convex_hull_area_safe(points):
    points = np.asarray(points, float)
    if len(points) < 3:
        return np.nan
    try:
        hull = ConvexHull(points)
        return float(hull.volume)
    except Exception:
        return np.nan


def _fiber_ids_for_df(df):
    d = df.copy()
    if 'FiberID' in d.columns:
        fiber_ids = d['FiberID'].astype(str).str.strip()
        bad = fiber_ids.isna() | fiber_ids.eq('') | fiber_ids.eq('nan')
        if bad.any():
            fiber_ids.loc[bad] = d.loc[bad, 'ID'].map(_extract_fiber_id)
    else:
        fiber_ids = d['ID'].map(_extract_fiber_id)
    return fiber_ids.astype(str).str.strip()


def summarize_within_fiber_spread(df, coords, label, min_boutons=4):
    d = df.copy().reset_index(drop=True)
    xy = np.asarray(coords, float)

    n = min(len(d), len(xy))
    d = d.iloc[:n].copy()
    xy = xy[:n, :2]

    d['FiberID'] = _fiber_ids_for_df(d).iloc[:n].values
    counts = d['FiberID'].value_counts()
    valid_fibers = counts[counts >= min_boutons].index.tolist()

    rows = []
    for fid in valid_fibers:
        idx = np.where(d['FiberID'].values == fid)[0]
        pts = xy[idx]
        nb = len(pts)
        if nb < min_boutons:
            continue

        centroid = np.nanmean(pts, axis=0)
        radial = np.sqrt(np.sum((pts - centroid) ** 2, axis=1))
        rms_radius = float(np.sqrt(np.nanmean(radial ** 2)))
        mean_radius = float(np.nanmean(radial))
        hull_area = _convex_hull_area_safe(pts)

        if nb >= 2:
            D = cdist(pts, pts)
            tri = np.triu_indices(nb, k=1)
            pairwise = D[tri]
            max_pairwise = float(np.nanmax(pairwise))
            mean_pairwise = float(np.nanmean(pairwise))
        else:
            max_pairwise = np.nan
            mean_pairwise = np.nan

        rows.append({
            'Dataset': label,
            'FiberID': fid,
            'n_boutons': nb,
            'HullArea': hull_area,
            'RMSRadius': rms_radius,
            'MeanRadius': mean_radius,
            'MaxPairwise': max_pairwise,
            'MeanPairwise': mean_pairwise,
        })

    return pd.DataFrame(rows)


def _size_matched_wt_sampling_plan(wt_df, syn_sizes):
    available_sizes = sorted(pd.Series(wt_df['n_boutons']).dropna().astype(int).unique())
    if not available_sizes:
        return None

    plan = []
    fallback_counts = {}
    for size in syn_sizes:
        size = int(size)
        if size in available_sizes:
            matched_size = size
        else:
            matched_size = min(available_sizes, key=lambda x: (abs(x - size), x))
            fallback_counts[(size, matched_size)] = fallback_counts.get((size, matched_size), 0) + 1
        plan.append(matched_size)
    return plan, fallback_counts


def _draw_size_matched_wt_sample(wt_df, size_plan, rng):
    chosen = []
    for size in size_plan:
        pool = wt_df[wt_df['n_boutons'].astype(int) == int(size)]
        if len(pool) == 0:
            continue
        pick = int(rng.integers(0, len(pool)))
        chosen.append(pool.iloc[[pick]])
    if not chosen:
        return pd.DataFrame(columns=wt_df.columns)
    return pd.concat(chosen, ignore_index=True)


wt_fiber_spread = summarize_within_fiber_spread(
    PCA_Data_WT_Pooled,
    pca_coordinates,
    label='WT pooled',
    min_boutons=MIN_BOUTONS_PER_FIBER_SPREAD,
)

synii_fiber_spread = summarize_within_fiber_spread(
    PCA_Data_SynII,
    pca_data['SynII'],
    label='SynII',
    min_boutons=MIN_BOUTONS_PER_FIBER_SPREAD,
)

print(f"=== WITHIN-FIBER PCA SPREAD ({SPREAD_METRIC}, fibers with >= {MIN_BOUTONS_PER_FIBER_SPREAD} boutons) ===")
print(f"WT pooled fibers analyzed: {len(wt_fiber_spread)}")
print(f"SynII fibers analyzed:     {len(synii_fiber_spread)}")

if len(wt_fiber_spread) == 0 or len(synii_fiber_spread) == 0:
    print('Not enough fibers to compare WT and SynII with the current threshold.')
else:
    syn_sizes = synii_fiber_spread['n_boutons'].astype(int).tolist()
    plan_out = _size_matched_wt_sampling_plan(wt_fiber_spread, syn_sizes)
    if plan_out is None:
        print('No WT fibers available for size matching.')
    else:
        size_plan, fallback_counts = plan_out
        rng_display = np.random.default_rng(WT_SIZE_MATCH_SEED)
        wt_matched_display = _draw_size_matched_wt_sample(wt_fiber_spread, size_plan, rng_display)

        syn_vals = synii_fiber_spread[SPREAD_METRIC].dropna().to_numpy(float)
        wt_vals = wt_matched_display[SPREAD_METRIC].dropna().to_numpy(float)

        rng_shuffle = np.random.default_rng(WT_SIZE_MATCH_SEED)
        wt_shuffle_medians = []
        wt_shuffle_means = []
        for _ in range(N_WT_SIZE_MATCH_SHUFFLES):
            wt_sample = _draw_size_matched_wt_sample(wt_fiber_spread, size_plan, rng_shuffle)
            vals = wt_sample[SPREAD_METRIC].dropna().to_numpy(float)
            if len(vals):
                wt_shuffle_medians.append(float(np.nanmedian(vals)))
                wt_shuffle_means.append(float(np.nanmean(vals)))

        wt_shuffle_medians = np.asarray(wt_shuffle_medians, float)
        wt_shuffle_means = np.asarray(wt_shuffle_means, float)

        syn_median = float(np.nanmedian(syn_vals)) if len(syn_vals) else np.nan
        wt_display_median = float(np.nanmedian(wt_vals)) if len(wt_vals) else np.nan
        wt_shuffle_center = float(np.nanmedian(wt_shuffle_medians)) if len(wt_shuffle_medians) else np.nan

        if len(syn_vals) and len(wt_vals):
            u_stat, p_mwu = mannwhitneyu(syn_vals, wt_vals, alternative='two-sided')
        else:
            u_stat, p_mwu = np.nan, np.nan

        if len(wt_shuffle_medians):
            p_emp_lower = (1.0 + np.sum(wt_shuffle_medians <= syn_median)) / (len(wt_shuffle_medians) + 1.0)
            p_emp_two = (1.0 + np.sum(np.abs(wt_shuffle_medians - wt_shuffle_center) >= np.abs(syn_median - wt_shuffle_center))) / (len(wt_shuffle_medians) + 1.0)
        else:
            p_emp_lower = np.nan
            p_emp_two = np.nan

        fig, axes = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
        ax_left, ax_right = axes[0, 0], axes[0, 1]
        colors = {'WT matched': get_wt_ca_color('2.5mM'), 'SynII': SYNII_COLOR}

        bp = ax_left.boxplot(
            [wt_vals, syn_vals],
            positions=[0, 1],
            widths=0.6,
            patch_artist=True,
            showfliers=False,
        )
        for patch, fc in zip(bp['boxes'], [colors['WT matched'], colors['SynII']]):
            patch.set_facecolor(fc)
            patch.set_alpha(0.45)

        for xpos, vals, c in [(0, wt_vals, colors['WT matched']), (1, syn_vals, colors['SynII'])]:
            if len(vals):
                xj = np.full(len(vals), xpos, float) + np.random.uniform(-0.10, 0.10, len(vals))
                ax_left.scatter(xj, vals, c=c, s=28, alpha=0.75, edgecolors='black', linewidths=0.3)

        if len(wt_vals) and len(syn_vals):
            y_top = np.nanmax(np.r_[wt_vals, syn_vals])
            y_pad = 0.08 * y_top if y_top > 0 else 0.1
            ax_left.plot([0, 1], [y_top + y_pad, y_top + y_pad], color='black', lw=1)
            ax_left.text(0.5, y_top + 1.15 * y_pad, f'p={p_mwu:.2g}', ha='center', va='bottom', fontsize=8)

        ax_left.set_xticks([0, 1])
        ax_left.set_xticklabels([f'WT matched\n(n={len(wt_vals)})', f'SynII\n(n={len(syn_vals)})'])
        style_ax(ax_left, 'Dataset', SPREAD_LABEL, f'{SPREAD_LABEL}: matched sample')

        wt_shuffle_bins = np.arange(np.nanmin(wt_shuffle_medians), np.nanmax(wt_shuffle_medians) + WT_SHUFFLE_MEDIAN_BIN_WIDTH, WT_SHUFFLE_MEDIAN_BIN_WIDTH, dtype=float)
        if wt_shuffle_bins.size < 2:
            center = float(np.nanmean(wt_shuffle_medians))
            wt_shuffle_bins = np.array([center - WT_SHUFFLE_MEDIAN_BIN_WIDTH / 2.0, center + WT_SHUFFLE_MEDIAN_BIN_WIDTH / 2.0], dtype=float)
        elif wt_shuffle_bins[-1] < np.nanmax(wt_shuffle_medians):
            wt_shuffle_bins = np.append(wt_shuffle_bins, wt_shuffle_bins[-1] + WT_SHUFFLE_MEDIAN_BIN_WIDTH)
        ax_right.hist(wt_shuffle_medians, bins=wt_shuffle_bins, color=colors['WT matched'], alpha=0.75, edgecolor='white')
        if np.isfinite(syn_median):
            ax_right.axvline(syn_median, color=colors['SynII'], lw=2, label='SynII median')
        if np.isfinite(wt_display_median):
            ax_right.axvline(wt_display_median, color='black', lw=1.2, ls='--', label='Displayed WT median')
        style_ax(ax_right, f'Size-matched WT shuffle median {SPREAD_LABEL}', 'Count', f'{SPREAD_LABEL}: WT size-matched shuffle')
        add_legend(ax_right, loc='upper right')

        finalize_figure(
            fig,
            title=f'Within-fiber PCA spread: SynII vs size-matched WT ({SPREAD_LABEL})',
            save_path=OUTPUT_DIR / '10_25_synii_vs_wt_within_fiber_pca_spread.pdf',
        )

        syn_size_counts = pd.Series(syn_sizes).value_counts().sort_index()
        print('\nSize-matched comparison summary:')
        print('  SynII fiber-size profile: ' + '  '.join(f'n{int(k)}={int(v)}' for k, v in syn_size_counts.items()))
        if fallback_counts:
            print('  Fallback size matches: ' + '  '.join(f'{src}->{dst} x{count}' for (src, dst), count in sorted(fallback_counts.items())))
        else:
            print('  Fallback size matches: none (all exact)')
        print(f'  {SPREAD_LABEL} median | SynII={syn_median:.3f} | WT matched={wt_display_median:.3f}')
        print(f'  Displayed matched-sample Mann-Whitney U={u_stat:.3f}, p={p_mwu:.3g}')
        print(f'  Shuffle empirical p (SynII <= WT matched median) = {p_emp_lower:.4f}')
        print(f'  Shuffle empirical two-sided p = {p_emp_two:.4f}')
        print(f'  WT shuffle median distribution: median={np.nanmedian(wt_shuffle_medians):.3f}, IQR=[{np.nanpercentile(wt_shuffle_medians, 25):.3f}, {np.nanpercentile(wt_shuffle_medians, 75):.3f}]')

        spread_summary = pd.concat([
            wt_matched_display.assign(Dataset='WT matched'),
            synii_fiber_spread.assign(Dataset='SynII'),
        ], ignore_index=True)
        print('\nDisplayed per-fiber summary:')
        print(
            spread_summary.sort_values(['Dataset', 'n_boutons', 'FiberID'], ascending=[True, False, True])[
                ['Dataset', 'FiberID', 'n_boutons', SPREAD_METRIC]
            ].to_string(index=False)
        )


## 11. Stability Control

Stability measurements are placed near the end of the notebook because they validate the persistence of WT bouton properties without redefining the main biological axes. The goal is to show that the major WT features are not explained by recording drift over time.

These controls matter for both the diversity and the mechanistic analyses: if amplitudes, PPR profiles, or PCA positions drift strongly over minutes, then bouton-to-bouton heterogeneity would be harder to interpret as a stable biological property.


### 11.1 Stability Configuration

The configuration cell defines the pairing and boundary parameters used for the stability controls. Keeping these parameters explicit is important because the stability readouts are boundary- and pairing-dependent by construction.


In [ ]:
# Config + tiny helpers (set your "edge" controls here) 

ELLIPSE_ALPHA   = 0.95     # ellipse containment level
ALPHA_EXPANSION = 0.50     # alpha-shape expansion factor (0.0 → off)
KNN_K           = None     # k-NN neighbors (None → auto √N)

import numpy as np
from scipy.stats import chi2

def build_ellipse_models(X, y, n_clusters, alpha=0.95, ridge=1e-6):
    """Mean/cov/inv and chi2 threshold for each cluster."""
    thr = chi2.ppf(alpha, df=2)
    models = []
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts) > 2:
            mu  = pts.mean(axis=0)
            cov = np.cov(pts.T) + np.eye(2)*ridge
            inv = np.linalg.pinv(cov)
            models.append({'cluster': k, 'center': mu, 'cov': cov, 'inv_cov': inv})
    return models, thr

def mahalanobis_sq(x, mu, inv):
    d = x - mu
    return float(d.T @ inv @ d)

def ellipses_containing(x, models, thr):
    return {m['cluster'] for m in models if mahalanobis_sq(x, m['center'], m['inv_cov']) <= thr}

def nearest_ellipse_edge(x, models, thr):
    """Return (cluster_id, euclid_dist_to_edge)."""
    best = (None, np.inf)
    for m in models:
        md2 = mahalanobis_sq(x, m['center'], m['inv_cov'])
        if md2 <= thr:
            return m['cluster'], 0.0
        s = np.sqrt(thr/md2)
        x_proj = m['center'] + s*(x - m['center'])
        dist = float(np.linalg.norm(x - x_proj))
        if dist < best[1]:
            best = (m['cluster'], dist)
    return best


### 11.2 Paired Trajectories in WT PCA Space

Paired PCA trajectories show how the same boutons move between repeated recordings. This is the closest stability analogue of the perturbation trajectory figures used for calcium, frequency, and SynII analyses.


In [ ]:
# ==== Cell 1 : Before/After trajectories + summary ====

import matplotlib.pyplot as plt

# Data
before_coords = np.asarray(pca_data['stab_before'])
after_coords  = np.asarray(pca_data['stab_after'])
n_pairs       = int(min(len(before_coords), len(after_coords)))
assert n_pairs > 0, "No paired before/after points."

# Plot
make_figure(figsize=(8,6))
plot_pca_background(plt.gca(), alpha=0.4, s=20, label='WT background')
plot_pca_overlay_points(plt.gca(), before_coords[:], c=STABILITY_BEFORE_COLOR,  edgecolors='black', label=f'Before (n={len(before_coords)})')
plot_pca_overlay_points(plt.gca(), after_coords[:],  c=STABILITY_AFTER_COLOR,  edgecolors='black',   label=f'After  (n={len(after_coords)})')

# Pair links + mean arrow
moves = []
for i in range(n_pairs):
    plt.plot([before_coords[i,0], after_coords[i,0]],
             [before_coords[i,1], after_coords[i,1]], color='gray', alpha=0.6, lw=1)
    moves.append(float(np.linalg.norm(after_coords[i] - before_coords[i])))

diffs    = after_coords[:n_pairs] - before_coords[:n_pairs]
mean_vec = diffs.mean(axis=0)
center   = np.vstack([before_coords[:n_pairs], after_coords[:n_pairs]]).mean(axis=0)
plt.arrow(center[0], center[1], mean_vec[0], mean_vec[1], color='black',
          width=0.05, head_width=0.25, head_length=0.25, length_includes_head=True, label='Mean trajectory')

style_pca_axes(plt.gca(), title='Stability: Before vs After',  legend=False)
add_legend(plt.gca(), frameon=False); plt.tight_layout()
out = OUTPUT_DIR / "11_01_stability_before_after_trajectories.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print("=== STABILITY TRAJECTORY ANALYSIS ===")
print(f"Pairs: {n_pairs}")
print(f"Movement (mean±SD): {np.mean(moves):.3f} ± {np.std(moves):.3f}")
print(f"Mean trajectory |mag|: {np.linalg.norm(mean_vec):.3f}  dir=({mean_vec[0]:.3f}, {mean_vec[1]:.3f})")
print(f"✓ Saved {out}")


### 11.3 Distribution of Movement Magnitudes

The movement histogram condenses the paired PCA trajectories into a simple distribution of drift distances. It provides a compact benchmark for judging whether repeated recordings remain within the range expected for a stable WT bouton class. 


In [ ]:
# ==== Cell 2 : Movement distance histogram (auto bins) ====

# Freedman:Diaconis binning with fallback
md      = np.linalg.norm(diffs, axis=1).astype(float)
md      = md[np.isfinite(md)]
q25,q75 = np.percentile(md,[25,75]); iqr=float(q75-q25); n=len(md)
bw      = (2*iqr)/(n**(1/3)) if iqr>0 else 0.0
bins    = max(5, int(np.ceil((md.max()-md.min())/bw))) if bw>0 else max(5, int(np.ceil(np.sqrt(n))))

make_figure(figsize=(6,4))
plt.hist(md, bins=bins, edgecolor='black', alpha=0.85)
plt.axvline(md.mean(), ls='--', lw=2, label=f'Mean = {md.mean():.2f}')
plt.xlabel('Distance in PCA space'); plt.ylabel('Count'); plt.title('Before→After distances'); add_legend(plt.gca(), frameon=False)
if 'style_hist_axis' in globals():
    style_hist_axis(plt.gca())
plt.tight_layout()
out = OUTPUT_DIR / "11_02_stability_movement_distances.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print(f"n={n}  mean={md.mean():.3f}  sd={md.std(ddof=1):.3f}  median={np.median(md):.3f}  min={md.min():.3f}  max={md.max():.3f}  bins={bins}")
print(f"✓ Saved {out}")


### 11.4 Stability of Mean Traces

Mean-trace comparisons test whether the overall glutamate transient waveform is preserved when fibers are revisited. Stable traces strengthen the interpretation that the main bouton classes are not artifacts of acquisition order or time-dependent bleaching.


In [ ]:
# ==== Cell 3 : Mean traces (±SEM) for Stability_Before/_05 vs Stability_After/_05 ====

conds = STABILITY_2_5_20HZ_CONDITIONS
stats = {c: compute_trace_stats(condition_names=c, source=TRACE_MEAN_SOURCE) for c in conds}
counts = {c: stats[c]['n'] for c in conds}
colors = {c: get_stability_condition_color(c) for c in conds}
styles = {"Stability_Before": ('-', 2.0), "Stability_Before_05": ('--', 1.8), "Stability_After": ('-', 2.0), "Stability_After_05": ('--', 1.8)}
fig, ax = make_figure_grid(figsize=(8.5, 5.0))
for idx, c in enumerate(conds):
    linestyle, line_width = styles[c]
    plot_traces(ax=ax, rows=stats[c]['rows'], source=TRACE_MEAN_SOURCE, color=colors[c], label=f"{c} (n={stats[c]['n']})", show_average=True, show_sem=True, linestyle=linestyle, linewidth=line_width, sem_alpha=0.15, zero_line=(idx == len(conds) - 1), xlim=TRACE_XLIM_20HZ if idx == len(conds) - 1 else None, xlabel='Time (s)' if idx == len(conds) - 1 else None, ylabel='ΔF/F' if idx == len(conds) - 1 else None, title='Stability conditions: mean traces (±SEM)' if idx == len(conds) - 1 else None, legend=(idx == len(conds) - 1), legend_kwargs={'ncol': 2, 'fontsize': 9}, style_axis=(idx == len(conds) - 1))
ax.axvspan(*TRACE_XLIM_20HZ, color='gray', alpha=0.08)
plt.tight_layout()
out = OUTPUT_DIR / "11_03_stability_mean_traces_before_after.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print("Counts:", {c: counts[c] for c in conds})
print(f"✓ Saved {out}")


### 11.5 Elliptical WT Boundaries

Ellipse-based boundaries offer a conservative, low-dimensional way to ask whether repeated recordings remain in the same WT territory. This mirrors the use of the WT PCA space in the main results while turning it into a stability-control geometry.


In [ ]:
# ===== Cell 4 : Ellipses: PCA with hard edge + tolerance zone, and pie =====
# knobs
ELLIPSE_ALPHA = 0.8        # hard edge level
ELLIPSE_TOL   = 1.0       # inflate ellipse threshold by (1 + ELLIPSE_TOL)

import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# --- build models ---
def _ellipse_models(X, y, n_clusters, ridge=1e-6):
    models=[]
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts)>2:
            mu=pts.mean(axis=0); cov=np.cov(pts.T)+np.eye(2)*ridge; inv=np.linalg.pinv(cov)
            models.append({'cluster':k,'center':mu,'cov':cov,'inv_cov':inv})
    return models
def _md2(x, m): 
    d=x-m['center']; return float(d.T @ m['inv_cov'] @ d)

ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard       = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol        = thr_hard*(1.0 + ELLIPSE_TOL)
print(f"Thresholds - Hard: {thr_hard:.2f}, Tolerance: {thr_tol:.2f}")

# --- plot PCA with hard + tolerance ---
fig, ax = make_figure_grid(figsize=(8,6))
plot_pca_background(ax, alpha=0.35, s=20, label='WT')
uniq = np.unique(cluster_assignments)

def _draw_ellipse(ax, mean, cov, thr, color, fa, lw, ls='-'):
    U,s,_=np.linalg.svd(cov); ang=np.degrees(np.arctan2(U[1,0],U[0,0])); w,h=2*np.sqrt(thr*s)
    ax.add_patch(Ellipse(mean, w, h, angle=ang, facecolor=color, edgecolor=color, alpha=fa, lw=lw, ls=ls))

for cid in uniq:
    pts = pca_coordinates[cluster_assignments==cid]
    if len(pts)<3: continue
    m     = [mm for mm in ellipse_models if mm['cluster']==cid][0]
    color = get_cluster_color(cid)
    # tolerance ring: draw tol (filled light), then hard (outline)
    _draw_ellipse(ax, m['center'], m['cov'], thr_tol,  color, 0.10, 0.0)    # tol zone
    _draw_ellipse(ax, m['center'], m['cov'], thr_hard, color, 0.00, 2.0)    # hard edge

# overlay pairs
n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
before  = np.asarray(pca_data['stab_before'])[:n_pairs]
after   = np.asarray(pca_data['stab_after'])[:n_pairs]

# Calculate stability for line colors
def _label_all_for_plot(x, thr):
    """Return all clusters that contain point x (for plotting)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d       = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _label_all_for_plot(before[i], thr_tol)  
    a_clusters = _label_all_for_plot(after[i], thr_tol)   
    is_stable  = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

plot_pca_overlay_points(ax, before[:], c=STABILITY_BEFORE_COLOR, edgecolors='k', lw=0.5, label='Before')
plot_pca_overlay_points(ax, after[:],  c=STABILITY_AFTER_COLOR, marker='s', edgecolors='k', lw=0.5, label='After')

# Add legend entries for line colors
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

style_pca_axes(ax, title=f'Ellipses: hard α={ELLIPSE_ALPHA:.2f}, tol +{ELLIPSE_TOL*100:.0f}% (Conservative)',  legend=False)

# Combine existing legend with line color legend
handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
add_legend(ax, handles=handles); ax.grid(False); plt.tight_layout()
out = OUTPUT_DIR / "11_04_stability_ellipses_pca.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Conservative stability approach (hard-edge only) ---
def _inside_any(x, thr): 
    return any(_md2(x,m) <= thr for m in ellipse_models)

def _label_all(x, thr):
    """Return all clusters that contain point x (conservative approach)"""
    inside = [m['cluster'] for m in ellipse_models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    # If not inside any ellipse, assign to nearest-edge cluster
    d       = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in ellipse_models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

# Get all cluster memberships for each point
b_labs = [_label_all(before[i], thr_tol) for i in range(n_pairs)]
a_labs = [_label_all(after[i], thr_tol) for i in range(n_pairs)]

# Conservative stability: stable if any overlap between before and after cluster sets
stable   = sum(1 for i in range(n_pairs) if len(b_labs[i] & a_labs[i]) > 0)
unstable = n_pairs - stable

make_figure(figsize=(6.3,5.8))
plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title('Conservative Ellipse Stability (hard-edge)'); plt.tight_layout()
out = OUTPUT_DIR / "11_05_stability_ellipse_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
print(f"[Ellipses] Conservative: hard α={ELLIPSE_ALPHA:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

# Optional: Print details about overlapping cases for verification
overlap_cases = []
for i in range(n_pairs):
    if len(b_labs[i] & a_labs[i]) > 0 and (len(b_labs[i]) > 1 or len(a_labs[i]) > 1):
        overlap_cases.append((i, b_labs[i], a_labs[i], b_labs[i] & a_labs[i]))

if overlap_cases:
    print(f"Found {len(overlap_cases)} cases with ellipse overlaps:")
    for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
        print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
    if len(overlap_cases) > 5:
        print(f"  ... and {len(overlap_cases)-5} more cases")


### 11.6 Alpha-Shape WT Boundaries

Alpha-shape boundaries provide a less parametric complement to the ellipse analysis. The purpose is not to create a second classification system, but to verify that the stability conclusion does not depend on one specific geometric model of WT occupancy. 


In [ ]:
# ===== Cell : Alpha-shapes: PCA with hard polygon + tolerance zone, and pie =====
# knobs
ALPHA_EXPANSION = 2      # tolerance zone: expanded by sqrt(1+exp)
ALPHA_KNN_Q     = 0.2       # alpha heuristic quantile (0.8 for very small n)

import numpy as np, matplotlib.pyplot as plt
from shapely.affinity import scale as shp_scale
from shapely.geometry import MultiPoint, Polygon as ShapelyPolygon, MultiPolygon, Point


if alphashape is None:
    print('Skipping alpha-shape stability control: alphashape is not installed.')
else:
    def _alpha_for(pts):
        n=len(pts); d2=np.sum((pts[:,None,:]-pts[None,:,:])**2, axis=2); np.fill_diagonal(d2, np.inf)
        kth=np.partition(d2,1,axis=1)[:,1]; base=np.sqrt(kth)
        return float(1.5*np.quantile(base, ALPHA_KNN_Q if n>=10 else 0.8))

    def _make_alpha_shapes(X,y,expansion):
        s = float(np.sqrt(1.0+expansion)) if expansion>0 else 1.0
        res={}
        for cid in np.unique(y):
            pts = X[y==cid]
            if len(pts)<3: res[cid]=None; continue
            a=_alpha_for(pts)
            poly = alphashape.alphashape([tuple(r) for r in pts], a)
            if poly is None or getattr(poly,'is_empty',True): poly = MultiPoint([tuple(r) for r in pts]).convex_hull
            res[cid]={'hard':poly, 'tol': (shp_scale(poly, xfact=s, yfact=s, origin='centroid') if expansion>0 else poly)}
        return res

    shapes = _make_alpha_shapes(pca_coordinates, cluster_assignments, ALPHA_EXPANSION)
    # --- PCA: draw tol (light fill) + hard (outline) ---
    fig, ax = make_figure_grid(figsize=(8,6))
    plot_pca_background(ax, alpha=0.35, s=20, label='WT')
    for cid,sh in shapes.items():
        if not sh: continue
        for tag, (fa, ls, lw) in dict(tol=(0.10,'-',0.0), hard=(0.00,'-',2.0)).items():
            g = sh[tag]
            geoms=[g] if g.geom_type=='Polygon' else (list(g.geoms) if g.geom_type=='MultiPolygon' else [])
            for gg in geoms:
                X,Y = np.array(gg.exterior.coords).T
                if fa>0: ax.fill(X,Y,color=get_cluster_color(cid),alpha=fa)
                ax.plot(X,Y,color=get_cluster_color(cid),ls=ls,lw=lw)

    # pairs with conservative coloring
    n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
    before = np.asarray(pca_data['stab_before'])[:n_pairs]
    after  = np.asarray(pca_data['stab_after'])[:n_pairs]

    # Conservative approach: find ALL clusters that contain each point
    def _in_shape_all(x, use_tol=False, paired_point=None, paired_clusters=None):
        """Return set of all clusters that contain point x (conservative approach)"""
        p = Point(float(x[0]), float(x[1]))
        clusters = set()
    
        # Check containment in all shapes
        for cid, sh in shapes.items():
            if sh is None: continue
            g = sh['tol'] if use_tol else sh['hard']
            if g.geom_type == 'Polygon':
                if g.contains(p):
                    clusters.add(cid)
            elif g.geom_type == 'MultiPolygon':
                if any(gg.contains(p) for gg in g.geoms):
                    clusters.add(cid)
    
        # If not inside any shape, assign to nearest boundary
        if not clusters:
            distances = []
            for cid, sh in shapes.items():
                if sh is None: continue
                g = sh['tol'] if use_tol else sh['hard']
                dist = p.distance(g)
                distances.append((cid, dist))
        
            if distances:
                # If paired point is inside shapes, prefer those clusters when distances are close
                if paired_clusters:
                    # Find distances to paired clusters
                    paired_dists = [(cid, d) for cid, d in distances if cid in paired_clusters]
                    if paired_dists:
                        min_paired_dist = min(paired_dists, key=lambda t: t[1])[1]
                        overall_min_dist = min(distances, key=lambda t: t[1])[1]
                        # If paired cluster is within 20% of nearest, prefer it
                        if min_paired_dist <= overall_min_dist * 1.2:
                            nearest_cid = min(paired_dists, key=lambda t: t[1])[0]
                            clusters = {nearest_cid}
                            return clusters
            
                # Otherwise use nearest
                nearest_cid = min(distances, key=lambda t: t[1])[0]
                clusters = {nearest_cid}
    
        return clusters

    # Draw connecting lines with appropriate colors
    for i in range(n_pairs):
        b_clusters = _in_shape_all(before[i], use_tol=True)
        a_clusters = _in_shape_all(after[i], use_tol=True)
    
        # Check stability
        is_stable = len(b_clusters & a_clusters) > 0
    
        line_color = 'gray' if is_stable else 'red'
        line_alpha = 0.6 if is_stable else 0.8
        line_width = 1 if is_stable else 1.2
    
        ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
                color=line_color, alpha=line_alpha, lw=line_width)

    plot_pca_overlay_points(ax, before[:], c=STABILITY_BEFORE_COLOR, edgecolors='k', lw=0.5, label='Before')
    plot_pca_overlay_points(ax, after[:],  c=STABILITY_AFTER_COLOR, marker='s', edgecolors='k', lw=0.5, label='After')

    # Add legend entries for line colors
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
        Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
    ]

    style_pca_axes(ax, title=f'Alpha-shapes: hard + tol (exp={ALPHA_EXPANSION:.2f}) (Conservative)',  legend=False)

    # Combine existing legend with line color legend
    handles, labels = ax.get_legend_handles_labels()
    handles.extend(legend_elements)
    add_legend(ax, handles=handles); ax.grid(False); plt.tight_layout()

    out = OUTPUT_DIR / "11_06_stability_alphashapes_pca.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

    # --- Conservative stability (tolerance boundary) ---
    b_clusters_all = [_in_shape_all(before[i], use_tol=True) for i in range(n_pairs)]
    a_clusters_all = [_in_shape_all(after[i], use_tol=True) for i in range(n_pairs)]

    # Conservative stability: stable if any overlap between before and after cluster sets
    stable = sum(1 for i in range(n_pairs) if len(b_clusters_all[i] & a_clusters_all[i]) > 0)
    unstable = n_pairs - stable

    make_figure(figsize=(6.3,5.8))
    plt.pie([stable,unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
            colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
    plt.title(f'Conservative Alpha-shape Stability (exp={ALPHA_EXPANSION:.2f})'); plt.tight_layout()
    out = OUTPUT_DIR / "11_07_stability_alphashapes_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
    print(f"[Alpha] Conservative: exp={ALPHA_EXPANSION:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)  ✓ {out}")

    # Optional: Print details about overlapping cases for verification
    overlap_cases = []
    for i in range(n_pairs):
        if len(b_clusters_all[i] & a_clusters_all[i]) > 0 and (len(b_clusters_all[i]) > 1 or len(a_clusters_all[i]) > 1):
            overlap_cases.append((i, b_clusters_all[i], a_clusters_all[i], b_clusters_all[i] & a_clusters_all[i]))

    # After the stability calculation, add this diagnostic
    print("\nDiagnostic for UNSTABLE pairs (red lines):")
    unstable_indices = [i for i in range(n_pairs) if len(b_clusters_all[i] & a_clusters_all[i]) == 0]

    for idx in unstable_indices[:10]:  # Show first 10 unstable pairs
        b_pos = before[idx]
        a_pos = after[idx]
        b_clusters = b_clusters_all[idx]
        a_clusters = a_clusters_all[idx]
    
        print(f"\nPair {idx}: UNSTABLE")
        print(f"  Before {b_pos}: assigned to clusters={b_clusters}")
        print(f"  After  {a_pos}: assigned to clusters={a_clusters}")
    
        # Show distances to all shape boundaries
        for cid, sh in shapes.items():
            if sh is None: continue
            g = sh['tol']
            p_before = Point(float(b_pos[0]), float(b_pos[1]))
            p_after = Point(float(a_pos[0]), float(a_pos[1]))
            b_dist = p_before.distance(g)
            a_dist = p_after.distance(g)
            print(f"    Cluster {cid}: Before dist={b_dist:.3f}, After dist={a_dist:.3f}")

    # After creating shapes, verify tolerance expansion
    print("\nVerifying tolerance zone sizes:")
    for cid, sh in shapes.items():
        if sh is None: continue
        hard_area = sh['hard'].area
        tol_area = sh['tol'].area
        expansion_ratio = np.sqrt(tol_area / hard_area)
        print(f"Cluster {cid}: area ratio = {expansion_ratio:.3f} (expected: {np.sqrt(1+ALPHA_EXPANSION):.3f})")
    if overlap_cases:
        print(f"Found {len(overlap_cases)} cases with alpha-shape overlaps:")
        for i, before_clusters, after_clusters, shared in overlap_cases[:5]:  # Show first 5
            print(f"  Pair {i}: Before={before_clusters}, After={after_clusters}, Shared={shared}")
        if len(overlap_cases) > 5:
            print(f"  ... and {len(overlap_cases)-5} more cases")


### 11.7 Bootstrap Stability Estimates

Bootstrap-based controls quantify how much apparent stability is expected under random pairing or resampling. They provide a statistical background against which the observed paired stability can be interpreted. 


In [ ]:
# ===== Cell 4 : Ellipses: PCA with hard edge + tolerance zone, and pie =====
# knobs
ELLIPSE_ALPHA = 0.8        # hard edge level
ELLIPSE_TOL   = 1.0       # inflate ellipse threshold by (1 + ELLIPSE_TOL)
N_BOOTSTRAP   = 2000      # bootstrap iterations
BOOTSTRAP_STABILITY_BIN_WIDTH = 2.5

# --- Bootstrap stability analysis ---
n_pairs = 26
rng = np.random.default_rng(852)
rng_viz = np.random.default_rng(852)

# --- build models ---
ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard       = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol        = thr_hard*(1.0 + ELLIPSE_TOL)
print(f"Thresholds - Hard: {thr_hard:.2f}, Tolerance: {thr_tol:.2f}")



def _label_all(x, thr, models):
    """Return all clusters that contain point x"""
    inside = [m['cluster'] for m in models if _md2(x, m) <= thr]
    if inside:
        return set(inside)
    d = [(m['cluster'], np.sqrt(_md2(x, m)/thr)-1.0) for m in models]
    nearest = min(d, key=lambda t: t[1])[0]
    return {nearest}

def compute_stability(before, after, thr, models):
    """Compute number of stable pairs"""
    stable = 0
    for i in range(len(before)):
        b_clusters = _label_all(before[i], thr, models)
        a_clusters = _label_all(after[i], thr, models)
        if len(b_clusters & a_clusters) > 0:
            stable += 1
    return stable

# Bootstrap
bootstrap_stabilities = []
for b in range(N_BOOTSTRAP):
    # Sample random pairs with replacement
    indices = rng.choice(len(pca_coordinates), size=(n_pairs, 2), replace=True)
    before = pca_coordinates[indices[:, 0]]
    after = pca_coordinates[indices[:, 1]]
    stable = compute_stability(before, after, thr_tol, ellipse_models)
    bootstrap_stabilities.append(stable / n_pairs * 100)

bootstrap_stabilities = np.array(bootstrap_stabilities)
mean_stability = np.mean(bootstrap_stabilities)
ci_low = np.percentile(bootstrap_stabilities, 2.5)
ci_high = np.percentile(bootstrap_stabilities, 97.5)

print(f"\nBootstrap Results ({N_BOOTSTRAP} iterations):")
print(f"  Mean stability: {mean_stability:.1f}%")
print(f"  95% CI: [{ci_low:.1f}%, {ci_high:.1f}%]")
print(f"  Std: {np.std(bootstrap_stabilities):.1f}%")

####
# --- Generate one set of pairs for visualization ---
####
indices_viz = rng_viz.choice(len(pca_coordinates), size=(n_pairs, 2), replace=False)
before = pca_coordinates[indices_viz[:, 0]]
after = pca_coordinates[indices_viz[:, 1]]

# --- plot PCA with hard + tolerance ---
fig, ax = make_figure_grid(figsize=(8,6))
plot_pca_background(ax, alpha=0.35, s=20, label='WT')
uniq = np.unique(cluster_assignments)

def _draw_ellipse(ax, mean, cov, thr, color, fa, lw, ls='-'):
    U,s,_=np.linalg.svd(cov); ang=np.degrees(np.arctan2(U[1,0],U[0,0])); w,h=2*np.sqrt(thr*s)
    ax.add_patch(Ellipse(mean, w, h, angle=ang, facecolor=color, edgecolor=color, alpha=fa, lw=lw, ls=ls))

for cid in uniq:
    pts = pca_coordinates[cluster_assignments==cid]
    if len(pts)<3: continue
    m     = [mm for mm in ellipse_models if mm['cluster']==cid][0]
    color = get_cluster_color(cid)
    # tolerance ring: draw tol (filled light), then hard (outline)
    _draw_ellipse(ax, m['center'], m['cov'], thr_tol,  color, 0.10, 0.0)    # tol zone
    _draw_ellipse(ax, m['center'], m['cov'], thr_hard, color, 0.00, 2.0)    # hard edge

# Draw connecting lines with appropriate colors
for i in range(n_pairs):
    b_clusters = _label_all(before[i], thr_tol, ellipse_models)
    a_clusters = _label_all(after[i], thr_tol, ellipse_models)
    is_stable  = len(b_clusters & a_clusters) > 0
    
    line_color = 'gray' if is_stable else 'red'
    line_alpha = 0.6 if is_stable else 0.8
    line_width = 1 if is_stable else 1.2
    
    ax.plot([before[i,0], after[i,0]], [before[i,1], after[i,1]], 
            color=line_color, alpha=line_alpha, lw=line_width)

plot_pca_overlay_points(ax, before[:], c=STABILITY_BEFORE_COLOR, edgecolors='k', lw=0.5, label='Random A')
plot_pca_overlay_points(ax, after[:],  c=STABILITY_AFTER_COLOR, marker='s', edgecolors='k', lw=0.5, label='Random B')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='gray', alpha=0.6, lw=1, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.8, lw=1.2, label='Unstable pairs')
]

style_pca_axes(ax, title=f'Ellipses: hard α={ELLIPSE_ALPHA:.2f}, tol +{ELLIPSE_TOL*100:.0f}% (Random Pairs)',  legend=False)

handles, labels = ax.get_legend_handles_labels()
handles.extend(legend_elements)
add_legend(ax, handles=handles); ax.grid(False); plt.tight_layout()
out = OUTPUT_DIR / "11_08_stability_random_pairs_ellipses_pca.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Bootstrap distribution histogram ---
fig, ax = make_figure_grid(figsize=(8,5))
bootstrap_bins = np.arange(np.nanmin(bootstrap_stabilities), np.nanmax(bootstrap_stabilities) + BOOTSTRAP_STABILITY_BIN_WIDTH, BOOTSTRAP_STABILITY_BIN_WIDTH, dtype=float)
if bootstrap_bins.size < 2:
    center = float(np.nanmean(bootstrap_stabilities))
    bootstrap_bins = np.array([center - BOOTSTRAP_STABILITY_BIN_WIDTH / 2.0, center + BOOTSTRAP_STABILITY_BIN_WIDTH / 2.0], dtype=float)
elif bootstrap_bins[-1] < np.nanmax(bootstrap_stabilities):
    bootstrap_bins = np.append(bootstrap_bins, bootstrap_bins[-1] + BOOTSTRAP_STABILITY_BIN_WIDTH)
ax.hist(bootstrap_stabilities, bins=bootstrap_bins, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(mean_stability, color='red', linestyle='--', lw=2, label=f'Mean: {mean_stability:.1f}%')
ax.axvline(ci_low, color='orange', linestyle=':', lw=2, label=f'95% CI: [{ci_low:.1f}%, {ci_high:.1f}%]')
ax.axvline(ci_high, color='orange', linestyle=':', lw=2)
ax.set_xlabel('Stability (%)')
ax.set_ylabel('Frequency')
ax.set_title(f'Bootstrap Distribution of Stability ({N_BOOTSTRAP} iterations, {n_pairs} pairs each)')
add_legend(ax, frameon=False)
if 'style_hist_axis' in globals():
    style_hist_axis(ax)
ax.grid(False)
plt.tight_layout()
out = OUTPUT_DIR / "11_09_stability_ellipse_bootstrap_distribution.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

# --- Pie chart with bootstrap CI ---
stable_pct = mean_stability
unstable_pct = 100 - mean_stability

make_figure(figsize=(6.3,5.8))
plt.pie([stable_pct, unstable_pct], 
        labels=[f"Stable ({stable_pct:.1f}%)", f"Unstable ({unstable_pct:.1f}%)"],
        colors=['#4CAF50','#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
plt.title(f'Random Pairs Stability (Bootstrap: {N_BOOTSTRAP} iter)\n95% CI: [{ci_low:.1f}%, {ci_high:.1f}%]')
plt.tight_layout()
out = OUTPUT_DIR / "11_10_stability_ellipse_bootstrap_pie.pdf"; plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()

print(f"\n[Ellipses Bootstrap] α={ELLIPSE_ALPHA:.2f}, tol={ELLIPSE_TOL*100:.0f}%")
print(f"  Mean stability: {mean_stability:.1f}% (95% CI: [{ci_low:.1f}%, {ci_high:.1f}%])")


In [ ]:
# Pie plots side by side for ellipse stability and bootstrap results

fig, (ax1, ax2) = make_figure_grid(1, 2, figsize=(12, 5.8))

# Pie plot 1: Ellipse stability (conservative, hard-edge)
ax1.pie([stable, unstable], labels=[f"Stable ({stable})", f"Unstable ({unstable})"],
        colors=['#4CAF50', '#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
ax1.set_title('Conservative Ellipse Stability (hard-edge)')

# Pie plot 2: Bootstrap results
ax2.pie([stable_pct, unstable_pct], labels=[f"Stable ({stable_pct:.1f}%)", f"Unstable ({unstable_pct:.1f}%)"],
        colors=['#4CAF50', '#FF5722'], autopct='%.1f%%', startangle=90, textprops={'fontsize':11})
ax2.set_title('Bootstrap Results')

# Equal aspect ratio for both
ax1.set_aspect('equal')
ax2.set_aspect('equal')

plt.tight_layout()
out = OUTPUT_DIR / "11_11_stability_ellipse_bootstrap_side_by_side.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()

print(f"[Ellipses] Conservative: hard α={ELLIPSE_ALPHA:.2f}  stable={stable}/{n_pairs} ({100*stable/n_pairs:.1f}%)")
print(f"✓ Saved {out}")

### 11.8 Stability of WT Plasticity Profiles

Because the manuscript is centered on train dynamics, stability must also be assessed in normalized PPR space rather than only in the PCA or amplitude domain. These panels therefore ask whether the same boutons preserve their STP phenotype over time. 


In [ ]:
# Compare PPR profiles before vs after treatment
from scipy.stats import ttest_rel


def plot_stability_ppr_profile():
    """Plot PPR profile with statistical testing."""
    if 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'finalize_ppr_axis' not in globals():
        raise RuntimeError('Run the shared PPR helper cell first (cell defining ppr_profile_stats).')

    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_Stability_Before.columns]

    pulse_numbers, ppr_before, ppr_before_sem, n_before = ppr_profile_stats(
        PCA_Data_Stability_Before,
        ppr_column_names=ppr_cols,
    )
    _, ppr_after, ppr_after_sem, n_after = ppr_profile_stats(
        PCA_Data_Stability_After,
        ppr_column_names=ppr_cols,
    )

    fig, ax = make_figure_grid(figsize=(8, 6))

    plot_ppr_mean_sem(
        ax,
        pulse_numbers,
        ppr_before,
        ppr_before_sem,
        color=STABILITY_BEFORE_COLOR,
        marker='o',
        label=f'Before (n={n_before})',
        linewidth=2,
        sem_alpha=0.2,
    )
    plot_ppr_mean_sem(
        ax,
        pulse_numbers,
        ppr_after,
        ppr_after_sem,
        color=STABILITY_AFTER_COLOR,
        marker='s',
        label=f'After (n={n_after})',
        linewidth=2,
        sem_alpha=0.2,
    )

    finalize_ppr_axis(
        ax,
        pulse_numbers,
        title='PPR Profile: Before vs After',
        ylabel='PPR (A_n/A_1)',
        unity_kwargs={'color': 'gray', 'linestyle': 'dotted', 'linewidth': 2},
        legend=True,
        legend_loc='best',
)
    plt.tight_layout()

    output_file = OUTPUT_DIR / '11_12_stability_ppr_profile.pdf'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print('=== PPR STATISTICAL ANALYSIS ===')
    for i, col in enumerate(ppr_cols, start=2):
        t_stat, p_val = ttest_rel(PCA_Data_Stability_Before[col], PCA_Data_Stability_After[col], nan_policy='omit')
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        print(f'Pulse {i}: t={t_stat:.3f}, p={p_val:.3e} ({sig})')

    return output_file


plot_output = plot_stability_ppr_profile()
print(f'✓ Saved PPR profile to {plot_output}')


### 11.9 Paired WT Metric Comparisons

Paired scalar comparisons summarize whether the main WT descriptors drift systematically between recordings. They provide the numerical counterpart to the trajectory and trace-based stability views.


In [ ]:
from scipy.stats import wilcoxon, ttest_rel
import numpy as np


def _pair_stability_metric(before_df, after_df, value_col):
    b = before_df[['BaseID', value_col]].dropna().groupby('BaseID', as_index=False).mean()
    a = after_df[['BaseID', value_col]].dropna().groupby('BaseID', as_index=False).mean()
    paired = b.merge(a, on='BaseID', how='inner', suffixes=('_before', '_after'))
    return paired


def _paired_baseline(raw_traces_df, common_time):
    d = raw_traces_df.copy()
    if 'BaseID' not in d.columns:
        d['BaseID'] = d['ID'].map(_normalize_bouton_id)

    before_conds = STABILITY_BEFORE_CONDITIONS
    after_conds = STABILITY_AFTER_CONDITIONS

    d_before = d[d['Condition'].isin(before_conds)].copy()
    d_after = d[d['Condition'].isin(after_conds)].copy()

    baseline_end = int(0.5 * len(common_time))
    d_before['Baseline'] = d_before['Avg'].map(lambda x: float(np.nanmean(x[:baseline_end])))
    d_after['Baseline'] = d_after['Avg'].map(lambda x: float(np.nanmean(x[:baseline_end])))

    b = d_before[['BaseID', 'Baseline']].dropna().groupby('BaseID', as_index=False).mean()
    a = d_after[['BaseID', 'Baseline']].dropna().groupby('BaseID', as_index=False).mean()
    return b.merge(a, on='BaseID', how='inner', suffixes=('_before', '_after'))


def plot_stability_comparisons():
    """Create paired boxplots with proper BaseID pairing and display p-values."""

    def add_p_value(ax, p_value, positions=[0, 1]):
        y_max = ax.get_ylim()[1]
        y_min = ax.get_ylim()[0]
        y_sig = y_max + 0.05 * (y_max - y_min)
        p_text = f"p = {p_value:.3g}"
        ax.plot(positions, [y_sig, y_sig], color='black', linewidth=1.5)
        ax.text(np.mean(positions), y_sig + 0.01 * (y_max - y_min), p_text,
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    paired_amp1 = _pair_stability_metric(PCA_Data_Stability_Before, PCA_Data_Stability_After, 'AMP1')
    paired_ppr = _pair_stability_metric(PCA_Data_Stability_Before, PCA_Data_Stability_After, 'PPR2/1')
    paired_baseline = _paired_baseline(RAW_TRACES_DF, COMMON_TIME)

    fig, axes = make_figure_grid(1, 3, figsize=(15, 5))

    # AMP1
    amp1_long = pd.DataFrame({
        'Condition': ['Before'] * len(paired_amp1) + ['After'] * len(paired_amp1),
        'AMP1': pd.concat([paired_amp1['AMP1_before'], paired_amp1['AMP1_after']], ignore_index=True)
    })
    sns.boxplot(data=amp1_long, x='Condition', y='AMP1', hue='Condition', ax=axes[0], palette=[STABILITY_BEFORE_COLOR, STABILITY_AFTER_COLOR], legend=False)
    sns.stripplot(data=amp1_long, x='Condition', y='AMP1', ax=axes[0], color='black', size=3, alpha=0.6)
    for _, row in paired_amp1.iterrows():
        axes[0].plot([0, 1], [row['AMP1_before'], row['AMP1_after']], color='gray', alpha=0.4, linewidth=1)

    t_stat_amp1, p_val_amp1 = ttest_rel(paired_amp1['AMP1_before'], paired_amp1['AMP1_after'])
    add_p_value(axes[0], p_val_amp1)
    axes[0].set_title(f'AMP1: Before vs After (paired n={len(paired_amp1)})')

    # PPR2/1
    ppr_long = pd.DataFrame({
        'Condition': ['Before'] * len(paired_ppr) + ['After'] * len(paired_ppr),
        'PPR2/1': pd.concat([paired_ppr['PPR2/1_before'], paired_ppr['PPR2/1_after']], ignore_index=True)
    })
    sns.boxplot(data=ppr_long, x='Condition', y='PPR2/1', hue='Condition', ax=axes[1], palette=[STABILITY_BEFORE_COLOR, STABILITY_AFTER_COLOR], legend=False)
    sns.stripplot(data=ppr_long, x='Condition', y='PPR2/1', ax=axes[1], color='black', size=3, alpha=0.6)
    for _, row in paired_ppr.iterrows():
        axes[1].plot([0, 1], [row['PPR2/1_before'], row['PPR2/1_after']], color='gray', alpha=0.4, linewidth=1)

    t_stat_ppr, p_val_ppr = ttest_rel(paired_ppr['PPR2/1_before'], paired_ppr['PPR2/1_after'])
    add_p_value(axes[1], p_val_ppr)
    axes[1].set_title(f'PPR2/1: Before vs After (paired n={len(paired_ppr)})')

    # Baseline F0
    f0_long = pd.DataFrame({
        'Condition': ['Before'] * len(paired_baseline) + ['After'] * len(paired_baseline),
        'F0': pd.concat([paired_baseline['Baseline_before'], paired_baseline['Baseline_after']], ignore_index=True)
    })
    sns.boxplot(data=f0_long, x='Condition', y='F0', hue='Condition', ax=axes[2], palette=[STABILITY_BEFORE_COLOR, STABILITY_AFTER_COLOR], legend=False)
    sns.stripplot(data=f0_long, x='Condition', y='F0', ax=axes[2], color='black', size=3, alpha=0.6)
    for _, row in paired_baseline.iterrows():
        axes[2].plot([0, 1], [row['Baseline_before'], row['Baseline_after']], color='gray', alpha=0.4, linewidth=1)

    stat_f0, p_val_f0 = wilcoxon(paired_baseline['Baseline_before'], paired_baseline['Baseline_after'])
    add_p_value(axes[2], p_val_f0)
    axes[2].set_title(f'Baseline F0: Before vs After (paired n={len(paired_baseline)})')
    axes[2].set_ylabel('F0 (a.u.)')

    plt.tight_layout()
    output_file = OUTPUT_DIR / "11_13_stability_statistical_comparisons.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print("=== STABILITY STATISTICAL RESULTS ===")
    print(f"AMP1: t={t_stat_amp1:.3f}, p={p_val_amp1:.4g}")
    print(f"PPR2/1: t={t_stat_ppr:.3f}, p={p_val_ppr:.4g}")
    print(f"Baseline F0 (Wilcoxon): W={stat_f0:.3f}, p={p_val_f0:.4g}")

    baseline_before = paired_baseline['Baseline_before'].tolist()
    baseline_after = paired_baseline['Baseline_after'].tolist()
    return output_file, baseline_after, baseline_before


comparison_output = plot_stability_comparisons()
print(f"Comparisons saved to {comparison_output[0]}")

# Additional plot: Distribution of baseline values
fig2, ax2 = make_figure_grid(figsize=(8, 5))

baseline_before = comparison_output[2]
baseline_after = comparison_output[1]
baseline_diff = [after - before for after, before in zip(baseline_after, baseline_before)]

sns.histplot(baseline_diff, kde=True, color='purple', alpha=0.7, ax=ax2)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='No change')

ax2.set_xlabel('ΔBaseline F0 (After - Before)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Baseline Difference (After - Before)')
add_legend(ax2, )

plt.tight_layout()
distribution_output = OUTPUT_DIR / "11_14_stability_baseline_difference_distribution.pdf"
plt.savefig(distribution_output, dpi=300, bbox_inches='tight')
plt.show()

print(f"Baseline difference distribution saved to {distribution_output}")


### 11.10 Stability Summary

The summary cell consolidates the main stability outputs into a concise statement that can be compared directly with the main biological perturbation results. The purpose is to show that the perturbation trajectories exceed the intrinsic drift measured in repeated WT recordings.


In [ ]:
# Final stability analysis summary
def stability_summary():
    """Generate comprehensive stability analysis summary."""
    
    print("=" * 60)
    print("COMPREHENSIVE STABILITY ANALYSIS SUMMARY")
    print("=" * 60)
    
    # Sample sizes
    n_before = len(PCA_Data_Stability_Before)
    n_after = len(PCA_Data_Stability_After)
    n_pairs = min(n_before, n_after)
    
    print(f"\nSample sizes:")
    print(f"  Before: {n_before} boutons")
    print(f"  After: {n_after} boutons")
    print(f"  Paired: {n_pairs} boutons")
    
    # Key parameter changes
    print(f"\nKey parameter changes (Before → After):")
    
    # AMP1
    amp1_before_mean = PCA_Data_Stability_Before['AMP1'].mean()
    amp1_after_mean = PCA_Data_Stability_After['AMP1'].mean()
    amp1_change = ((amp1_after_mean - amp1_before_mean) / amp1_before_mean) * 100
    print(f"  AMP1: {amp1_before_mean:.3f} → {amp1_after_mean:.3f} ({amp1_change:+.1f}%)")
    
    # PPR2/1
    ppr_before_mean = PCA_Data_Stability_Before['PPR2/1'].mean()
    ppr_after_mean = PCA_Data_Stability_After['PPR2/1'].mean()
    ppr_change = ((ppr_after_mean - ppr_before_mean) / ppr_before_mean) * 100
    print(f"  PPR2/1: {ppr_before_mean:.3f} → {ppr_after_mean:.3f} ({ppr_change:+.1f}%)")
    
    # PCA movement analysis (if available)
    if 'movement_distances' in locals():
        print(f"\nPCA space movement:")
        print(f"  Mean distance: {np.mean(movement_distances):.3f} ± {np.std(movement_distances):.3f}")
        print(f"  Max distance: {np.max(movement_distances):.3f}")
        print(f"  Boutons with large movement (>mean): {np.sum(movement_distances > np.mean(movement_distances))}/{len(movement_distances)}")
    
    # Cluster stability (if available)
    if 'stable_pairs' in locals() and 'unstable_pairs' in locals():
        total_analyzed = stable_pairs + unstable_pairs
        stability_pct = (stable_pairs / total_analyzed) * 100
        print(f"\nCluster assignment stability:")
        print(f"  Stable pairs: {stable_pairs}/{total_analyzed} ({stability_pct:.1f}%)")
        print(f"  Unstable pairs: {unstable_pairs}/{total_analyzed} ({100-stability_pct:.1f}%)")
    
    print(f"\n" + "=" * 60)
    print("Analysis complete - all figures saved to OUTPUT_DIR")
    print("=" * 60)

# Run summary
stability_summary()

### 11.11 Global Trace Overview Across Conditions

This atlas groups all recording conditions in one place after the WT, calcium, frequency, SynII, and stability sections have already introduced the corresponding datasets. It is kept late in the notebook as a compact global QC view rather than as an early mixed-topic result panel.


In [ ]:
## Plot processed traces by condition. Extracellular Ca2+, temporal traces

condition_names_plot = list(NORM_TRACES_DATAFRAME['Condition'].unique())
n_conditions = len(condition_names_plot)
n_rows = (n_conditions + 4) // 5
fig, axes = make_figure_grid(n_rows, 5, figsize=(20, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

for plot_idx, condition_name in enumerate(condition_names_plot):
    if plot_idx >= len(axes):
        break
    ax = axes[plot_idx]
    stats = plot_traces(ax=ax, condition_names=condition_name, source='normalized', show_average=True, show_sem=False, show_individuals=True, alignment='grid', color='black', individual_color='gray', linewidth=2.0, individual_alpha=0.2, individual_lw=0.5, zero_line=True, zero_kwargs={'color': 'black', 'linestyle': '--', 'alpha': 0.3}, event_time=1.0, event_kwargs={'color': 'red', 'linestyle': '--', 'alpha': 0.5}, xlim=(0, CROP_END), style_axis=True, return_data=True)
    ax.set_title(f'{condition_name}\n(n={stats["n"]})', fontsize=10, pad=10)
    if plot_idx >= (n_rows - 1) * 5:
        ax.set_xlabel('Time (s)')
    else:
        ax.set_xlabel('')
    if plot_idx % 5 == 0:
        ax.set_ylabel('ΔF/F')
    else:
        ax.set_ylabel('')

for empty_idx in range(n_conditions, len(axes)):
    fig.delaxes(axes[empty_idx])

fig.tight_layout()
fig.suptitle('Preprocessed bouton traces by condition (from extract_metrics)', y=1.02, fontsize=14, fontweight='bold')
output_file = OUTPUT_DIR / '11_15_global_trace_overview_all_conditions.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved to {output_file}')
print(f"Time range: {COMMON_TIME[0]:.2f} - {COMMON_TIME[-1]:.2f}s ({len(COMMON_TIME)} points)")
print(f"Conditions: {', '.join(condition_names_plot)}")


## 12. Random Forest Validation

Random-forest classification is kept at the end as a validation step rather than as a primary biological analysis. It quantifies how reproducibly the WT bouton classes can be discriminated from the feature set used for clustering and provides a classifier-based check that the WT classes are not arbitrary partitions of the PCA space.


In [ ]:

# ============================================================================
# Random Forest Classification for 2 to 10 clusters
# Using hierarchical clustering to create different cluster numbers
# ============================================================================
from scipy.cluster.hierarchy import fcluster
from scipy.stats import ttest_ind
from Func_RF import random_forest_classification

# Feature columns for RF classification
feature_cols = ["AMP1", "AMP2", 
                "PPR2/1", "PPR3/1", "PPR4/1", "PPR5/1", "PPR6/1",
                "PPR7/1", "PPR8/1", "PPR9/1", "PPR10/1",
                "%Fail1", "%Fail2"]

# Store results for comparison
rf_comparison = {}

# Test cluster numbers from 2 to 10
n_clusters_list = list(range(2, 11))  # [2, 3, 4, 5, 6, 7, 8, 9, 10]
#n_clusters_list = [6]  # For quick testing, use only 6 clusters (comment out to run all)
# Get AMP1 values for reordering clusters
amp1_values = PCA_Data_WT_Pooled["AMP1"].values

for n_clust in n_clusters_list:
    print(f"\n{'='*60}")
    print(f"  Running Random Forest with {n_clust} clusters")
    print(f"{'='*60}")
    
    # Create cluster assignments for this number of clusters
    cluster_raw = fcluster(linkage_matrix, n_clust, criterion='maxclust')
    
    # Reorder clusters by ascending mean AMP1 (same as cell 191)
    cluster_amp1_means = {}
    for cid in range(1, n_clust + 1):
        mask = cluster_raw == cid
        cluster_amp1_means[cid] = np.nanmean(amp1_values[mask])
    sorted_clusters = sorted(cluster_amp1_means.keys(), key=lambda c: cluster_amp1_means[c])
    cluster_map = {old: new for new, old in enumerate(sorted_clusters, start=1)}
    cluster_assignments = np.array([cluster_map[c] for c in cluster_raw])
    
    # Create temporary DataFrame with new cluster assignments
    df_temp = PCA_Data_WT_Pooled.copy()
    df_temp['HC_Cluster'] = cluster_assignments
    
    # Run RF classification - show figures only for key numbers
    show_figure = n_clust in [3, 5, 8]  # Show figures for selected cluster counts
    
    fig, results = random_forest_classification(
        df_temp,
        feature_cols=feature_cols,
        target_col="HC_Cluster",
        n_splits=5,
        n_iter=3,
        verbose=show_figure  # Show detailed output only for key cluster counts
    )
    
    # Save all figures to PDF
    fig.savefig(OUTPUT_DIR / f"12_01_rf_confusion_matrix_k{n_clust}.pdf", 
                format='pdf', dpi=300, bbox_inches='tight')
    print(f"  Figures saved to: {OUTPUT_DIR}")
    plt.show()

    
    # Store results (compute std from scores if not in results)
    std_actual = results.get('std_actual', np.std(results['scores_actual']))
    std_shuffled = results.get('std_shuffled', np.std(results['scores_shuffled']))
    p_value = results.get('p_value', ttest_ind(results['scores_actual'], results['scores_shuffled'])[1])
    
    rf_comparison[n_clust] = {
        'accuracy_actual': results['accuracy_actual'],
        'accuracy_shuffled': results['accuracy_shuffled'],
        'std_actual': std_actual,
        'std_shuffled': std_shuffled,
        'p_value': p_value,
        'scores_actual': results['scores_actual'],
        'scores_shuffled': results['scores_shuffled'],
        'figure': fig
    }
    
    # Print short summary for all
    improvement = results['accuracy_actual'] - results['accuracy_shuffled']
    print(f"\n  Summary: {n_clust} clusters - Accuracy = {results['accuracy_actual']:.1%} ± {std_actual:.1%}")
    print(f"           Improvement = +{improvement:.1%}, p = {p_value:.2e}")

print("\n" + "="*60)
print("  All RF classifications completed (2-10 clusters)")
print(f"  Figures saved to: {OUTPUT_DIR}")
print("="*60)

In [ ]:
# ============================================================================
# Comparison summary plot
# ============================================================================
fig_comparison, axes = make_figure_grid(1, 2, figsize=(12, 5))

# --- Subplot 1: Bar chart of accuracies ---
ax1 = axes[0]
x_pos = np.arange(len(n_clusters_list))
bar_width = 0.35

# Actual accuracies
acc_actual = [rf_comparison[n]['accuracy_actual'] for n in n_clusters_list]
std_actual = [rf_comparison[n]['std_actual'] for n in n_clusters_list]
bars1 = ax1.bar(x_pos - bar_width/2, acc_actual, bar_width, 
                yerr=std_actual, label='Actual', color='#4CAF50', capsize=5, alpha=0.8)

# Shuffled accuracies
acc_shuffled = [rf_comparison[n]['accuracy_shuffled'] for n in n_clusters_list]
std_shuffled = [rf_comparison[n]['std_shuffled'] for n in n_clusters_list]
bars2 = ax1.bar(x_pos + bar_width/2, acc_shuffled, bar_width,
                yerr=std_shuffled, label='Shuffled', color='#9E9E9E', capsize=5, alpha=0.8)

ax1.set_xlabel('Number of Clusters')
ax1.set_ylabel('Classification Accuracy')
ax1.set_title('RF Classification: Actual vs Shuffled')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([str(n) for n in n_clusters_list])
add_legend(ax1, )
ax1.set_ylim(0, 1.0)
ax1.axhline(y=1/4, color='gray', linestyle='--', alpha=0.5, label='Chance (4cl)')
ax1.axhline(y=1/5, color='gray', linestyle=':', alpha=0.5, label='Chance (5cl)')
ax1.axhline(y=1/6, color='gray', linestyle='-.', alpha=0.5, label='Chance (6cl)')

# Add value labels on bars
for bar, val in zip(bars1, acc_actual):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, 
             f'{val:.0%}', ha='center', va='bottom', fontsize=9)

# --- Subplot 2: Improvement over chance ---
ax2 = axes[1]
improvement = [rf_comparison[n]['accuracy_actual'] - rf_comparison[n]['accuracy_shuffled'] 
               for n in n_clusters_list]
colors_imp = ['#2196F3' if imp > 0 else '#F44336' for imp in improvement]
bars3 = ax2.bar(x_pos, improvement, 0.6, color=colors_imp, alpha=0.8)

ax2.set_xlabel('Number of Clusters')
ax2.set_ylabel('Improvement over Shuffled')
ax2.set_title('Classification Improvement')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([str(n) for n in n_clusters_list])
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

for bar, val in zip(bars3, improvement):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'+{val:.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()

# Save comparison figure to PDF
fig_comparison.savefig(OUTPUT_DIR / "12_02_rf_cluster_count_comparison.pdf", 
                       format='pdf', dpi=300, bbox_inches='tight')
print(f"✓ Comparison figure saved: {OUTPUT_DIR / '12_02_rf_cluster_count_comparison.pdf'}")

plt.show()

# --- Separate figure for summary table ---
fig_table, ax_table = make_figure_grid(figsize=(10, 4))
ax_table.axis('off')

# Create summary table
table_data = []
for n in n_clusters_list:
    res = rf_comparison[n]
    row = [
        f"{n}",
        f"{res['accuracy_actual']:.1%} ± {res['std_actual']:.1%}",
        f"{res['accuracy_shuffled']:.1%} ± {res['std_shuffled']:.1%}",
        f"+{res['accuracy_actual'] - res['accuracy_shuffled']:.1%}",
        f"{res['p_value']:.2e}"
    ]
    table_data.append(row)

columns = ['Clusters', 'Actual', 'Shuffled', 'Improvement', 'p-value']
table = ax_table.table(cellText=table_data, colLabels=columns, loc='center', 
                       cellLoc='center', colColours=['#E0E0E0']*5)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.4, 1.8)

# Highlight best result
best_idx = np.argmax(improvement)
ax_table.set_title(f'Summary: Best = {n_clusters_list[best_idx]} clusters', 
                   fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()

# Save table figure to PDF
fig_table.savefig(OUTPUT_DIR / "12_03_rf_summary_table.pdf", 
                  format='pdf', dpi=300, bbox_inches='tight')
print(f"✓ Summary table saved: {OUTPUT_DIR / '12_03_rf_summary_table.pdf'}")

plt.show()

# Print conclusion
print("\n" + "="*60)
print("  CONCLUSION")
print("="*60)
best_n = n_clusters_list[best_idx]
best_acc = rf_comparison[best_n]['accuracy_actual']
best_imp = rf_comparison[best_n]['accuracy_actual'] - rf_comparison[best_n]['accuracy_shuffled']
print(f"\n  : Best number of clusters: {best_n}")
print(f"  : Classification accuracy: {best_acc:.1%}")
print(f"  : Improvement over chance: +{best_imp:.1%}")
print(f"  : p-value: {rf_comparison[best_n]['p_value']:.4e}")
print("\n  The higher the improvement over shuffled, the better the cluster separation.")
print(f"\n  All PDF figures saved to: {OUTPUT_DIR}")